# 4.5.2 Train Transformer-Based Time-Series Forecasting Model

This notebook extends the neural forecasting framework established in 4.5.1 with a compact **patch-based, encoder-only Transformer**. The forecasting problem remains unchanged: five mobility targets are forecast at **h = 1, 2, and 5 Dayparts** under the **Post-CP Primary** and **Full-History Regime-Aware** tracks, using the same benchmark, leakage protections, model-facing information, and ordinary-validation boundaries established upstream.

A Transformer differs from the MLP, GRU, and TCN mainly in how it learns relationships across recent mobility history. Instead of treating the entire lag window as one flat vector or processing the sequence recurrently, the Transformer converts short blocks of consecutive Dayparts into tokens and uses **self-attention** to learn relationships among those historical blocks.

The Transformer receives the same underlying forecasting information available to the finalized neural branch: five recent mobility measures with explicit observation masks, forecast-time supporting context including the leakage-safe weekly STL seasonal state, and Taxi Zone identity. What changes is how the recent history is represented and processed.

The one-week, **35-Daypart** sequence used in 4.5.1 remains an important reference but is not assumed to be optimal for attention. This notebook therefore tests whether a compact Transformer benefits from **one, two, or four weeks** of recent history and from a small number of alternative patch representations.

Transformer development remains deliberately bounded. Rather than grid-searching individual hyperparameters, the notebook evaluates a handful of coherent Transformer specifications through three restartable screening batches. The resulting architecture family may retain **different selected profiles by forecasting target** when the evidence supports doing so; it does not force one numerical hyperparameter vector across five very different mobility measures.

The notebook ultimately produces the Transformer-family candidate for downstream comparison in 4.6.x. It does not choose the overall forecasting winner, construct the Chapter 5 no-policy counterfactual, or score the final holdout beginning **January 5, 2026**.

<img src="image-20260909-133549.png" width="" align="" />

## Notebook Setup

The setup layer centralizes imports, filesystem locations, fixed forecasting scope, execution controls, reproducibility settings, and shared utilities used throughout the notebook.

The 4.5.1 final-table filenames are treated as a frozen upstream interface. This notebook reads those artifacts but does not rename, remove, or repurpose them. Transformer-specific checkpoints and outputs are written only to the 4.5.2 directories.

All shared helpers are defined before Section 4.5.2.1 so the numbered notebook sections can focus on forecasting methodology and evidence rather than utility code.

In [1]:
# ---------------------------------------------------------------------
# 4.5.2.0A — Imports
# ---------------------------------------------------------------------

import gc
import hashlib
import json
import math
import os
import random
import shutil
import time
import uuid
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

from sklearn.metrics import mean_absolute_error, mean_squared_error
from tqdm import tqdm

import duckdb

# Reader-facing tables should stay compact, while analytical objects retain
# their original numeric precision in memory and on disk.
pd.options.display.float_format = "{:,.3f}".format

from project_branding import (
    BRAND_COLORS,
    BRAND_COLOR_SEQUENCE,
    BRAND_DIVERGING_SEQUENCE,
    BRAND_SEQUENTIAL_HEATMAP_SCALE,
    BRAND_CLUSTER_PALETTE,
    apply_branding,
)

The notebook uses three groups of settings.

**Filesystem settings** identify the frozen 4.5.1 handoff and the new 4.5.2 output locations. The Transformer branch receives its own intermediate and final directories so its experiments cannot overwrite the MLP branch.

**Forecasting-scope settings** preserve the two ordinary modeling tracks, five targets, three horizons, policy boundary, and final-holdout boundary already established upstream.

**Execution controls** keep expensive experiments restartable. A `RUN_*` control allows unfinished work to execute, while the corresponding `RESET_*` control is the explicit way to discard durable current-experiment work. Successful current-contract checkpoints will later be recovered before new fitting begins.

The candidate Transformer lookbacks are **35, 70, and 140 Dayparts**, representing one, two, and four complete weeks. These values define the question that will be tested later; they do not pre-select a longer history.

In [2]:
# ---------------------------------------------------------------------
# 4.5.2.0B — Paths, forecasting scope, toggles, and design constants
# ---------------------------------------------------------------------

PROJECT_ROOT = Path(".")
PIPELINE_DIR = PROJECT_ROOT / "pipeline_data"

INPUT_451_FINAL_DIR = PIPELINE_DIR / "4.5.1.final_tables"
INPUT_412_FINAL_DIR = PIPELINE_DIR / "4.1.2.final_tables"
INPUT_423_FINAL_DIR = PIPELINE_DIR / "4.2.3.final_tables"
INPUT_424_FINAL_DIR = PIPELINE_DIR / "4.2.4.final_tables"
INPUT_424_INTERMEDIATE_DIR = PIPELINE_DIR / "4.2.4.intermediate_tables"

ROLLING_ORIGIN_PATH = INPUT_412_FINAL_DIR / "rolling_origin_registry.parquet"
TRACK_CONTRACT_PATH = INPUT_412_FINAL_DIR / "forecasting_track_contract_registry.parquet"
SUPPORTED_MODELING_SCOPE_PATH = INPUT_424_FINAL_DIR / "supported_modeling_scope.parquet"
ADVANCED_BENCHMARK_REGISTRY_PATH = INPUT_424_FINAL_DIR / "advanced_model_benchmark_registry.parquet"

INTERMEDIATE_DIR = PIPELINE_DIR / "4.5.2.intermediate_tables"
FINAL_DIR = PIPELINE_DIR / "4.5.2.final_tables"

INTERMEDIATE_DIR.mkdir(parents=True, exist_ok=True)
FINAL_DIR.mkdir(parents=True, exist_ok=True)


# ---------------------------------------------------------------------
# Frozen 4.5.1 downstream interface
# ---------------------------------------------------------------------
# These filenames belong to 4.5.1. Their contents may reflect the final
# repaired methodology, but 4.5.2 must never rename or repurpose them.

NEURAL_FRAMEWORK_REGISTRY_PATH = (
    INPUT_451_FINAL_DIR / "neural_forecasting_framework_registry.parquet"
)
NEURAL_FINAL_MODEL_REGISTRY_PATH = (
    INPUT_451_FINAL_DIR / "neural_final_model_registry.parquet"
)
NEURAL_MODEL_DEFINITION_PATH = (
    INPUT_451_FINAL_DIR / "neural_model_definition_contract.parquet"
)
NEURAL_INPUT_REGISTRY_PATH = (
    INPUT_451_FINAL_DIR / "neural_input_registry.parquet"
)
NEURAL_SEQUENCE_CONTRACT_PATH = (
    INPUT_451_FINAL_DIR / "neural_sequence_preprocessing_contract.parquet"
)
NEURAL_OUTPUT_CONTRACT_PATH = (
    INPUT_451_FINAL_DIR / "neural_forecast_output_contract.parquet"
)
NEURAL_TRAINING_POLICY_EVIDENCE_PATH = (
    INPUT_451_FINAL_DIR / "neural_training_policy_evidence.parquet"
)
NEURAL_TRAINING_CURVE_PATH = (
    INPUT_451_FINAL_DIR / "neural_training_curve_diagnostic.parquet"
)
NEURAL_ATTRIBUTION_METHOD_PATH = (
    INPUT_451_FINAL_DIR / "neural_attribution_method_contract.parquet"
)
NEURAL_INPUT_IMPORTANCE_PATH = (
    INPUT_451_FINAL_DIR / "neural_grouped_input_importance.parquet"
)
NEURAL_DEVELOPMENT_SUMMARY_PATH = (
    INPUT_451_FINAL_DIR / "neural_sensitivity_development_summary.parquet"
)
NEURAL_HISTORY_STATE_PATH = (
    INPUT_451_FINAL_DIR / "neural_mobility_history_state.npz"
)
NEURAL_SELECTED_PREDICTIONS_PATH = (
    INPUT_451_FINAL_DIR / "neural_selected_validation_predictions.parquet"
)
NEURAL_CLOSEOUT_QA_PATH = (
    INPUT_451_FINAL_DIR / "neural_closeout_qa.parquet"
)
NEURAL_HANDOFF_MANIFEST_PATH = (
    INPUT_451_FINAL_DIR / "neural_handoff_manifest.parquet"
)


# ---------------------------------------------------------------------
# 4.5.2-owned experiment directories and eventual handoff artifacts
# ---------------------------------------------------------------------

TRANSFORMER_REPRESENTATION_SCREEN_DIR = INTERMEDIATE_DIR / "transformer_representation_screen"
TRANSFORMER_PROFILE_SCREEN_DIR = INTERMEDIATE_DIR / "transformer_profile_screen"
TRANSFORMER_CONFIRMATION_SCREEN_DIR = INTERMEDIATE_DIR / "transformer_confirmation_screen"
TRANSFORMER_DIAGNOSTIC_DIR = INTERMEDIATE_DIR / "transformer_diagnostics"
TRANSFORMER_CADENCE_SCREEN_DIR = INTERMEDIATE_DIR / "transformer_cadence_screen"
TRANSFORMER_VALIDATION_DIR = INTERMEDIATE_DIR / "transformer_rolling_validation"

TRANSFORMER_FINAL_MODEL_REGISTRY_PATH = (
    FINAL_DIR / "transformer_final_model_registry.parquet"
)
TRANSFORMER_MODEL_DEFINITION_PATH = (
    FINAL_DIR / "transformer_model_definition_contract.parquet"
)
TRANSFORMER_TRAINING_CURVE_PATH = (
    FINAL_DIR / "transformer_training_curve_diagnostic.parquet"
)
TRANSFORMER_GROUPED_INPUT_IMPORTANCE_PATH = (
    FINAL_DIR / "transformer_grouped_input_importance.parquet"
)
TRANSFORMER_SELECTED_PREDICTIONS_PATH = (
    FINAL_DIR / "transformer_selected_validation_predictions.parquet"
)
TRANSFORMER_CLOSEOUT_QA_PATH = (
    FINAL_DIR / "transformer_closeout_qa.parquet"
)
TRANSFORMER_HANDOFF_MANIFEST_PATH = (
    FINAL_DIR / "transformer_handoff_manifest.parquet"
)


# ---------------------------------------------------------------------
# Frozen ordinary forecasting scope
# ---------------------------------------------------------------------

POLICY_START_DATE = pd.Timestamp("2025-01-05")
FINAL_HOLDOUT_START_DATE = pd.Timestamp("2026-01-05")

ORDINARY_TRACKS = [
    "post_cp_primary",
    "full_history_regime_aware",
]

TRACK_LABELS = {
    "post_cp_primary": "Post-CP Primary",
    "full_history_regime_aware": "Full-History Regime-Aware",
}

FORECAST_TARGETS = [
    "taxi_trip_count",
    "taxi_avg_trip_speed",
    "fhvhv_trip_count",
    "fhvhv_avg_trip_speed",
    "subway_ridership",
]

TARGET_LABELS = {
    "taxi_trip_count": "Taxi trips",
    "taxi_avg_trip_speed": "Taxi average speed",
    "fhvhv_trip_count": "FHVHV trips",
    "fhvhv_avg_trip_speed": "FHVHV average speed",
    "subway_ridership": "Subway ridership",
}

FORECAST_HORIZONS = [1, 2, 5]

EXPECTED_BENCHMARK_LABEL = "Previous week · same Daypart"

FORECASTING_MATRIX_PATHS = {
    "post_cp_primary": INPUT_423_FINAL_DIR / "forecasting_matrix__post_cp_primary.parquet",
    "full_history_regime_aware": INPUT_423_FINAL_DIR / "forecasting_matrix__full_history_regime_aware.parquet",
}
BASELINE_PREDICTION_PATHS = {
    track: INPUT_424_INTERMEDIATE_DIR / f"baseline_predictions__{track}.parquet"
    for track in ORDINARY_TRACKS
}

# ---------------------------------------------------------------------
# Expensive-work reset controls
# ---------------------------------------------------------------------
# Chapter 3 always progresses through Batch A → B → C when its cells execute.
#
# False = recover every valid saved fit and compute only missing/failed work.
# True  = delete that batch's saved artifacts first and recompute it from scratch.
#
# Separate RUN switches are intentionally unnecessary: restartable checkpoints
# already prevent completed expensive work from being repeated.
RESET_TRANSFORMER_REPRESENTATION_SCREEN = False
RESET_TRANSFORMER_PROFILE_SCREEN = False
RESET_TRANSFORMER_CONFIRMATION_SCREEN = False

# Interpretation remains separately gated because it is optional diagnostic
# inference after model selection rather than part of the A → B → C tournament.
RUN_TRANSFORMER_RELIANCE = False
RESET_TRANSFORMER_RELIANCE = False


# ---------------------------------------------------------------------
# Transformer development scope
# ---------------------------------------------------------------------
# Five canonical Dayparts form one day. The three candidate lookbacks
# therefore represent exactly 1, 2, and 4 complete weeks.

DAYPARTS_PER_DAY = 5
TRANSFORMER_LOOKBACK_STEPS = [35, 70, 140]

TRANSFORMER_SENTINEL_KEYS = [
    ("post_cp_primary", "taxi_trip_count", 1),
    ("full_history_regime_aware", "taxi_avg_trip_speed", 5),
    ("post_cp_primary", "fhvhv_trip_count", 2),
    ("full_history_regime_aware", "fhvhv_avg_trip_speed", 1),
    ("post_cp_primary", "subway_ridership", 5),
]

TRANSFORMER_REPRESENTATION_CANDIDATES = [
    {"representation_id": "one_week_daily", "lookback_steps": 35, "patch_length": 5, "patch_stride": 5},
    {"representation_id": "two_week_daily", "lookback_steps": 70, "patch_length": 5, "patch_stride": 5},
    {"representation_id": "four_week_daily", "lookback_steps": 140, "patch_length": 5, "patch_stride": 5},
    {"representation_id": "two_week_overlapping_two_day", "lookback_steps": 70, "patch_length": 10, "patch_stride": 5},
]

TRANSFORMER_PROFILE_CANDIDATES = [
    {"profile_id": "compact", "d_model": 48, "n_heads": 4, "n_layers": 2, "d_ff": 96, "dropout": 0.10, "zone_embedding_dim": 8, "learning_rate": 5e-4, "weight_decay": 1e-4},
    {"profile_id": "balanced", "d_model": 64, "n_heads": 4, "n_layers": 2, "d_ff": 128, "dropout": 0.10, "zone_embedding_dim": 16, "learning_rate": 3e-4, "weight_decay": 1e-4},
    {"profile_id": "higher_capacity_regularized", "d_model": 96, "n_heads": 6, "n_layers": 3, "d_ff": 192, "dropout": 0.15, "zone_embedding_dim": 16, "learning_rate": 2e-4, "weight_decay": 5e-4},
]

TRANSFORMER_POSITION_ENCODING = "learned"
TRANSFORMER_POOLING = "mean"
TRANSFORMER_LOSS = "huber"
TRANSFORMER_OPTIMIZER = "adamw"
TRANSFORMER_GRAD_CLIP_NORM = 1.0
TRANSFORMER_MAX_EPOCHS = 30
TRANSFORMER_PATIENCE = 5
TRANSFORMER_MIN_DELTA = 1e-4
TRANSFORMER_BATCH_SIZE_CANDIDATES = [1024, 512, 256]
TRANSFORMER_INNER_VALIDATION_DAYS = 28
TRANSFORMER_MAX_TRAIN_ORIGINS = 192

DAYPART_ORDER = [
    "overnight",
    "am_peak",
    "midday",
    "pm_peak",
    "evening",
]

TRANSFORMER_DEVELOPMENT_CONTRACT_ID = "transformer_sequence_scope_v1"
TRANSFORMER_REPRESENTATION_CONTRACT_ID = "transformer_representation_screen_v1"
TRANSFORMER_PROFILE_CONTRACT_ID = "transformer_profile_screen_v1"
TRANSFORMER_CONFIRMATION_CONTRACT_ID = "transformer_confirmation_screen_v1"
TRANSFORMER_RELIANCE_CONTRACT_ID = "transformer_reliance_diagnostic_v1"


# ---------------------------------------------------------------------
# Reproducibility and device configuration
# ---------------------------------------------------------------------

GLOBAL_SEED = 696
TORCH_DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)
TORCH_DTYPE = torch.float32


# ---------------------------------------------------------------------
# Future post-decision controls remain disabled in this notebook.
# ---------------------------------------------------------------------

RUN_TRANSFORMER_CADENCE_SCREEN = False
RESET_TRANSFORMER_CADENCE_SCREEN = False
RUN_TRANSFORMER_FULL_VALIDATION = False
RESET_TRANSFORMER_FULL_VALIDATION = False
RUN_TRANSFORMER_ATTRIBUTION = False
RESET_TRANSFORMER_ATTRIBUTION = False

WRITE_TRANSFORMER_INTERMEDIATES = True
WRITE_TRANSFORMER_FINAL_OUTPUTS = True

The shared utilities below support four recurring needs.

First, external 4.5.1 artifacts are checked before use so a missing file or incompatible upstream schema fails clearly rather than producing a misleading downstream result.

Second, experiment outputs are written **atomically**: the complete file is written to a temporary location and only then moved into its final name. This prevents an interrupted write from looking like valid completed work.

Third, expensive experiments use durable logical-fit checkpoints. A logical fit is the smallest independently recoverable model-training unit. Later runs can therefore reuse successful work and execute only missing or failed units.

Finally, the shared evaluation helpers preserve the project's forecasting conventions, including exact common-support comparisons and separate raw versus non-negative final predictions.

In [3]:
# ---------------------------------------------------------------------
# 4.5.2.0C — External-input, persistence, restart, and evaluation helpers
# ---------------------------------------------------------------------

def require_external_columns(df, required_columns, artifact_name):
    """Fail clearly when an upstream DataFrame violates its declared schema."""
    missing = sorted(set(required_columns) - set(df.columns))
    if missing:
        raise ValueError(f"{artifact_name} is missing required columns: {missing}")


def require_external_parquet_columns(path, required_columns):
    """Validate an upstream Parquet schema without materializing its rows."""
    schema_columns = set(pq.ParquetFile(path).schema_arrow.names)
    missing = sorted(set(required_columns) - schema_columns)
    if missing:
        raise ValueError(f"{Path(path).name} is missing required columns: {missing}")
    return schema_columns


def require_external_files(path_by_name):
    """Fail once with the complete list of missing authoritative inputs."""
    missing = [f"{name}: {path}" for name, path in path_by_name.items() if not Path(path).exists()]
    if missing:
        raise FileNotFoundError(
            "Required 4.5.1 handoff artifacts are unavailable:\n- " + "\n- ".join(missing)
        )


def external_single_value(df, column, artifact_name):
    """Recover one invariant setting from an external contract table."""
    require_external_columns(df, [column], artifact_name)
    values = df[column].dropna().unique()
    if len(values) != 1:
        raise ValueError(
            f"{artifact_name}.{column} must contain one non-null contract value; "
            f"observed {values.tolist()}."
        )
    return values[0]


def seed_experiment(seed=GLOBAL_SEED):
    """Seed Python, NumPy, and PyTorch for comparable logical fits."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def atomic_write_parquet(df, path):
    """Write a Parquet artifact atomically so partial files never count as outputs."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temp_path = path.with_name(f".{path.stem}.{uuid.uuid4().hex}.tmp{path.suffix}")
    try:
        df.to_parquet(temp_path, index=False, compression="zstd")
        os.replace(temp_path, path)
    finally:
        if temp_path.exists():
            temp_path.unlink()


def atomic_write_json(payload, path):
    """Write small experiment metadata atomically."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temp_path = path.with_name(f".{path.stem}.{uuid.uuid4().hex}.tmp{path.suffix}")
    try:
        with temp_path.open("w", encoding="utf-8") as handle:
            json.dump(payload, handle, indent=2, default=str)
        os.replace(temp_path, path)
    finally:
        if temp_path.exists():
            temp_path.unlink()


def atomic_torch_save(payload, path):
    """Write model state atomically before a fit is marked successful."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temp_path = path.with_name(f".{path.stem}.{uuid.uuid4().hex}.tmp{path.suffix}")
    try:
        torch.save(payload, temp_path)
        os.replace(temp_path, path)
    finally:
        if temp_path.exists():
            temp_path.unlink()


def logical_key_token(record, logical_key_columns):
    """Create a stable short token from a logical-fit identity."""
    key_text = "|".join(f"{column}={record[column]}" for column in logical_key_columns)
    return hashlib.sha1(key_text.encode("utf-8")).hexdigest()[:16]


def reset_experiment_artifacts(checkpoint_dir, consolidated_paths=()):
    """Reset only one explicitly selected experiment and its consolidated outputs."""
    checkpoint_dir = Path(checkpoint_dir)
    if checkpoint_dir.exists():
        shutil.rmtree(checkpoint_dir)
    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    for path in consolidated_paths:
        path = Path(path)
        if path.exists():
            path.unlink()


def recover_successful_checkpoints(checkpoint_dir, contract_id, logical_key_columns, required_path_columns=()):
    """Recover only readable, complete, successful checkpoints of this contract."""
    checkpoint_dir = Path(checkpoint_dir)
    recovered_records = []
    required_columns = {"contract_id", "status", *logical_key_columns, *required_path_columns}

    for path in sorted(checkpoint_dir.glob("fit*.parquet")):
        try:
            part = pd.read_parquet(path)
            if len(part) != 1 or not required_columns.issubset(part.columns):
                continue
            row = part.iloc[0]
            if row["contract_id"] != contract_id or row["status"] != "success":
                continue
            if any(pd.isna(row[column]) or not Path(row[column]).exists() for column in required_path_columns):
                continue
            recovered_records.append(row.to_dict())
        except (OSError, ValueError, KeyError, IndexError, TypeError):
            continue

    if not recovered_records:
        return pd.DataFrame(columns=sorted(required_columns))

    successful_df = pd.DataFrame(recovered_records)
    duplicate_mask = successful_df.duplicated(logical_key_columns, keep=False)
    if duplicate_mask.any():
        duplicate_keys = successful_df.loc[duplicate_mask, logical_key_columns].drop_duplicates().to_dict("records")
        raise ValueError(f"Multiple successful current-contract checkpoints exist for the same logical fit: {duplicate_keys[:10]}")
    return successful_df.reset_index(drop=True)


def validate_exact_success_roster(successful_df, expected_roster_df, logical_key_columns):
    """Require completed checkpoints to equal the planned logical-fit roster exactly."""
    expected = set(map(tuple, expected_roster_df[logical_key_columns].drop_duplicates().to_numpy()))
    actual = set(map(tuple, successful_df[logical_key_columns].drop_duplicates().to_numpy()))
    missing = expected - actual
    extra = actual - expected
    if missing or extra:
        raise ValueError(
            "Completed checkpoints do not match the planned experiment. "
            f"Missing logical fits: {len(missing):,}; unexpected logical fits: {len(extra):,}."
        )
    return True


def count_trainable_parameters(model):
    """Return the number of parameters eligible for optimization."""
    return int(sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad))


def transformer_fit_parameter_count(model):
    """Return the trainable parameter count used in experiment summaries."""
    return count_trainable_parameters(model)


def regression_metrics(y_true, y_pred):
    """Calculate MAE, RMSE, bias, and evaluated-row count in native units."""
    actual = np.asarray(y_true, dtype=float)
    predicted = np.asarray(y_pred, dtype=float)
    valid = np.isfinite(actual) & np.isfinite(predicted)
    if not valid.any():
        return {"evaluated_rows": 0, "mae": np.nan, "rmse": np.nan, "bias": np.nan}
    error = predicted[valid] - actual[valid]
    return {
        "evaluated_rows": int(valid.sum()),
        "mae": float(np.mean(np.abs(error))),
        "rmse": float(np.sqrt(np.mean(np.square(error)))),
        "bias": float(np.mean(error)),
    }


def exact_common_support(df, prediction_columns):
    """Keep only physical forecast rows scored by every compared prediction."""
    required_values = [df["target_observed_value"].notna(), *[df[column].notna() for column in prediction_columns]]
    return df.loc[np.logical_and.reduce(required_values)].copy()


def exact_common_support_many(frames, key_columns, value_columns):
    """Inner-join prediction tables on exact keys and reject duplicate key rows."""
    if len(frames) != len(value_columns):
        raise ValueError("One value-column list is required for each prediction frame.")
    merged = frames[0][key_columns + value_columns[0]].copy()
    if merged.duplicated(key_columns).any():
        raise ValueError("The first prediction frame has duplicate comparison keys.")
    for frame, columns in zip(frames[1:], value_columns[1:]):
        if frame.duplicated(key_columns).any():
            raise ValueError("A prediction frame has duplicate comparison keys.")
        merged = merged.merge(frame[key_columns + columns], on=key_columns, how="inner", validate="one_to_one")
    return merged


def apply_prediction_output_rule(prediction_raw):
    """Apply the nonnegative production rule without overwriting raw forecasts."""
    return np.maximum(0.0, np.asarray(prediction_raw, dtype=float))


def clear_device_memory():
    """Release temporary model memory between independent fits."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def reset_cuda_peak_memory():
    """Reset CUDA allocation accounting before a preflight trial."""
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()


def read_cuda_peak_memory_mb():
    """Return peak allocated CUDA memory, or NaN on CPU-only runs."""
    return float(torch.cuda.max_memory_allocated() / (1024 ** 2)) if torch.cuda.is_available() else np.nan


def progress_message(index, total, label):
    """Emit one concise Deepnote-safe status line per logical fit."""
    print(f"[{index}/{total}] {label}", flush=True)


def stable_transformer_fit_seed(contract_id, track, metric, horizon, candidate_id, temporal_window):
    """Derive a deterministic seed from the complete logical-fit identity."""
    parts = [contract_id, track, metric, horizon, candidate_id, temporal_window, GLOBAL_SEED]
    digest = hashlib.sha256("|".join(map(str, parts)).encode("utf-8")).digest()
    return int.from_bytes(digest[:4], "big") % (2**31 - 1)


def thin_training_origins(origin_df, max_origins, origin_column="origin_date"):
    """Select complete origin groups spread across legal history only."""
    if max_origins <= 0:
        raise ValueError("max_origins must be positive.")
    required = {origin_column}
    if not required.issubset(origin_df.columns):
        raise ValueError(f"Origin table is missing {sorted(required - set(origin_df.columns))}.")
    ordered = origin_df.sort_values(origin_column, kind="stable")
    origin_values = ordered[origin_column].drop_duplicates().tolist()
    if len(origin_values) <= max_origins:
        return ordered.copy()
    positions = np.linspace(0, len(origin_values) - 1, max_origins).round().astype(int)
    selected = set(np.asarray(origin_values, dtype=object)[np.unique(positions)].tolist())
    return ordered.loc[ordered[origin_column].isin(selected)].copy()


def recover_transformer_fit(checkpoint_path, contract_id, logical_key, prediction_path=None, curve_path=None, state_path=None):
    """Recover one fit only when metadata, identity, and companion artifacts are valid."""
    checkpoint_path = Path(checkpoint_path)
    if not checkpoint_path.exists():
        return None
    try:
        checkpoint = pd.read_parquet(checkpoint_path)
        if len(checkpoint) != 1:
            return None
        row = checkpoint.iloc[0]
        if row.get("contract_id") != contract_id or row.get("status") != "success":
            return None
        if any(str(row.get(column)) != str(value) for column, value in logical_key.items()):
            return None
        for path in [prediction_path, curve_path, state_path]:
            if path is not None:
                candidate = Path(path)
                if not candidate.exists():
                    return None
                if candidate.suffix == ".parquet" and pd.read_parquet(candidate).empty:
                    return None
                if candidate.suffix in {".pt", ".pth"}:
                    torch.load(candidate, map_location="cpu", weights_only=False)
        return row.to_dict()
    except (OSError, ValueError, KeyError, IndexError, TypeError, RuntimeError):
        return None


Before any forecasting artifact is loaded, perform a small environment check. This establishes which PyTorch device is available and confirms that the new 4.5.2 output directories are writable.

This is operational QA only. No forecasting evidence is evaluated, and no model is constructed or trained.

In [4]:
# ---------------------------------------------------------------------
# 4.5.2.0D — Environment preflight
# ---------------------------------------------------------------------

device_label = (
    torch.cuda.get_device_name(0)
    if torch.cuda.is_available()
    else "CPU"
)

if not INTERMEDIATE_DIR.exists() or not FINAL_DIR.exists():
    raise RuntimeError(
        "The 4.5.2 output directories could not be initialized."
    )

print(
    "4.5.2 setup ready | "
    f"PyTorch {torch.__version__} | "
    f"device={device_label} | "
    f"seed={GLOBAL_SEED}"
)

print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {device_label}")
if torch.cuda.is_available():
    print(f"Total GPU memory MB: {torch.cuda.get_device_properties(0).total_memory / (1024 ** 2):.3f}")
for directory in [TRANSFORMER_REPRESENTATION_SCREEN_DIR, TRANSFORMER_PROFILE_SCREEN_DIR, TRANSFORMER_CONFIRMATION_SCREEN_DIR, TRANSFORMER_DIAGNOSTIC_DIR]:
    directory.mkdir(parents=True, exist_ok=True)
print(f"Representation candidates: {len(TRANSFORMER_REPRESENTATION_CANDIDATES)}")
print(f"Profile candidates: {len(TRANSFORMER_PROFILE_CANDIDATES)}")
print(f"Sentinel jobs: {len(TRANSFORMER_SENTINEL_KEYS)}")

4.5.2 setup ready | PyTorch 2.1.2+cu121 | device=CPU | seed=696
CUDA available: False
GPU: CPU
Representation candidates: 4
Profile candidates: 3
Sentinel jobs: 5


**Findings.** The Transformer environment initialized successfully with PyTorch 2.1.2 on CPU. Four history representations, three Transformer capacity profiles, and five representative forecasting problems are registered for model selection.

CUDA is not available in this Deepnote session, so GPU-memory measurements are not applicable. The CPU environment is nevertheless sufficient for the bounded Transformer screen, and the restartable artifact directories are ready to preserve completed work.

# 4.5.2.1 Establish the Transformer Forecasting Contract

The Transformer should not redefine the forecasting problem that earlier model families already established. Before introducing patches, self-attention, positional information, or encoder depth, this section separates the information and evaluation rules inherited from 4.5.1 from the choices that genuinely belong to the Transformer.

Three boundaries matter.

First, **the forecasting information remains shared**. Recent multimodal mobility, observation masks, forecast-time supporting context, leakage-safe weekly STL seasonal state, Taxi Zone identity, target definitions, horizons, benchmark semantics, leakage protections, and raw/final forecast semantics come from the finalized forecasting framework.

Second, **Transformer architecture and numerical training settings are not inherited from the MLP**. The 35-Daypart history is retained as a reference candidate, but history length, patch geometry, representation width, attention structure, encoder depth, feed-forward width, regularization, Taxi Zone embedding width, learning rate, and weight decay are evaluated as Transformer-specific choices.

Third, **4.5.2 develops only the Transformer family**. It may retain different Transformer profiles by target when the bounded evidence supports doing so, but the comparison that chooses among classical, tree-based, MLP, and Transformer forecasts remains the responsibility of 4.6.x.

## 4.5.2.1.1 Load and Audit the 4.5.1 Handoff

The finalized 4.5.1 handoff contains both reusable forecasting contracts and evidence specific to its selected Lag-window MLP. 4.5.2 consumes the reusable contracts without allowing the MLP's selected architecture or numerical hyperparameters to become Transformer defaults.

This section loads the shared forecasting-framework registry, model-facing input registry, sequence/preprocessing contract, forecast-output contract, training-policy evidence, closeout QA, handoff manifest, and dense mobility-history state. It also checks the authoritative upstream forecasting paths before any Transformer-specific representation is constructed.

In [5]:
# ---------------------------------------------------------------------
# 4.5.2.1.1A — Load the reusable 4.5.1 neural handoff
# ---------------------------------------------------------------------

required_451_paths = {
    "forecasting framework registry": NEURAL_FRAMEWORK_REGISTRY_PATH,
    "neural input registry": NEURAL_INPUT_REGISTRY_PATH,
    "sequence/preprocessing contract": NEURAL_SEQUENCE_CONTRACT_PATH,
    "forecast-output contract": NEURAL_OUTPUT_CONTRACT_PATH,
    "training-policy evidence": NEURAL_TRAINING_POLICY_EVIDENCE_PATH,
    "dense mobility-history state": NEURAL_HISTORY_STATE_PATH,
    "closeout QA": NEURAL_CLOSEOUT_QA_PATH,
    "handoff manifest": NEURAL_HANDOFF_MANIFEST_PATH,
    "rolling-origin registry": ROLLING_ORIGIN_PATH,
    "track contract": TRACK_CONTRACT_PATH,
    "supported modeling scope": SUPPORTED_MODELING_SCOPE_PATH,
    "advanced benchmark registry": ADVANCED_BENCHMARK_REGISTRY_PATH,
}

require_external_files(required_451_paths)

neural_framework_registry_df = pd.read_parquet(NEURAL_FRAMEWORK_REGISTRY_PATH)
neural_input_registry_df = pd.read_parquet(NEURAL_INPUT_REGISTRY_PATH)
neural_sequence_contract_df = pd.read_parquet(NEURAL_SEQUENCE_CONTRACT_PATH)
neural_output_contract_df = pd.read_parquet(NEURAL_OUTPUT_CONTRACT_PATH)
neural_training_policy_evidence_df = pd.read_parquet(
    NEURAL_TRAINING_POLICY_EVIDENCE_PATH
)
neural_closeout_qa_df = pd.read_parquet(NEURAL_CLOSEOUT_QA_PATH)
neural_handoff_manifest_df = pd.read_parquet(NEURAL_HANDOFF_MANIFEST_PATH)
ordinary_track_contract_df = pd.read_parquet(TRACK_CONTRACT_PATH)
supported_modeling_scope_df = pd.read_parquet(SUPPORTED_MODELING_SCOPE_PATH)
advanced_model_benchmark_registry_df = pd.read_parquet(
    ADVANCED_BENCHMARK_REGISTRY_PATH
)

history_state_size_mb = NEURAL_HISTORY_STATE_PATH.stat().st_size / (1024 ** 2)
print(
    "4.5.1 reusable handoff available | "
    f"contract tables loaded | history state={history_state_size_mb:.3f} MB"
)


4.5.1 reusable handoff available | contract tables loaded | history state=16.723 MB


**Findings.** The finalized 4.5.1 forecasting handoff loaded successfully, including its reusable forecasting contracts and the 16.723 MB dense mobility-history state.

This gives the Transformer branch the same underlying forecasting problem, ordered mobility history, context definitions, and leakage rules used by the completed neural-sequence work. 4.5.2 can therefore focus on changing how that information is represented and modeled rather than rebuilding the upstream forecasting system.

## 4.5.2.1.2 Freeze the Inherited Forecasting Problem

The Transformer inherits the same forecasting task rather than creating a parallel definition of it.

The fixed scope is:

* **2 ordinary forecasting tracks:** Post-CP Primary and Full-History Regime-Aware.
* **5 targets:** Taxi trips, Taxi average speed, FHVHV trips, FHVHV average speed, and Subway ridership.
* **3 forecast horizons:** 1, 2, and 5 Dayparts ahead.
* **10 recent-mobility sequence channels:** five mobility values plus five matching observation masks.
* **Supporting forecast-time context:** the finalized 4.5.1 context, including the leakage-safe weekly STL seasonal state.
* **Taxi Zone identity:** the shared spatial identity input.
* **Forecast output:** preserve `prediction_raw` and separately create `prediction_final = max(0, prediction_raw)`.
* **Final holdout:** no target dated January 5, 2026 or later may be used for development or ordinary validation.

The inherited 35-Daypart history is a reference candidate, not a frozen Transformer choice.

The next cell verifies that the external 4.5.1 framework registry actually matches this inherited scope.

It checks the complete set of **Track × Target × Horizon** forecasting jobs rather than relying only on row counts. It also recovers the live sequence width, context width, Taxi Zone embedding width, training-duration policy, and prediction rule from the handoff itself.

The context width is deliberately not hard-coded here. If 4.5.1 publishes a different dimension from the finalized methodology, the live contract should determine what 4.5.2 receives. The detailed context-feature roster, including the weekly STL fields, will be validated when those values are assembled into Transformer tensors in Section 4.5.2.2.

In [6]:
# ---------------------------------------------------------------------
# 4.5.2.1.2A — Validate the inherited forecasting scope
# ---------------------------------------------------------------------

FRAMEWORK_ARTIFACT_NAME = (
    "neural_forecasting_framework_registry.parquet"
)

require_external_columns(
    neural_framework_registry_df,
    [
        "modeling_track",
        "metric",
        "horizon",
        "sequence_window_steps",
        "sequence_input_dim",
        "context_input_dim",
        "zone_embedding_dim",
        "refit_cadence",
        "production_training_duration_policy",
        "prediction_output_rule",
        "raw_prediction_preserved",
        "final_holdout_scored",
    ],
    FRAMEWORK_ARTIFACT_NAME,
)


# ---------------------------------------------------------------------
# Require the exact ordinary forecasting-job roster
# ---------------------------------------------------------------------
# There should be one definition for every combination of the two ordinary
# tracks, five targets, and three horizons: 2 × 5 × 3 = 30 forecasting jobs.

expected_jobs = {
    (track, metric, horizon)
    for track in ORDINARY_TRACKS
    for metric in FORECAST_TARGETS
    for horizon in FORECAST_HORIZONS
}

observed_jobs = set(
    neural_framework_registry_df[
        ["modeling_track", "metric", "horizon"]
    ]
    .drop_duplicates()
    .itertuples(index=False, name=None)
)

if observed_jobs != expected_jobs:
    missing_jobs = expected_jobs - observed_jobs
    extra_jobs = observed_jobs - expected_jobs

    raise ValueError(
        "The 4.5.1 framework does not match the frozen 30-job ordinary "
        "forecasting scope. "
        f"Missing jobs={len(missing_jobs):,}; "
        f"unexpected jobs={len(extra_jobs):,}."
    )


# ---------------------------------------------------------------------
# Recover invariant neural-framework dimensions from the external contract
# ---------------------------------------------------------------------
# These values describe the information shared across neural model families.
# Reading them from 4.5.1 prevents 4.5.2 from maintaining a competing copy.

INHERITED_SEQUENCE_WINDOW_STEPS = int(
    external_single_value(
        neural_framework_registry_df,
        "sequence_window_steps",
        FRAMEWORK_ARTIFACT_NAME,
    )
)

NEURAL_SEQUENCE_INPUT_DIM = int(
    external_single_value(
        neural_framework_registry_df,
        "sequence_input_dim",
        FRAMEWORK_ARTIFACT_NAME,
    )
)

NEURAL_CONTEXT_INPUT_DIM = int(
    external_single_value(
        neural_framework_registry_df,
        "context_input_dim",
        FRAMEWORK_ARTIFACT_NAME,
    )
)

INHERITED_451_ZONE_EMBEDDING_DIM = int(
    external_single_value(
        neural_framework_registry_df,
        "zone_embedding_dim",
        FRAMEWORK_ARTIFACT_NAME,
    )
)

INHERITED_451_REFIT_CADENCE = str(
    external_single_value(
        neural_framework_registry_df,
        "refit_cadence",
        FRAMEWORK_ARTIFACT_NAME,
    )
)

INHERITED_TRAINING_DURATION_POLICY = str(
    external_single_value(
        neural_framework_registry_df,
        "production_training_duration_policy",
        FRAMEWORK_ARTIFACT_NAME,
    )
)

PREDICTION_OUTPUT_RULE = str(
    external_single_value(
        neural_framework_registry_df,
        "prediction_output_rule",
        FRAMEWORK_ARTIFACT_NAME,
    )
)


# ---------------------------------------------------------------------
# Enforce the shared information and output contracts
# ---------------------------------------------------------------------

if INHERITED_SEQUENCE_WINDOW_STEPS != 35:
    raise ValueError(
        "The inherited neural reference window is no longer 35 Dayparts. "
        "Reconcile the Transformer lookback design before proceeding."
    )

if NEURAL_CONTEXT_INPUT_DIM != 52:
    raise ValueError(
        "The inherited context contract must contain exactly 52 values."
    )

if NEURAL_SEQUENCE_INPUT_DIM != 10:
    raise ValueError(
        "The inherited sequence no longer contains the expected five mobility "
        "values plus five observation masks."
    )

if PREDICTION_OUTPUT_RULE != "max(0, prediction_raw)":
    raise ValueError(
        "The inherited neural final-prediction rule changed and must be "
        "reconciled before Transformer evaluation."
    )

if not neural_framework_registry_df[
    "raw_prediction_preserved"
].fillna(False).all():
    raise ValueError(
        "The 4.5.1 handoff does not preserve raw neural predictions."
    )

if neural_framework_registry_df[
    "final_holdout_scored"
].fillna(True).any():
    raise ValueError(
        "The 4.5.1 handoff indicates that the final holdout was scored."
    )


print(
    "Inherited forecasting contract validated | "
    f"jobs={len(observed_jobs)} | "
    f"reference history={INHERITED_SEQUENCE_WINDOW_STEPS} Dayparts | "
    f"sequence channels={NEURAL_SEQUENCE_INPUT_DIM} | "
    f"context values={NEURAL_CONTEXT_INPUT_DIM} | "
    f"reference MLP embedding={INHERITED_451_ZONE_EMBEDDING_DIM} | "
    "final holdout untouched"
)

Inherited forecasting contract validated | jobs=30 | reference history=35 Dayparts | sequence channels=10 | context values=52 | reference MLP embedding=8 | final holdout untouched


**Findings.** The inherited forecasting contract contains the expected **30 ordinary forecasting jobs: 2 modeling tracks × 5 mobility targets × 3 forecast horizons**.

The shared neural interface contains 10 historical channels—five mobility values and their five observation masks—plus 52 supporting context values and Taxi Zone identity. The 35-Daypart sequence from 4.5.1 remains a reference representation rather than a Transformer requirement.

Raw neural predictions remain separate from the downstream nonnegative forecast, and the final holdout beginning January 5, 2026 remains untouched by model selection.

## 4.5.2.1.3 Freeze the Transformer Development Plan

4.5.2 inherits the **bounded experimental strategy** established in the neural branch, not the Lag-window MLP's winning numerical hyperparameters.

| Stage | Question | Transformer-specific choice |
| --- | --- | --- |
| 4.5.2.3.1 | How much history and what patch geometry? | Four complete representations |
| 4.5.2.3.2 | Which complete Transformer specification? | Four coherent representation × profile specifications |
| 4.5.2.3.3 | Does the choice persist through time? | Two finalists across early, middle, and late windows |

The Transformer keeps learned positional encoding, mean pooling, Huber loss, AdamW, and gradient clipping fixed for this MVP. Profiles are complete specifications, not one-variable ablations. Architecture selection remains inside the Transformer family; the overall forecasting-family comparison remains 4.6.x.

### Section Summary

Section 4.5.2.1 establishes a strict boundary between the forecasting framework inherited from 4.5.1 and the choices that remain specific to the Transformer.

The Transformer keeps the same ordinary forecasting tracks, five targets, three horizons, mobility channels and observation masks, supporting forecast-time context, Taxi Zone identity, leakage-safe training-duration methodology, benchmark semantics, and raw/final forecast contract. The supporting-context width is recovered directly from the finalized 4.5.1 handoff rather than duplicated as a 4.5.2 constant.

The one-week, 35-Daypart history used by the earlier neural models is retained as the inherited reference rather than assumed to be optimal for attention. The next section therefore loads the shared dense mobility-history state, reconstructs longer legal recent-history windows, and verifies that the Transformer can receive one-, two-, and four-week histories without changing temporal ordering, missingness treatment, policy boundaries, or forecast alignment.

No Transformer has been trained, no architecture choice has been made, and the final holdout remains untouched.

# 4.5.2.2 Build and Validate Transformer-Ready Inputs

The Transformer uses the same forecasting information as the finalized neural branch but reorganizes the recent mobility history into longer, patchable sequences.

This section performs no model selection. It restores the authoritative dense mobility-history state, recovers the published context and preprocessing contracts, reconstructs legal 35-, 70-, and 140-Daypart windows, and verifies that patching changes only the representation—not the forecast timing, missingness semantics, policy boundary, or eligible target rows.

The most important comparison rule is **common support**. A longer lookback cannot appear better merely because it drops difficult early forecast rows. Whenever representations are compared later, they must be scored on the exact same eligible forecast population.

## 4.5.2.2.1 Restore the Shared Mobility-History and Context State

The reusable dense history state contains the canonical mobility values and observation masks needed to reconstruct longer recent-history windows without rebuilding the source panel.

Load that state directly from `neural_mobility_history_state.npz`. Require the published arrays, validate their dimensions and ordering, and preserve the five mobility channels in the finalized order used by 4.5.1. The supporting context is recovered from the finalized model-facing input contract rather than reconstructed from a hand-written feature list. Weekly STL is never refit here.

This first step restores the dense \*\*Taxi Zone × observation sequence × mobility metric\*\* arrays and verifies the structural assumptions that later sequence slicing depends on\. In particular, Taxi Zone IDs and observation\-sequence IDs remain explicit identities; the lookup dictionaries below translate those identities into NumPy positions only when an array must be indexed\.

**Restore the finalized dense mobility-history state.** The Transformer inherits its historical mobility state directly from the finalized 4.5.1 neural handoff rather than rebuilding history from the forecasting matrices.

This first step restores the dense **Taxi Zone × observation sequence × mobility metric** arrays and verifies the structural assumptions that later sequence slicing depends on. In particular, Taxi Zone IDs and observation-sequence IDs remain explicit identities; the lookup dictionaries below translate those identities into NumPy positions only when an array must be indexed.

In [7]:
# ---------------------------------------------------------------------
# 4.5.2.2.1A — Restore the finalized dense mobility-history state
# ---------------------------------------------------------------------

required_history_arrays = {"values", "masks", "zone_ids", "sequence_ids"}

with np.load(NEURAL_HISTORY_STATE_PATH, allow_pickle=False) as history_state:
    missing_arrays = sorted(required_history_arrays - set(history_state.files))
    if missing_arrays:
        raise ValueError(
            f"The finalized dense history is missing required arrays: {missing_arrays}"
        )

    # Validate the raw mask values before casting to bool. Casting first would
    # turn any nonzero value into True and could hide a malformed upstream mask.
    raw_masks = np.asarray(history_state["masks"])
    if not np.isin(raw_masks, [False, True]).all():
        raise ValueError("Observation masks must contain only binary 0/1 values.")

    transformer_history_values = np.asarray(history_state["values"], dtype=np.float32)
    transformer_history_masks = raw_masks.astype(bool, copy=False)
    transformer_zone_ids = np.asarray(history_state["zone_ids"], dtype=np.int64)
    transformer_sequence_ids = np.asarray(history_state["sequence_ids"], dtype=np.int64)


# Values and masks describe the same historical cells, so their complete
# three-dimensional geometry must agree exactly.
if transformer_history_values.shape != transformer_history_masks.shape:
    raise ValueError("History values and observation masks must have identical shapes.")

if transformer_history_values.ndim != 3 or transformer_history_values.shape[-1] != 5:
    raise ValueError("Dense history must have shape [Taxi Zone, sequence position, 5].")

if len(transformer_zone_ids) != transformer_history_values.shape[0]:
    raise ValueError("Taxi Zone IDs do not align with the dense history's zone axis.")

if len(transformer_sequence_ids) != transformer_history_values.shape[1]:
    raise ValueError(
        "Canonical sequence IDs do not align with the dense history's time axis."
    )

if len(np.unique(transformer_zone_ids)) != len(transformer_zone_ids):
    raise ValueError("Taxi Zone IDs must be unique.")

if len(transformer_sequence_ids) == 0:
    raise ValueError("The dense history must contain at least one sequence position.")

# Later lookback slicing uses arithmetic on sequence positions. Requiring a
# contiguous calendar here prevents a missing sequence ID from masquerading as
# an ordinary one-step transition.
expected_sequence_ids = np.arange(
    transformer_sequence_ids.min(),
    transformer_sequence_ids.max() + 1,
    dtype=np.int64,
)

if not np.array_equal(transformer_sequence_ids, expected_sequence_ids):
    raise ValueError("Canonical sequence IDs must be strictly ordered and contiguous.")


# These dictionaries separate canonical IDs from physical array positions.
# That prevents downstream code from assuming, for example, that Taxi Zone 1
# necessarily occupies array position 1.
transformer_zone_to_index = {
    int(zone_id): index for index, zone_id in enumerate(transformer_zone_ids)
}
transformer_sequence_to_index = {
    int(sequence_id): index
    for index, sequence_id in enumerate(transformer_sequence_ids)
}

**Restore target-specific sequence and shared context contracts.** 
The dense history has one physical five-metric order, but the finalized neural interface is **target-specific**: the metric being forecast comes first, the other four mobility histories follow, and the five observation masks must follow that same order.

The 52-value context interface is different. Its feature order and preprocessing semantics are shared across all five targets. Rather than reconstructing those rules manually, this cell reads the authoritative 4.5.1 input registry and converts its published preprocessing descriptions into the three Transformer roles used downstream.

In [8]:
# ---------------------------------------------------------------------
# 4.5.2.2.1B — Restore target-specific sequence and shared context contracts
# ---------------------------------------------------------------------

require_external_columns(
    neural_input_registry_df,
    [
        "metric",
        "representation",
        "input_name",
        "input_group",
        "input_index",
        "preprocessing",
    ],
    "neural_input_registry.parquet",
)

sequence_value_channels_by_target = {}
sequence_mask_channels_by_target = {}
context_orders_by_target = {}
first_target_context = None

for metric in FORECAST_TARGETS:
    metric_registry = neural_input_registry_df.loc[
        neural_input_registry_df["metric"].eq(metric)
    ].copy()

    values = metric_registry.loc[
        metric_registry["representation"].eq("sequence_value")
    ].sort_values("input_index")

    masks = metric_registry.loc[
        metric_registry["representation"].eq("sequence_mask")
    ].sort_values("input_index")

    context = metric_registry.loc[
        metric_registry["representation"].eq("context")
    ].sort_values("input_index")

    if len(values) != 5 or len(masks) != 5:
        raise ValueError(f"{metric} must publish five sequence values and five masks.")

    value_names = values["input_name"].astype(str).tolist()
    mask_names = masks["input_name"].astype(str).tolist()

    # 4.5.1 deliberately puts the target's own history first. Preserving that
    # ordering is important because the same physical dense array is reordered
    # differently depending on which metric the Transformer is forecasting.
    if value_names[0] != metric:
        raise ValueError(f"{metric} must appear first in its model-facing value order.")

    mask_sources = [name.removeprefix("observed_mask__") for name in mask_names]
    if mask_sources != value_names:
        raise ValueError(f"{metric} mask order does not match its value-channel order.")

    if len(context) != 52:
        raise ValueError(f"{metric} must publish a 52-value context interface.")

    sequence_value_channels_by_target[metric] = value_names
    sequence_mask_channels_by_target[metric] = mask_names
    context_orders_by_target[metric] = context["input_name"].astype(str).tolist()

    if first_target_context is None:
        first_target_context = context.copy()


# Unlike sequence history, context order must be identical for every target.
# One shared Transformer preprocessor can therefore interpret the same 52 slots.
if len({tuple(order) for order in context_orders_by_target.values()}) != 1:
    raise ValueError("All five targets must share one ordered 52-value context interface.")


preprocessing_role_map = {
    "Training-only median completion and scaling": "ordinary_continuous",
    "State-aware training-only fitted-offset scaling": "weekly_stl_offset",
    "Bounded deterministic/binary passthrough": "passthrough",
}

first_target_context["role"] = first_target_context["preprocessing"].map(
    preprocessing_role_map
)

unknown_preprocessing = sorted(
    first_target_context.loc[
        first_target_context["role"].isna(), "preprocessing"
    ].astype(str).unique()
)

if unknown_preprocessing:
    raise ValueError(
        f"Unknown finalized context preprocessing contract: {unknown_preprocessing}"
    )

transformer_context_contract_df = (
    first_target_context.rename(columns={"input_name": "feature"})
    [["feature", "input_group", "role", "input_index", "preprocessing"]]
    .sort_values("input_index")
    .reset_index(drop=True)
)

TRANSFORMER_SEQUENCE_VALUE_CHANNELS_BY_TARGET = sequence_value_channels_by_target
TRANSFORMER_SEQUENCE_MASK_CHANNELS_BY_TARGET = sequence_mask_channels_by_target
TRANSFORMER_CONTEXT_FEATURE_ORDER = transformer_context_contract_df["feature"].tolist()

**Validate and summarize the restored Transformer interface.**
A few compact boundary checks now confirm that the restored registry still describes the finalized 4.5.1 interface expected by this notebook: **24 ordinary continuous context values, one state-aware weekly-STL offset, and 27 passthrough values**, for 52 context values total.

The weekly-STL offset is checked together with its enabled flag and missingness indicator because those three fields jointly distinguish a fitted seasonal effect, a legitimate disabled zero, and an enabled-but-not-yet-available value.

In [9]:
# ---------------------------------------------------------------------
# 4.5.2.2.1C — Validate and summarize the restored Transformer interface
# ---------------------------------------------------------------------

role_counts = transformer_context_contract_df["role"].value_counts()
continuous_count = int(role_counts.get("ordinary_continuous", 0))
stl_count = int(role_counts.get("weekly_stl_offset", 0))
passthrough_count = int(role_counts.get("passthrough", 0))

if (continuous_count, stl_count, passthrough_count) != (24, 1, 27):
    raise ValueError(
        "The final context contract must contain 24 ordinary continuous, "
        "1 weekly-STL offset, and 27 passthrough fields."
    )

required_stl_features = {
    "weekly_stl_seasonal_offset",
    "weekly_stl_enabled",
    "missing_mask__weekly_stl_seasonal_offset",
}
stl_bundle_present = required_stl_features.issubset(TRANSFORMER_CONTEXT_FEATURE_ORDER)

if not stl_bundle_present:
    raise ValueError("The finalized input contract lacks the weekly-STL semantic bundle.")

transformer_context_feature_count = len(TRANSFORMER_CONTEXT_FEATURE_ORDER)

if transformer_context_feature_count != 52:
    raise ValueError("The finalized Transformer context contract must contain 52 values.")

transformer_input_summary_df = pd.DataFrame([{
    "History zones": len(transformer_zone_ids),
    "Canonical positions": len(transformer_sequence_ids),
    "Mobility value channels": 5,
    "Observation-mask channels": 5,
    "Final sequence channels": NEURAL_SEQUENCE_INPUT_DIM,
    "Context values": transformer_context_feature_count,
    "Ordinary continuous context": continuous_count,
    "Weekly STL offset": stl_count,
    "Passthrough context": passthrough_count,
    "Weekly STL state present": stl_bundle_present,
    "Final holdout used?": "No",
}])

display(transformer_input_summary_df.round(3))

,History zones,Canonical positions,Mobility value channels,Observation-mask channels,Final sequence channels,Context values,Ordinary continuous context,Weekly STL offset,Passthrough context,Weekly STL state present,Final holdout used?
0,258,5930,5,5,10,52,24,1,27,True,No


**Findings.** The restored mobility-history state contains **258 Taxi Zones × 5,930 canonical observation positions**, with five mobility values and five matching observation masks.

The model-facing sequence therefore retains 10 channels, while the supporting context contains the expected 52 values: 24 ordinary continuous features, one specially handled weekly-STL seasonal offset, and 27 deterministic, binary, or missingness values. The complete weekly-STL state is present.

Nothing Transformer-specific has changed the available forecasting information yet. This section confirms that patches and attention will operate on the same leakage-safe mobility, context, and Taxi Zone information inherited from 4.5.1.

## 4.5.2.2.2 Reconstruct One-, Two-, and Four-Week Legal Histories

A Transformer lookback is defined in canonical Dayparts, not in the number of non-missing observations. Every historical position remains present even when one or more mobility measurements were unavailable.

For a forecast issued at canonical origin t, a lookback of L contains the literal sequence positions from t-L+1 through t. The five mobility values and their five masks remain aligned at every historical position. Longer lookbacks may reduce early eligibility, but that support change is handled explicitly and never by padding or dropping missing Dayparts.

**Define legal history windows and track boundaries.**
The four Transformer representations require either 35, 70, or 140 historical Dayparts. A row is usable only when that entire history genuinely exists before the forecast origin; this notebook does not pad, backfill, or manufacture earlier observations.

The two ordinary modeling tracks also begin at different historical boundaries. Their start dates are therefore translated into the same canonical observation-sequence coordinate system used by the dense history before row-level eligibility is evaluated.

In [10]:
# ---------------------------------------------------------------------
# 4.5.2.2.2A — Define legal history windows and track boundaries
# ---------------------------------------------------------------------

def build_transformer_history_window(values, masks, origin_position, lookback_steps):
    """Return the literal t-L+1...t history, or None when it does not fully exist."""
    if lookback_steps not in {35, 70, 140}:
        raise ValueError(f"Unsupported Transformer lookback: {lookback_steps}")

    start = int(origin_position) - int(lookback_steps) + 1
    end = int(origin_position) + 1

    # No padding or backfilling is allowed. A candidate becomes eligible only
    # when its complete requested history physically exists in the dense state.
    if start < 0 or end > values.shape[0]:
        return None

    return values[start:end].copy(), masks[start:end].copy()


def _track_history_start_sequence_ids():
    """Translate each track's published history-start date to a sequence ID."""
    calendar = pd.read_parquet(
        ROLLING_ORIGIN_PATH,
        columns=["modeling_track", "date", "observation_sequence_id"],
    )
    calendar["date"] = pd.to_datetime(calendar["date"]).dt.normalize()
    calendar["observation_sequence_id"] = pd.to_numeric(
        calendar["observation_sequence_id"], errors="raise"
    ).astype("int64")

    track_contract = pd.read_parquet(
        TRACK_CONTRACT_PATH,
        columns=["modeling_track", "training_history_start_date"],
    )
    track_contract["training_history_start_date"] = pd.to_datetime(
        track_contract["training_history_start_date"]
    ).dt.normalize()

    starts = {}

    for track in ORDINARY_TRACKS:
        start_dates = track_contract.loc[
            track_contract["modeling_track"].eq(track), "training_history_start_date"
        ]

        if len(start_dates) != 1:
            raise ValueError(f"Expected exactly one training-history start for {track}.")

        # Restrict the calendar to the same modeling track before translating
        # its start date. This keeps track semantics explicit even if sequence
        # IDs happen to share the same global calendar.
        candidates = calendar.loc[
            calendar["modeling_track"].eq(track)
            & calendar["date"].ge(start_dates.iloc[0])
        ].sort_values("observation_sequence_id")

        if candidates.empty:
            raise ValueError(f"No canonical sequence position resolves {track}.")

        starts[track] = int(candidates.iloc[0]["observation_sequence_id"])

    return starts


TRACK_HISTORY_START_SEQUENCE_ID = _track_history_start_sequence_ids()

**Compute legal eligibility for one candidate/sentinel.**
Eligibility is evaluated against the **actual finalized forecasting matrix**, not inferred from calendar dates alone.

For a row to enter Transformer development evaluation, five conditions must agree: it belongs to the frozen rolling-validation surface, its target label is available, the modeling track permits the row, the candidate's complete lookback exists, and the target date precedes the untouched final holdout. Both the track-relative sequence rule and the physical dense-array position are checked because they protect different parts of the contract.

In [11]:
# ---------------------------------------------------------------------
# 4.5.2.2.2B — Compute legal eligibility for one candidate/sentinel
# ---------------------------------------------------------------------

def compute_transformer_eligibility(track, metric, horizon, lookback_steps):
    """Return development evaluation rows with the candidate's full legal history."""
    required_columns = [
        "modeling_track",
        "stage",
        "forecast_sequence_id",
        "taxi_zone_id",
        "metric",
        "horizon",
        "origin_observation_sequence_id",
        "target_observation_sequence_id",
        "target_date",
        "target_observed_value",
        "target_label_available",
        "native_track_eligible",
    ]

    matrix = pd.read_parquet(
        FORECASTING_MATRIX_PATHS[track],
        columns=required_columns,
        filters=[
            ("modeling_track", "==", track),
            ("metric", "==", metric),
            ("horizon", "==", int(horizon)),
            ("stage", "in", ["initial_training", "rolling_validation"]),
            ("native_track_eligible", "==", True),
        ],
    )

    matrix["target_date"] = pd.to_datetime(matrix["target_date"]).dt.normalize()

    for column in [
        "origin_observation_sequence_id",
        "target_observation_sequence_id",
    ]:
        matrix[column] = pd.to_numeric(matrix[column], errors="raise").astype("int64")

    labels_available = (
        matrix["target_label_available"].fillna(False).astype(bool)
        & matrix["target_observed_value"].notna()
    )

    # Track legality asks whether enough history exists AFTER this modeling
    # track officially begins. Longer-lookback candidates naturally become
    # eligible later without changing the track's underlying start boundary.
    first_legal_origin = (
        TRACK_HISTORY_START_SEQUENCE_ID[track] + int(lookback_steps) - 1
    )
    legal_track_history = matrix["origin_observation_sequence_id"].ge(
        first_legal_origin
    )

    # Physical-array legality is checked separately. A canonical ID may be
    # conceptually legal but still cannot be sliced unless the dense state
    # contains a complete lookback ending at that position.
    matrix["origin_position"] = matrix["origin_observation_sequence_id"].map(
        transformer_sequence_to_index
    )
    matrix["history_start_position"] = (
        matrix["origin_position"] - int(lookback_steps) + 1
    )

    eligible = matrix.loc[
        matrix["stage"].eq("rolling_validation")
        & matrix["target_date"].lt(FINAL_HOLDOUT_START_DATE)
        & labels_available
        & legal_track_history
        & matrix["origin_position"].notna()
        & matrix["history_start_position"].ge(0)
    ].copy()

    # Keep the holdout check visible in the returned audit rather than merely
    # assuming the date filter above behaved as intended.
    eligible["holdout_rows"] = eligible["target_date"].ge(
        FINAL_HOLDOUT_START_DATE
    ).astype(int)

    return eligible

**Audit Batch A variable-history eligibility.**
The eligibility helper is now applied to the **five frozen sentinels × four representation candidates**, giving the 20 combinations that Batch A may later compare.

This is still a cheap contract check rather than model fitting. The table shows how much legal development support each representation has and verifies that none of the candidate surfaces reaches into the final holdout.

In [12]:
# ---------------------------------------------------------------------
# 4.5.2.2.2C — Audit Batch A variable-history eligibility
# ---------------------------------------------------------------------

eligibility_rows = []

for track, metric, horizon in TRANSFORMER_SENTINEL_KEYS:
    for representation in TRANSFORMER_REPRESENTATION_CANDIDATES:
        lookback_steps = int(representation["lookback_steps"])

        eligible = compute_transformer_eligibility(
            track, metric, horizon, lookback_steps
        )

        eligibility_rows.append({
            "Track": TRACK_LABELS[track],
            "Modeling track": track,
            "Metric": metric,
            "Horizon": int(horizon),
            "Representation": representation["representation_id"],
            "Lookback": lookback_steps,
            "Eligible evaluation rows": int(len(eligible)),
            "First eligible forecast origin": (
                int(eligible["origin_observation_sequence_id"].min())
                if not eligible.empty else None
            ),
            "Last eligible forecast origin": (
                int(eligible["origin_observation_sequence_id"].max())
                if not eligible.empty else None
            ),
            "Holdout rows": int(eligible["holdout_rows"].sum()),
        })

transformer_history_eligibility_df = pd.DataFrame(eligibility_rows)

# Batch A has exactly five sentinel forecasting problems crossed with four
# historical representations. Catching a missing combination here prevents an
# apparently successful screen from ranking an incomplete candidate roster.
if len(transformer_history_eligibility_df) != 20:
    raise ValueError("Transformer history eligibility must contain exactly 20 rows.")

if transformer_history_eligibility_df["Holdout rows"].ne(0).any():
    raise ValueError("Transformer eligibility must exclude every final-holdout row.")

display(transformer_history_eligibility_df.round(3))

,Track,Modeling track,Metric,Horizon,Representation,Lookback,Eligible evaluation rows,First eligible forecast origin,Last eligible forecast origin,Holdout rows
0,Post-CP Primary,post_cp_primary,taxi_trip_count,1,one_week_daily,35,353022,4125,5499,0
1,Post-CP Primary,post_cp_primary,taxi_trip_count,1,two_week_daily,70,353022,4125,5499,0
2,Post-CP Primary,post_cp_primary,taxi_trip_count,1,four_week_daily,140,353022,4125,5499,0
3,Post-CP Primary,post_cp_primary,taxi_trip_count,1,two_week_overlapping_two_day,70,353022,4125,5499,0
4,Full-History Regime-Aware,full_history_regime_aware,taxi_avg_trip_speed,5,one_week_daily,35,168696,4121,5495,0
5,Full-History Regime-Aware,full_history_regime_aware,taxi_avg_trip_speed,5,two_week_daily,70,168696,4121,5495,0
6,Full-History Regime-Aware,full_history_regime_aware,taxi_avg_trip_speed,5,four_week_daily,140,168696,4121,5495,0
7,Full-History Regime-Aware,full_history_regime_aware,taxi_avg_trip_speed,5,two_week_overlapping_two_day,70,168696,4121,5495,0
8,Post-CP Primary,post_cp_primary,fhvhv_trip_count,2,one_week_daily,35,353022,4124,5498,0
9,Post-CP Primary,post_cp_primary,fhvhv_trip_count,2,two_week_daily,70,353022,4124,5498,0


**Findings.** All four proposed history representations are legally usable for every sentinel forecasting problem, including the longest four-week lookback.

Within each sentinel, the one-week, two-week, four-week, and overlapping two-week candidates retain exactly the same number of evaluation rows. This is important: later accuracy differences cannot be explained by one candidate receiving an easier or larger future scoring population.

Longer lookbacks therefore change how much historical information the Transformer receives, not which future rows it is allowed to predict. No final-holdout rows enter any candidate surface.

## 4.5.2.2.3 Define the Patch-Based Encoder-Only Transformer

The Transformer does not look up mobility values in a vocabulary. Each patch is a short block of processed numeric sequence values. A patch contains all ten mobility and observation-mask channels, is projected into a learned token, receives learned positional information, and is processed by one compact encoder-only Transformer family.

Mean pooling joins the encoded history with finalized forecast-time context and Taxi Zone identity before a scalar forecast head. The design is PatchTST-inspired, not canonical PatchTST: the temporal patch is multivariate rather than channel-independent.

**Transformer Class.**
Before PyTorch sees any values, each logical fit needs its own **training-only numerical preparation state**.

The five mobility-history channels are standardized using only observations that were actually present in the legal training sample; their five masks remain explicit so an imputed numerical zero is never confused with an observed zero. The 52-value context follows the finalized 4.5.1 contract rather than one blanket scaling rule: 24 ordinary continuous values use training-only completion and scaling, the weekly-STL offset uses its separate state-aware rule, and 27 deterministic/binary/missingness values pass through unchanged.

The target is also standardized from training labels only. Validation and evaluation rows may be transformed by this fitted state, but they never contribute to it.

In [13]:
# ---------------------------------------------------------------------
# 4.5.2.2.3A — Transformer preprocessor
# ---------------------------------------------------------------------

WEEKLY_STL_OFFSET_FEATURE = "weekly_stl_seasonal_offset"
WEEKLY_STL_ENABLED_FEATURE = "weekly_stl_enabled"
WEEKLY_STL_MISSING_MASK_FEATURE = "missing_mask__weekly_stl_seasonal_offset"
WEEKLY_STL_SCALE_EPS = 1e-8


class TransformerPreprocessor:
    """
    Learn leakage-safe numerical transforms for one Transformer fit.

    The finalized neural input contract is preserved:
      • 5 mobility histories are scaled from observed training cells only;
      • 5 matching observation masks remain explicit model inputs;
      • 24 ordinary continuous context values use training-only completion/scaling;
      • the weekly-STL offset uses its own state-aware scaling rule;
      • 27 deterministic/binary/missingness context values pass through unchanged;
      • the target is standardized from legal training labels only.

    Validation and evaluation rows never influence fitted preprocessing state.
    """

    def __init__(self, context_contract):
        contract = context_contract.sort_values("input_index").reset_index(drop=True).copy()

        required = {"feature", "role", "input_index", "preprocessing"}
        missing = sorted(required - set(contract.columns))
        if missing:
            raise ValueError(f"Transformer context contract is missing required fields: {missing}")

        self.context_contract = contract
        self.feature_order = contract["feature"].astype(str).tolist()
        self.feature_to_index = {feature: i for i, feature in enumerate(self.feature_order)}

        role = contract["role"].astype(str).str.lower()
        self.continuous_features = contract.loc[role.eq("ordinary_continuous"), "feature"].astype(str).tolist()
        self.stl_features = contract.loc[role.eq("weekly_stl_offset"), "feature"].astype(str).tolist()
        self.passthrough_features = contract.loc[role.eq("passthrough"), "feature"].astype(str).tolist()

        role_counts = (
            len(self.continuous_features),
            len(self.stl_features),
            len(self.passthrough_features),
        )
        if role_counts != (24, 1, 27):
            raise ValueError(
                "Transformer context must contain exactly 24 ordinary continuous values, "
                "1 weekly-STL offset, and 27 passthrough values."
            )

        if self.stl_features != [WEEKLY_STL_OFFSET_FEATURE]:
            raise ValueError(
                "weekly_stl_seasonal_offset must be the only state-aware continuous context field."
            )

        required_stl = {
            WEEKLY_STL_OFFSET_FEATURE,
            WEEKLY_STL_ENABLED_FEATURE,
            WEEKLY_STL_MISSING_MASK_FEATURE,
        }
        if not required_stl.issubset(self.feature_order):
            raise ValueError("The weekly-STL semantic bundle is incomplete.")

    # -----------------------------------------------------------------
    # Shared validation helpers
    # -----------------------------------------------------------------

    def _ordered_context(self, context_frame):
        """Validate the 52-value context contract and restore canonical feature order."""
        missing = sorted(set(self.feature_order) - set(context_frame.columns))
        if missing:
            raise ValueError(f"Transformer context is missing required fields: {missing}")

        return context_frame[self.feature_order].copy()

    @staticmethod
    def _require_binary(values, name):
        """Protect semantic indicators from silently drifting away from 0/1."""
        if not np.isin(values, [0.0, 1.0]).all():
            raise ValueError(f"{name} must remain binary.")

    def _require_fitted(self):
        """Prevent transformation before legal training state has been learned."""
        if not getattr(self, "is_fitted_", False):
            raise RuntimeError("TransformerPreprocessor must be fitted first.")

    # -----------------------------------------------------------------
    # Fit training-only preprocessing state
    # -----------------------------------------------------------------

    def fit(self, sequence_values, sequence_masks, context_frame, target_values):
        """
        Learn all numerical state from legal training rows only.

        Mobility statistics use observed cells only. Ordinary context is
        median-completed before scaling. Weekly STL excludes disabled/missing
        semantic zeros from its fitted seasonal magnitude distribution.
        """
        sequence_values = np.asarray(sequence_values, dtype=np.float64)
        sequence_masks = np.asarray(sequence_masks, dtype=bool)

        if sequence_values.shape != sequence_masks.shape:
            raise ValueError("Sequence values and observation masks must have identical shapes.")

        if sequence_values.ndim != 3 or sequence_values.shape[-1] != 5:
            raise ValueError("Raw Transformer history must have shape [rows, lookback, 5].")

        # ----- Mobility-history scaling --------------------------------
        # Each target-specific mobility channel learns its own scale from
        # genuinely observed training cells; missing values teach nothing.
        channel_count = sequence_values.shape[-1]
        self.sequence_means_ = np.zeros(channel_count, dtype=np.float64)
        self.sequence_stds_ = np.ones(channel_count, dtype=np.float64)

        for channel_index in range(channel_count):
            observed = sequence_values[..., channel_index][sequence_masks[..., channel_index]]
            observed = observed[np.isfinite(observed)]

            if observed.size == 0:
                raise ValueError(
                    f"Historical mobility channel {channel_index} has no observed training values."
                )

            channel_mean = float(observed.mean())
            channel_std = float(observed.std(ddof=0))

            self.sequence_means_[channel_index] = channel_mean
            self.sequence_stds_[channel_index] = channel_std if channel_std > 1e-8 else 1.0

        # ----- Ordinary continuous context -----------------------------
        # Missing continuous context is completed and scaled independently
        # using only this fit's legal training sample.
        context_frame = self._ordered_context(context_frame)
        continuous_values = (
            context_frame[self.continuous_features]
            .apply(pd.to_numeric, errors="coerce")
            .to_numpy(dtype=np.float64)
        )
        continuous_values[~np.isfinite(continuous_values)] = np.nan

        continuous_width = len(self.continuous_features)
        self.context_medians_ = np.zeros(continuous_width, dtype=np.float64)
        self.context_means_ = np.zeros(continuous_width, dtype=np.float64)
        self.context_stds_ = np.ones(continuous_width, dtype=np.float64)

        for feature_index, feature_name in enumerate(self.continuous_features):
            values = continuous_values[:, feature_index]
            finite = values[np.isfinite(values)]

            if finite.size == 0:
                raise ValueError(
                    f"Continuous context has no legal training value for {feature_name}."
                )

            median = float(np.median(finite))
            completed = np.where(np.isfinite(values), values, median)
            mean = float(completed.mean())
            std = float(completed.std(ddof=0))

            self.context_medians_[feature_index] = median
            self.context_means_[feature_index] = mean
            self.context_stds_[feature_index] = std if std > 1e-8 else 1.0

        # ----- State-aware weekly-STL scaling --------------------------
        # Disabled STL rows intentionally carry a zero offset. Those semantic
        # zeros must not shrink the scale learned from genuine fitted offsets.
        stl_values = pd.to_numeric(
            context_frame[WEEKLY_STL_OFFSET_FEATURE], errors="coerce"
        ).to_numpy(dtype=np.float64)
        stl_enabled = pd.to_numeric(
            context_frame[WEEKLY_STL_ENABLED_FEATURE], errors="raise"
        ).to_numpy(dtype=np.float64)
        stl_missing = pd.to_numeric(
            context_frame[WEEKLY_STL_MISSING_MASK_FEATURE], errors="raise"
        ).to_numpy(dtype=np.float64)

        self._require_binary(stl_enabled, WEEKLY_STL_ENABLED_FEATURE)
        self._require_binary(stl_missing, WEEKLY_STL_MISSING_MASK_FEATURE)

        stl_available = (
            (stl_enabled == 1.0)
            & (stl_missing == 0.0)
            & np.isfinite(stl_values)
        )
        stl_training_values = stl_values[stl_available]

        self.stl_training_support_ = int(stl_training_values.size)
        self.stl_mean_ = (
            float(stl_training_values.mean())
            if self.stl_training_support_ > 0 else 0.0
        )
        self.stl_std_ = (
            float(stl_training_values.std(ddof=0))
            if self.stl_training_support_ >= 2 else 0.0
        )
        self.weekly_stl_scaler_active_ = bool(
            self.stl_training_support_ >= 2
            and np.isfinite(self.stl_std_)
            and self.stl_std_ > WEEKLY_STL_SCALE_EPS
        )

        # ----- Passthrough context ------------------------------------
        # Calendar flags, regime state, cyclic encodings, and missingness
        # indicators are already model-ready and must remain numerically intact.
        passthrough = context_frame[self.passthrough_features].to_numpy(dtype=np.float64)
        if not np.isfinite(passthrough).all():
            raise ValueError(
                "Deterministic, binary, and missingness context must already be finite."
            )

        # ----- Target scaling -----------------------------------------
        # The forecast target gets its own training-only standardization so
        # Huber optimization operates on a stable numerical scale.
        target = np.asarray(target_values, dtype=np.float64)
        target = target[np.isfinite(target)]

        if target.size == 0:
            raise ValueError("Training target contains no finite observations.")

        self.target_mean_ = float(target.mean())
        self.target_std_ = float(target.std(ddof=0))

        if not np.isfinite(self.target_std_) or self.target_std_ <= 1e-8:
            raise ValueError("Training target has zero or non-finite variance.")

        self.is_fitted_ = True
        return self

    # -----------------------------------------------------------------
    # Apply fitted state
    # -----------------------------------------------------------------

    def transform_sequence(self, sequence_values, sequence_masks):
        """
        Standardize the five mobility histories and append their five masks.

        Missing histories become numerical zero only after standardization, so
        zero means "training mean" while the appended mask tells the model
        whether that value was actually observed.
        """
        self._require_fitted()

        sequence_values = np.asarray(sequence_values, dtype=np.float32)
        sequence_masks = np.asarray(sequence_masks, dtype=bool)

        if sequence_values.shape != sequence_masks.shape:
            raise ValueError("Sequence values and masks must have identical shapes.")

        if sequence_values.shape[-1] != 5:
            raise ValueError("Transformer history must contain five raw mobility channels.")

        standardized = (
            sequence_values - self.sequence_means_.astype("float32")
        ) / self.sequence_stds_.astype("float32")

        standardized = np.where(
            sequence_masks & np.isfinite(standardized),
            standardized,
            0.0,
        ).astype("float32")

        return np.concatenate(
            [standardized, sequence_masks.astype("float32")],
            axis=-1,
        )

    def transform_context(self, context_frame):
        """
        Apply frozen training state to the 52-value context vector.

        Ordinary continuous inputs use learned completion/scaling. Weekly STL
        is active only when fitted training history supported a meaningful
        seasonal scale. Passthrough features—including the CP regime flag—
        retain their original model-facing values.
        """
        self._require_fitted()
        context_frame = self._ordered_context(context_frame)

        output = np.zeros(
            (len(context_frame), len(self.feature_order)),
            dtype=np.float32,
        )

        # ----- Ordinary continuous context -----------------------------
        for feature_index, feature_name in enumerate(self.continuous_features):
            values = pd.to_numeric(
                context_frame[feature_name], errors="coerce"
            ).to_numpy(dtype=np.float32)

            median = np.float32(self.context_medians_[feature_index])
            mean = np.float32(self.context_means_[feature_index])
            std = np.float32(self.context_stds_[feature_index])

            values = np.where(np.isfinite(values), values, median)
            output[:, self.feature_to_index[feature_name]] = (values - mean) / std

        # ----- Weekly STL ----------------------------------------------
        stl_values = pd.to_numeric(
            context_frame[WEEKLY_STL_OFFSET_FEATURE], errors="coerce"
        ).to_numpy(dtype=np.float32)
        stl_enabled = pd.to_numeric(
            context_frame[WEEKLY_STL_ENABLED_FEATURE], errors="raise"
        ).to_numpy(dtype=np.float32)
        stl_missing = pd.to_numeric(
            context_frame[WEEKLY_STL_MISSING_MASK_FEATURE], errors="raise"
        ).to_numpy(dtype=np.float32)

        self._require_binary(stl_enabled, WEEKLY_STL_ENABLED_FEATURE)
        self._require_binary(stl_missing, WEEKLY_STL_MISSING_MASK_FEATURE)

        stl_available = (
            (stl_enabled == 1.0)
            & (stl_missing == 0.0)
            & np.isfinite(stl_values)
        )

        # Persisted support and scale are enough to determine whether the
        # numerical STL channel carries usable standardized information.
        stl_scaler_active = bool(
            getattr(
                self,
                "weekly_stl_scaler_active_",
                self.stl_training_support_ >= 2
                and np.isfinite(self.stl_std_)
                and self.stl_std_ > WEEKLY_STL_SCALE_EPS,
            )
        )

        stl_index = self.feature_to_index[WEEKLY_STL_OFFSET_FEATURE]
        if stl_scaler_active:
            output[stl_available, stl_index] = (
                stl_values[stl_available] - np.float32(self.stl_mean_)
            ) / np.float32(self.stl_std_)

        # If scaling is inactive, the numerical STL channel remains zero.
        # The enabled/missingness indicators still preserve its semantic state.

        # ----- Deterministic / binary / missingness passthrough --------
        passthrough = context_frame[self.passthrough_features].to_numpy(dtype=np.float32)
        if not np.isfinite(passthrough).all():
            raise ValueError("Transformer passthrough context contains non-finite values.")

        passthrough_indices = [
            self.feature_to_index[feature]
            for feature in self.passthrough_features
        ]
        output[:, passthrough_indices] = passthrough

        if not np.isfinite(output).all():
            raise ValueError("Model-facing Transformer context contains non-finite values.")

        return output

    def transform_target(self, target_values):
        """Standardize targets using this fit's legal training-label state."""
        self._require_fitted()
        target = np.asarray(target_values, dtype=np.float32)

        return ((target - self.target_mean_) / self.target_std_).astype("float32")

    def inverse_target(self, standardized_values):
        """Return standardized predictions to the target's native unit."""
        self._require_fitted()
        values = np.asarray(standardized_values, dtype=np.float32)

        return (values * self.target_std_ + self.target_mean_).astype("float32")

    # -----------------------------------------------------------------
    # Durable fitted state
    # -----------------------------------------------------------------

    def export_state(self):
        """
        Persist the exact preprocessing state needed for later inference.

        Saving these statistics beside the Transformer weights lets evaluation
        and diagnostics replay the fitted model without relearning anything
        from future observations.
        """
        self._require_fitted()

        return {
            "sequence_means": self.sequence_means_.copy(),
            "sequence_stds": self.sequence_stds_.copy(),
            "context_medians": self.context_medians_.copy(),
            "context_means": self.context_means_.copy(),
            "context_stds": self.context_stds_.copy(),
            "stl_training_support": int(self.stl_training_support_),
            "stl_mean": float(self.stl_mean_),
            "stl_std": float(self.stl_std_),
            "weekly_stl_scaler_active": bool(self.weekly_stl_scaler_active_),
            "target_mean": float(self.target_mean_),
            "target_std": float(self.target_std_),
        }

**Transformer Dataset Class.**
After preprocessing, `TransformerDataset` performs the narrow handoff from NumPy arrays to PyTorch.

Each example contains three model inputs—the prepared 10-channel historical sequence, the 52-value supporting context, and Taxi Zone identity—plus the standardized target during supervised training. The adapter also checks row and shape alignment before batching so malformed inputs fail here rather than deep inside a training loop.

In [14]:
# ---------------------------------------------------------------------
# 4.5.2.2.3B — Transformer dataset
# ---------------------------------------------------------------------

class TransformerDataset(Dataset):
    """
    Expose already-prepared Transformer examples to PyTorch.

    Each row contains:
      • one [lookback, 10] mobility-history tensor;
      • one 52-value supporting-context vector;
      • one Taxi Zone index;
      • optionally, one standardized forecast target.

    Preprocessing happens before this class. Its responsibility is only to
    preserve row alignment and provide explicit PyTorch-ready dtypes.
    """

    def __init__(self, sequence, context, zone_index, target=None):
        self.sequence = np.asarray(sequence, dtype=np.float32)
        self.context = np.asarray(context, dtype=np.float32)
        self.zone_index = np.asarray(zone_index, dtype=np.int64)
        self.target = None if target is None else np.asarray(target, dtype=np.float32)

        row_count = len(self.sequence)

        # Every model input must describe the same forecasting example. Catching
        # row misalignment here prevents silent mixing of one zone/time row with
        # another row's history or context.
        if len(self.context) != row_count or len(self.zone_index) != row_count:
            raise ValueError(
                "Sequence, context, and Taxi Zone arrays must align by row."
            )

        if self.target is not None and len(self.target) != row_count:
            raise ValueError("Target rows must align with model inputs.")

        # The preprocessor should already have transformed 5 mobility values +
        # 5 observation masks into the finalized 10-channel history tensor.
        if self.sequence.ndim != 3 or self.sequence.shape[-1] != 10:
            raise ValueError(
                "Prepared sequence must have shape [rows, lookback, 10]."
            )

        if (
            self.context.ndim != 2
            or self.context.shape[1] != transformer_context_feature_count
        ):
            raise ValueError(
                "Prepared context width does not match the finalized "
                "52-value contract."
            )

    def __len__(self):
        """Return the number of supervised or inference examples."""
        return len(self.sequence)

    def __getitem__(self, index):
        """Return one model-ready example using the required PyTorch dtypes."""
        model_inputs = (
            torch.as_tensor(self.sequence[index], dtype=torch.float32),
            torch.as_tensor(self.context[index], dtype=torch.float32),
            torch.as_tensor(self.zone_index[index], dtype=torch.long),
        )

        if self.target is None:
            return model_inputs

        return (
            *model_inputs,
            torch.as_tensor(self.target[index], dtype=torch.float32),
        )

**Patch Transformer Regressor Class.**
The forecasting model is a **patch-based, encoder-only Transformer**. Each temporal patch contains all ten historical channels together: five mobility values plus their five observation masks. The patch is flattened and projected into the Transformer's hidden dimension, learned positional information is added, and self-attention lets the historical patches exchange information.

After the encoder, the historical tokens are mean-pooled into one representation. Only then are the 52 supporting context values and the learned Taxi Zone identity fused into the scalar forecast head.

This is therefore *PatchTST-inspired*, not canonical PatchTST: modalities are deliberately combined inside each temporal token rather than being encoded as separate independent variable streams.

In [15]:
# ---------------------------------------------------------------------
# 4.5.2.2.3C — Patch-based Transformer regressor
# ---------------------------------------------------------------------

class PatchTransformerRegressor(nn.Module):
    """
    Forecast one standardized target from patched mobility history and context.

    The architecture follows five conceptual steps:

      1. Split the legal history into temporal patches.
      2. Flatten each patch across all 10 sequence channels.
      3. Project patches into d_model-sized tokens and add learned position.
      4. Use an encoder-only Transformer to contextualize those history tokens.
      5. Mean-pool history, append context + Taxi Zone identity, and forecast.

    This is PatchTST-inspired rather than canonical PatchTST: each patch contains
    all five mobility values and all five observation masks together instead of
    modeling each variable as an independent token stream.
    """

    def __init__(
        self,
        sequence_input_dim,
        context_dim,
        n_zones,
        representation,
        profile,
    ):
        super().__init__()

        if sequence_input_dim != 10:
            raise ValueError("The Transformer expects exactly 10 sequence channels.")
        if context_dim != 52:
            raise ValueError("The Transformer expects exactly 52 context values.")
        if n_zones <= 0:
            raise ValueError("Taxi Zone vocabulary must be non-empty.")

        self.sequence_input_dim = int(sequence_input_dim)
        self.context_dim = int(context_dim)

        self.lookback_steps = int(representation["lookback_steps"])
        self.patch_length = int(representation["patch_length"])
        self.patch_stride = int(representation["patch_stride"])

        d_model = int(profile["d_model"])
        n_heads = int(profile["n_heads"])
        n_layers = int(profile["n_layers"])
        d_ff = int(profile["d_ff"])
        dropout = float(profile["dropout"])
        zone_embedding_dim = int(profile["zone_embedding_dim"])

        # Candidate representations must tile the requested history exactly.
        # We deliberately avoid padding/truncation so every token corresponds to
        # a real, interpretable span of legally available history.
        if self.lookback_steps < self.patch_length:
            raise ValueError("Patch length cannot exceed lookback length.")

        if (self.lookback_steps - self.patch_length) % self.patch_stride != 0:
            raise ValueError(
                "Patch geometry must cover the lookback exactly without "
                "padding or truncation."
            )

        if d_model % n_heads != 0:
            raise ValueError("d_model must divide evenly by n_heads.")

        self.n_patches = (
            1 + (self.lookback_steps - self.patch_length) // self.patch_stride
        )
        patch_width = self.patch_length * self.sequence_input_dim

        # Every flattened patch is a continuous numerical vector, so a Linear
        # projection—not an embedding lookup—maps it into Transformer space.
        self.patch_projection = nn.Linear(patch_width, d_model)
        self.position = nn.Parameter(torch.zeros(1, self.n_patches, d_model))

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=d_ff,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.history_norm = nn.LayerNorm(d_model)

        # Taxi Zone identity enters after historical encoding. This lets one
        # global model share temporal patterns across zones while retaining a
        # learned location-specific signature.
        self.zone_embedding = nn.Embedding(n_zones, zone_embedding_dim)

        fused_dim = d_model + self.context_dim + zone_embedding_dim
        self.head = nn.Sequential(
            nn.LayerNorm(fused_dim),
            nn.Linear(fused_dim, d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model, 1),
        )

    def forward(self, sequence, context, zone_index):
        """
        Encode legal historical patches and return one standardized forecast.

        Shape flow:
            [B, lookback, 10]
            → [B, patches, patch_length, 10]
            → [B, patches, patch_length × 10]
            → [B, patches, d_model]
            → [B, d_model]
            → [B]

        No causal attention mask is needed because every token comes entirely
        from history at or before the forecast origin; no token contains future
        mobility information.
        """
        expected_sequence_shape = (self.lookback_steps, self.sequence_input_dim)

        if sequence.ndim != 3 or sequence.shape[1:] != expected_sequence_shape:
            raise ValueError(
                "Sequence shape does not match the selected Transformer representation."
            )

        if context.ndim != 2 or context.shape[1] != self.context_dim:
            raise ValueError(
                "Context shape does not match the finalized contract."
            )

        # unfold() returns [B, patches, channels, patch_length]. Reorder once to
        # the conceptual [B, patches, patch_length, channels] form, then flatten.
        patches = sequence.unfold(1, self.patch_length, self.patch_stride)
        patches = patches.permute(0, 1, 3, 2).contiguous()
        patches = patches.reshape(sequence.shape[0], self.n_patches, -1)

        tokens = self.patch_projection(patches) + self.position
        encoded = self.encoder(tokens)

        # Mean pooling is deliberately fixed. The tournament compares history
        # representation and model capacity, not competing pooling strategies.
        pooled_history = self.history_norm(encoded.mean(dim=1))
        zone_signature = self.zone_embedding(zone_index)

        fused = torch.cat([pooled_history, context, zone_signature], dim=1)
        return self.head(fused).squeeze(-1)

**Cheap model-contract smoke test.**
Before any real Transformer training, every approved **representation × profile** combination is instantiated once with two synthetic examples.

This smoke test asks only whether the frozen architecture is internally coherent: can each representation form the expected number of patches, can each profile accept the resulting tensors, and does the network return one finite scalar prediction per example? It does **not** train a model, score real targets, or touch the final holdout.

In [16]:
# ---------------------------------------------------------------------
# 4.5.2.2.3D — Cheap Transformer model-contract smoke test
# ---------------------------------------------------------------------
# Instantiate every approved representation × profile combination once with
# two synthetic rows. This checks architecture compatibility, patch geometry,
# token counts, and output shape only. It performs no training and touches no
# development or holdout targets.

transformer_shape_smoke_rows = []

for representation in TRANSFORMER_REPRESENTATION_CANDIDATES:
    for profile in TRANSFORMER_PROFILE_CANDIDATES:
        model = PatchTransformerRegressor(
            NEURAL_SEQUENCE_INPUT_DIM,
            transformer_context_feature_count,
            len(transformer_zone_ids),
            representation,
            profile,
        )

        dummy_sequence = torch.zeros(
            2,
            representation["lookback_steps"],
            NEURAL_SEQUENCE_INPUT_DIM,
        )
        dummy_context = torch.zeros(2, transformer_context_feature_count)
        dummy_zones = torch.zeros(2, dtype=torch.long)

        with torch.no_grad():
            output = model(dummy_sequence, dummy_context, dummy_zones)

        expected_tokens = (
            1
            + (representation["lookback_steps"] - representation["patch_length"])
            // representation["patch_stride"]
        )

        # A valid candidate must map two complete model inputs to exactly two
        # finite scalar forecasts and expose the token count implied by its
        # declared patch geometry.
        candidate_label = (
            f"{representation['representation_id']} × {profile['profile_id']}"
        )

        if output.shape != (2,):
            raise ValueError(
                f"Transformer smoke-test output has the wrong shape for {candidate_label}."
            )

        if not torch.isfinite(output).all():
            raise ValueError(
                f"Transformer smoke-test output is non-finite for {candidate_label}."
            )

        if model.n_patches != expected_tokens:
            raise ValueError(
                f"Transformer token count is incorrect for {candidate_label}."
            )

        transformer_shape_smoke_rows.append({
            "Representation": representation["representation_id"],
            "Profile": profile["profile_id"],
            "Tokens": model.n_patches,
            "Output shape": str(tuple(output.shape)),
        })

        # These objects are tiny, but releasing them each iteration keeps the
        # smoke test representative of the later create-fit-release lifecycle.
        del model, dummy_sequence, dummy_context, dummy_zones, output

transformer_shape_smoke_df = pd.DataFrame(transformer_shape_smoke_rows)

if len(transformer_shape_smoke_df) != 12:
    raise ValueError("Expected exactly 12 Transformer shape smoke tests.")

display(transformer_shape_smoke_df.round(3))

/root/venv/lib/python3.11/site-packages/torch/nn/modules/transformer.py:282: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(f"enable_nested_tensor is True, but self.use_nested_tensor is False because {why_not_sparsity_fast_path}")


,Representation,Profile,Tokens,Output shape
0,one_week_daily,compact,7,"(2,)"
1,one_week_daily,balanced,7,"(2,)"
2,one_week_daily,higher_capacity_regularized,7,"(2,)"
3,two_week_daily,compact,14,"(2,)"
4,two_week_daily,balanced,14,"(2,)"
5,two_week_daily,higher_capacity_regularized,14,"(2,)"
6,four_week_daily,compact,28,"(2,)"
7,four_week_daily,balanced,28,"(2,)"
8,four_week_daily,higher_capacity_regularized,28,"(2,)"
9,two_week_overlapping_two_day,compact,13,"(2,)"


**Findings.** All **12 representation × Transformer-profile combinations** passed the model-contract smoke test and produced one finite scalar forecast per example.

The patching schemes create meaningfully different sequence lengths for self-attention. One week of daily patches produces 7 tokens, two weeks produces 14, and four weeks produces 28. The overlapping two-week representation produces 13 tokens because each two-day patch advances by one day.

These token counts change the Transformer representation while preserving the same underlying 10 sequence channels, 52 context values, and Taxi Zone identity.

## 4.5.2.2.4 Preflight Runtime, Memory, and Planned Workload

The model-design questions are bundled into three restartable runs, but each run contains multiple logical fits. Before training, make that workload explicit and verify only a representative forward/backward pass. This is a feasibility check, not an accuracy experiment.

In [17]:
# ---------------------------------------------------------------------
# 4.5.2.2.4A — Preflight Transformer memory and planned workload
# ---------------------------------------------------------------------

largest_representation = max(
    TRANSFORMER_REPRESENTATION_CANDIDATES,
    key=lambda spec: spec["lookback_steps"],
)

largest_profile = next(
    spec
    for spec in TRANSFORMER_PROFILE_CANDIDATES
    if spec["profile_id"] == "higher_capacity_regularized"
)


def run_transformer_preflight(candidate_batch_size):
    """
    Test one forward/backward pass for the largest planned Transformer.

    On CUDA, the candidate batch size itself is tested because GPU memory is
    the limiting resource we need to measure. On CPU, a small smoke batch is
    enough to validate graph construction without turning a notebook preflight
    into a long-running training job.
    """
    using_cuda = TORCH_DEVICE.type == "cuda"

    probe_batch_size = (
        candidate_batch_size
        if using_cuda
        else min(candidate_batch_size, 32)
    )

    reset_cuda_peak_memory()

    model = PatchTransformerRegressor(
        NEURAL_SEQUENCE_INPUT_DIM,
        transformer_context_feature_count,
        len(transformer_zone_ids),
        largest_representation,
        largest_profile,
    ).to(TORCH_DEVICE)

    sequence = torch.zeros(
        probe_batch_size,
        largest_representation["lookback_steps"],
        NEURAL_SEQUENCE_INPUT_DIM,
        dtype=TORCH_DTYPE,
        device=TORCH_DEVICE,
    )

    context = torch.zeros(
        probe_batch_size,
        transformer_context_feature_count,
        dtype=TORCH_DTYPE,
        device=TORCH_DEVICE,
    )

    zones = torch.zeros(
        probe_batch_size,
        dtype=torch.long,
        device=TORCH_DEVICE,
    )

    target = torch.zeros(
        probe_batch_size,
        dtype=TORCH_DTYPE,
        device=TORCH_DEVICE,
    )

    # Complete one miniature optimization step, not merely graph construction.
    # This confirms that the same optimizer, loss, and gradient-clipping path
    # used by training can execute before any expensive screen is launched.
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=largest_profile["learning_rate"],
        weight_decay=largest_profile["weight_decay"],
    )
    optimizer.zero_grad(set_to_none=True)

    prediction = model(sequence, context, zones)

    if prediction.shape != (probe_batch_size,):
        raise ValueError(
            "Transformer preflight returned an unexpected prediction shape: "
            f"{tuple(prediction.shape)}"
        )

    loss = nn.HuberLoss()(prediction, target)

    if not torch.isfinite(loss):
        raise ValueError("Transformer preflight produced a non-finite loss.")

    loss.backward()
    torch.nn.utils.clip_grad_norm_(
        model.parameters(),
        TRANSFORMER_GRAD_CLIP_NORM,
    )
    optimizer.step()

    peak_memory_mb = read_cuda_peak_memory_mb()

    del optimizer, model, sequence, context, zones, target, prediction, loss
    clear_device_memory()

    return {
        "Requested batch size": candidate_batch_size,
        "Probe batch size": probe_batch_size,
        "Succeeded": True,
        "Peak GPU memory MB": peak_memory_mb,
    }


preflight_rows = []
selected_transformer_batch_size = None

if TORCH_DEVICE.type == "cuda":
    # GPU execution tests the actual approved candidates from largest to
    # smallest and retains the largest one that successfully backpropagates.
    for candidate_batch_size in TRANSFORMER_BATCH_SIZE_CANDIDATES:
        try:
            result = run_transformer_preflight(candidate_batch_size)
            preflight_rows.append(result)
            selected_transformer_batch_size = candidate_batch_size
            break

        except RuntimeError as exc:
            if "out of memory" not in str(exc).lower():
                raise

            preflight_rows.append({
                "Requested batch size": candidate_batch_size,
                "Probe batch size": candidate_batch_size,
                "Succeeded": False,
                "Peak GPU memory MB": read_cuda_peak_memory_mb(),
            })

            clear_device_memory()

else:
    # CPU mode is only a functional smoke test. Testing batches of 1024 on a
    # large Transformer can take minutes while telling us nothing useful about
    # the GPU workload that will eventually run the expensive screen.
    cpu_training_batch_size = min(TRANSFORMER_BATCH_SIZE_CANDIDATES)

    preflight_rows.append(
        run_transformer_preflight(cpu_training_batch_size)
    )

    selected_transformer_batch_size = cpu_training_batch_size


if selected_transformer_batch_size is None:
    raise RuntimeError(
        "No approved Transformer batch size passed the preflight."
    )


transformer_preflight_df = pd.DataFrame(preflight_rows)

transformer_workload_df = pd.DataFrame([
    {
        "Batch": "A",
        "Logical fits": 20,
        "Purpose": "Representation screen",
    },
    {
        "Batch": "B",
        "Logical fits": 20,
        "Purpose": "Complete Transformer-profile screen",
    },
    {
        "Batch": "C",
        "Logical fits": 30,
        "Purpose": "Early/middle/late confirmation",
    },
    {
        "Batch": "Total",
        "Logical fits": 70,
        "Purpose": "Three restartable launches",
    },
])

print(
    "Transformer preflight complete | "
    f"device={TORCH_DEVICE.type.upper()} | "
    f"training batch size={selected_transformer_batch_size}"
)

display(transformer_preflight_df.round(3))
display(transformer_workload_df.round(3))

/root/venv/lib/python3.11/site-packages/torch/nn/modules/transformer.py:282: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(f"enable_nested_tensor is True, but self.use_nested_tensor is False because {why_not_sparsity_fast_path}")
Transformer preflight complete | device=CPU | training batch size=256


,Requested batch size,Probe batch size,Succeeded,Peak GPU memory MB
0,256,32,True,NaN


,Batch,Logical fits,Purpose
0,A,20,Representation screen
1,B,20,Complete Transformer-profile screen
2,C,30,Early/middle/late confirmation
3,Total,70,Three restartable launches


**Findings.** The CPU functional preflight successfully completed the required forward pass, backward pass, gradient clipping, and AdamW update. The real model-selection runs use a batch size of 256; the preflight itself used only 32 rows so feasibility testing did not become an expensive benchmark.

The complete Transformer tournament contains **70 restartable logical fits**: 20 representation-screen fits in Batch A, 20 complete-specification fits in Batch B, and 30 early/middle/late confirmation fits in Batch C.

These are three dependent stages in one automatic restartable pipeline—not 70 manually launched experiments. Completed fit-level artifacts are recovered on rerun, while only missing or invalid work is recomputed.

### Section Summary

Section 4.5.2.2 converts the finalized neural information contract into a Transformer-ready representation without changing the forecasting problem.

The same five mobility histories, five observation masks, forecast-time context, weekly STL state, Taxi Zone identity, forecast timing, benchmark, and holdout boundary remain intact. The Transformer can reconstruct one-, two-, and four-week histories directly from the shared dense mobility state while keeping every canonical Daypart in order.

Patching is a representational transformation: consecutive processed Dayparts across all ten sequence channels are projected into Transformer tokens and passed through one encoder-only family.

The next section uses three bounded training batches to narrow the representation and profile choices. No cadence experiment or full rolling validation begins until the resulting evidence is reviewed.

# 4.5.2.3 Screen Compact Transformer Designs

The Transformer now has a valid forecasting interface, but three design questions are still unresolved:

1. **How much recent mobility history should it see, and how should that history be divided into patches?**
2. **How large does the Transformer need to be?**
3. **Do the strongest choices remain reliable at different points in the historical validation period?**

Testing every representation, model size, target, and historical window at once would create a large and difficult-to-interpret search. This chapter instead uses a **three-stage tournament**, with each batch answering one question before the next spends more computation.

**Batch A chooses the history representation.** Four ways of presenting recent mobility history are compared while the Transformer architecture is held fixed at the balanced profile. The five sentinel forecasting problems each test all four representations in the same middle historical window, producing **20 logical fits**. The two strongest representations advance.

**Batch B chooses the complete Transformer specification.** The strongest representation is paired with compact, balanced, and higher-capacity Transformer profiles, while the runner-up representation receives one balanced-profile chance to remain competitive. The same five sentinels compare these four complete specifications, producing another **20 logical fits**. Two finalists advance.

**Batch C checks whether those finalists are robust through time.** Rather than trusting one middle validation window, both finalists are tested on early, middle, and late historical windows for every sentinel. This produces **30 confirmation fits**. The pooled prediction error across those windows is then used to propose the stronger finalist **separately for each mobility target**.

The full pre-decision screen therefore contains **70 restartable logical fits: 20 in Batch A, 20 in Batch B, and 30 in Batch C**. Each stage deliberately narrows the question before expanding the evidence.

This chapter is still **model selection, not the final forecasting evaluation**. Its purpose is to decide which Transformer specification should be carried forward for Taxi trips, Taxi speed, FHVHV trips, FHVHV speed, and Subway ridership. Later forecasting sections can then evaluate those choices without continuing to tune the Transformer against the same validation evidence.

Throughout all three batches, candidate comparisons use the same physical future rows within each forecasting problem, the final holdout beginning January 5, 2026 remains untouched, and every logical fit is saved independently so an interrupted screen can resume without repeating completed work.

<img src="image-20260909-150545.png" width="" align="" />

## 4.5.2.3.0 Common Restartable Screening Engine

Each screening batch uses the same legal-history, inner-validation, training-only preprocessing, selected-epoch, raw/final forecast, checkpoint, and common-support rules inherited from 4.5.1. The code below records logical rosters and leaves expensive fitting behind explicit run toggles.

**Define logical-fit roster and payload contracts.**
The Transformer screens are organized around **logical fits**: one modeling track, target, horizon, candidate specification, and evaluation window. That logical identity is also the restart unit. In practical terms, it means that if a 20-fit screen stops after fit 17, the notebook can prove which 17 are complete and resume only the remaining three.

Before any fit begins, the payload is also checked for row alignment. Sequence history, context features, Taxi Zone identity, and the target must all describe the same examples; otherwise the model could train successfully while learning from mismatched rows.

In [18]:
# ---------------------------------------------------------------------
# 4.5.2.3.0A — Define logical-fit roster and payload contracts
# ---------------------------------------------------------------------

# These five fields uniquely identify one expensive Transformer experiment.
# Derived metrics and file paths are deliberately excluded: they describe a
# completed fit, but they should not change which fit the notebook is asking for.
TRANSFORMER_LOGICAL_KEY_COLUMNS = [
    "track", "metric", "horizon", "candidate_id", "temporal_window"
]

TRANSFORMER_BATCH_CONTRACT_IDS = {
    "A": TRANSFORMER_REPRESENTATION_CONTRACT_ID,
    "B": TRANSFORMER_PROFILE_CONTRACT_ID,
    "C": TRANSFORMER_CONFIRMATION_CONTRACT_ID,
}


def build_transformer_roster(batch, candidates, temporal_windows):
    """Build the complete checklist of logical fits for one Transformer batch."""
    if batch not in TRANSFORMER_BATCH_CONTRACT_IDS:
        raise ValueError(f"Unknown Transformer batch: {batch}")

    rows = []
    contract_id = TRANSFORMER_BATCH_CONTRACT_IDS[batch]

    for track, metric, horizon in TRANSFORMER_SENTINEL_KEYS:
        for candidate in candidates:
            candidate_id = candidate.get("representation_id") or candidate.get("spec_id")

            if not candidate_id:
                raise ValueError("Every Transformer candidate needs an explicit ID.")

            for temporal_window in temporal_windows:
                rows.append({
                    "batch": batch,
                    "track": track,
                    "metric": metric,
                    "horizon": int(horizon),
                    "candidate_id": candidate_id,
                    "temporal_window": temporal_window,
                    "contract_id": contract_id,
                    "status": "planned",
                })

    return pd.DataFrame(rows)


def _require_fit_payload(payload):
    """Fail before training if any required split is missing or row-misaligned."""
    required = {
        "train_sequence_values", "train_sequence_masks", "train_context",
        "train_zone_index", "train_target",
        "inner_sequence_values", "inner_sequence_masks", "inner_context",
        "inner_zone_index", "inner_target",
        "final_sequence_values", "final_sequence_masks", "final_context",
        "final_zone_index", "final_target",
        "eval_sequence_values", "eval_sequence_masks", "eval_context",
        "eval_zone_index", "eval_target",
        "eval_keys", "benchmark_prediction",
    }

    missing = sorted(required - set(payload))
    if missing:
        raise ValueError(f"Transformer fit payload is missing: {missing}")

    # Each row is one forecasting example. History, context, zone identity,
    # and target must therefore have exactly the same row count within a split.
    for split in ["train", "inner", "final", "eval"]:
        row_counts = {
            len(payload[f"{split}_sequence_values"]),
            len(payload[f"{split}_sequence_masks"]),
            len(payload[f"{split}_context"]),
            len(payload[f"{split}_zone_index"]),
            len(payload[f"{split}_target"]),
        }

        if len(row_counts) != 1:
            raise ValueError(
                f"{split} payload arrays do not share one row count."
            )

    eval_rows = len(payload["eval_target"])

    if len(payload["eval_keys"]) != eval_rows:
        raise ValueError("Evaluation keys and target rows must align one-to-one.")

    if len(payload["benchmark_prediction"]) != eval_rows:
        raise ValueError("Benchmark predictions and evaluation rows must align one-to-one.")

**Select training duration and perform fresh final refit.**
Each logical fit uses the same two-stage training procedure finalized in 4.5.1.

First, a **temporary selector model** uses a historical 28-day validation window to answer only one question: *how many epochs should this specification train?* That model is then discarded. A completely fresh model and fresh preprocessing state are fitted on all legally available pre-evaluation history for exactly the selected number of epochs.

This separation matters because the inner-validation period may choose training duration, but it must not quietly donate fitted weights or preprocessing statistics to the final model.

In [19]:
# ---------------------------------------------------------------------
# 4.5.2.3.0B — Select training duration and perform fresh final refit
# ---------------------------------------------------------------------

def _select_transformer_epoch(model, train_loader, validation_loader, profile):
    """Choose the epoch count using historical inner-validation performance."""
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=profile["learning_rate"],
        weight_decay=profile["weight_decay"],
    )
    criterion = nn.HuberLoss()

    curve_rows = []
    best_epoch = 1
    best_validation_mae = np.inf
    stale_epochs = 0

    for epoch in range(1, TRANSFORMER_MAX_EPOCHS + 1):
        # ---- Selector training ------------------------------------------------
        model.train()
        train_losses = []

        for sequence, context, zones, target in train_loader:
            sequence = sequence.to(TORCH_DEVICE)
            context = context.to(TORCH_DEVICE)
            zones = zones.to(TORCH_DEVICE)
            target = target.to(TORCH_DEVICE)

            optimizer.zero_grad(set_to_none=True)

            prediction = model(sequence, context, zones)
            loss = criterion(prediction, target)
            loss.backward()

            # Clipping limits an unusually large gradient update from destabilizing
            # training. It is a fixed safety rule, not another tuned hyperparameter.
            torch.nn.utils.clip_grad_norm_(
                model.parameters(), TRANSFORMER_GRAD_CLIP_NORM
            )

            optimizer.step()
            train_losses.append(float(loss.detach().cpu()))

        # ---- Historical inner validation -------------------------------------
        model.eval()
        validation_losses = []
        validation_predictions = []
        validation_targets = []

        with torch.no_grad():
            for sequence, context, zones, target in validation_loader:
                sequence = sequence.to(TORCH_DEVICE)
                context = context.to(TORCH_DEVICE)
                zones = zones.to(TORCH_DEVICE)
                target_device = target.to(TORCH_DEVICE)

                prediction = model(sequence, context, zones)
                validation_loss = criterion(prediction, target_device)

                validation_losses.append(float(validation_loss.cpu()))
                validation_predictions.extend(prediction.cpu().numpy())
                validation_targets.extend(target.numpy())

        # Targets were standardized by training-only preprocessing, so this MAE
        # has a comparable scale for deciding when training has stopped improving.
        validation_mae = regression_metrics(
            validation_targets, validation_predictions
        )["mae"]

        improved = (
            np.isfinite(validation_mae)
            and validation_mae < best_validation_mae - TRANSFORMER_MIN_DELTA
        )

        if improved:
            best_epoch = epoch
            best_validation_mae = validation_mae
            stale_epochs = 0
        else:
            stale_epochs += 1

        curve_rows.append({
            "epoch": epoch,
            "train_huber": float(np.mean(train_losses)) if train_losses else np.nan,
            "inner_validation_huber": (
                float(np.mean(validation_losses)) if validation_losses else np.nan
            ),
            "inner_validation_standardized_mae": validation_mae,
            "best_validation_mae_so_far": best_validation_mae,
            "selected_epoch": False,
        })

        if stale_epochs >= TRANSFORMER_PATIENCE:
            break

    curve_df = pd.DataFrame(curve_rows)
    curve_df.loc[curve_df["epoch"].eq(best_epoch), "selected_epoch"] = True

    return best_epoch, curve_df


def _fit_transformer_fixed_epochs(model, train_loader, profile, epochs):
    """Train a fresh final model for exactly the selector-chosen duration."""
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=profile["learning_rate"],
        weight_decay=profile["weight_decay"],
    )
    criterion = nn.HuberLoss()

    # There is intentionally no second early-stopping rule here. The selector
    # already chose the duration; changing it during final refit would let a
    # second validation decision alter the experiment we just selected.
    for _ in range(int(epochs)):
        model.train()

        for sequence, context, zones, target in train_loader:
            sequence = sequence.to(TORCH_DEVICE)
            context = context.to(TORCH_DEVICE)
            zones = zones.to(TORCH_DEVICE)
            target = target.to(TORCH_DEVICE)

            optimizer.zero_grad(set_to_none=True)

            prediction = model(sequence, context, zones)
            loss = criterion(prediction, target)
            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(), TRANSFORMER_GRAD_CLIP_NORM
            )
            optimizer.step()

    return model

**Run and save one complete logical fit.**
The next function executes one complete logical fit from end to end: selector training, fresh final refit, outer-window prediction, scoring, and durable checkpointing.

The saved state contains more than model weights. It also preserves the exact representation, Transformer profile, target-specific sequence-channel order, context order, Taxi Zone vocabulary, and fitted preprocessing state. That makes later restart recovery and predictive-reliance analysis possible without reconstructing hidden in-memory assumptions.

In [20]:
# ---------------------------------------------------------------------
# 4.5.2.3.0C — Run and save one complete logical fit
# ---------------------------------------------------------------------

def run_transformer_logical_fit(logical_row, representation, profile, payload, output_dir):
    """Run selector → fresh refit → outer forecast for one restartable fit."""
    _require_fit_payload(payload)

    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    seed_override = logical_row.get("fit_seed_override", None)

    if seed_override is not None and pd.notna(seed_override):
        # Matched experiments may deliberately share stochastic initialization
        # so the modeling choice under test is not confounded by a different seed.
        seed = int(seed_override)
    else:
        seed = stable_transformer_fit_seed(
            logical_row["contract_id"],
            logical_row["track"],
            logical_row["metric"],
            logical_row["horizon"],
            logical_row["candidate_id"],
            logical_row["temporal_window"],
        )

    seed_experiment(seed)
    reset_cuda_peak_memory()
    fit_started = time.perf_counter()

    # =================================================================
    # 1. TEMPORARY SELECTOR
    # =================================================================
    # Preprocessing learns only from selector-training rows. The inner window
    # is transformed using that already-fitted state, keeping validation truly
    # out of the statistics used to prepare training data.
    selector_preprocessor = TransformerPreprocessor(transformer_context_contract_df)
    selector_preprocessor.fit(
        payload["train_sequence_values"],
        payload["train_sequence_masks"],
        payload["train_context"],
        payload["train_target"],
    )

    selector_train_sequence = selector_preprocessor.transform_sequence(
        payload["train_sequence_values"], payload["train_sequence_masks"]
    )
    selector_inner_sequence = selector_preprocessor.transform_sequence(
        payload["inner_sequence_values"], payload["inner_sequence_masks"]
    )
    selector_train_context = selector_preprocessor.transform_context(payload["train_context"])
    selector_inner_context = selector_preprocessor.transform_context(payload["inner_context"])
    selector_train_target = selector_preprocessor.transform_target(payload["train_target"])
    selector_inner_target = selector_preprocessor.transform_target(payload["inner_target"])

    selector_train_dataset = TransformerDataset(
        selector_train_sequence,
        selector_train_context,
        payload["train_zone_index"],
        selector_train_target,
    )
    selector_inner_dataset = TransformerDataset(
        selector_inner_sequence,
        selector_inner_context,
        payload["inner_zone_index"],
        selector_inner_target,
    )

    selector_train_loader = DataLoader(
        selector_train_dataset,
        batch_size=selected_transformer_batch_size,
        shuffle=True,
    )
    selector_inner_loader = DataLoader(
        selector_inner_dataset,
        batch_size=selected_transformer_batch_size,
        shuffle=False,
    )

    selector_model = PatchTransformerRegressor(
        NEURAL_SEQUENCE_INPUT_DIM,
        transformer_context_feature_count,
        len(transformer_zone_ids),
        representation,
        profile,
    ).to(TORCH_DEVICE)

    selected_epoch, curve_df = _select_transformer_epoch(
        selector_model, selector_train_loader, selector_inner_loader, profile
    )

    # Only the selected epoch count survives. Selector weights and preprocessing
    # are disposable evidence and must not leak into the production-style refit.
    del (
        selector_model,
        selector_preprocessor,
        selector_train_loader,
        selector_inner_loader,
        selector_train_dataset,
        selector_inner_dataset,
    )
    clear_device_memory()

    # =================================================================
    # 2. FRESH FINAL REFIT
    # =================================================================
    # Now preprocessing is learned again from all bounded history legally
    # available before the outer evaluation window.
    final_preprocessor = TransformerPreprocessor(transformer_context_contract_df)
    final_preprocessor.fit(
        payload["final_sequence_values"],
        payload["final_sequence_masks"],
        payload["final_context"],
        payload["final_target"],
    )

    final_train_sequence = final_preprocessor.transform_sequence(
        payload["final_sequence_values"], payload["final_sequence_masks"]
    )
    final_train_context = final_preprocessor.transform_context(payload["final_context"])
    final_train_target = final_preprocessor.transform_target(payload["final_target"])

    eval_sequence = final_preprocessor.transform_sequence(
        payload["eval_sequence_values"], payload["eval_sequence_masks"]
    )
    eval_context = final_preprocessor.transform_context(payload["eval_context"])

    final_dataset = TransformerDataset(
        final_train_sequence,
        final_train_context,
        payload["final_zone_index"],
        final_train_target,
    )
    final_loader = DataLoader(
        final_dataset,
        batch_size=selected_transformer_batch_size,
        shuffle=True,
    )

    final_model = PatchTransformerRegressor(
        NEURAL_SEQUENCE_INPUT_DIM,
        transformer_context_feature_count,
        len(transformer_zone_ids),
        representation,
        profile,
    ).to(TORCH_DEVICE)

    _fit_transformer_fixed_epochs(
        final_model, final_loader, profile, selected_epoch
    )

    # =================================================================
    # 3. OUTER FORECAST
    # =================================================================
    final_model.eval()

    with torch.no_grad():
        prediction_standardized = final_model(
            torch.as_tensor(eval_sequence, dtype=torch.float32, device=TORCH_DEVICE),
            torch.as_tensor(eval_context, dtype=torch.float32, device=TORCH_DEVICE),
            torch.as_tensor(
                payload["eval_zone_index"], dtype=torch.long, device=TORCH_DEVICE
            ),
        ).cpu().numpy()

    prediction_raw = final_preprocessor.inverse_target(prediction_standardized)

    # Raw predictions show what the network actually produced. The final view
    # applies the project's production rule that mobility outcomes cannot be
    # negative. Both are retained because they answer different questions.
    prediction_final = apply_prediction_output_rule(prediction_raw)

    prediction_df = payload["eval_keys"].copy()
    prediction_df["target_observed_value"] = payload["eval_target"]
    prediction_df["benchmark_prediction"] = payload["benchmark_prediction"]
    prediction_df["prediction_raw"] = prediction_raw
    prediction_df["prediction_final"] = prediction_final
    prediction_df["track"] = logical_row["track"]
    prediction_df["metric"] = logical_row["metric"]
    prediction_df["horizon"] = int(logical_row["horizon"])
    prediction_df["candidate_id"] = logical_row["candidate_id"]
    prediction_df["temporal_window"] = logical_row["temporal_window"]

    # Curves need their own identity because they are reloaded independently
    # after a kernel restart for learning-dynamics analysis.
    curve_df["track"] = logical_row["track"]
    curve_df["metric"] = logical_row["metric"]
    curve_df["horizon"] = int(logical_row["horizon"])
    curve_df["candidate_id"] = logical_row["candidate_id"]
    curve_df["temporal_window"] = logical_row["temporal_window"]

    raw_metrics = regression_metrics(payload["eval_target"], prediction_raw)
    benchmark_metrics = regression_metrics(
        payload["eval_target"], payload["benchmark_prediction"]
    )

    benchmark_mae = benchmark_metrics["mae"]
    benchmark_skill_pct = (
        100.0 * (benchmark_mae - raw_metrics["mae"]) / benchmark_mae
        if benchmark_mae > 0
        else np.nan
    )

    # =================================================================
    # 4. DURABLE ARTIFACTS
    # =================================================================
    token = logical_key_token(logical_row, TRANSFORMER_LOGICAL_KEY_COLUMNS)

    prediction_path = output_dir / f"prediction_{token}.parquet"
    curve_path = output_dir / f"curve_{token}.parquet"
    state_path = output_dir / f"state_{token}.pt"
    checkpoint_path = output_dir / f"fit_{token}.parquet"

    result = {
        **logical_row.to_dict(),
        "status": "success",
        "selected_epoch": int(selected_epoch),
        "parameter_count": transformer_fit_parameter_count(final_model),
        "runtime_seconds": time.perf_counter() - fit_started,
        "peak_gpu_memory_mb": read_cuda_peak_memory_mb(),
        "train_rows": len(payload["train_target"]),
        "inner_validation_rows": len(payload["inner_target"]),
        "final_refit_rows": len(payload["final_target"]),
        "evaluation_rows": len(payload["eval_target"]),
        "raw_mae": raw_metrics["mae"],
        "benchmark_mae": benchmark_mae,
        "benchmark_skill_pct": benchmark_skill_pct,
        "negative_raw_predictions": int(np.sum(np.asarray(prediction_raw) < 0)),
        "prediction_path": str(prediction_path),
        "curve_path": str(curve_path),
        "state_path": str(state_path),
    }

    # Companion artifacts are written first. Only after all companions exist do
    # we write the small success checkpoint. Recovery therefore cannot mistake
    # a crash halfway through serialization for a completed logical fit.
    atomic_write_parquet(prediction_df, prediction_path)
    atomic_write_parquet(curve_df, curve_path)

    atomic_torch_save(
        {
            "contract_id": logical_row["contract_id"],
            "logical_key": {
                column: logical_row[column]
                for column in TRANSFORMER_LOGICAL_KEY_COLUMNS
            },
            "model_state_dict": final_model.state_dict(),
            "representation": dict(representation),
            "profile": dict(profile),
            "selected_epoch": int(selected_epoch),
            "context_feature_order": list(TRANSFORMER_CONTEXT_FEATURE_ORDER),
            "sequence_value_channels": list(
                TRANSFORMER_SEQUENCE_VALUE_CHANNELS_BY_TARGET[logical_row["metric"]]
            ),
            "sequence_mask_channels": list(
                TRANSFORMER_SEQUENCE_MASK_CHANNELS_BY_TARGET[logical_row["metric"]]
            ),
            "zone_ids": np.asarray(transformer_zone_ids, dtype=np.int64),
            "sequence_ids": np.asarray(transformer_sequence_ids, dtype=np.int64),
            "context_contract": transformer_context_contract_df.to_dict("records"),
            "preprocessor_state": {
                "sequence_means": final_preprocessor.sequence_means_,
                "sequence_stds": final_preprocessor.sequence_stds_,
                "context_medians": final_preprocessor.context_medians_,
                "context_means": final_preprocessor.context_means_,
                "context_stds": final_preprocessor.context_stds_,
                "stl_training_support": final_preprocessor.stl_training_support_,
                "stl_mean": final_preprocessor.stl_mean_,
                "stl_std": final_preprocessor.stl_std_,
                "target_mean": final_preprocessor.target_mean_,
                "target_std": final_preprocessor.target_std_,
            },
        },
        state_path,
    )

    atomic_write_parquet(pd.DataFrame([result]), checkpoint_path)

    del final_model, final_loader, final_dataset
    clear_device_memory()

    return result, prediction_df, curve_df

**Reconstruct the finalized 52-value context.**
The Transformer uses the same finalized **52-value supporting context** as the neural workflow, but this step performs only deterministic reconstruction. Learned completion and scaling remain inside `TransformerPreprocessor`.

That distinction is important for leakage safety: this function may identify missing values and construct calendar encodings, but it never learns a mean, median, standard deviation, or STL scale from evaluation data.

In [21]:
# ---------------------------------------------------------------------
# 4.5.2.3.0D — Reconstruct the finalized 52-value context
# ---------------------------------------------------------------------

# These lists describe the raw forecast-matrix sources used to reconstruct the
# model-facing context. They are grouped semantically so future readers can see
# what information the model receives without inspecting a 52-column list.
MULTIMODAL_CONTEXT_SOURCES = [
    "mm_taxi_avg_trip_duration_origin_value",
    "mm_fhvhv_avg_trip_duration_origin_value",
    "mm_subway_transfers_origin_value",
    "mm_bus_trip_count_origin_value",
    "mm_avg_bus_speed_origin_value",
]

SPATIAL_CONTEXT_SOURCES = [
    f"spatial_connected__{metric}" for metric in FORECAST_TARGETS
]

WEATHER_CONTEXT_SOURCES = [
    "weather_origin__temperature",
    "weather_origin__precipitation",
    "weather_origin__visibility",
    "weather_origin__wind_speed",
    "weather_target_proxy__temperature",
    "weather_target_proxy__precipitation",
    "weather_target_proxy__visibility",
    "weather_target_proxy__wind_speed",
]

CALENDAR_CONTEXT_SOURCES = [
    "target_daypart",
    "target_weekday_index",
    "target_is_weekend",
    "annual_fourier_sin",
    "annual_fourier_cos",
    "target_pre_post_cp",
]

STL_CONTEXT_SOURCES = [
    "weekly_stl_enabled",
    "weekly_stl_seasonal_offset",
]

GEOGRAPHIC_CONTEXT_SOURCES = [
    "is_cbd_zone",
    "is_cbd_gateway_zone",
    "is_cbd_adjacent_zone",
    "distance_to_cbd_miles",
    "distance_to_gateway_miles",
    "connected_neighbor_count",
    "network_betweenness_centrality",
    "network_closeness_centrality",
    "network_clustering_coefficient",
]

TRANSFORMER_CONTEXT_SOURCE_FEATURES = list(dict.fromkeys(
    MULTIMODAL_CONTEXT_SOURCES
    + SPATIAL_CONTEXT_SOURCES
    + WEATHER_CONTEXT_SOURCES
    + CALENDAR_CONTEXT_SOURCES
    + STL_CONTEXT_SOURCES
    + GEOGRAPHIC_CONTEXT_SOURCES
))


def derive_transformer_context_frame(frame):
    """Rebuild the 52 model-facing context values without learning from them."""
    missing_sources = sorted(
        set(TRANSFORMER_CONTEXT_SOURCE_FEATURES) - set(frame.columns)
    )
    if missing_sources:
        raise ValueError(
            f"Forecast matrix lacks final context sources: {missing_sources}"
        )

    result = pd.DataFrame(index=frame.index)
    derived_sources = {
        "target_daypart",
        "target_weekday_index",
        "target_pre_post_cp",
    }

    # Preserve ordinary numeric sources in their native units. Missing values
    # stay missing here so the fitted preprocessor—not evaluation data—decides
    # how continuous features are completed and scaled.
    for feature in TRANSFORMER_CONTEXT_SOURCE_FEATURES:
        if feature not in derived_sources:
            result[feature] = pd.to_numeric(frame[feature], errors="coerce")

    # Missingness is recorded BEFORE later completion. Otherwise an imputed zero
    # would be indistinguishable from a genuine observed zero.
    for feature in TRANSFORMER_CONTEXT_FEATURE_ORDER:
        if not feature.startswith("missing_mask__"):
            continue

        source = feature.removeprefix("missing_mask__")
        if source not in frame.columns:
            raise ValueError(f"Missingness source is absent from forecast matrix: {source}")

        result[feature] = (
            pd.to_numeric(frame[source], errors="coerce").isna().astype("float32")
        )

    # Dayparts and weekdays are circular: the final category is adjacent to the
    # first. Sine/cosine coordinates preserve that geometry for the model.
    daypart_lookup = {name: index for index, name in enumerate(DAYPART_ORDER)}
    daypart_index = frame["target_daypart"].map(daypart_lookup)

    if daypart_index.isna().any():
        raise ValueError("Forecast matrix contains an unknown target Daypart.")

    daypart_angle = 2.0 * np.pi * daypart_index.astype(float) / len(DAYPART_ORDER)
    result["target_daypart_sin"] = np.sin(daypart_angle)
    result["target_daypart_cos"] = np.cos(daypart_angle)

    weekday = pd.to_numeric(
        frame["target_weekday_index"], errors="raise"
    ).astype(int)

    if not weekday.between(0, 6).all():
        raise ValueError("target_weekday_index must be in the canonical 0..6 range.")

    weekday_angle = 2.0 * np.pi * weekday / 7.0
    result["target_weekday_sin"] = np.sin(weekday_angle)
    result["target_weekday_cos"] = np.cos(weekday_angle)

    # The policy regime is deterministic context, not a learned category.
    regime = frame["target_pre_post_cp"].astype("string")
    if not regime.isin(["pre_cp", "post_cp"]).all():
        raise ValueError("Forecast matrix contains an unknown policy regime.")

    result["target_cp_regime"] = regime.map({"pre_cp": 0.0, "post_cp": 1.0})

    result["target_is_weekend"] = pd.to_numeric(
        frame["target_is_weekend"], errors="raise"
    )
    result["annual_fourier_sin"] = pd.to_numeric(
        frame["annual_fourier_sin"], errors="raise"
    )
    result["annual_fourier_cos"] = pd.to_numeric(
        frame["annual_fourier_cos"], errors="raise"
    )

    # Weekly STL arrives from the upstream forecasting feature contract.
    # Disabled STL and unavailable STL are semantically different states, so
    # both the enabled flag and the possibly-missing offset are preserved.
    result["weekly_stl_enabled"] = pd.to_numeric(
        frame["weekly_stl_enabled"], errors="raise"
    )
    result["weekly_stl_seasonal_offset"] = pd.to_numeric(
        frame["weekly_stl_seasonal_offset"], errors="coerce"
    )

    # Geographic membership flags are binary context. Missing membership means
    # the zone is not tagged with that role, so zero is the intended semantic value.
    for feature in ["is_cbd_zone", "is_cbd_gateway_zone", "is_cbd_adjacent_zone"]:
        result[feature] = pd.to_numeric(
            frame[feature], errors="coerce"
        ).fillna(0.0)

    missing_final = sorted(
        set(TRANSFORMER_CONTEXT_FEATURE_ORDER) - set(result.columns)
    )
    if missing_final:
        raise ValueError(
            f"Context reconstruction missed finalized fields: {missing_final}"
        )

    return result[TRANSFORMER_CONTEXT_FEATURE_ORDER].copy()

**Extract target-first history and legal temporal windows.**
Transformer representations differ in how much mobility history they can see, so history extraction has two jobs.

First, it retrieves the exact legal sequence ending at each forecast origin and then reorders the five mobility values plus five masks into the **target-specific model-facing order** established in 4.5.1. Second, it defines legal 28-day early, middle, and late evaluation windows, each with a complete 28-day historical inner-validation period immediately before it.

In [22]:
# ---------------------------------------------------------------------
# 4.5.2.3.0E — Extract target-first history and legal temporal windows
# ---------------------------------------------------------------------

def _extract_transformer_history(frame, metric, lookback_steps):
    """Extract one legal history window and reorder its channels target-first."""
    zone_positions = frame["taxi_zone_id"].map(transformer_zone_to_index)
    origin_positions = frame["origin_observation_sequence_id"].map(
        transformer_sequence_to_index
    )

    if zone_positions.isna().any() or origin_positions.isna().any():
        raise ValueError("Forecast rows contain unknown Taxi Zone or sequence IDs.")

    starts = origin_positions.to_numpy(dtype=int) - int(lookback_steps) + 1

    # A negative start means the representation is asking for history that did
    # not yet exist. Padding it would silently change the frozen experiment.
    if (starts < 0).any():
        raise ValueError("A requested history window precedes the dense state.")

    offsets = np.arange(int(lookback_steps), dtype=int)
    positions = starts[:, None] + offsets[None, :]
    zones = zone_positions.to_numpy(dtype=int)

    values = transformer_history_values[zones[:, None], positions, :]
    masks = transformer_history_masks[zones[:, None], positions, :]

    # The dense NPZ has one physical modality order, but 4.5.1 finalized a
    # target-specific model order: the target's own history comes first.
    # Values and masks move together so each mask still describes its modality.
    target_first_order = TRANSFORMER_SEQUENCE_VALUE_CHANNELS_BY_TARGET[metric]
    channel_positions = [
        FORECAST_TARGETS.index(channel) for channel in target_first_order
    ]

    return (
        values[:, :, channel_positions],
        masks[:, :, channel_positions],
    )


def _transformer_window_bounds(frame, temporal_window):
    """Return a legal 28-day outer window with a complete 28-day inner tail."""
    rolling_dates = pd.DatetimeIndex(sorted(
        frame.loc[
            frame["stage"].eq("rolling_validation"), "target_date"
        ].dropna().unique()
    ))
    available_dates = set(rolling_dates)
    feasible_starts = []

    for start in rolling_dates:
        inner_dates = pd.date_range(
            start - pd.Timedelta(days=28), periods=28, freq="D"
        )
        outer_dates = pd.date_range(start, periods=28, freq="D")

        # Chapter 4 development must stop before the untouched final holdout.
        if outer_dates[-1] >= FINAL_HOLDOUT_START_DATE:
            continue

        if set(inner_dates).issubset(available_dates) and set(outer_dates).issubset(
            available_dates
        ):
            feasible_starts.append(pd.Timestamp(start))

    if not feasible_starts:
        raise ValueError(
            "No legal 28-day outer window has a complete preceding inner tail."
        )

    window_position = {
        "early": 0,
        "middle": (len(feasible_starts) - 1) // 2,
        "late": len(feasible_starts) - 1,
    }

    if temporal_window not in window_position:
        raise ValueError(f"Unknown Transformer temporal window: {temporal_window}")

    start = feasible_starts[window_position[temporal_window]]
    end = start + pd.Timedelta(days=27)

    return start, end


def _build_split_payload(frame, metric, lookback_steps):
    """Convert one already-legal row partition into raw Transformer inputs."""
    raw_values, raw_masks = _extract_transformer_history(
        frame, metric, lookback_steps
    )

    zone_index = frame["taxi_zone_id"].map(transformer_zone_to_index)
    if zone_index.isna().any():
        raise ValueError(
            "A split contains Taxi Zones outside the embedding vocabulary."
        )

    target = pd.to_numeric(
        frame["target_observed_value"], errors="raise"
    ).to_numpy(dtype=np.float32)

    return {
        "sequence_values": raw_values,
        "sequence_masks": raw_masks,
        "context": derive_transformer_context_frame(frame),
        "zone_index": zone_index.to_numpy(dtype=np.int64),
        "target": target,
    }

**Build one leakage-safe logical-fit payload.**
Batch A deliberately separates **training eligibility** from **comparison eligibility**.

A 35-step representation may train whenever 35 historical steps are legally available; it should not lose training examples merely because another candidate needs 140 steps. The longest lookback is used only to define a common **outer evaluation population**, ensuring that all four representations answer the same future forecasting questions.

The bounded development cap is also applied by **forecast origin**, not by individual Taxi Zone row, so a retained origin keeps its complete cross-zone panel intact.

In [23]:
# ---------------------------------------------------------------------
# 4.5.2.3.0F — Build one leakage-safe logical-fit payload
# ---------------------------------------------------------------------

def build_transformer_fit_payload(logical_row, representation, profile):
    """Assemble selector, inner-validation, final-refit, and outer-evaluation data."""
    # Capacity settings do not change which historical rows are legal; the
    # representation's lookback is what determines sequence eligibility.
    del profile

    track = logical_row["track"]
    metric = logical_row["metric"]
    horizon = int(logical_row["horizon"])

    lookback_steps = int(representation["lookback_steps"])
    comparison_lookback_steps = int(
        logical_row.get("comparison_lookback_steps", lookback_steps)
    )

    source_columns = list(dict.fromkeys([
        "modeling_track",
        "stage",
        "forecast_sequence_id",
        "taxi_zone_id",
        "metric",
        "origin_observation_sequence_id",
        "target_observation_sequence_id",
        "target_date",
        "target_observed_value",
        "target_label_available",
        "native_track_eligible",
        "horizon",
        *TRANSFORMER_CONTEXT_SOURCE_FEATURES,
    ]))

    matrix = pd.read_parquet(
        FORECASTING_MATRIX_PATHS[track],
        columns=source_columns,
        filters=[
            ("modeling_track", "==", track),
            ("metric", "==", metric),
            ("horizon", "==", horizon),
            ("native_track_eligible", "==", True),
            ("stage", "in", ["initial_training", "rolling_validation"]),
        ],
    )

    matrix["target_date"] = pd.to_datetime(matrix["target_date"]).dt.normalize()

    for column in [
        "origin_observation_sequence_id",
        "target_observation_sequence_id",
    ]:
        matrix[column] = pd.to_numeric(
            matrix[column], errors="raise"
        ).astype("int64")

    labels_available = (
        matrix["target_label_available"].fillna(False).astype(bool)
        & matrix["target_observed_value"].notna()
    )

    # -------------------------------------------------------------
    # Two different lookback rules answer two different questions.
    # -------------------------------------------------------------
    # OWN lookback: "Could this candidate legally have made this example?"
    own_history_start = TRACK_HISTORY_START_SEQUENCE_ID[track] + lookback_steps - 1
    own_history_eligible = matrix["origin_observation_sequence_id"].ge(
        own_history_start
    )

    # COMMON lookback: "Is this future row fair to compare across candidates?"
    # It affects only the outer comparison calendar—not training history.
    comparison_history_start = (
        TRACK_HISTORY_START_SEQUENCE_ID[track] + comparison_lookback_steps - 1
    )
    comparison_eligible = matrix["origin_observation_sequence_id"].ge(
        comparison_history_start
    )

    window_source = matrix.loc[
        labels_available & comparison_eligible
    ].copy()

    eval_start, eval_end = _transformer_window_bounds(
        window_source, logical_row["temporal_window"]
    )
    inner_start = eval_start - pd.Timedelta(days=28)
    inner_end = eval_start - pd.Timedelta(days=1)

    # -------------------------------------------------------------
    # Selector training: legal history before the inner window
    # -------------------------------------------------------------
    selector_train_df = matrix.loc[
        labels_available
        & own_history_eligible
        & matrix["target_date"].lt(inner_start)
    ].copy()

    # Development screens cap expensive history at 192 complete origins.
    # Sampling origins instead of rows keeps every Taxi Zone from that origin together.
    selector_train_df = thin_training_origins(
        selector_train_df,
        TRANSFORMER_MAX_TRAIN_ORIGINS,
        "origin_observation_sequence_id",
    )

    # -------------------------------------------------------------
    # Inner validation: complete historical 28-day rehearsal window
    # -------------------------------------------------------------
    inner_df = matrix.loc[
        labels_available
        & own_history_eligible
        & matrix["stage"].eq("rolling_validation")
        & matrix["target_date"].between(inner_start, inner_end)
    ].copy()

    # -------------------------------------------------------------
    # Fresh final refit: all legal history before outer evaluation
    # -------------------------------------------------------------
    final_refit_df = matrix.loc[
        labels_available
        & own_history_eligible
        & matrix["target_date"].lt(eval_start)
    ].copy()

    final_refit_df = thin_training_origins(
        final_refit_df,
        TRANSFORMER_MAX_TRAIN_ORIGINS,
        "origin_observation_sequence_id",
    )

    # -------------------------------------------------------------
    # Outer evaluation: exact common future population
    # -------------------------------------------------------------
    evaluation_df = matrix.loc[
        labels_available
        & own_history_eligible
        & comparison_eligible
        & matrix["stage"].eq("rolling_validation")
        & matrix["target_date"].between(eval_start, eval_end)
        & matrix["target_date"].lt(FINAL_HOLDOUT_START_DATE)
    ].copy()

    if any(
        frame.empty
        for frame in [selector_train_df, inner_df, final_refit_df, evaluation_df]
    ):
        raise ValueError(
            f"No complete legal Transformer split for {track} · {metric} · h={horizon}."
        )

    # A partial 28-day window could make one candidate appear better simply
    # because difficult dates vanished. Require the entire calendar window.
    if inner_df["target_date"].nunique() != 28:
        raise ValueError(
            f"{track} · {metric} · h={horizon}: "
            "inner validation is not a complete 28-day window."
        )

    if evaluation_df["target_date"].nunique() != 28:
        raise ValueError(
            f"{track} · {metric} · h={horizon}: "
            "outer evaluation is not a complete 28-day window."
        )

    # -------------------------------------------------------------
    # Frozen previous-week benchmark
    # -------------------------------------------------------------
    benchmark_info = advanced_model_benchmark_registry_df.loc[
        advanced_model_benchmark_registry_df["modeling_track"].eq(track)
        & advanced_model_benchmark_registry_df["metric"].eq(metric)
        & advanced_model_benchmark_registry_df["horizon"].astype(int).eq(horizon)
    ]

    if len(benchmark_info) != 1:
        raise ValueError("Expected exactly one frozen benchmark contract.")

    benchmark_id = str(benchmark_info.iloc[0]["baseline_id"])
    baseline_column = f"baseline__{benchmark_id}"

    join_keys = [
        "forecast_sequence_id",
        "origin_observation_sequence_id",
        "target_observation_sequence_id",
        "horizon",
        "target_date",
    ]

    baseline_df = pd.read_parquet(
        BASELINE_PREDICTION_PATHS[track],
        columns=join_keys + [baseline_column],
        filters=[
            ("metric", "==", metric),
            ("horizon", "==", horizon),
            ("stage", "==", "rolling_validation"),
        ],
    )
    baseline_df["target_date"] = pd.to_datetime(
        baseline_df["target_date"]
    ).dt.normalize()

    # One-to-one keys guarantee that model and benchmark are being scored on
    # the same forecasting question rather than merely similar-looking rows.
    if evaluation_df.duplicated(join_keys).any():
        raise ValueError("Transformer evaluation keys are not unique.")

    if baseline_df.duplicated(join_keys).any():
        raise ValueError("Frozen benchmark evaluation keys are not unique.")

    evaluation_df = evaluation_df.merge(
        baseline_df[join_keys + [baseline_column]],
        on=join_keys,
        how="inner",
        validate="one_to_one",
    )

    if evaluation_df.empty:
        raise ValueError("No common Transformer/benchmark evaluation rows remain.")

    # Only deterministic arrays are built here. No scaler or model has yet seen
    # the inner-validation or outer-evaluation distributions.
    train_payload = _build_split_payload(
        selector_train_df, metric, lookback_steps
    )
    inner_payload = _build_split_payload(inner_df, metric, lookback_steps)
    final_payload = _build_split_payload(final_refit_df, metric, lookback_steps)
    eval_payload = _build_split_payload(evaluation_df, metric, lookback_steps)

    eval_keys = evaluation_df[
        [
            "forecast_sequence_id",
            "taxi_zone_id",
            "origin_observation_sequence_id",
            "target_observation_sequence_id",
            "target_date",
        ]
    ].copy()

    return {
        "train_sequence_values": train_payload["sequence_values"],
        "train_sequence_masks": train_payload["sequence_masks"],
        "train_context": train_payload["context"],
        "train_zone_index": train_payload["zone_index"],
        "train_target": train_payload["target"],

        "inner_sequence_values": inner_payload["sequence_values"],
        "inner_sequence_masks": inner_payload["sequence_masks"],
        "inner_context": inner_payload["context"],
        "inner_zone_index": inner_payload["zone_index"],
        "inner_target": inner_payload["target"],

        "final_sequence_values": final_payload["sequence_values"],
        "final_sequence_masks": final_payload["sequence_masks"],
        "final_context": final_payload["context"],
        "final_zone_index": final_payload["zone_index"],
        "final_target": final_payload["target"],

        "eval_sequence_values": eval_payload["sequence_values"],
        "eval_sequence_masks": eval_payload["sequence_masks"],
        "eval_context": eval_payload["context"],
        "eval_zone_index": eval_payload["zone_index"],
        "eval_target": eval_payload["target"],

        "eval_keys": eval_keys,
        "benchmark_prediction": pd.to_numeric(
            evaluation_df[baseline_column], errors="raise"
        ).to_numpy(dtype=np.float32),
    }


def get_transformer_fit_payload(logical_row, representation, profile):
    """Provide one stable notebook entry point for payload construction."""
    return build_transformer_fit_payload(logical_row, representation, profile)

**Recover companion artifacts and enforce common support.**
Restartability requires recovering more than a one-row score summary. Completed logical fits must bring back their **row-level predictions** and **epoch curves**, because later candidate comparisons and diagnostics depend on those artifacts.

Candidate accuracy is then recomputed on **exact common future rows** within each sentinel. This is the forecasting equivalent of giving every candidate the same exam questions: a model cannot win simply because it happened to be evaluated on an easier subset.

In [24]:
# ---------------------------------------------------------------------
# 4.5.2.3.0G — Recover companion artifacts and enforce common support
# ---------------------------------------------------------------------

def _existing_artifact_paths(results_df, column):
    """Return recorded companion-artifact paths that still exist on disk."""
    if results_df.empty or column not in results_df.columns:
        return []

    paths = results_df[column].dropna().astype(str).map(Path)
    return [path for path in paths if path.exists()]


def load_transformer_prediction_artifacts(results_df):
    """Reload row-level forecasts needed for exact-support comparisons."""
    paths = _existing_artifact_paths(results_df, "prediction_path")
    frames = [pd.read_parquet(path) for path in paths]

    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


def load_transformer_curve_artifacts(results_df):
    """Reload saved selector learning curves without retraining models."""
    paths = _existing_artifact_paths(results_df, "curve_path")
    frames = [pd.read_parquet(path) for path in paths]

    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


def common_support_result_table(results_df, prediction_df):
    """Recompute candidate scores using identical future rows per sentinel."""
    if prediction_df.empty:
        raise ValueError("Exact-support ranking requires saved prediction rows.")

    support_keys = [
        "forecast_sequence_id",
        "origin_observation_sequence_id",
        "target_observation_sequence_id",
        "target_date",
    ]
    sentinel_keys = ["track", "metric", "horizon", "temporal_window"]
    result_keys = sentinel_keys + ["candidate_id"]

    required = set(
        support_keys
        + result_keys
        + ["target_observed_value", "prediction_raw", "benchmark_prediction"]
    )
    missing = sorted(required - set(prediction_df.columns))

    if missing:
        raise ValueError(f"Prediction table lacks exact-support fields: {missing}")

    rows = []

    for sentinel, sentinel_df in prediction_df.groupby(
        sentinel_keys, dropna=False
    ):
        support_sets = []

        for candidate_id, candidate_df in sentinel_df.groupby("candidate_id"):
            if candidate_df.duplicated(support_keys).any():
                raise ValueError(
                    f"Duplicate future-row keys for candidate {candidate_id}."
                )

            support_sets.append(set(map(
                tuple, candidate_df[support_keys].to_numpy()
            )))

        common_support = (
            set.intersection(*support_sets) if support_sets else set()
        )

        if not common_support:
            raise ValueError(f"No exact common support for sentinel {sentinel}.")

        common_index = pd.MultiIndex.from_tuples(
            common_support, names=support_keys
        )

        for candidate_id, candidate_df in sentinel_df.groupby("candidate_id"):
            aligned = candidate_df.set_index(support_keys)
            aligned = aligned.loc[
                aligned.index.intersection(common_index)
            ].copy()

            if len(aligned) != len(common_support):
                raise ValueError(
                    f"Candidate {candidate_id} misses common-support rows."
                )

            raw_mae = regression_metrics(
                aligned["target_observed_value"], aligned["prediction_raw"]
            )["mae"]

            benchmark_mae = regression_metrics(
                aligned["target_observed_value"], aligned["benchmark_prediction"]
            )["mae"]

            benchmark_skill_pct = (
                100.0 * (benchmark_mae - raw_mae) / benchmark_mae
                if benchmark_mae > 0
                else np.nan
            )

            rows.append({
                "track": sentinel[0],
                "metric": sentinel[1],
                "horizon": int(sentinel[2]),
                "temporal_window": sentinel[3],
                "candidate_id": candidate_id,
                "raw_mae": raw_mae,
                "benchmark_mae": benchmark_mae,
                "benchmark_skill_pct": benchmark_skill_pct,
                "common_support_rows": len(common_support),
            })

    common_results_df = pd.DataFrame(rows)

    # Runtime, model size, and selected duration describe a whole logical fit.
    # They are joined only after prediction error has been recomputed on the
    # exact shared future population.
    if not results_df.empty:
        metadata_columns = [
            column
            for column in [
                *result_keys,
                "runtime_seconds",
                "parameter_count",
                "peak_gpu_memory_mb",
                "selected_epoch",
            ]
            if column in results_df.columns
        ]

        metadata = results_df[metadata_columns].drop_duplicates(result_keys)

        common_results_df = common_results_df.merge(
            metadata,
            on=result_keys,
            how="left",
            validate="one_to_one",
        )

    return common_results_df

**Run restartable batches and instantiate Batch A.**
The shared runner checks durable evidence **before doing any expensive data preparation or training**. If a logical fit already has a valid checkpoint, predictions, learning curve, and saved state under the current contract, those artifacts are reloaded immediately. Only genuinely unfinished fits proceed to payload construction and model training.

Batch A then instantiates the first 20-fit experiment: five sentinels × four temporal representations, all evaluated in the middle historical window. The 140-step maximum lookback is attached only as the common outer-comparison requirement; each candidate keeps its own legal training history.

In [25]:
# ---------------------------------------------------------------------
# 4.5.2.3.0H — Run restartable batches and instantiate Batch A
# ---------------------------------------------------------------------

def run_transformer_batch(roster_df, candidate_lookup, output_dir):
    """Recover valid logical fits first, then train only unfinished work."""
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    result_rows = []
    prediction_frames = []
    curve_frames = []

    for index, logical_row in roster_df.iterrows():
        label = (
            f"Batch {logical_row['batch']} · {logical_row['track']} · "
            f"{logical_row['metric']} · h={int(logical_row['horizon'])} · "
            f"{logical_row['candidate_id']} · {logical_row['temporal_window']}"
        )
        progress_message(index + 1, len(roster_df), label)

        logical_key = {
            column: logical_row[column]
            for column in TRANSFORMER_LOGICAL_KEY_COLUMNS
        }
        token = logical_key_token(logical_row, TRANSFORMER_LOGICAL_KEY_COLUMNS)

        checkpoint_path = output_dir / f"fit_{token}.parquet"
        prediction_path = output_dir / f"prediction_{token}.parquet"
        curve_path = output_dir / f"curve_{token}.parquet"
        state_path = output_dir / f"state_{token}.pt"

        # Recovery comes first. A completed fit should cost artifact reads—not
        # another large Parquet load, payload reconstruction, or GPU training run.
        recovered = recover_transformer_fit(
            checkpoint_path,
            logical_row["contract_id"],
            logical_key,
            prediction_path,
            curve_path,
            state_path,
        )

        if recovered is not None:
            result_rows.append(recovered)
            prediction_frames.append(pd.read_parquet(prediction_path))
            curve_frames.append(pd.read_parquet(curve_path))
            continue

        candidate_id = logical_row["candidate_id"]
        if candidate_id not in candidate_lookup:
            raise KeyError(
                f"No Transformer candidate definition for {candidate_id}."
            )

        candidate = candidate_lookup[candidate_id]
        representation = candidate["representation"]
        profile = candidate["profile"]

        payload = get_transformer_fit_payload(
            logical_row, representation, profile
        )

        result, predictions, curves = run_transformer_logical_fit(
            logical_row,
            representation,
            profile,
            payload,
            output_dir,
        )

        result_rows.append(result)
        prediction_frames.append(predictions)
        curve_frames.append(curves)

    results_df = pd.DataFrame(result_rows)

    # A batch is complete only if its durable success rows reproduce the exact
    # planned roster—no missing fits, unexpected extras, or silent duplicates.
    validate_exact_success_roster(
        results_df,
        roster_df,
        TRANSFORMER_LOGICAL_KEY_COLUMNS,
    )

    predictions_df = pd.concat(prediction_frames, ignore_index=True)
    curves_df = pd.concat(curve_frames, ignore_index=True)

    return results_df, predictions_df, curves_df


# ---------------------------------------------------------------------
# Batch A · 5 sentinels × 4 representations × middle window = 20 fits
# ---------------------------------------------------------------------

transformer_batch_a_roster_df = build_transformer_roster(
    "A",
    TRANSFORMER_REPRESENTATION_CANDIDATES,
    ["middle"],
)

# All four representations will be SCORED only where the longest representation
# is legal. This creates one fair outer population without forcing the shorter
# candidates to surrender earlier legal training history.
transformer_batch_a_roster_df["comparison_lookback_steps"] = max(
    candidate["lookback_steps"]
    for candidate in TRANSFORMER_REPRESENTATION_CANDIDATES
)

# These checks describe the experiment we intend to launch. Failing here is much
# cheaper—and much easier to diagnose—than discovering halfway through training
# that a sentinel or representation was omitted.
if len(transformer_batch_a_roster_df) != 20:
    raise ValueError("Batch A must contain exactly 20 logical fits.")

if transformer_batch_a_roster_df["candidate_id"].nunique() != 4:
    raise ValueError("Batch A must compare exactly four representations.")

if transformer_batch_a_roster_df["metric"].nunique() != 5:
    raise ValueError("Batch A must cover all five forecasting targets.")

if transformer_batch_a_roster_df["track"].nunique() != 2:
    raise ValueError("Batch A must preserve both modeling tracks.")

if set(transformer_batch_a_roster_df["horizon"]) != {1, 2, 5}:
    raise ValueError("Batch A must preserve horizons 1, 2, and 5.")

if not transformer_batch_a_roster_df.groupby("metric").size().eq(4).all():
    raise ValueError(
        "Every Batch A sentinel must evaluate all four representations."
    )

# ---------------------------------------------------------------------
# Reader-facing Batch A experiment summary
# ---------------------------------------------------------------------

track_labels = {
    "post_cp_primary": "Post-CP Primary",
    "full_history_regime_aware": "Full-History Regime-Aware",
}
target_labels = {
    "taxi_trip_count": "Taxi trips",
    "taxi_avg_trip_speed": "Taxi avg speed",
    "fhvhv_trip_count": "FHVHV trips",
    "fhvhv_avg_trip_speed": "FHVHV avg speed",
    "subway_ridership": "Subway ridership",
}

# The full 20-row roster remains available for audit. For the reader, collapse
# the four repeated candidate rows into the five forecasting problems they test.
transformer_batch_a_display_df = (
    transformer_batch_a_roster_df
    .groupby(
        [
            "track",
            "metric",
            "horizon",
            "temporal_window",
            "comparison_lookback_steps",
        ],
        as_index=False,
    )
    .agg(Representations=("candidate_id", "nunique"))
    .assign(
        Track=lambda df: df["track"].map(track_labels),
        Target=lambda df: df["metric"].map(target_labels),
        Window=lambda df: df["temporal_window"].str.title(),
        **{
            "Common scoring history": lambda df: (
                df["comparison_lookback_steps"].astype(int).astype(str)
                + " Dayparts"
            )
        },
    )
    [["Track", "Target", "horizon", "Representations", "Window", "Common scoring history"]]
    .rename(columns={"horizon": "Horizon"})
)

display(transformer_batch_a_display_df)

,Track,Target,Horizon,Representations,Window,Common scoring history
0,Full-History Regime-Aware,FHVHV avg speed,1,4,Middle,140 Dayparts
1,Full-History Regime-Aware,Taxi avg speed,5,4,Middle,140 Dayparts
2,Post-CP Primary,FHVHV trips,2,4,Middle,140 Dayparts
3,Post-CP Primary,Subway ridership,5,4,Middle,140 Dayparts
4,Post-CP Primary,Taxi trips,1,4,Middle,140 Dayparts


**Findings.** Batch A is set up as **20 planned Transformer fits: five representative forecasting problems × four history representations**. The five problems collectively cover all five mobility targets, both modeling tracks, and horizons 1, 2, and 5.

All four representations are scored on a common **140-Daypart eligibility surface**, corresponding to the longest four-week candidate. This is a fairness rule for evaluation, not a restriction on training: shorter representations still retain their own legally available earlier training history. The comparison therefore asks which way of presenting recent history forecasts the **same future rows** most accurately rather than letting candidates compete on different prediction populations.

### 4.5.2.3.1 Batch A — Screen Recent-History Representation

Batch A compares the four complete history/patch representations on five comprehensive sentinel jobs using one middle validation window. The retained representations are selected by within-sentinel MAE rank, sentinel wins, benchmark skill, and runtime.

In [26]:
# ---------------------------------------------------------------------
# 4.5.2.3.1A — Batch A representation screen
# ---------------------------------------------------------------------

transformer_representation_results_df = pd.DataFrame()
transformer_representation_predictions_df = pd.DataFrame()
transformer_representation_curves_df = pd.DataFrame()
transformer_representation_summary_df = pd.DataFrame()
transformer_retained_representation_registry_df = pd.DataFrame()

balanced_profile = next(
    profile
    for profile in TRANSFORMER_PROFILE_CANDIDATES
    if profile["profile_id"] == "balanced"
)

representation_lookup = {
    representation["representation_id"]: {
        "representation": representation,
        "profile": balanced_profile,
    }
    for representation in TRANSFORMER_REPRESENTATION_CANDIDATES
}


# ---------------------------------------------------------------------
# Recover completed work, then run only what is missing
# ---------------------------------------------------------------------

if RESET_TRANSFORMER_REPRESENTATION_SCREEN:
    # RESET is the explicit "start this batch over" control. With RESET=False,
    # valid logical-fit artifacts survive and are reused automatically.
    reset_experiment_artifacts(TRANSFORMER_REPRESENTATION_SCREEN_DIR)

# The runner always executes as notebook logic, but expensive fitting does not
# necessarily occur. Each of the 20 logical fits is checked on disk first:
#
#   valid saved fit  -> reload checkpoint + predictions + curve + state
#   missing/bad fit  -> build its payload and train only that fit
#
# Therefore an interrupted 17/20 run resumes with the remaining three fits.
(
    transformer_representation_results_df,
    transformer_representation_predictions_df,
    transformer_representation_curves_df,
) = run_transformer_batch(
    transformer_batch_a_roster_df,
    representation_lookup,
    TRANSFORMER_REPRESENTATION_SCREEN_DIR,
)


# ---------------------------------------------------------------------
# Rank representations without mixing incomparable native-unit MAEs
# ---------------------------------------------------------------------

if not transformer_representation_results_df.empty:
    screen_results = common_support_result_table(
        transformer_representation_results_df,
        transformer_representation_predictions_df,
    )

    screen_results["within_sentinel_rank"] = (
        screen_results.groupby(
            ["track", "metric", "horizon"]
        )["raw_mae"]
        .rank(method="min")
    )

    screen_results["sentinel_win"] = (
        screen_results["within_sentinel_rank"].eq(1)
    )

    runtime_summary = (
        transformer_representation_results_df
        .groupby("candidate_id", as_index=False)
        .agg(
            median_runtime_seconds=(
                "runtime_seconds",
                "median",
            )
        )
    )

    transformer_representation_summary_df = (
        screen_results
        .groupby("candidate_id", as_index=False)
        .agg(
            median_within_sentinel_rank=(
                "within_sentinel_rank",
                "median",
            ),
            sentinel_wins=(
                "sentinel_win",
                "sum",
            ),
            median_benchmark_skill_pct=(
                "benchmark_skill_pct",
                "median",
            ),
        )
        .merge(
            runtime_summary,
            on="candidate_id",
            how="left",
            validate="one_to_one",
        )
        .sort_values(
            [
                "median_within_sentinel_rank",
                "sentinel_wins",
                "median_benchmark_skill_pct",
                "median_runtime_seconds",
            ],
            ascending=[True, False, False, True],
        )
        .reset_index(drop=True)
    )

    # The first two representations advance. The ordering above implements the
    # precommitted cross-target hierarchy rather than averaging native MAE.
    transformer_retained_representation_registry_df = (
        transformer_representation_summary_df
        .head(2)
        .copy()
    )

    transformer_retained_representation_registry_df[
        "contract_id"
    ] = TRANSFORMER_REPRESENTATION_CONTRACT_ID


# ---------------------------------------------------------------------
# Reader-facing summary
# ---------------------------------------------------------------------

if not transformer_representation_summary_df.empty:
    representation_plot_df = transformer_representation_summary_df.copy()
    representation_plot_df["Representation"] = (
        representation_plot_df["candidate_id"]
        .str.replace("_", " ", regex=False)
        .str.title()
    )

    # Give each historical representation a stable project-brand color. The
    # layout helper alone cannot reliably recolor traces already created by
    # Plotly Express, so the categorical palette is supplied up front.
    fig = px.scatter(
        representation_plot_df,
        x="median_runtime_seconds",
        y="median_benchmark_skill_pct",
        color="Representation",
        text="Representation",
        color_discrete_sequence=BRAND_COLOR_SEQUENCE,
        hover_data={
            "median_within_sentinel_rank": ":.3f",
            "sentinel_wins": True,
            "median_runtime_seconds": ":.3f",
            "median_benchmark_skill_pct": ":.3f",
            "candidate_id": False,
        },
        labels={
            "median_runtime_seconds": "Median fit time (seconds)",
            "median_benchmark_skill_pct": "Median skill vs benchmark (%)",
        },
        title=(
            "Batch A · Representation accuracy versus runtime"
            "<br><sup>Higher benchmark skill is better; "
            "runtime is only the final tie-break.</sup>"
        ),
    )

    fig.update_traces(textposition="top center")
    fig = apply_branding(fig)

    fig.update_layout(
        legend={
            "title_text": "",
            "orientation": "h",
            "x": 0.5,
            "xanchor": "center",
            "y": -0.18,
            "yanchor": "top",
        },
        margin={"l": 80, "r": 40, "t": 85, "b": 125},
    )

    fig.show()

display(transformer_representation_summary_df.round(3))

[1/20] Batch A · post_cp_primary · taxi_trip_count · h=1 · one_week_daily · middle
[2/20] Batch A · post_cp_primary · taxi_trip_count · h=1 · two_week_daily · middle
[3/20] Batch A · post_cp_primary · taxi_trip_count · h=1 · four_week_daily · middle
[4/20] Batch A · post_cp_primary · taxi_trip_count · h=1 · two_week_overlapping_two_day · middle
[5/20] Batch A · full_history_regime_aware · taxi_avg_trip_speed · h=5 · one_week_daily · middle
[6/20] Batch A · full_history_regime_aware · taxi_avg_trip_speed · h=5 · two_week_daily · middle
[7/20] Batch A · full_history_regime_aware · taxi_avg_trip_speed · h=5 · four_week_daily · middle
[8/20] Batch A · full_history_regime_aware · taxi_avg_trip_speed · h=5 · two_week_overlapping_two_day · middle
[9/20] Batch A · post_cp_primary · fhvhv_trip_count · h=2 · one_week_daily · middle
[10/20] Batch A · post_cp_primary · fhvhv_trip_count · h=2 · two_week_daily · middle
[11/20] Batch A · post_cp_primary · fhvhv_trip_count · h=2 · four_week_daily · mi

,candidate_id,median_within_sentinel_rank,sentinel_wins,median_benchmark_skill_pct,median_runtime_seconds
0,two_week_overlapping_two_day,2.000,2,1.315,104.161
1,two_week_daily,3.000,1,-2.488,128.020
2,four_week_daily,3.000,0,-1.867,113.922
3,one_week_daily,4.000,2,6.651,96.814


**Findings.** Batch A favors the **two-week overlapping two-day representation**, which achieved the best overall median within-sentinel rank at 2.000, won 2 of the 5 sentinel problems, and produced +1.315% median skill relative to the frozen weekly benchmark.

The result does not support the idea that simply giving the Transformer more history is better. Four-week daily history won no sentinel problems, while the shorter one-week representation actually won two individual sentinels despite ranking less consistently overall.

The second advancement slot is therefore less decisive than the first. Two-week daily history advances under the precommitted ranking hierarchy because its performance is more consistently ranked across the five sentinels, while one-week history shows stronger results on some individual problems. The clearest conclusion is that **overlapping two-week history is the strongest representation**, not that longer lookbacks universally improve forecasting.

### 4.5.2.3.2 Batch B — Screen Complete Transformer Profiles

Batch B evaluates four complete Transformer specifications on the same five sentinels and middle window: the best representation paired with each of the three profiles, plus the runner-up representation paired with the balanced profile. It does not create independent one-variable ablations.

In [27]:
# ---------------------------------------------------------------------
# 4.5.2.3.2A — Batch B Transformer-profile screen
# ---------------------------------------------------------------------

transformer_profile_results_df = pd.DataFrame()
transformer_profile_predictions_df = pd.DataFrame()
transformer_profile_curves_df = pd.DataFrame()
transformer_profile_summary_df = pd.DataFrame()
transformer_retained_profile_registry_df = pd.DataFrame()
transformer_batch_b_roster_df = pd.DataFrame()

if len(transformer_retained_representation_registry_df) == 2:
    retained_representation_ids = transformer_retained_representation_registry_df[
        "candidate_id"
    ].tolist()
    best_representation_id, runner_up_representation_id = retained_representation_ids

    profile_by_id = {
        profile["profile_id"]: profile
        for profile in TRANSFORMER_PROFILE_CANDIDATES
    }

    required_profile_ids = {
        "compact",
        "balanced",
        "higher_capacity_regularized",
    }
    missing_profiles = sorted(required_profile_ids - set(profile_by_id))

    if missing_profiles:
        raise ValueError(
            f"Batch B is missing required Transformer profiles: {missing_profiles}"
        )

    # Exactly four coherent specifications enter Batch B. Capacity settings move
    # together as precommitted profiles; this is not a Cartesian hyperparameter sweep.
    profile_specs = [
        (best_representation_id, profile_by_id["compact"]),
        (best_representation_id, profile_by_id["balanced"]),
        (best_representation_id, profile_by_id["higher_capacity_regularized"]),
        (runner_up_representation_id, profile_by_id["balanced"]),
    ]

    transformer_batch_b_roster_df = pd.DataFrame([
        {
            "batch": "B",
            "track": track,
            "metric": metric,
            "horizon": horizon,
            "candidate_id": f"{representation_id}__{profile['profile_id']}",
            "representation_id": representation_id,
            "profile_id": profile["profile_id"],
            "temporal_window": "middle",
            "contract_id": TRANSFORMER_PROFILE_CONTRACT_ID,
            "status": "planned",
        }
        for track, metric, horizon in TRANSFORMER_SENTINEL_KEYS
        for representation_id, profile in profile_specs
    ])

    if len(transformer_batch_b_roster_df) != 20:
        raise ValueError("Batch B must contain exactly 20 logical fits.")
    if transformer_batch_b_roster_df["candidate_id"].nunique() != 4:
        raise ValueError("Batch B must compare exactly four complete specifications.")
    if transformer_batch_b_roster_df["metric"].nunique() != 5:
        raise ValueError("Batch B must contain all five targets.")
    if transformer_batch_b_roster_df["track"].nunique() != 2:
        raise ValueError("Batch B must preserve both modeling tracks.")
    if set(transformer_batch_b_roster_df["horizon"]) != {1, 2, 5}:
        raise ValueError("Batch B must preserve horizons 1, 2, and 5.")
    if not transformer_batch_b_roster_df.groupby("metric").size().eq(4).all():
        raise ValueError(
            "Every Batch B sentinel must evaluate exactly four specifications."
        )

    representation_by_id = {
        spec["representation_id"]: spec
        for spec in TRANSFORMER_REPRESENTATION_CANDIDATES
    }
    profile_lookup = {
        f"{representation_id}__{profile['profile_id']}": {
            "representation": representation_by_id[representation_id],
            "profile": profile,
        }
        for representation_id, profile in profile_specs
    }

    # -------------------------------------------------------------
    # Recover completed work, then run only what is missing
    # -------------------------------------------------------------

    if RESET_TRANSFORMER_PROFILE_SCREEN:
        reset_experiment_artifacts(TRANSFORMER_PROFILE_SCREEN_DIR)

    # Batch A has already identified the two representations that are allowed to
    # enter this stage. Batch B can therefore proceed immediately in the same
    # notebook run; no additional human approval is required between A and B.
    #
    # Existing valid profile-screen fits are reloaded. Only specifications without
    # complete durable artifacts are trained.
    (
        transformer_profile_results_df,
        transformer_profile_predictions_df,
        transformer_profile_curves_df,
    ) = run_transformer_batch(
        transformer_batch_b_roster_df,
        profile_lookup,
        TRANSFORMER_PROFILE_SCREEN_DIR,
    )

    # -------------------------------------------------------------
    # Rank complete specifications without mixing target units
    # -------------------------------------------------------------
    if not transformer_profile_results_df.empty:
        profile_common_results = common_support_result_table(
            transformer_profile_results_df,
            transformer_profile_predictions_df,
        )

        profile_common_results["within_sentinel_rank"] = (
            profile_common_results.groupby(
                ["track", "metric", "horizon"]
            )["raw_mae"].rank(method="min")
        )
        profile_common_results["sentinel_win"] = (
            profile_common_results["within_sentinel_rank"].eq(1)
        )
        profile_common_results["benchmark_win"] = (
            profile_common_results["raw_mae"].lt(
                profile_common_results["benchmark_mae"]
            )
        )

        transformer_profile_summary_df = (
            profile_common_results.groupby("candidate_id", as_index=False)
            .agg(
                median_within_sentinel_rank=("within_sentinel_rank", "median"),
                sentinel_wins=("sentinel_win", "sum"),
                median_benchmark_skill_pct=("benchmark_skill_pct", "median"),
                benchmark_win_share=("benchmark_win", "mean"),
                median_runtime_seconds=("runtime_seconds", "median"),
            )
            .sort_values(
                [
                    "median_within_sentinel_rank",
                    "sentinel_wins",
                    "median_benchmark_skill_pct",
                    "benchmark_win_share",
                    "median_runtime_seconds",
                ],
                ascending=[True, False, False, False, True],
            )
            .reset_index(drop=True)
        )

        # The first two complete specifications advance to temporal confirmation.
        transformer_retained_profile_registry_df = (
            transformer_profile_summary_df.head(2).copy()
        )
        transformer_retained_profile_registry_df[
            "contract_id"
        ] = TRANSFORMER_PROFILE_CONTRACT_ID

else:
    print(
        "Batch B waiting: Batch A must retain exactly two representations first."
    )


# ---------------------------------------------------------------------
# Reader-facing summary
# ---------------------------------------------------------------------

if not transformer_profile_summary_df.empty:
    profile_plot_df = transformer_profile_summary_df.copy()
    profile_plot_df["Transformer specification"] = (
        profile_plot_df["candidate_id"]
        .str.replace("__", " · ", regex=False)
        .str.replace("_", " ", regex=False)
        .str.title()
    )

    # Batch B compares four complete Transformer specifications. Supplying the
    # shared categorical sequence here keeps their colors inside the project
    # palette rather than inheriting Plotly's default blue/red/green scheme.
    fig = px.scatter(
        profile_plot_df,
        x="median_runtime_seconds",
        y="median_benchmark_skill_pct",
        color="Transformer specification",
        text="Transformer specification",
        color_discrete_sequence=BRAND_COLOR_SEQUENCE,
        hover_data={
            "median_within_sentinel_rank": ":.3f",
            "sentinel_wins": True,
            "benchmark_win_share": ":.3f",
            "median_runtime_seconds": ":.3f",
            "median_benchmark_skill_pct": ":.3f",
            "candidate_id": False,
        },
        labels={
            "median_runtime_seconds": "Median fit time (seconds)",
            "median_benchmark_skill_pct": "Median skill vs benchmark (%)",
        },
        title=(
            "Batch B · Complete Transformer specifications"
            "<br><sup>Accuracy ranks first; benchmark consistency and runtime "
            "break progressively narrower ties.</sup>"
        ),
    )

    fig.update_traces(textposition="top center")
    fig = apply_branding(fig)

    fig.update_layout(
        legend={
            "title_text": "",
            "orientation": "h",
            "x": 0.5,
            "xanchor": "center",
            "y": -0.18,
            "yanchor": "top",
        },
        margin={"l": 80, "r": 40, "t": 85, "b": 145},
    )

    fig.show()

display(transformer_profile_summary_df.round(3))

[1/20] Batch B · post_cp_primary · taxi_trip_count · h=1 · two_week_overlapping_two_day__compact · middle
[2/20] Batch B · post_cp_primary · taxi_trip_count · h=1 · two_week_overlapping_two_day__balanced · middle
[3/20] Batch B · post_cp_primary · taxi_trip_count · h=1 · two_week_overlapping_two_day__higher_capacity_regularized · middle
[4/20] Batch B · post_cp_primary · taxi_trip_count · h=1 · two_week_daily__balanced · middle
[5/20] Batch B · full_history_regime_aware · taxi_avg_trip_speed · h=5 · two_week_overlapping_two_day__compact · middle
[6/20] Batch B · full_history_regime_aware · taxi_avg_trip_speed · h=5 · two_week_overlapping_two_day__balanced · middle
[7/20] Batch B · full_history_regime_aware · taxi_avg_trip_speed · h=5 · two_week_overlapping_two_day__higher_capacity_regularized · middle
[8/20] Batch B · full_history_regime_aware · taxi_avg_trip_speed · h=5 · two_week_daily__balanced · middle
[9/20] Batch B · post_cp_primary · fhvhv_trip_count · h=2 · two_week_overlapping

,candidate_id,median_within_sentinel_rank,sentinel_wins,median_benchmark_skill_pct,benchmark_win_share,median_runtime_seconds
0,two_week_overlapping_two_day__higher_capacity_...,1.000,3,2.613,0.600,142.764
1,two_week_overlapping_two_day__compact,2.000,1,3.832,0.800,101.907
2,two_week_overlapping_two_day__balanced,3.000,0,-3.382,0.400,100.932
3,two_week_daily__balanced,4.000,1,5.742,0.600,78.074


**Findings.** Batch B reinforces the representation result: both surviving complete Transformer specifications use the **two-week overlapping two-day history**.

The higher-capacity regularized Transformer ranks first overall and wins 3 of the 5 sentinel problems. The compact Transformer ranks second and beats the frozen benchmark on 4 of 5 sentinels. The balanced profile does not justify an intermediate position between them and is eliminated.

The remaining modeling question is therefore no longer how history should be patched. It is whether an individual mobility target benefits enough from the larger Transformer to justify its extra computational cost.

### 4.5.2.3.3 Batch C — Confirm the Finalists Through Time

Batch C asks whether the two strongest complete Transformer specifications remain competitive when validation moves through early, middle, and late legal historical windows. It proposes one Transformer profile per target without creating horizon-specific settings.

**4.5.2.3.3A — Build the Batch C temporal-confirmation roster.**

Batches A and B narrowed the tournament from four historical representations to **two complete Transformer finalists**. Batch C no longer changes those models. Its job is to ask whether the same two finalists remain competitive at different points in the historical validation period.

Each of the five sentinel forecasting problems is therefore evaluated with both finalists in three 28-day windows: **early, middle, and late**. This produces **5 sentinels × 2 finalists × 3 windows = 30 logical fits**.

This cell only constructs and validates that 30-fit experiment plan. It also resolves each finalist ID back to its approved history representation and Transformer profile so the shared training engine knows exactly what architecture each ID represents. No model fitting occurs here.

In [28]:
# ---------------------------------------------------------------------
# 4.5.2.3.3A — Build the Batch C temporal-confirmation roster
# ---------------------------------------------------------------------

transformer_batch_c_roster_df = pd.DataFrame()
finalist_lookup = {}

if len(transformer_retained_profile_registry_df) != 2:
    print(
        "Batch C waiting: Batch B must retain exactly two complete "
        "Transformer specifications first."
    )

else:
    finalist_ids = transformer_retained_profile_registry_df["candidate_id"].tolist()

    # Reuse the shared roster builder rather than reconstructing the same
    # Track × Target × Horizon expansion by hand. Batch C simply adds three
    # historical windows to each of the two surviving specifications.
    transformer_batch_c_roster_df = build_transformer_roster(
        batch="C",
        candidates=[{"spec_id": finalist_id} for finalist_id in finalist_ids],
        temporal_windows=["early", "middle", "late"],
    )

    # These checks describe the experiment we intend to run. If one finalist,
    # target, or temporal window disappears, failing here is far cheaper than
    # discovering an incomplete tournament after expensive fitting.
    if len(transformer_batch_c_roster_df) != 30:
        raise ValueError("Batch C must contain exactly 30 logical fits.")

    if transformer_batch_c_roster_df["candidate_id"].nunique() != 2:
        raise ValueError("Batch C must compare exactly two finalist specifications.")

    if transformer_batch_c_roster_df["temporal_window"].nunique() != 3:
        raise ValueError("Batch C must cover early, middle, and late windows.")

    if transformer_batch_c_roster_df["metric"].nunique() != 5:
        raise ValueError("Batch C must preserve all five sentinel targets.")

    if not transformer_batch_c_roster_df.groupby("metric").size().eq(6).all():
        raise ValueError(
            "Every Batch C target must contribute 2 finalists × 3 windows = 6 fits."
        )

    representation_lookup = {
        spec["representation_id"]: spec
        for spec in TRANSFORMER_REPRESENTATION_CANDIDATES
    }
    profile_lookup = {
        spec["profile_id"]: spec
        for spec in TRANSFORMER_PROFILE_CANDIDATES
    }

    # Candidate IDs encode the complete specification as
    # <representation>__<profile>. Resolve that compact label once here so
    # downstream fitting code does not repeatedly rediscover model settings.
    for finalist_id in finalist_ids:
        candidate_parts = finalist_id.split("__", maxsplit=1)

        if len(candidate_parts) != 2:
            raise ValueError(
                f"Batch C finalist ID is malformed: {finalist_id}"
            )

        representation_id, profile_id = candidate_parts

        if representation_id not in representation_lookup:
            raise ValueError(
                f"Unknown Batch C representation: {representation_id}"
            )

        if profile_id not in profile_lookup:
            raise ValueError(f"Unknown Batch C Transformer profile: {profile_id}")

        finalist_lookup[finalist_id] = {
            "representation": representation_lookup[representation_id],
            "profile": profile_lookup[profile_id],
        }

display(transformer_batch_c_roster_df.round(3))

,batch,track,metric,horizon,candidate_id,temporal_window,contract_id,status
0,C,post_cp_primary,taxi_trip_count,1,two_week_overlapping_two_day__higher_capacity_...,early,transformer_confirmation_screen_v1,planned
1,C,post_cp_primary,taxi_trip_count,1,two_week_overlapping_two_day__higher_capacity_...,middle,transformer_confirmation_screen_v1,planned
2,C,post_cp_primary,taxi_trip_count,1,two_week_overlapping_two_day__higher_capacity_...,late,transformer_confirmation_screen_v1,planned
3,C,post_cp_primary,taxi_trip_count,1,two_week_overlapping_two_day__compact,early,transformer_confirmation_screen_v1,planned
4,C,post_cp_primary,taxi_trip_count,1,two_week_overlapping_two_day__compact,middle,transformer_confirmation_screen_v1,planned
5,C,post_cp_primary,taxi_trip_count,1,two_week_overlapping_two_day__compact,late,transformer_confirmation_screen_v1,planned
6,C,full_history_regime_aware,taxi_avg_trip_speed,5,two_week_overlapping_two_day__higher_capacity_...,early,transformer_confirmation_screen_v1,planned
7,C,full_history_regime_aware,taxi_avg_trip_speed,5,two_week_overlapping_two_day__higher_capacity_...,middle,transformer_confirmation_screen_v1,planned
8,C,full_history_regime_aware,taxi_avg_trip_speed,5,two_week_overlapping_two_day__higher_capacity_...,late,transformer_confirmation_screen_v1,planned
9,C,full_history_regime_aware,taxi_avg_trip_speed,5,two_week_overlapping_two_day__compact,early,transformer_confirmation_screen_v1,planned


**Findings.** Batch C produces a **target-specific capacity split** rather than one universally best Transformer.

The higher-capacity specification is proposed for **Taxi trips** and **FHVHV average speed**. The compact specification is proposed for **Taxi average speed, FHVHV trips, and Subway ridership**.

The strongest Transformer improves on the frozen weekly benchmark by **14.894% for Taxi average speed, 13.802% for FHVHV average speed, and 3.013% for FHVHV trips**. Taxi trips remains **1.589% worse than the benchmark**, while Subway ridership remains **36.105% worse** even with its stronger Transformer finalist.

Taxi average speed is especially close: compact achieves a pooled MAE of 2.011 versus 2.019 for the higher-capacity model, a difference of only about 0.4%. Because compact is also substantially smaller, there is little practical reason to choose the larger model for that target.

The pooled MAEs combine exact-common-support errors from all three confirmation windows. This proves that early, middle, and late evidence all contribute to the selection; it does **not by itself prove that the same finalist wins every individual window**, so window-specific performance should remain visible as supporting temporal-stability evidence.

**4.5.2.3.3B — Run or recover the 30 Batch C confirmation fits.**

Once the two finalists exist, there is no additional modeling decision to make before temporal confirmation. Batch C can therefore start automatically in the same notebook run.

The shared runner checks each logical fit against its saved checkpoint, row-level predictions, learning curve, and fitted model state **before doing any expensive work**. A valid completed fit is simply reloaded. A missing or invalid fit is recomputed and saved immediately.

`RESET_TRANSFORMER_CONFIRMATION_SCREEN=False` therefore means **reuse completed work and fill any gaps**. Setting it to `True` deliberately discards the current Batch C artifacts and starts this stage over.

In [29]:
# ---------------------------------------------------------------------
# 4.5.2.3.3B — Run or recover the 30 Batch C confirmation fits
# ---------------------------------------------------------------------

transformer_confirmation_results_df = pd.DataFrame()
transformer_confirmation_predictions_df = pd.DataFrame()
transformer_confirmation_curves_df = pd.DataFrame()

if transformer_batch_c_roster_df.empty:
    print("Batch C execution waiting: the finalist roster is not available.")

else:
    if RESET_TRANSFORMER_CONFIRMATION_SCREEN:
        # RESET means "intentionally recompute this experiment." With RESET
        # off, valid fit-level artifacts are preserved and reused automatically.
        reset_experiment_artifacts(TRANSFORMER_CONFIRMATION_SCREEN_DIR)

    # run_transformer_batch() always reconciles the requested roster with the
    # durable artifacts first. In a fresh run this trains 30 fits; after a
    # partial run it trains only the unfinished fits; after completion it only
    # reloads the saved evidence.
    (
        transformer_confirmation_results_df,
        transformer_confirmation_predictions_df,
        transformer_confirmation_curves_df,
    ) = run_transformer_batch(
        transformer_batch_c_roster_df,
        finalist_lookup,
        TRANSFORMER_CONFIRMATION_SCREEN_DIR,
    )

[1/30] Batch C · post_cp_primary · taxi_trip_count · h=1 · two_week_overlapping_two_day__higher_capacity_regularized · early
[2/30] Batch C · post_cp_primary · taxi_trip_count · h=1 · two_week_overlapping_two_day__higher_capacity_regularized · middle
[3/30] Batch C · post_cp_primary · taxi_trip_count · h=1 · two_week_overlapping_two_day__higher_capacity_regularized · late
[4/30] Batch C · post_cp_primary · taxi_trip_count · h=1 · two_week_overlapping_two_day__compact · early
[5/30] Batch C · post_cp_primary · taxi_trip_count · h=1 · two_week_overlapping_two_day__compact · middle
[6/30] Batch C · post_cp_primary · taxi_trip_count · h=1 · two_week_overlapping_two_day__compact · late
[7/30] Batch C · full_history_regime_aware · taxi_avg_trip_speed · h=5 · two_week_overlapping_two_day__higher_capacity_regularized · early
[8/30] Batch C · full_history_regime_aware · taxi_avg_trip_speed · h=5 · two_week_overlapping_two_day__higher_capacity_regularized · middle
[9/30] Batch C · full_history_r

**4.5.2.3.3C — Pool temporal evidence and propose one finalist per target.**

Batch C is not asking which finalist wins a single convenient month. It asks which finalist makes the smaller errors **across the early, middle, and late confirmation periods together**.

Within each historical window, the two finalists are first rescored on the exact same future rows. The window-level MAE is then converted back into total absolute error by multiplying MAE by the number of common rows. Summing those absolute errors across all three windows and dividing by the combined row count is mathematically equivalent to pooling the row-level absolute errors and computing MAE once.

The lower pooled raw MAE becomes the **proposed finalist for that mobility target**. This is still a proposal rather than a frozen downstream decision; the remaining diagnostics provide evidence for the decision point that follows.

In [30]:
# ---------------------------------------------------------------------
# 4.5.2.3.3C — Pool temporal evidence and propose one finalist per target
# ---------------------------------------------------------------------

transformer_target_profile_evidence_df = pd.DataFrame()
transformer_proposed_target_profile_registry_df = pd.DataFrame()

if transformer_confirmation_results_df.empty:
    print("Batch C proposal waiting: confirmation results are not available.")

else:
    # First force the two finalists onto identical future rows within every
    # early/middle/late sentinel. This prevents a model from winning merely
    # because it happened to be scored on an easier subset.
    confirmation_common_results = common_support_result_table(
        transformer_confirmation_results_df,
        transformer_confirmation_predictions_df,
    )

    # MAE × row count recovers the total absolute error represented by each
    # window. Those totals can then be pooled without averaging already-
    # aggregated MAEs or mixing unequal support sizes.
    confirmation_common_results["raw_absolute_error"] = (
        confirmation_common_results["raw_mae"]
        * confirmation_common_results["common_support_rows"]
    )
    confirmation_common_results["benchmark_absolute_error"] = (
        confirmation_common_results["benchmark_mae"]
        * confirmation_common_results["common_support_rows"]
    )

    transformer_target_profile_evidence_df = (
        confirmation_common_results
        .groupby(["metric", "candidate_id"], as_index=False)
        .agg(
            pooled_raw_error=("raw_absolute_error", "sum"),
            pooled_benchmark_error=("benchmark_absolute_error", "sum"),
            pooled_support_rows=("common_support_rows", "sum"),
            temporal_windows=("temporal_window", "nunique"),
        )
    )

    evidence = transformer_target_profile_evidence_df

    # Each target/finalist must contribute all three confirmation windows.
    # Otherwise its pooled MAE would summarize a different temporal surface.
    if not evidence["temporal_windows"].eq(3).all():
        raise ValueError(
            "Every Batch C target/finalist must contribute all three temporal windows."
        )

    evidence["pooled_raw_mae"] = (
        evidence["pooled_raw_error"] / evidence["pooled_support_rows"]
    )
    evidence["pooled_benchmark_mae"] = (
        evidence["pooled_benchmark_error"] / evidence["pooled_support_rows"]
    )
    evidence["pooled_benchmark_skill_pct"] = (
        100.0
        * (evidence["pooled_benchmark_mae"] - evidence["pooled_raw_mae"])
        / evidence["pooled_benchmark_mae"]
    )

    # The target proposal uses raw MAE because architecture selection should
    # reflect the network's actual numerical behavior before the downstream
    # nonnegative output rule is applied.
    evidence["target_winner"] = (
        evidence["pooled_raw_mae"]
        .eq(evidence.groupby("metric")["pooled_raw_mae"].transform("min"))
    )

    transformer_proposed_target_profile_registry_df = evidence.loc[
        evidence["target_winner"]
    ].copy()

    # No tie-break was precommitted for an exact pooled-MAE tie. If one occurs,
    # stop rather than silently inventing a new selection rule after seeing data.
    winner_counts = (
        transformer_proposed_target_profile_registry_df
        .groupby("metric")
        .size()
    )

    if not winner_counts.reindex(FORECAST_TARGETS, fill_value=0).eq(1).all():
        raise ValueError(
            "Batch C produced an unresolved pooled-MAE tie or missing target proposal."
        )

display(transformer_target_profile_evidence_df.round(3))

,metric,candidate_id,pooled_raw_error,pooled_benchmark_error,pooled_support_rows,temporal_windows,pooled_raw_mae,pooled_benchmark_mae,pooled_benchmark_skill_pct,target_winner
0,fhvhv_avg_trip_speed,two_week_overlapping_two_day__compact,"135,586.707","150,440.359",107742,3,1.258,1.396,9.873,False
1,fhvhv_avg_trip_speed,two_week_overlapping_two_day__higher_capacity_...,"129,677.133","150,440.359",107742,3,1.204,1.396,13.802,True
2,fhvhv_trip_count,two_week_overlapping_two_day__compact,"8,174,886.510","8,428,857.448",107819,3,75.820,78.176,3.013,True
3,fhvhv_trip_count,two_week_overlapping_two_day__higher_capacity_...,"8,504,859.733","8,428,857.448",107819,3,78.881,78.176,-0.902,False
4,subway_ridership,two_week_overlapping_two_day__compact,"49,585,682.853","36,431,910.595",62548,3,792.762,582.463,-36.105,True
5,subway_ridership,two_week_overlapping_two_day__higher_capacity_...,"51,428,617.292","36,431,910.595",62548,3,822.226,582.463,-41.164,False
6,taxi_avg_trip_speed,two_week_overlapping_two_day__compact,"103,626.776","121,761.801",51534,3,2.011,2.363,14.894,True
7,taxi_avg_trip_speed,two_week_overlapping_two_day__higher_capacity_...,"104,048.733","121,761.801",51534,3,2.019,2.363,14.547,False
8,taxi_trip_count,two_week_overlapping_two_day__compact,"2,613,630.805","2,451,819.558",107819,3,24.241,22.740,-6.600,False
9,taxi_trip_count,two_week_overlapping_two_day__higher_capacity_...,"2,490,787.582","2,451,819.558",107819,3,23.102,22.740,-1.589,True


**Findings.** Batch C pools prediction errors across the **early, middle, and late** confirmation windows after forcing both finalists onto the exact same future rows. The resulting **pooled MAE** is the average size of the model's miss across those combined rows, expressed in each target's native units. Lower is better.

The higher-capacity Transformer is proposed for **Taxi trips** and **FHVHV average speed**. The compact Transformer is proposed for **Taxi average speed, FHVHV trips, and Subway ridership**.

The benchmark-skill column puts those errors into a more intuitive frame: **positive skill means the Transformer improves on the frozen “previous week, same Daypart” benchmark; negative skill means it performs worse**. The selected Transformers improve on that benchmark by **14.894% for Taxi average speed, 13.802% for FHVHV average speed, and 3.013% for FHVHV trips**. Taxi trips remains **1.589% worse than the benchmark**, while Subway ridership remains substantially worse at **−36.105%**.

Taxi average speed is essentially a capacity tie: compact records a pooled MAE of **2.011** versus **2.019** for higher capacity. FHVHV average speed shows a clearer benefit from the larger model, while FHVHV trips and Subway favor compact. These results support **target-specific model capacity rather than one Transformer size for every mobility outcome**.

**4.5.2.3.3D — Visualize whether the finalists remain competitive through time.**

The pooled Batch C result determines which finalist advances for each mobility target, but pooling can hide *when* one model was stronger or weaker. This view therefore keeps the **early, middle, and late** confirmation windows separate.

Because Taxi trips, speed, FHVHV demand, and Subway ridership have very different numerical scales, their raw MAEs should not be compared directly across targets. Instead, each cell shows the model's **MAE penalty relative to the better finalist for that same Target × Window**.

A value of **0%** means that finalist had the lower raw MAE in that historical window. Positive values show how much worse it was than the competing finalist. A robust choice should remain close to the window winner even when it does not win every period.


In [31]:
# ---------------------------------------------------------------------
# 4.5.2.3.3D — Visualize finalist robustness across temporal windows
# ---------------------------------------------------------------------

transformer_confirmation_robustness_df = pd.DataFrame()

if confirmation_common_results.empty:
    print("Batch C robustness view waiting: common-support results are unavailable.")

else:
    transformer_confirmation_robustness_df = confirmation_common_results.copy()

    # Raw MAE cannot be compared across mobility targets because each target
    # uses different units. Measure each finalist against the better model
    # within the SAME target and historical window instead.
    best_window_mae = (
        transformer_confirmation_robustness_df
        .groupby(["metric", "temporal_window"])["raw_mae"]
        .transform("min")
    )

    transformer_confirmation_robustness_df["mae_penalty_pct"] = (
        100
        * (
            transformer_confirmation_robustness_df["raw_mae"]
            - best_window_mae
        )
        / best_window_mae
    )

    target_labels = {
        "taxi_trip_count": "Taxi trips",
        "taxi_avg_trip_speed": "Taxi avg speed",
        "fhvhv_trip_count": "FHVHV trips",
        "fhvhv_avg_trip_speed": "FHVHV avg speed",
        "subway_ridership": "Subway ridership",
    }
    window_labels = {
        "early": "Early",
        "middle": "Middle",
        "late": "Late",
    }
    representation_labels = {
        "one_week_daily": "1-week daily",
        "two_week_daily": "2-week daily",
        "four_week_daily": "4-week daily",
        "two_week_overlapping_two_day": "2-week overlapping",
    }
    profile_labels = {
        "compact": "Compact",
        "balanced": "Balanced",
        "higher_capacity_regularized": "Higher capacity",
    }

    candidate_parts = transformer_confirmation_robustness_df[
        "candidate_id"
    ].str.split("__", n=1, expand=True)

    transformer_confirmation_robustness_df["Finalist"] = (
        candidate_parts[0].map(representation_labels).fillna(candidate_parts[0])
        + " · "
        + candidate_parts[1].map(profile_labels).fillna(candidate_parts[1])
    )
    transformer_confirmation_robustness_df["Target"] = (
        transformer_confirmation_robustness_df["metric"]
        .map(target_labels)
        .fillna(transformer_confirmation_robustness_df["metric"])
    )
    transformer_confirmation_robustness_df["Window"] = (
        transformer_confirmation_robustness_df["temporal_window"]
        .map(window_labels)
        .fillna(transformer_confirmation_robustness_df["temporal_window"])
    )
    transformer_confirmation_robustness_df["Target · Window"] = (
        transformer_confirmation_robustness_df["Target"]
        + " · "
        + transformer_confirmation_robustness_df["Window"]
    )

    target_window_order = [
        f"{target_labels[metric]} · {window}"
        for metric in FORECAST_TARGETS
        for window in ["Early", "Middle", "Late"]
    ]

    heatmap_df = (
        transformer_confirmation_robustness_df
        .pivot(
            index="Target · Window",
            columns="Finalist",
            values="mae_penalty_pct",
        )
        .reindex(target_window_order)
    )

    # The heatmap answers a simple question: how much accuracy was sacrificed
    # when a finalist was not the best model in that particular historical
    # window? Darker concentration therefore means a larger relative penalty.
    fig = px.imshow(
        heatmap_df,
        text_auto=".1f",
        aspect="auto",
        color_continuous_scale=BRAND_SEQUENTIAL_HEATMAP_SCALE,
        labels={
            "x": "Transformer finalist",
            "y": "",
            "color": "MAE penalty (%)",
        },
        title=(
            "Batch C · Do the finalists remain competitive through time?"
            "<br><sup>0% = lower-MAE finalist within that target and window; "
            "larger values indicate a greater accuracy penalty.</sup>"
        ),
    )

    fig.update_traces(
        texttemplate="%{z:.1f}%",
        hovertemplate=(
            "<b>%{y}</b><br>"
            "%{x}<br>"
            "MAE penalty vs window winner: %{z:.3f}%"
            "<extra></extra>"
        ),
    )

    fig = apply_branding(fig)

    fig.update_layout(
        height=720,
        coloraxis_colorbar={
            "title": "MAE penalty<br>(%)",
            "ticksuffix": "%",
        },
        margin={"l": 145, "r": 70, "t": 95, "b": 90},
    )

    fig.show()

**Findings.** The temporal heatmap shows that Batch C is testing genuine robustness rather than repeatedly producing the same ranking. A value of **0%** means that finalist had the lower MAE for that target and historical window; positive values show how much accuracy the other finalist gave up.

For **Taxi trips**, compact wins the early window, but higher capacity wins the middle and late windows. The compact penalty reaches **12.0% in the middle window** and **6.7% in the late window**, which explains why higher capacity wins after the three periods are pooled.

**Taxi average speed** is much closer. Compact wins early and late, while higher capacity wins the middle window, and the losing penalties remain small—no more than **2.6%** in the visible comparison. That temporal closeness reinforces the pooled result that compact and higher capacity are nearly tied for this target.

**FHVHV trips** illustrates why Batch C pools the actual errors rather than simply counting window wins. Higher capacity wins the early and middle windows, but only by small margins: compact trails by **2.0%** and **1.3%**. In the late window, compact wins by much more, leaving higher capacity **11.1% worse**. That larger late-period advantage is strong enough for compact to produce the lower pooled MAE overall.

For **FHVHV average speed**, higher capacity wins the early and late windows, while compact wins the middle period. The middle-window penalty for higher capacity is only **1.5%**, compared with compact penalties of **5.4% early** and **8.6% late**, making the pooled higher-capacity choice comparatively strong.

Subway also shows meaningful temporal separation: the higher-capacity model is **4.2% worse in the early window and 12.8% worse in the middle window**, consistent with compact producing the lower pooled Subway error.

Overall, the finalists do not rank identically in every historical period, but the pooled choices are supported by the **size of their errors across time**, not merely by which model wins the most windows.

### 4.5.2.3.4 Diagnose Training and Validation Learning Dynamics

Forecast accuracy alone does not show whether a Transformer learned cleanly. Use the saved Batch C selector curves for training Huber, inner-validation Huber, and standardized MAE, with the selected epoch marked. This section reuses fitted states and does not launch another experiment.

In [32]:
# ---------------------------------------------------------------------
# 4.5.2.3.4A — Diagnose Batch C learning dynamics across temporal windows
# ---------------------------------------------------------------------

transformer_learning_dynamics_df = pd.DataFrame()
transformer_runtime_capacity_df = pd.DataFrame()
transformer_runtime_capacity_display_df = pd.DataFrame()

if transformer_confirmation_curves_df.empty:
    print("Learning diagnostics waiting: Batch C curve artifacts are not available.")

else:
    transformer_learning_dynamics_df = transformer_confirmation_curves_df.copy()

    target_labels = {
        "taxi_trip_count": "Taxi trips",
        "taxi_avg_trip_speed": "Taxi avg speed",
        "fhvhv_trip_count": "FHVHV trips",
        "fhvhv_avg_trip_speed": "FHVHV avg speed",
        "subway_ridership": "Subway ridership",
    }
    window_labels = {
        "early": "Early",
        "middle": "Middle",
        "late": "Late",
    }
    profile_labels = {
        "compact": "Compact",
        "higher_capacity_regularized": "Higher capacity",
    }

    curves = transformer_learning_dynamics_df
    curves["Target"] = curves["metric"].map(target_labels)
    curves["Window"] = curves["temporal_window"].map(window_labels)
    curves["Profile"] = (
        curves["candidate_id"]
        .str.rsplit("__", n=1).str[-1]
        .map(profile_labels)
    )
    curves["Selected marker"] = np.where(curves["selected_epoch"], "★", "")

    # -----------------------------------------------------------------
    # Validation accuracy by target
    # -----------------------------------------------------------------
    # Standardized MAE is comparable enough to plot across targets, but each
    # target receives its own facet so unrelated epoch sequences are never
    # joined into one line. The star marks the epoch chosen by inner validation.
    mae_fig = px.line(
        curves,
        x="epoch",
        y="inner_validation_standardized_mae",
        color="Profile",
        line_dash="Window",
        text="Selected marker",
        facet_col="Target",
        facet_col_wrap=2,
        category_orders={
            "Target": list(target_labels.values()),
            "Profile": ["Compact", "Higher capacity"],
            "Window": ["Early", "Middle", "Late"],
        },
        color_discrete_sequence=BRAND_COLOR_SEQUENCE,
        title=(
            "Batch C · When does validation accuracy stop improving?"
            "<br><sup>Lower standardized MAE is better; ★ marks the selected epoch.</sup>"
        ),
    )

    mae_fig.for_each_annotation(
        lambda annotation: annotation.update(
            text=annotation.text.split("=")[-1]
        )
    )

    # Apply the shared project styling first, then make only chart-specific
    # layout adjustments such as global axis titles and bottom legend placement.
    mae_fig = apply_branding(mae_fig)
    mae_fig.update_traces(textposition="top center")
    mae_fig.update_xaxes(title_text=None)
    mae_fig.update_yaxes(title_text=None)

    mae_fig.update_layout(
        height=480,
        legend={
            "title_text": "",
            "orientation": "h",
            "x": 0.5,
            "xanchor": "center",
            "y": -0.12,
            "yanchor": "top",
        },
        margin={"l": 95, "r": 45, "t": 80, "b": 30},
    )

    mae_fig.add_annotation(
        text="Training epoch",
        x=0.5,
        y=-0.075,
        xref="paper",
        yref="paper",
        showarrow=False,
    )
    mae_fig.add_annotation(
        text="Standardized validation MAE",
        textangle=-90,
        x=-0.065,
        y=0.5,
        xref="paper",
        yref="paper",
        showarrow=False,
    )

    mae_fig.show()

    # # -----------------------------------------------------------------
    # # Training versus validation Huber loss
    # # -----------------------------------------------------------------
    # # Huber loss is the quantity optimized during fitting. Aggregate by median
    # # across the five targets so the chart answers one simple diagnostic:
    # # does validation loss broadly follow training loss or begin to diverge?
    # huber_long = curves.melt(
    #     id_vars=["epoch", "metric", "Profile", "Window"],
    #     value_vars=["train_huber", "inner_validation_huber"],
    #     var_name="Loss series",
    #     value_name="Huber loss",
    # )
    # huber_long["Loss series"] = huber_long["Loss series"].map({
    #     "train_huber": "Training",
    #     "inner_validation_huber": "Inner validation",
    # })

    # huber_summary_df = (
    #     huber_long
    #     .groupby(["Profile", "Window", "Loss series", "epoch"], as_index=False)
    #     .agg(
    #         median_huber_loss=("Huber loss", "median"),
    #         contributing_targets=("metric", "nunique"),
    #     )
    # )

    # huber_fig = px.line(
    #     huber_summary_df,
    #     x="epoch",
    #     y="median_huber_loss",
    #     color="Profile",
    #     line_dash="Loss series",
    #     facet_row="Window",
    #     category_orders={
    #         "Profile": ["Compact", "Higher capacity"],
    #         "Window": ["Late", "Middle", "Early"],
    #         "Loss series": ["Training", "Inner validation"],
    #     },
    #     color_discrete_sequence=BRAND_COLOR_SEQUENCE,
    #     hover_data={"contributing_targets": True},
    #     title=(
    #         "Batch C · Do training and validation loss remain aligned?"
    #         "<br><sup>Median Huber loss across the five targets; lower is better.</sup>"
    #     ),
    # )

    # huber_fig.for_each_annotation(
    #     lambda annotation: annotation.update(
    #         text=annotation.text.split("=")[-1]
    #     )
    # )

    # huber_fig = apply_branding(huber_fig)
    # huber_fig.update_xaxes(title_text=None)
    # huber_fig.update_yaxes(title_text=None)

    # huber_fig.update_layout(
    #     height=650,
    #     legend={
    #         "title_text": "",
    #         "orientation": "h",
    #         "x": 0.5,
    #         "xanchor": "center",
    #         "y": -0.15,
    #         "yanchor": "top",
    #     },
    #     margin={"l": 95, "r": 45, "t": 90, "b": 35},
    # )

    # huber_fig.add_annotation(
    #     text="Training epoch",
    #     x=0.5,
    #     y=-0.09,
    #     xref="paper",
    #     yref="paper",
    #     showarrow=False,
    # )
    # huber_fig.add_annotation(
    #     text="Median Huber loss",
    #     textangle=-90,
    #     x=-0.065,
    #     y=0.5,
    #     xref="paper",
    #     yref="paper",
    #     showarrow=False,
    # )

    # huber_fig.show()

    # -----------------------------------------------------------------
    # Compact operational summary
    # -----------------------------------------------------------------
    transformer_runtime_capacity_df = (
        transformer_confirmation_results_df
        .groupby(["metric", "candidate_id", "temporal_window"], as_index=False)
        .agg(
            selected_epoch=("selected_epoch", "median"),
            parameter_count=("parameter_count", "median"),
            runtime_seconds=("runtime_seconds", "median"),
            peak_gpu_memory_mb=("peak_gpu_memory_mb", "median"),
        )
    )

    transformer_runtime_capacity_display_df = (
        transformer_runtime_capacity_df
        .assign(
            Target=lambda df: df["metric"].map(target_labels),
            Profile=lambda df: (
                df["candidate_id"]
                .str.rsplit("__", n=1).str[-1]
                .map(profile_labels)
            ),
        )
        .groupby(["Target", "Profile"], as_index=False)
        .agg(
            **{
                "Median selected epoch": ("selected_epoch", "median"),
                "Earliest selected epoch": ("selected_epoch", "min"),
                "Latest selected epoch": ("selected_epoch", "max"),
                "Parameters": ("parameter_count", "first"),
                "Median runtime (s)": ("runtime_seconds", "median"),
                "Peak GPU memory (MB)": ("peak_gpu_memory_mb", "max"),
            }
        )
    )

    # CPU runs have no meaningful GPU-memory measurement, so avoid displaying
    # an all-missing column merely because the durable schema preserves it.
    if transformer_runtime_capacity_display_df["Peak GPU memory (MB)"].isna().all():
        transformer_runtime_capacity_display_df = (
            transformer_runtime_capacity_display_df
            .drop(columns="Peak GPU memory (MB)")
        )

display(transformer_runtime_capacity_display_df.round(3))

,Target,Profile,Median selected epoch,Earliest selected epoch,Latest selected epoch,Parameters,Median runtime (s)
0,FHVHV avg speed,Compact,7.000,2.000,11.000,"51,049.000",70.477
1,FHVHV avg speed,Higher capacity,10.000,7.000,10.000,"255,881.000",151.698
2,FHVHV trips,Compact,14.000,9.000,14.000,"51,049.000",120.591
3,FHVHV trips,Higher capacity,11.000,10.000,26.000,"255,881.000",165.576
4,Subway ridership,Compact,13.000,8.000,19.000,"51,049.000",65.767
5,Subway ridership,Higher capacity,16.000,15.000,17.000,"255,881.000",129.378
6,Taxi avg speed,Compact,15.000,7.000,17.000,"51,049.000",59.233
7,Taxi avg speed,Higher capacity,16.000,14.000,19.000,"255,881.000",103.154
8,Taxi trips,Compact,17.000,10.000,22.000,"51,049.000",141.361
9,Taxi trips,Higher capacity,14.000,12.000,26.000,"255,881.000",197.949


**Findings.** The 30 Batch C selector runs choose best training durations between **2 and 26 epochs**, with a median selected epoch of **14**.

No Transformer selects the 30-epoch search ceiling as its best state. The target-level validation curves generally flatten or worsen after their selected epochs, while the patience rule allows several additional epochs before stopping. This provides no evidence that the 30-epoch search envelope is systematically truncating still-improving models.

Training duration varies materially across target, historical window, and model capacity, reinforcing the value of selecting epochs from historical inner validation rather than imposing one fixed duration on every Transformer.

The capacity tradeoff is also substantial. The compact model contains **51,049 trainable parameters**, while the higher-capacity model contains **255,881—about five times as many**. The higher-capacity profile is also consistently slower to fit across the five targets, so its additional complexity is worth carrying forward only where it produces a meaningful accuracy advantage. Because this run used CPU, peak GPU-memory evidence is not available.

### 4.5.2.3.5 Explain Predictive Reliance and Temporal Sensitivity

Forecast accuracy tells us whether a Transformer predicts well. This section asks a different question: **what information does the fitted Transformer actually depend on to make those predictions?**

The analysis uses **inference-only grouped row permutation**. For each of the five proposed target-specific Transformers, the saved **middle-window** model remains completely fixed while one coherent input group is reassigned across the same evaluation rows. If forecast error increases, the correspondence carried by that information was helping the fitted model.

The five mobility-history groups always move with their matching observation masks. Related context fields also move together—for example, the weekly-STL offset, enabled flag, and missingness indicator form one semantic bundle. Taxi Zone identity is tested by permuting zone identities while leaving the learned embedding table unchanged.

The primary measure is the increase in **final-output MAE**, matching the nonnegative forecasts used downstream. Raw-output MAE damage is retained separately so the unconstrained neural behavior remains auditable.

A second diagnostic asks **which parts of the historical lookback matter most**. One complete patch-sized historical span is permuted at a time across all mobility values and masks. This is temporal sensitivity, not an interpretation of attention weights, and neither diagnostic implies causality.

**Select the saved Transformer states used for diagnostics.**

Batch C evaluates two finalists across early, middle, and late history, producing many saved fitted states. Running interpretation across all of them would answer a broader—and much more expensive—question than this MVP requires.

For each mobility target, this cell therefore identifies the **proposed Batch C winner** and retrieves exactly its **middle-window saved state**. The result should contain five states total: one for Taxi trips, Taxi speed, FHVHV trips, FHVHV speed, and Subway ridership.

These are already-fitted models. Nothing in this section retrains them.

In [33]:
# ---------------------------------------------------------------------
# 4.5.2.3.5A — Select the five saved states used for diagnostics
# ---------------------------------------------------------------------

transformer_grouped_reliance_df = pd.DataFrame()
transformer_patch_sensitivity_df = pd.DataFrame()
transformer_reliance_state_registry_df = pd.DataFrame()

SEQUENCE_RELIANCE_GROUPS = {
    "Taxi demand history": "taxi_trip_count",
    "Taxi speed history": "taxi_avg_trip_speed",
    "FHVHV demand history": "fhvhv_trip_count",
    "FHVHV speed history": "fhvhv_avg_trip_speed",
    "Subway ridership history": "subway_ridership",
}

# These names come from the finalized 4.5.1 semantic input registry. Using the
# saved group labels means 4.5.2 does not invent a second interpretation scheme.
CONTEXT_RELIANCE_GROUP_ORDER = [
    "Annual / calendar timing",
    "Weekly STL seasonal state",
    "Weather",
    "Congestion-pricing regime",
    "Multimodal / connected context",
    "Spatial / network context",
]


def build_transformer_reliance_state_registry():
    """Select one proposed-winner middle-window state for every mobility target."""
    if (
        transformer_proposed_target_profile_registry_df.empty
        or transformer_confirmation_results_df.empty
    ):
        return pd.DataFrame()

    required_proposal = {"metric", "candidate_id"}
    missing_proposal = sorted(
        required_proposal - set(transformer_proposed_target_profile_registry_df.columns)
    )
    if missing_proposal:
        raise ValueError(
            f"Target-profile proposal is missing fields: {missing_proposal}"
        )

    required_results = {
        "track", "metric", "horizon", "candidate_id",
        "temporal_window", "state_path", "status",
    }
    missing_results = sorted(
        required_results - set(transformer_confirmation_results_df.columns)
    )
    if missing_results:
        raise ValueError(
            f"Batch C results are missing diagnostic fields: {missing_results}"
        )

    selected_profiles = (
        transformer_proposed_target_profile_registry_df[["metric", "candidate_id"]]
        .drop_duplicates()
        .copy()
    )

    # The interpretation surface is target-specific. A tie that leaves two
    # proposed finalists for one target must be resolved before reliance is run.
    if selected_profiles["metric"].duplicated().any():
        raise ValueError(
            "Grouped reliance requires exactly one proposed finalist per target."
        )

    middle_states = transformer_confirmation_results_df.loc[
        transformer_confirmation_results_df["temporal_window"].eq("middle")
        & transformer_confirmation_results_df["status"].eq("success"),
        [
            "track", "metric", "horizon", "candidate_id",
            "temporal_window", "state_path",
        ],
    ].copy()

    registry = selected_profiles.merge(
        middle_states,
        on=["metric", "candidate_id"],
        how="left",
        validate="one_to_one",
    )

    # Five targets should resolve to five saved models—no averaging across the
    # 30 Batch C fits and no accidental reuse of early/late states.
    complete_target_set = (
        len(registry) == len(FORECAST_TARGETS)
        and set(registry["metric"]) == set(FORECAST_TARGETS)
    )
    if not complete_target_set:
        raise ValueError(
            "Grouped reliance requires one selected middle-window state for each target."
        )

    if registry["state_path"].isna().any():
        missing_targets = registry.loc[
            registry["state_path"].isna(), "metric"
        ].tolist()
        raise ValueError(
            f"Selected targets are missing saved Batch C states: {missing_targets}"
        )

    missing_files = [
        path
        for path in registry["state_path"].astype(str)
        if not Path(path).exists()
    ]
    if missing_files:
        raise FileNotFoundError(
            f"Selected Transformer state files are missing: {missing_files}"
        )

    return registry.sort_values("metric").reset_index(drop=True)

# ---------------------------------------------------------------------
# Reader-facing diagnostic-state registry
# ---------------------------------------------------------------------

transformer_reliance_state_registry_df = build_transformer_reliance_state_registry()

if not transformer_reliance_state_registry_df.empty:
    target_labels = {
        "taxi_trip_count": "Taxi trips",
        "taxi_avg_trip_speed": "Taxi avg speed",
        "fhvhv_trip_count": "FHVHV trips",
        "fhvhv_avg_trip_speed": "FHVHV avg speed",
        "subway_ridership": "Subway ridership",
    }
    track_labels = {
        "post_cp_primary": "Post-CP Primary",
        "full_history_regime_aware": "Full-History Regime-Aware",
    }

    transformer_reliance_state_display_df = (
        transformer_reliance_state_registry_df
        .assign(
            Target=lambda df: df["metric"].map(target_labels),
            Track=lambda df: df["track"].map(track_labels),
            Finalist=lambda df: (
                df["candidate_id"]
                .str.replace("__", " · ", regex=False)
                .str.replace("_", " ", regex=False)
                .str.title()
            ),
            Window=lambda df: df["temporal_window"].str.title(),
            **{"Saved state": "Ready"},
        )
        [["Target", "Track", "horizon", "Finalist", "Window", "Saved state"]]
        .rename(columns={"horizon": "Horizon"})
        .reset_index(drop=True)
    )

    display(transformer_reliance_state_display_df)
else:
    print(
        "Diagnostic-state registry waiting: Batch C must first propose "
        "one complete Transformer finalist per target."
    )

,Target,Track,Horizon,Finalist,Window,Saved state
0,FHVHV avg speed,Full-History Regime-Aware,1,Two Week Overlapping Two Day · Higher Capacity...,Middle,Ready
1,FHVHV trips,Post-CP Primary,2,Two Week Overlapping Two Day · Compact,Middle,Ready
2,Subway ridership,Post-CP Primary,5,Two Week Overlapping Two Day · Compact,Middle,Ready
3,Taxi avg speed,Full-History Regime-Aware,5,Two Week Overlapping Two Day · Compact,Middle,Ready
4,Taxi trips,Post-CP Primary,1,Two Week Overlapping Two Day · Higher Capacity...,Middle,Ready


**Findings.** The interpretation stage resolves exactly **five saved Transformer states—one proposed finalist for each mobility target**—and all five are available for inference-only diagnostics.

Taxi trips, FHVHV trips, and Subway ridership use the **Post-CP Primary** modeling track, while Taxi average speed and FHVHV average speed use the **Full-History Regime-Aware** track. The selected capacity also varies by target: Taxi trips and FHVHV average speed use the higher-capacity regularized Transformer, while Taxi average speed, FHVHV trips, and Subway ridership use the compact profile.

All five selected models share the same **two-week overlapping two-day history representation** and are restored from their middle Batch C confirmation window. The diagnostics that follow therefore examine the actual target-specific Transformer finalists rather than averaging behavior across discarded architectures.

#### 4.5.2.3.5B — Restore a saved Transformer and reproduce its forecasts

Predictive reliance is meaningful only if the diagnostic uses the **same fitted forecasting function** that produced the original Batch C result.

This cell restores three pieces together: the Transformer weights, the training-only preprocessing state, and the exact middle-window evaluation payload. The preprocessor is not refitted; saved means, standard deviations, medians, weekly-STL scaling state, and target scaling are reused exactly as they were learned during training.

Inference is performed in ordinary batches rather than sending the entire evaluation surface through the network at once. This changes only memory usage—not the forecasts—and keeps repeated reliance tests practical on either CPU or GPU.

In [34]:
# ---------------------------------------------------------------------
# 4.5.2.3.5B — Restore saved Transformer state and reproduce inference
# ---------------------------------------------------------------------

def restore_transformer_preprocessor(state):
    """Restore the training-only numerical state saved beside one Transformer."""
    contract = pd.DataFrame(state["context_contract"])
    preprocessor = TransformerPreprocessor(contract)
    saved = state["preprocessor_state"]

    array_fields = {
        "sequence_means_": "sequence_means",
        "sequence_stds_": "sequence_stds",
        "context_medians_": "context_medians",
        "context_means_": "context_means",
        "context_stds_": "context_stds",
    }
    scalar_fields = {
        "stl_training_support_": "stl_training_support",
        "stl_mean_": "stl_mean",
        "stl_std_": "stl_std",
        "target_mean_": "target_mean",
        "target_std_": "target_std",
    }

    required_state = set(array_fields.values()) | set(scalar_fields.values())
    missing_state = sorted(required_state - set(saved))
    if missing_state:
        raise ValueError(
            f"Saved preprocessor state is incomplete: {missing_state}"
        )

    for attribute, key in array_fields.items():
        setattr(
            preprocessor,
            attribute,
            np.asarray(saved[key], dtype=np.float64),
        )

    for attribute, key in scalar_fields.items():
        setattr(preprocessor, attribute, saved[key])

    # STL scaling is active only when the saved training sample contained enough
    # genuinely fitted seasonal variation to define a stable numerical scale.
    preprocessor.weekly_stl_scaler_active_ = bool(saved.get(
        "weekly_stl_scaler_active",
        preprocessor.stl_training_support_ >= 2
        and np.isfinite(preprocessor.stl_std_)
        and preprocessor.stl_std_ > WEEKLY_STL_SCALE_EPS,
    ))

    preprocessor.is_fitted_ = True
    return preprocessor


def restore_transformer_diagnostic_bundle(state_path):
    """Restore one saved middle-window model and its exact scoring inputs."""
    state = torch.load(
        state_path,
        map_location=TORCH_DEVICE,
        weights_only=False,
    )

    required_state = {
        "contract_id",
        "logical_key",
        "model_state_dict",
        "representation",
        "profile",
        "context_feature_order",
        "sequence_value_channels",
        "sequence_mask_channels",
        "zone_ids",
        "context_contract",
        "preprocessor_state",
    }
    missing_state = sorted(required_state - set(state))
    if missing_state:
        raise ValueError(
            f"Saved Transformer state is incomplete: {missing_state}"
        )

    if state["contract_id"] != TRANSFORMER_CONFIRMATION_CONTRACT_ID:
        raise ValueError(
            "Reliance diagnostics require a current-contract Batch C state."
        )

    logical = pd.Series(state["logical_key"])

    if logical["temporal_window"] != "middle":
        raise ValueError(
            "Reliance diagnostics are restricted to middle-window states."
        )

    model = PatchTransformerRegressor(
        NEURAL_SEQUENCE_INPUT_DIM,
        len(state["context_feature_order"]),
        len(state["zone_ids"]),
        state["representation"],
        state["profile"],
    ).to(TORCH_DEVICE)

    model.load_state_dict(state["model_state_dict"])
    model.eval()

    preprocessor = restore_transformer_preprocessor(state)

    # Rebuild the same legal evaluation population used by this logical fit.
    # No scaler is fitted and no model parameter changes during reconstruction.
    payload = build_transformer_fit_payload(
        logical,
        state["representation"],
        state["profile"],
    )

    return state, model, preprocessor, logical, payload


def predict_saved_transformer(
    model,
    preprocessor,
    payload,
    sequence_values=None,
    sequence_masks=None,
    context_frame=None,
    zone_index=None,
):
    """Score original or perturbed rows through one completely fixed model."""
    sequence_values = (
        payload["eval_sequence_values"]
        if sequence_values is None
        else sequence_values
    )
    sequence_masks = (
        payload["eval_sequence_masks"]
        if sequence_masks is None
        else sequence_masks
    )
    context_frame = (
        payload["eval_context"]
        if context_frame is None
        else context_frame
    )
    zone_index = (
        payload["eval_zone_index"]
        if zone_index is None
        else zone_index
    )

    prepared_sequence = preprocessor.transform_sequence(
        sequence_values, sequence_masks
    )
    prepared_context = preprocessor.transform_context(context_frame)
    zone_index = np.asarray(zone_index, dtype=np.int64)

    row_count = len(zone_index)
    if len(prepared_sequence) != row_count or len(prepared_context) != row_count:
        raise ValueError(
            "Saved-state inference inputs are not aligned by evaluation row."
        )

    # Reliance requires many repeated forward passes. Batching keeps those
    # passes memory-safe without changing which evaluation rows are scored.
    standardized_batches = []

    with torch.no_grad():
        for start in range(0, row_count, selected_transformer_batch_size):
            stop = min(start + selected_transformer_batch_size, row_count)

            sequence_tensor = torch.as_tensor(
                prepared_sequence[start:stop],
                dtype=torch.float32,
                device=TORCH_DEVICE,
            )
            context_tensor = torch.as_tensor(
                prepared_context[start:stop],
                dtype=torch.float32,
                device=TORCH_DEVICE,
            )
            zone_tensor = torch.as_tensor(
                zone_index[start:stop],
                dtype=torch.long,
                device=TORCH_DEVICE,
            )

            standardized_batches.append(
                model(
                    sequence_tensor,
                    context_tensor,
                    zone_tensor,
                ).cpu().numpy()
            )

    standardized = np.concatenate(standardized_batches)
    prediction_raw = preprocessor.inverse_target(standardized)
    prediction_final = apply_prediction_output_rule(prediction_raw)

    return prediction_final, prediction_raw


def deterministic_diagnostic_permutation(row_count, *identity_parts):
    """Create one reproducible row shuffle for a diagnostic input group."""
    if row_count < 2:
        return np.arange(row_count)

    seed_parts = ["transformer_reliance", *identity_parts, GLOBAL_SEED]
    digest = hashlib.sha256(
        "|".join(map(str, seed_parts)).encode("utf-8")
    ).digest()

    seed = int.from_bytes(digest[:4], "big") % (2**31 - 1)
    permutation = np.random.default_rng(seed).permutation(row_count)

    # A shuffle should actually break row correspondence. This matters only for
    # tiny samples, but enforcing it makes the diagnostic definition explicit.
    if np.array_equal(permutation, np.arange(row_count)):
        permutation = np.roll(permutation, 1)

    return permutation

def arrays_equal_with_missing(left, right):
    """Test whether two diagnostic arrays are unchanged, including missing values.

    Reliance perturbations operate on several kinds of data: continuous values,
    Boolean observation masks, and context groups that Pandas can represent
    with a mixed/object dtype. NumPy's `equal_nan=True` comparison assumes a
    numeric dtype and can fail on that mixed context.

    This helper treats missing values in the same locations as equal, then
    compares only the genuinely observed cells. It lets us ask the simple
    diagnostic question we actually care about: did the row permutation change
    this information group?
    """
    left = np.asarray(left)
    right = np.asarray(right)

    if left.shape != right.shape:
        return False

    left_missing = pd.isna(left)
    right_missing = pd.isna(right)

    # If one array is missing where the other is observed, the arrays changed.
    if not np.array_equal(left_missing, right_missing):
        return False

    observed = ~left_missing

    # Two entirely missing arrays carry the same information for this check.
    if not observed.any():
        return True

    return np.array_equal(
        left[observed],
        right[observed],
    )

def summarize_prediction_damage(
    logical,
    group,
    target,
    baseline_final,
    baseline_raw,
    perturbed_final=None,
    perturbed_raw=None,
    permutation_identifiable=True,
):
    """Translate one perturbation into before/after MAE and percentage damage."""
    baseline_final_mae = regression_metrics(target, baseline_final)["mae"]
    baseline_raw_mae = regression_metrics(target, baseline_raw)["mae"]

    if permutation_identifiable:
        perturbed_final_mae = regression_metrics(
            target, perturbed_final
        )["mae"]
        perturbed_raw_mae = regression_metrics(
            target, perturbed_raw
        )["mae"]

        final_increase = perturbed_final_mae - baseline_final_mae
        raw_increase = perturbed_raw_mae - baseline_raw_mae
    else:
        # A constant group cannot be meaningfully shuffled. N/A is more honest
        # than reporting zero and implying the fitted model learned no reliance.
        perturbed_final_mae = np.nan
        perturbed_raw_mae = np.nan
        final_increase = np.nan
        raw_increase = np.nan

    final_increase_pct = (
        100.0 * final_increase / baseline_final_mae
        if np.isfinite(final_increase) and baseline_final_mae > 0
        else np.nan
    )
    raw_increase_pct = (
        100.0 * raw_increase / baseline_raw_mae
        if np.isfinite(raw_increase) and baseline_raw_mae > 0
        else np.nan
    )

    return {
        "logical_key": logical_key_token(
            logical, TRANSFORMER_LOGICAL_KEY_COLUMNS
        ),
        "track": logical["track"],
        "metric": logical["metric"],
        "horizon": int(logical["horizon"]),
        "candidate_id": logical["candidate_id"],
        "temporal_window": logical["temporal_window"],
        "group": group,
        "permutation_identifiable": bool(permutation_identifiable),
        "baseline_final_mae": baseline_final_mae,
        "perturbed_final_mae": perturbed_final_mae,
        "final_mae_increase": final_increase,
        "final_mae_increase_pct": final_increase_pct,
        "baseline_raw_mae": baseline_raw_mae,
        "perturbed_raw_mae": perturbed_raw_mae,
        "raw_mae_increase": raw_increase,
        "raw_mae_increase_pct": raw_increase_pct,
        "retrained": False,
    }

**Measure grouped predictive reliance.**

Grouped permutation asks a simple counterfactual-of-information question: **if this fitted Transformer no longer knew which row a particular input belonged to, how much worse would its forecasts become?**

For mobility history, an entire modality sequence is reassigned to another evaluation row together with its observation mask. This preserves the donor sequence's internal time pattern while breaking its relationship to the forecast being made.

Context uses the semantic groups already published by the finalized 4.5.1 input registry. Calendar fields move together, weather fields move together, connected mobility context moves together, and the three weekly-STL fields always move as one unit. Taxi Zone identity is handled separately by permuting zone indices while keeping the learned embedding weights fixed.

A positive MAE increase means the original row-to-input relationship was useful to the model. A negative value can occur when scrambling unexpectedly improves forecasts. A constant group is marked **not permutation-identifiable** rather than being interpreted as zero reliance.

In [35]:
# ---------------------------------------------------------------------
# 4.5.2.3.5C — Measure grouped predictive reliance
# ---------------------------------------------------------------------

def transformer_context_reliance_groups(state):
    """Recover the six semantic context groups published by the saved contract."""
    groups = {group: [] for group in CONTEXT_RELIANCE_GROUP_ORDER}

    for record in state["context_contract"]:
        feature = str(record["feature"])
        group = str(record["input_group"])

        if group not in groups:
            raise ValueError(
                f"Unexpected context reliance group for {feature}: {group}"
            )

        groups[group].append(feature)

    assigned_features = [
        feature
        for group_features in groups.values()
        for feature in group_features
    ]
    feature_order = list(state["context_feature_order"])

    # Every one of the 52 context fields should belong to exactly one semantic
    # group. Overlap would double-count information; omissions would hide it.
    valid_partition = (
        len(assigned_features) == len(set(assigned_features))
        and set(assigned_features) == set(feature_order)
    )
    if not valid_partition:
        raise ValueError(
            "Saved context groups must partition the 52-value context exactly once."
        )

    expected_stl_bundle = {
        "weekly_stl_seasonal_offset",
        "weekly_stl_enabled",
        "missing_mask__weekly_stl_seasonal_offset",
    }
    if set(groups["Weekly STL seasonal state"]) != expected_stl_bundle:
        raise ValueError(
            "Weekly STL must remain one three-field semantic perturbation bundle."
        )

    return groups


def compute_transformer_grouped_reliance(
    state,
    model,
    preprocessor,
    logical,
    payload,
    baseline_final,
    baseline_raw,
):
    """Shuffle one coherent information group at a time through a fixed model."""
    target = payload["eval_target"]
    row_count = len(target)
    logical_token = logical_key_token(
        logical, TRANSFORMER_LOGICAL_KEY_COLUMNS
    )
    rows = []

    # -----------------------------------------------------------------
    # Five mobility-history groups
    # -----------------------------------------------------------------
    # Each donor contributes the WHOLE history for one modality, not isolated
    # time points. Its mask moves with it so "was this observed?" remains paired
    # with the value it describes.
    value_channels = list(state["sequence_value_channels"])
    mask_channels = list(state["sequence_mask_channels"])

    for group, metric_name in SEQUENCE_RELIANCE_GROUPS.items():
        if metric_name not in value_channels:
            raise ValueError(
                f"Saved sequence is missing modality {metric_name}."
            )

        channel_index = value_channels.index(metric_name)
        expected_mask = f"observed_mask__{metric_name}"

        if mask_channels[channel_index] != expected_mask:
            raise ValueError(
                f"Saved value/mask order is misaligned for {metric_name}."
            )

        permutation = deterministic_diagnostic_permutation(
            row_count, logical_token, group
        )

        sequence_values = payload["eval_sequence_values"].copy()
        sequence_masks = payload["eval_sequence_masks"].copy()

        original_values = sequence_values[:, :, channel_index].copy()
        original_masks = sequence_masks[:, :, channel_index].copy()

        permuted_values = original_values[permutation]
        permuted_masks = original_masks[permutation]

        # A permutation is informative only when it actually changes either the
        # modality history or its observation pattern. Values and masks are
        # checked together because they form one model-facing information source.
        identifiable = not (
            arrays_equal_with_missing(original_values, permuted_values)
            and arrays_equal_with_missing(original_masks, permuted_masks)
        )

        if identifiable:
            sequence_values[:, :, channel_index] = permuted_values
            sequence_masks[:, :, channel_index] = permuted_masks

            perturbed_final, perturbed_raw = predict_saved_transformer(
                model,
                preprocessor,
                payload,
                sequence_values=sequence_values,
                sequence_masks=sequence_masks,
            )
        else:
            perturbed_final, perturbed_raw = None, None

        rows.append(summarize_prediction_damage(
            logical,
            group,
            target,
            baseline_final,
            baseline_raw,
            perturbed_final,
            perturbed_raw,
            permutation_identifiable=identifiable,
        ))

    # -----------------------------------------------------------------
    # Six supporting-context groups
    # -----------------------------------------------------------------
    # All columns belonging to a semantic group receive the SAME row
    # permutation. This preserves relationships inside the group while breaking
    # the group's correspondence with the forecast row.
    context_groups = transformer_context_reliance_groups(state)

    for group, features in context_groups.items():
        permutation = deterministic_diagnostic_permutation(
            row_count, logical_token, group
        )

        context_frame = payload["eval_context"].copy()
        original_values = context_frame[features].to_numpy()
        permuted_values = context_frame.iloc[permutation][features].to_numpy()

        # Context groups can contain a mixture of continuous, binary, and
        # missingness fields. Use the dtype-safe comparison rather than asking
        # NumPy to apply a numeric isnan() operation to the whole mixed array.
        identifiable = not arrays_equal_with_missing(
            original_values,
            permuted_values,
        )

        if identifiable:
            context_frame.loc[:, features] = permuted_values

            perturbed_final, perturbed_raw = predict_saved_transformer(
                model,
                preprocessor,
                payload,
                context_frame=context_frame,
            )
        else:
            perturbed_final, perturbed_raw = None, None

        rows.append(summarize_prediction_damage(
            logical,
            group,
            target,
            baseline_final,
            baseline_raw,
            perturbed_final,
            perturbed_raw,
            permutation_identifiable=identifiable,
        ))

    # -----------------------------------------------------------------
    # Taxi Zone identity
    # -----------------------------------------------------------------
    # Zone identity reaches the model through an embedding lookup rather than
    # the 52-column context matrix. Permuting IDs therefore breaks "which zone
    # is this?" while leaving the learned embedding table completely untouched.
    group = "Taxi Zone identity"
    permutation = deterministic_diagnostic_permutation(
        row_count, logical_token, group
    )

    original_zones = np.asarray(
        payload["eval_zone_index"], dtype=np.int64
    )
    permuted_zones = original_zones[permutation]
    identifiable = not np.array_equal(
        original_zones, permuted_zones
    )

    if identifiable:
        perturbed_final, perturbed_raw = predict_saved_transformer(
            model,
            preprocessor,
            payload,
            zone_index=permuted_zones,
        )
    else:
        perturbed_final, perturbed_raw = None, None

    rows.append(summarize_prediction_damage(
        logical,
        group,
        target,
        baseline_final,
        baseline_raw,
        perturbed_final,
        perturbed_raw,
        permutation_identifiable=identifiable,
    ))

    return rows

#### 4.5.2.3.5D — Measure temporal patch sensitivity

Grouped reliance identifies **what kind of information** matters. Patch sensitivity asks **where in the historical lookback that useful information sits**.

For each saved Transformer, this cell takes one patch-sized span of history and reassigns that entire span across evaluation rows. All five mobility values and all five matching observation masks move together. The model, preprocessing state, context, Taxi Zone identity, and physical evaluation rows remain fixed.

`relative_patch_age = 0` denotes the most recent patch-sized span, with larger values moving farther back in history. For the overlapping representation, neighboring Transformer tokens share some physical history, so this diagnostic should be interpreted as sensitivity to the **historical span associated with a patch position**, not as a pure token-level attention attribution.

In [36]:
# ---------------------------------------------------------------------
# 4.5.2.3.5D — Measure temporal patch sensitivity
# ---------------------------------------------------------------------

def compute_transformer_patch_sensitivity(
    model,
    preprocessor,
    logical,
    payload,
    baseline_final,
    baseline_raw,
):
    """Permute each patch-sized historical span while keeping the model fixed."""
    target = payload["eval_target"]
    row_count = len(target)
    logical_token = logical_key_token(
        logical, TRANSFORMER_LOGICAL_KEY_COLUMNS
    )
    rows = []

    for patch_index in range(int(model.n_patches)):
        start = patch_index * int(model.patch_stride)
        stop = start + int(model.patch_length)

        permutation = deterministic_diagnostic_permutation(
            row_count,
            logical_token,
            "patch",
            patch_index,
        )

        sequence_values = payload["eval_sequence_values"].copy()
        sequence_masks = payload["eval_sequence_masks"].copy()

        # A patch is a temporal bundle. Move every mobility channel and every
        # matching mask together so the donor span remains internally coherent.
        original_values = sequence_values[:, start:stop, :].copy()
        original_masks = sequence_masks[:, start:stop, :].copy()

        permuted_values = original_values[permutation]
        permuted_masks = original_masks[permutation]

        # A patch perturbation counts only when the donor rows genuinely alter
        # the historical span. The comparison remains missing-value aware and
        # keeps the ten-channel value/mask bundle conceptually intact.
        identifiable = not (
            arrays_equal_with_missing(original_values, permuted_values)
            and arrays_equal_with_missing(original_masks, permuted_masks)
        )

        if identifiable:
            sequence_values[:, start:stop, :] = permuted_values
            sequence_masks[:, start:stop, :] = permuted_masks

            perturbed_final, perturbed_raw = predict_saved_transformer(
                model,
                preprocessor,
                payload,
                sequence_values=sequence_values,
                sequence_masks=sequence_masks,
            )
        else:
            perturbed_final, perturbed_raw = None, None

        row = summarize_prediction_damage(
            logical,
            f"Patch {patch_index}",
            target,
            baseline_final,
            baseline_raw,
            perturbed_final,
            perturbed_raw,
            permutation_identifiable=identifiable,
        )

        row.update({
            "patch_index": patch_index,
            "relative_patch_age": int(model.n_patches - 1 - patch_index),
            "patch_start_step": start,
            "patch_stop_step_exclusive": stop,
        })
        rows.append(row)

    return rows

**4.5.2.3.5E — Run or recover the five selected-state diagnostics.**

The interpretation stage uses exactly **five already-fitted Transformers**: the proposed Batch C winner for each mobility target, restored from its middle-window saved state.

For each state, the notebook first checks whether both required diagnostic artifacts already exist and belong to the current diagnostic contract. If they do, they are reloaded. If either artifact is missing, unreadable, or stale, only that target's inference diagnostics are recomputed and written immediately.

This makes interpretation restartable at the same natural unit as the analysis itself. `RESET_TRANSFORMER_RELIANCE=False` means **reuse valid target-level diagnostics and compute only missing work**; `True` deliberately clears this diagnostic cache and rebuilds all five targets. No optimizer is created and no Transformer is retrained.

In [37]:
# ---------------------------------------------------------------------
# 4.5.2.3.5E — Run or recover the five selected-state diagnostics
# ---------------------------------------------------------------------

TRANSFORMER_RELIANCE_CONTRACT_ID = (
    f"{TRANSFORMER_CONFIRMATION_CONTRACT_ID}__grouped_reliance_v1"
)
TRANSFORMER_RELIANCE_DIR = (
    Path(TRANSFORMER_CONFIRMATION_SCREEN_DIR).parent
    / "transformer_grouped_reliance_v1"
)

transformer_grouped_reliance_df = pd.DataFrame()
transformer_patch_sensitivity_df = pd.DataFrame()
transformer_reliance_state_registry_df = build_transformer_reliance_state_registry()


def transformer_diagnostic_paths(state_row):
    """Return the two durable diagnostic files belonging to one selected model."""
    logical = pd.Series({
        column: getattr(state_row, column)
        for column in TRANSFORMER_LOGICAL_KEY_COLUMNS
    })
    token = logical_key_token(logical, TRANSFORMER_LOGICAL_KEY_COLUMNS)

    return (
        TRANSFORMER_RELIANCE_DIR / f"reliance_{token}.parquet",
        TRANSFORMER_RELIANCE_DIR / f"patch_sensitivity_{token}.parquet",
    )


def recover_transformer_diagnostics(reliance_path, patch_path, logical_key):
    """Recover one target only when both diagnostic artifacts are complete and current."""
    if not reliance_path.exists() or not patch_path.exists():
        return None

    try:
        reliance_df = pd.read_parquet(reliance_path)
        patch_df = pd.read_parquet(patch_path)
    except Exception:
        # A partially written or unreadable file represents unfinished work.
        # Returning None lets the caller recompute only this selected model.
        return None

    for frame in [reliance_df, patch_df]:
        required = {"diagnostic_contract_id", "logical_key", "retrained"}

        if frame.empty or not required.issubset(frame.columns):
            return None

        if not frame["diagnostic_contract_id"].eq(
            TRANSFORMER_RELIANCE_CONTRACT_ID
        ).all():
            return None

        if not frame["logical_key"].eq(logical_key).all():
            return None

        if frame["retrained"].any():
            return None

    return reliance_df, patch_df


if transformer_reliance_state_registry_df.empty:
    print(
        "Reliance diagnostics waiting: the five proposed middle-window "
        "Transformer states are not available."
    )

else:
    if RESET_TRANSFORMER_RELIANCE:
        # RESET is the explicit request to discard valid cached diagnostics.
        # With RESET=False, completed target-level work is reused automatically.
        reset_experiment_artifacts(TRANSFORMER_RELIANCE_DIR)

    TRANSFORMER_RELIANCE_DIR.mkdir(parents=True, exist_ok=True)

    reliance_frames = []
    patch_frames = []

    for state_row in tqdm(
        transformer_reliance_state_registry_df.itertuples(index=False),
        total=len(transformer_reliance_state_registry_df),
        desc="Transformer diagnostic states",
    ):
        logical = pd.Series({
            column: getattr(state_row, column)
            for column in TRANSFORMER_LOGICAL_KEY_COLUMNS
        })
        logical_key = logical_key_token(
            logical, TRANSFORMER_LOGICAL_KEY_COLUMNS
        )

        reliance_path, patch_path = transformer_diagnostic_paths(state_row)

        # Recovery happens before any model restoration or inference. A target
        # with valid saved diagnostics should cost only two small artifact reads.
        recovered = recover_transformer_diagnostics(
            reliance_path,
            patch_path,
            logical_key,
        )

        if recovered is not None:
            reliance_df, patch_df = recovered

        else:
            # Restore the exact fitted Transformer, preprocessing state, and
            # middle-window evaluation population chosen by Batch C. Nothing
            # is fitted again during interpretation.
            state, model, preprocessor, logical, payload = (
                restore_transformer_diagnostic_bundle(state_row.state_path)
            )

            # Every perturbation for this model is compared with one untouched
            # baseline forecast. Computing it once avoids redundant inference
            # and ensures every damage estimate uses the same reference.
            baseline_final, baseline_raw = predict_saved_transformer(
                model,
                preprocessor,
                payload,
            )

            reliance_df = pd.DataFrame(
                compute_transformer_grouped_reliance(
                    state,
                    model,
                    preprocessor,
                    logical,
                    payload,
                    baseline_final,
                    baseline_raw,
                )
            )

            patch_df = pd.DataFrame(
                compute_transformer_patch_sensitivity(
                    model,
                    preprocessor,
                    logical,
                    payload,
                    baseline_final,
                    baseline_raw,
                )
            )

            # Persist the diagnostic contract beside every row. On a later
            # notebook run, stale results from an older methodology will fail
            # recovery and only that target will be recomputed.
            for frame in [reliance_df, patch_df]:
                frame["diagnostic_contract_id"] = TRANSFORMER_RELIANCE_CONTRACT_ID
                frame["state_path"] = str(state_row.state_path)

            # Save immediately after each target finishes. If execution stops
            # during target 4 of 5, targets 1–3 remain complete and recoverable.
            atomic_write_parquet(reliance_df, reliance_path)
            atomic_write_parquet(patch_df, patch_path)

            del state, model, preprocessor, logical, payload
            clear_device_memory()

        reliance_frames.append(reliance_df)
        patch_frames.append(patch_df)

    transformer_grouped_reliance_df = pd.concat(
        reliance_frames,
        ignore_index=True,
    )
    transformer_patch_sensitivity_df = pd.concat(
        patch_frames,
        ignore_index=True,
    )

Transformer diagnostic states: 100%|██████████| 5/5 [00:03<00:00,  1.66it/s]


**4.5.2.3.5F — Verify that the diagnostic evidence is complete and inference-only.**

A completed diagnostic run should have a very specific shape. Each of the five selected Transformers contributes **12 grouped-reliance tests**: five mobility-history groups, six supporting-context groups, and Taxi Zone identity.

The checks below also confirm that every grouped-reliance target is represented exactly once, all diagnostics come from the middle confirmation window, and no row claims that a model was retrained. Patch sensitivity is checked separately because the number of patch positions legitimately differs across representations.

These are integrity checks on interpretation evidence; they do not rank or select another Transformer.

In [38]:
# ---------------------------------------------------------------------
# 4.5.2.3.5F — Verify complete inference-only diagnostic evidence
# ---------------------------------------------------------------------

reliance_df = globals().get("transformer_grouped_reliance_df", pd.DataFrame())
patch_df = globals().get("transformer_patch_sensitivity_df", pd.DataFrame())
state_registry_df = globals().get(
    "transformer_reliance_state_registry_df",
    pd.DataFrame(),
)

if reliance_df.empty or patch_df.empty:
    print(
        "Diagnostic QA waiting: run 4.5.2.3.5E first so the saved reliance "
        "and patch-sensitivity artifacts are recovered into memory."
    )

else:
    expected_groups_per_target = (
        len(SEQUENCE_RELIANCE_GROUPS)
        + len(CONTEXT_RELIANCE_GROUP_ORDER)
        + 1  # Taxi Zone identity
    )
    expected_reliance_rows = len(state_registry_df) * expected_groups_per_target

    # Grouped reliance should contain one complete information-group roster for
    # each of the five selected Transformers. Missing or duplicated groups would
    # make the reader-facing comparison incomplete or misleading.
    if len(reliance_df) != expected_reliance_rows:
        raise ValueError(
            "Grouped reliance does not contain the complete "
            "selected-target × information-group roster."
        )

    group_counts = (
        reliance_df.groupby("metric")["group"]
        .nunique()
        .reindex(FORECAST_TARGETS, fill_value=0)
    )
    if not group_counts.eq(expected_groups_per_target).all():
        raise ValueError(
            "Every target must contain exactly one result for every reliance group."
        )

    # Both diagnostics must remain inference-only and use the deliberately
    # selected middle-window states rather than silently mixing fitted models
    # from other temporal confirmation windows.
    if reliance_df["retrained"].any():
        raise ValueError(
            "Grouped predictive reliance must never retrain a Transformer."
        )

    if patch_df["retrained"].any():
        raise ValueError(
            "Patch sensitivity must never retrain a Transformer."
        )

    if not reliance_df["temporal_window"].eq("middle").all():
        raise ValueError(
            "Grouped reliance must use only selected middle-window states."
        )

    if not patch_df["temporal_window"].eq("middle").all():
        raise ValueError(
            "Patch sensitivity must use only selected middle-window states."
        )

    # Representations can contain different numbers of patches. Within one
    # fitted model, however, every patch position must appear exactly once as
    # an uninterrupted 0..N-1 sequence.
    for logical_key, group in patch_df.groupby("logical_key"):
        patch_indices = sorted(group["patch_index"].astype(int).tolist())
        expected_indices = list(range(len(patch_indices)))

        if patch_indices != expected_indices:
            raise ValueError(
                f"Patch-sensitivity positions are incomplete for {logical_key}."
            )

    print(
        "Diagnostic QA passed: "
        f"{len(state_registry_df)} selected states · "
        f"{len(reliance_df)} grouped-reliance tests · "
        f"{len(patch_df)} patch-sensitivity tests · "
        "0 retrained models."
    )

Diagnostic QA passed: 5 selected states · 60 grouped-reliance tests · 65 patch-sensitivity tests · 0 retrained models.


**4.5.2.3.5G — Summarize what information and historical periods matter most.**

The full diagnostic tables are intentionally detailed, but the reader should not have to inspect dozens of rows to understand the main result.

The first view reorganizes grouped predictive reliance into a **target × information-source matrix**. Each value is the percentage increase in final-output MAE after that information group's row correspondence is scrambled. Larger positive values indicate stronger predictive dependence; negative values mean the scrambled version happened to forecast slightly better. Groups that cannot actually be permuted remain missing rather than being mislabeled as zero reliance.

The second view shows the **five most damaging historical patch perturbations for each target**. `relative_patch_age = 0` means the most recent historical span, with larger values reaching farther into the lookback. These results describe sensitivity of the fitted forecast—not causal effects and not attention-weight importance.

In [39]:
# ---------------------------------------------------------------------
# 4.5.2.3.5G — Build reader-facing reliance and patch-sensitivity views
# ---------------------------------------------------------------------

transformer_reliance_display_df = pd.DataFrame()
transformer_patch_display_df = pd.DataFrame()

if not transformer_grouped_reliance_df.empty:
    reliance_group_order = (
        list(SEQUENCE_RELIANCE_GROUPS)
        + CONTEXT_RELIANCE_GROUP_ORDER
        + ["Taxi Zone identity"]
    )

    transformer_reliance_display_df = (
        transformer_grouped_reliance_df
        .pivot(
            index="group",
            columns="metric",
            values="final_mae_increase_pct",
        )
        .reindex(
            index=reliance_group_order,
            columns=FORECAST_TARGETS,
        )
        .round(3)
    )

if not transformer_patch_sensitivity_df.empty:
    # Showing only the strongest five spans per target keeps the notebook
    # readable while the complete patch-level evidence remains available in
    # transformer_patch_sensitivity_df for downstream analysis.
    transformer_patch_display_df = (
        transformer_patch_sensitivity_df
        .sort_values(
            ["metric", "final_mae_increase_pct"],
            ascending=[True, False],
        )
        .groupby("metric", group_keys=False)
        .head(5)
        [[
            "metric",
            "candidate_id",
            "patch_index",
            "relative_patch_age",
            "final_mae_increase_pct",
            "raw_mae_increase_pct",
        ]]
        .reset_index(drop=True)
    )

display(transformer_reliance_display_df)
display(transformer_patch_display_df.round(3))

metric,taxi_trip_count,taxi_avg_trip_speed,fhvhv_trip_count,fhvhv_avg_trip_speed,subway_ridership
group,,,,,
Taxi demand history,500.759,1.355,3.228,1.938,6.793
Taxi speed history,11.033,93.012,4.704,0.158,8.757
FHVHV demand history,18.040,1.790,689.637,2.897,9.155
FHVHV speed history,20.280,9.115,6.787,268.094,17.102
Subway ridership history,6.885,0.307,2.797,2.108,669.198
Annual / calendar timing,0.788,15.846,3.414,19.505,53.752
Weekly STL seasonal state,2.167,1.147,8.662,0.924,19.010
Weather,-1.313,1.421,1.423,5.873,1.508
Congestion-pricing regime,NaN,NaN,NaN,NaN,NaN


,metric,candidate_id,patch_index,relative_patch_age,final_mae_increase_pct,raw_mae_increase_pct
0,fhvhv_avg_trip_speed,two_week_overlapping_two_day__higher_capacity_...,7,5,70.046,70.046
1,fhvhv_avg_trip_speed,two_week_overlapping_two_day__higher_capacity_...,6,6,66.611,66.611
2,fhvhv_avg_trip_speed,two_week_overlapping_two_day__higher_capacity_...,0,12,64.677,64.677
3,fhvhv_avg_trip_speed,two_week_overlapping_two_day__higher_capacity_...,12,0,39.211,39.211
4,fhvhv_avg_trip_speed,two_week_overlapping_two_day__higher_capacity_...,8,4,26.189,26.189
5,fhvhv_trip_count,two_week_overlapping_two_day__compact,7,5,237.825,234.353
6,fhvhv_trip_count,two_week_overlapping_two_day__compact,6,6,211.189,208.059
7,fhvhv_trip_count,two_week_overlapping_two_day__compact,0,12,196.614,193.635
8,fhvhv_trip_count,two_week_overlapping_two_day__compact,12,0,66.980,65.446
9,fhvhv_trip_count,two_week_overlapping_two_day__compact,8,4,49.456,48.174


**Findings.** The reliance diagnostics pass an important behavioral sanity check: every selected Transformer depends most strongly on the recent historical sequence for the mobility target it is forecasting.

Scrambling Taxi-demand history increases Taxi-trip MAE by **500.759%**. Scrambling Taxi-speed history increases Taxi-speed MAE by **93.012%**. The corresponding increases are **689.637% for FHVHV demand**, **268.094% for FHVHV speed**, and **669.198% for Subway ridership**. The Transformer is therefore not ignoring its primary time series and relying on unrelated contextual shortcuts.

Supporting context still matters. Multimodal and transportation-connected information increases error by **28.873% for Taxi trips, 25.454% for Taxi speed, 26.429% for FHVHV speed, and 47.711% for Subway** when its row correspondence is disrupted. Calendar timing is particularly influential for the two speed targets and Subway. Weekly STL provides a smaller but still useful longer-run seasonal signal, with reliance ranging from roughly 1% for the speed targets to **19.010% for Subway**.

Weather contributes comparatively little. The **−1.313% Taxi-trip result** means the perturbed forecast happened to perform slightly better on this diagnostic surface; it should be interpreted as weak predictive reliance rather than as evidence that weather causally worsens Taxi demand.

The **Congestion-pricing regime is N/A for this row-permutation diagnostic because `target_cp_regime` does not vary within the selected 28-day middle scoring surfaces**. For Post-CP Primary this is expected by construction. For Full-History Regime-Aware, the training history can span both pre- and post-CP observations, but permuting a constant regime value within one scoring window cannot reveal how strongly the fitted model uses that feature. N/A therefore means *not identifiable by this permutation design*, not *unused by the model*.

Patch sensitivity also shows that the Transformer does not depend only on the newest observations. Historical spans about **five to six patch positions behind the forecast origin** are among the most damaging perturbations for every target, while the oldest and often the newest spans also matter. Because the selected patches overlap, these percentage increases should be interpreted as sensitivity diagnostics rather than additive feature-importance shares.

### 4.5.2.3.6 Assemble the Proposed Target-Specific Transformer Profiles

Batch A narrows history representation, Batch B narrows complete Transformer configurations, and Batch C tests the finalists through time. The diagnostics then show whether the proposals train cleanly, remain practical, and rely on plausible forecasting information.

**4.5.2.3.6A — Resolve the proposed Transformer specification for each target.**

Batch C proposes one finalist for each of the five mobility targets based on pooled raw prediction error across the early, middle, and late confirmation windows. This cell translates each winning `candidate_id` back into the architecture that a reader can understand: how much history it sees, how that history is patched, and how large the Transformer itself is.

This is deliberately a **contract-resolution step**, not another model-selection calculation. The performance evidence has already chosen the proposed finalist; here we verify that each candidate maps cleanly back to one approved representation and one approved Transformer profile.

The result should contain exactly one proposed architecture for each of the five targets before runtime or interpretation evidence is attached.

In [40]:
# ---------------------------------------------------------------------
# 4.5.2.3.6A — Resolve the proposed Transformer specification for each target
# ---------------------------------------------------------------------

transformer_predecision_architecture_df = pd.DataFrame()

if transformer_proposed_target_profile_registry_df.empty:
    print("Architecture assembly waiting: Batch C target-profile evidence is not available.")

else:
    required_evidence = {"metric", "candidate_id", "pooled_raw_mae"}
    missing_evidence = sorted(
        required_evidence - set(transformer_proposed_target_profile_registry_df.columns)
    )

    if missing_evidence:
        raise ValueError(
            f"Batch C target-profile evidence is missing required fields: {missing_evidence}"
        )

    proposal_df = transformer_proposed_target_profile_registry_df.copy()

    # The pre-decision contract needs one concrete proposed architecture per
    # mobility target. An unresolved tie would leave more than one candidate
    # for a target and therefore needs human resolution before we continue.
    complete_target_roster = (
        len(proposal_df) == len(FORECAST_TARGETS)
        and set(proposal_df["metric"]) == set(FORECAST_TARGETS)
        and not proposal_df["metric"].duplicated().any()
    )

    if not complete_target_roster:
        raise ValueError(
            "The pre-decision profile contract requires exactly one proposed "
            "Batch C finalist for each of the five forecasting targets."
        )

    representation_lookup = {
        spec["representation_id"]: spec
        for spec in TRANSFORMER_REPRESENTATION_CANDIDATES
    }
    profile_lookup = {
        spec["profile_id"]: spec
        for spec in TRANSFORMER_PROFILE_CANDIDATES
    }

    architecture_rows = []

    for evidence in proposal_df.itertuples(index=False):
        finalist_id = str(evidence.candidate_id)
        candidate_parts = finalist_id.split("__", maxsplit=1)

        if len(candidate_parts) != 2:
            raise ValueError(
                f"Transformer finalist ID does not encode representation + profile: "
                f"{finalist_id}"
            )

        representation_id, profile_id = candidate_parts

        if representation_id not in representation_lookup:
            raise ValueError(
                f"Unknown Transformer representation in finalist ID: {representation_id}"
            )

        if profile_id not in profile_lookup:
            raise ValueError(
                f"Unknown Transformer profile in finalist ID: {profile_id}"
            )

        representation = representation_lookup[representation_id]
        profile = profile_lookup[profile_id]

        architecture_rows.append({
            "metric": evidence.metric,
            "candidate_id": finalist_id,
            "representation_id": representation_id,
            "lookback_steps": int(representation["lookback_steps"]),
            "patch_length": int(representation["patch_length"]),
            "patch_stride": int(representation["patch_stride"]),
            "profile_id": profile_id,
            "d_model": int(profile["d_model"]),
            "heads": int(profile["n_heads"]),
            "layers": int(profile["n_layers"]),
            "d_ff": int(profile["d_ff"]),
            "dropout": float(profile["dropout"]),
            "pooled_confirmation_raw_mae": float(evidence.pooled_raw_mae),
        })

    transformer_predecision_architecture_df = (
        pd.DataFrame(architecture_rows)
        .sort_values("metric")
        .reset_index(drop=True)
    )

**4.5.2.3.6B — Attach training and interpretation evidence to each proposed profile.**

Architecture alone does not tell us whether a proposed Transformer is operationally reasonable or sufficiently documented for downstream use. This cell therefore attaches the supporting evidence already produced elsewhere in Batch C rather than recomputing anything.

For training behavior, the selected epoch and runtime are summarized across the finalist's early, middle, and late confirmation fits. **Parameter count is different:** it is fixed by the architecture, so all saved confirmation states for the same candidate should report the same value; disagreement is treated as a contract error rather than averaged away.

For interpretation, the cell counts the grouped predictive-reliance tests produced by the selected candidate's **middle-window saved state**. This is a coverage check—not a feature-importance score. The actual reliance magnitudes remain in the detailed diagnostic table.

The resulting table is the compact pre-decision profile contract: what Transformer is proposed for each target, how it trained across confirmation windows, and whether its interpretation evidence is available.

In [41]:
# ---------------------------------------------------------------------
# 4.5.2.3.6B — Attach training and interpretation evidence
# ---------------------------------------------------------------------

transformer_predecision_target_profile_df = pd.DataFrame()

if transformer_predecision_architecture_df.empty:
    print("Profile assembly waiting: proposed target architectures are not available.")

else:
    profile_rows = []

    for architecture in transformer_predecision_architecture_df.itertuples(index=False):
        # Batch C fits the same finalist in early, middle, and late windows.
        # Median epoch/runtime summarizes that temporal variation without
        # pretending those three historical periods are identical.
        runtime_rows = transformer_runtime_capacity_df.loc[
            transformer_runtime_capacity_df["metric"].eq(architecture.metric)
            & transformer_runtime_capacity_df["candidate_id"].eq(architecture.candidate_id)
        ].copy()

        if runtime_rows.empty:
            median_selected_epoch = np.nan
            parameter_count = np.nan
            median_runtime_seconds = np.nan

        else:
            parameter_values = runtime_rows["parameter_count"].dropna().unique()

            # Parameter count comes from the architecture itself. The same
            # candidate should therefore have exactly one model size across
            # all three temporal confirmation windows.
            if len(parameter_values) > 1:
                raise ValueError(
                    f"{architecture.metric} · {architecture.candidate_id} reports "
                    "inconsistent parameter counts across Batch C windows."
                )

            parameter_count = (
                float(parameter_values[0])
                if len(parameter_values) == 1
                else np.nan
            )
            median_selected_epoch = runtime_rows["selected_epoch"].median()
            median_runtime_seconds = runtime_rows["runtime_seconds"].median()

        # Reliance is intentionally based on one selected middle-window state
        # per target. Counting only that candidate/window prevents early/late
        # diagnostics or the losing finalist from inflating coverage.
        if transformer_grouped_reliance_df.empty:
            reliance_rows = pd.DataFrame()

        else:
            required_reliance = {"metric", "candidate_id", "temporal_window"}
            missing_reliance = sorted(
                required_reliance - set(transformer_grouped_reliance_df.columns)
            )

            if missing_reliance:
                raise ValueError(
                    "Grouped reliance output is missing fields needed for the "
                    f"pre-decision contract: {missing_reliance}"
                )

            reliance_rows = transformer_grouped_reliance_df.loc[
                transformer_grouped_reliance_df["metric"].eq(architecture.metric)
                & transformer_grouped_reliance_df["candidate_id"].eq(
                    architecture.candidate_id
                )
                & transformer_grouped_reliance_df["temporal_window"].eq("middle")
            ].copy()

            # One selected state should contribute at most one row per semantic
            # information group. Duplicates would make the coverage count look
            # healthier without representing additional interpretation evidence.
            if not reliance_rows.empty and reliance_rows["group"].duplicated().any():
                raise ValueError(
                    f"{architecture.metric} has duplicate grouped-reliance rows "
                    "for its selected middle-window Transformer."
                )

        profile_rows.append({
            "metric": architecture.metric,
            "representation_id": architecture.representation_id,
            "lookback_steps": architecture.lookback_steps,
            "patch_length": architecture.patch_length,
            "patch_stride": architecture.patch_stride,
            "profile_id": architecture.profile_id,
            "d_model": architecture.d_model,
            "heads": architecture.heads,
            "layers": architecture.layers,
            "d_ff": architecture.d_ff,
            "dropout": architecture.dropout,
            "pooled_confirmation_raw_mae": architecture.pooled_confirmation_raw_mae,
            "median_selected_epoch": median_selected_epoch,
            "parameter_count": parameter_count,
            "median_runtime_seconds": median_runtime_seconds,
            "reliance_rows": int(len(reliance_rows)),
        })

    transformer_predecision_target_profile_df = (
        pd.DataFrame(profile_rows)
        .sort_values("metric")
        .reset_index(drop=True)
    )

display(transformer_predecision_target_profile_df.round(3))

,metric,representation_id,lookback_steps,patch_length,patch_stride,profile_id,d_model,heads,layers,d_ff,dropout,pooled_confirmation_raw_mae,median_selected_epoch,parameter_count,median_runtime_seconds,reliance_rows
0,fhvhv_avg_trip_speed,two_week_overlapping_two_day,70,10,5,higher_capacity_regularized,96,6,3,192,0.150,1.204,10.000,"255,881.000",151.698,12
1,fhvhv_trip_count,two_week_overlapping_two_day,70,10,5,compact,48,4,2,96,0.100,75.820,14.000,"51,049.000",120.591,12
2,subway_ridership,two_week_overlapping_two_day,70,10,5,compact,48,4,2,96,0.100,792.762,13.000,"51,049.000",65.767,12
3,taxi_avg_trip_speed,two_week_overlapping_two_day,70,10,5,compact,48,4,2,96,0.100,2.011,15.000,"51,049.000",59.233,12
4,taxi_trip_count,two_week_overlapping_two_day,70,10,5,higher_capacity_regularized,96,6,3,192,0.150,23.102,14.000,"255,881.000",197.949,12


**Findings.** All five proposed Transformer models converge on the same history representation: **70 Dayparts of recent mobility history divided into overlapping 10-Daypart patches with a five-Daypart stride**.

Capacity remains target-specific. Taxi trips and FHVHV average speed retain the higher-capacity regularized profile with **255,881 parameters**. Taxi average speed, FHVHV trips, and Subway ridership retain the compact **51,049-parameter** profile.

That difference is operationally meaningful: the larger Transformer contains about **five times as many trainable parameters** and generally requires more fitting time. The results therefore support spending that additional capacity only on targets where it produces a meaningful accuracy advantage rather than imposing the largest architecture everywhere.

These are the Transformer family's proposed target-level models, not the final forecasting winners for the project. In particular, the selected Taxi-trip and Subway Transformers still trail the frozen weekly benchmark. Their next test is therefore cross-family comparison against the classical, tree-based, and other neural candidates rather than another broad round of Transformer tuning.

> ✅ Decision — Freeze the target\-specific Transformer profiles

All five targets will use the two\-week overlapping two\-day representation: 70 Dayparts of history, 10\-Daypart patches, and a five\-Daypart stride\.

Higher\-capacity regularized Transformer: Taxi trips and FHVHV average speed\.

Compact Transformer: Taxi average speed, FHVHV trips, and Subway ridership\.

These target\-specific profiles will be shared across forecast horizons 1, 2, and 5\. Later validation may determine that another model family is more accurate, but it will not reopen the Transformer representation/profile search unless a downstream QA failure invalidates the current evidence\.

The decision reflects more than pooled accuracy alone\. Batch C confirms the finalists across early, middle, and late historical windows; selector curves show validation\-supported stopping well below the 30\-epoch ceiling; the compact and higher\-capacity profiles expose a meaningful size/runtime tradeoff; and inference\-only diagnostics show that the selected Transformers rely primarily on plausible recent mobility history rather than obvious contextual shortcuts\.

Taxi trips and Subway ridership still trail the frozen weekly benchmark\. That evidence is preserved for the later cross\-family comparison rather than treated as a reason to continue tuning the Transformer family\.


This closes representation and capacity selection\. It does \*\*not\*\* select a training\-history track or a refit cadence\. The next experiment keeps these five Transformer definitions fixed and asks only whether the learned weights should remain fixed from validation start or be refreshed monthly\.

### Section Summary ### 
The three-stage Transformer tournament narrows a deliberately small design space into five target-specific proposals without evidence that a broader hyperparameter search is necessary.

Batch A establishes **two-week overlapping patches** as the strongest history representation. Batch B reduces the architecture question to compact versus higher-capacity Transformers. Batch C then assigns those two capacity levels by target using pooled exact-common-support evidence from early, middle, and late historical confirmation windows.

The resulting models show plausible training behavior, choose validation-supported training durations well below the 30-epoch ceiling, and rely primarily on the historical mobility signal they are intended to forecast. Their supporting dependence on calendar, multimodal, spatial, and weekly-seasonal information is also interpretable rather than obviously pathological.

The evidence is therefore sufficient to freeze these five **Transformer-family proposals** for downstream validation. This does not imply that every Transformer is competitive with simpler alternatives: Taxi trips remains slightly behind the frozen weekly benchmark, and Subway ridership remains substantially behind it. Those weaknesses should be preserved as evidence for the later cross-family model comparison rather than hidden through additional Transformer tuning.

# 4.5.2.4 Choose the Transformer Refit Cadence

The Transformer architecture is now frozen target by target. One operating question remains before we compare Post-CP and Full-History training:

> **Should each selected Transformer keep one set of validation-start weights, or should it be refitted at the first legal forecast origin of each validation month?**

Both policies continue to receive the information legally available at each forecast row. A validation-start-only model still sees the later row's recent mobility history, calendar and weather context, congestion-pricing state, weekly STL seasonal state, and Taxi Zone identity. The only difference is whether the learned Transformer weights and fitted preprocessing are refreshed.

Because Chapter 3 selected different capacity profiles for different targets, this screen keeps **all five target sentinels**. Each uses the Track × Target × Horizon problem already assigned to it in Chapter 3 and its frozen target-specific Transformer specification.

The ten rolling-validation months create **50 physical Transformer fits: 5 targets × 10 monthly refit origins**. We do not train five extra copies for the fixed policy. The first monthly state for each target is also the validation-start-only state; that exact saved state is reused for inference through the remaining months.

The bounded-development cap of **192 training origins** remains in force for this cadence screen. As in Chapter 3, retained origins remain spread across the legal historical training surface rather than being sampled as disconnected individual rows. The latest 28 legal historical days form the inner-validation tail used to select training duration, and a fresh final model is then trained for exactly the selected number of epochs.

Cadence selection uses **raw MAE on the exact same future rows** as the primary accuracy evidence. MAE is the average absolute size of a forecasting error, so lower values mean the predictions are closer to what actually happened. Nonnegative final-MAE behavior, benchmark skill, RMSE, bias, negative raw forecasts, and runtime provide supporting diagnostics.

Because trip counts, ridership, and speeds use different units, their native MAEs are never averaged across targets. Cross-target synthesis instead uses scale-free percentage improvement and the number of targets favoring each cadence.

The final holdout beginning January 5, 2026 remains untouched. Training-history track selection also remains open: the next bounded experiment will compare Post-CP Primary with Full-History Regime-Aware before the expensive full rolling-validation run.

**4.5.2.4.1A — Freeze the bounded Transformer cadence experiment.**

Start from the five Transformer definitions selected in Chapter 3. This cell verifies that every target has exactly one frozen representation/profile pair, attaches that specification to its existing representative Track × Target × Horizon forecasting problem, and creates the versioned cadence artifact paths.

No model is trained here. The workload estimate uses fit times already observed in Chapter 3, so we can estimate the likely cost without launching another timing benchmark.

The cadence section follows the notebook's recover-or-compute convention. A normal rerun preserves valid artifacts and computes only missing work; `RESET_TRANSFORMER_CADENCE_SCREEN=True` deliberately clears this cadence experiment and rebuilds it.

In [42]:
# ---------------------------------------------------------------------
# 4.5.2.4.1A — Freeze the bounded Transformer cadence experiment
# ---------------------------------------------------------------------

TRANSFORMER_CADENCE_CONTRACT_ID = "transformer_cadence_v1_5target_10refit_192origin_adaptive"
TRANSFORMER_CADENCE_ROOT = INTERMEDIATE_DIR / "tr_cad_v1"
TRANSFORMER_CADENCE_SCHEDULE_PATH = TRANSFORMER_CADENCE_ROOT / "schedule.parquet"
TRANSFORMER_CADENCE_RESULTS_PATH = TRANSFORMER_CADENCE_ROOT / "results.parquet"
TRANSFORMER_CADENCE_FIXED_DIR = TRANSFORMER_CADENCE_ROOT / "fixed_predictions"
TRANSFORMER_CADENCE_FIXED_REGISTRY_PATH = TRANSFORMER_CADENCE_ROOT / "fixed_registry.parquet"
TRANSFORMER_CADENCE_COMPARISON_PATH = TRANSFORMER_CADENCE_ROOT / "comparison.parquet"

TRANSFORMER_CADENCE_EXPECTED_REFITS_PER_JOB = 10
TRANSFORMER_CADENCE_EXPECTED_JOBS = len(FORECAST_TARGETS)
TRANSFORMER_CADENCE_EXPECTED_FITS = (
    TRANSFORMER_CADENCE_EXPECTED_JOBS * TRANSFORMER_CADENCE_EXPECTED_REFITS_PER_JOB
)

# Keep Section 4 self-contained without requiring an edit to an earlier setup
# cell. If the notebook already defines this reset control, preserve its value.
if "RESET_TRANSFORMER_CADENCE_SCREEN" not in globals():
    RESET_TRANSFORMER_CADENCE_SCREEN = False

if RESET_TRANSFORMER_CADENCE_SCREEN:
    # RESET intentionally invalidates the whole cadence experiment so one run
    # cannot mix schedules, fitted states, or forecasts from different contracts.
    reset_experiment_artifacts(TRANSFORMER_CADENCE_ROOT)

TRANSFORMER_CADENCE_ROOT.mkdir(parents=True, exist_ok=True)
TRANSFORMER_CADENCE_FIXED_DIR.mkdir(parents=True, exist_ok=True)

required_profile_columns = {
    "metric", "representation_id", "lookback_steps", "patch_length",
    "patch_stride", "profile_id", "median_runtime_seconds",
}
missing_profile_columns = sorted(
    required_profile_columns - set(transformer_predecision_target_profile_df.columns)
)
if missing_profile_columns:
    raise ValueError(
        "The Chapter 3 target-profile handoff is missing cadence fields: "
        f"{missing_profile_columns}"
    )

# Chapter 3 must hand cadence exactly one complete Transformer definition per
# target. Cadence changes only the refresh schedule, never architecture.
transformer_selected_target_profile_df = transformer_predecision_target_profile_df.copy()

profile_counts = (
    transformer_selected_target_profile_df["metric"]
    .value_counts()
    .reindex(FORECAST_TARGETS, fill_value=0)
)
if not profile_counts.eq(1).all():
    raise ValueError(
        "Cadence requires exactly one frozen Transformer specification "
        "for each of the five forecasting targets."
    )

sentinel_df = pd.DataFrame(
    TRANSFORMER_SENTINEL_KEYS,
    columns=["track", "metric", "horizon"],
)
if sentinel_df["metric"].nunique() != len(FORECAST_TARGETS):
    raise ValueError("The cadence experiment expects exactly one sentinel per target.")

transformer_cadence_jobs_df = sentinel_df.merge(
    transformer_selected_target_profile_df,
    on="metric",
    how="left",
    validate="one_to_one",
)

transformer_cadence_jobs_df["candidate_id"] = (
    transformer_cadence_jobs_df["representation_id"].astype(str)
    + "__"
    + transformer_cadence_jobs_df["profile_id"].astype(str)
)
transformer_cadence_jobs_df["job_id"] = (
    transformer_cadence_jobs_df["metric"].astype(str)
    + "__h"
    + transformer_cadence_jobs_df["horizon"].astype(int).astype(str)
)

if transformer_cadence_jobs_df[
    ["representation_id", "profile_id", "candidate_id"]
].isna().any().any():
    raise ValueError("At least one cadence job lacks a frozen Transformer design.")

# Estimate the 10-refit workload from completed Chapter 3 timings rather than
# spending compute on another benchmark whose result would not affect selection.
runtime_seconds = pd.to_numeric(
    transformer_cadence_jobs_df["median_runtime_seconds"],
    errors="coerce",
)
if not np.isfinite(runtime_seconds).all():
    raise ValueError(
        "Chapter 3 runtime evidence is required for the cadence workload estimate."
    )

transformer_cadence_jobs_df["projected_10_fit_minutes"] = (
    10.0 * runtime_seconds / 60.0
)

display_columns = [
    "metric", "track", "horizon", "representation_id", "profile_id",
    "lookback_steps", "patch_length", "patch_stride",
    "projected_10_fit_minutes",
]
transformer_cadence_plan_display_df = transformer_cadence_jobs_df[
    display_columns
].copy()

transformer_cadence_plan_display_df["metric"] = (
    transformer_cadence_plan_display_df["metric"].map(TARGET_LABELS)
)
transformer_cadence_plan_display_df["track"] = (
    transformer_cadence_plan_display_df["track"].map(TRACK_LABELS)
)
transformer_cadence_plan_display_df["projected_10_fit_minutes"] = (
    transformer_cadence_plan_display_df["projected_10_fit_minutes"].round(3)
)

display(transformer_cadence_plan_display_df)

projected_total_hours = (
    transformer_cadence_jobs_df["projected_10_fit_minutes"].sum() / 60.0
)
print(
    f"Cadence plan: {TRANSFORMER_CADENCE_EXPECTED_JOBS} target-specific jobs × "
    f"{TRANSFORMER_CADENCE_EXPECTED_REFITS_PER_JOB} monthly states = "
    f"{TRANSFORMER_CADENCE_EXPECTED_FITS} physical Transformer fits."
)
print(
    "Validation-start-only reuses each job's first monthly fitted state; "
    "it does not add five duplicate training fits."
)
print(f"Chapter 3 timing projection: approximately {projected_total_hours:.3f} fit-hours.")

,metric,track,horizon,representation_id,profile_id,lookback_steps,patch_length,patch_stride,projected_10_fit_minutes
0,Taxi trips,Post-CP Primary,1,two_week_overlapping_two_day,higher_capacity_regularized,70,10,5,32.991
1,Taxi average speed,Full-History Regime-Aware,5,two_week_overlapping_two_day,compact,70,10,5,9.872
2,FHVHV trips,Post-CP Primary,2,two_week_overlapping_two_day,compact,70,10,5,20.099
3,FHVHV average speed,Full-History Regime-Aware,1,two_week_overlapping_two_day,higher_capacity_regularized,70,10,5,25.283
4,Subway ridership,Post-CP Primary,5,two_week_overlapping_two_day,compact,70,10,5,10.961


Cadence plan: 5 target-specific jobs × 10 monthly states = 50 physical Transformer fits.
Validation-start-only reuses each job's first monthly fitted state; it does not add five duplicate training fits.
Chapter 3 timing projection: approximately 1.653 fit-hours.


**Findings.** The cadence experiment carries forward exactly **five frozen Transformer specifications—one for each mobility target**. All five use the selected two-week overlapping history representation, while capacity remains target-specific: Taxi trips and FHVHV average speed use the higher-capacity regularized profile, and the other three targets use the compact profile.

Testing monthly refitting requires **50 physical Transformer fits: five target-specific forecasting problems × ten monthly fitted states**. The validation-start-only comparison does not require five additional fits because each target's first monthly state is also its fixed-policy starting state.

Based on the completed Chapter 3 runtimes, the full bounded cadence screen was projected to require approximately **1.653 fit-hours**.

**4.5.2.4.1B — Resolve the monthly refit schedule without looking at future outcomes.**

Each selected Transformer is eligible to retrain at the first native rolling-validation forecast origin in each calendar month. The schedule is built only from forecast-origin identity and calendar date; target values do not decide when a refit occurs.

The first scheduled origin is shared by both policies. Monthly refitting trains again at the next nine month boundaries, while validation-start-only keeps the first fitted state unchanged.

In [43]:
# ---------------------------------------------------------------------
# 4.5.2.4.1B — Resolve the ten legal monthly refit origins per target
# ---------------------------------------------------------------------

def build_transformer_cadence_schedule(job):
    """Resolve the first legal rolling-validation forecast origin each month.

    Refit timing depends only on forecast-origin dates. Observed target values
    never decide when training occurs, preventing future outcomes from shaping
    the cadence schedule.
    """
    origin_surface_df = pd.read_parquet(
        FORECASTING_MATRIX_PATHS[job["track"]],
        columns=[
            "modeling_track", "stage", "metric", "horizon",
            "origin_observation_sequence_id", "origin_date",
            "native_track_eligible",
        ],
        filters=[
            ("modeling_track", "==", job["track"]),
            ("metric", "==", job["metric"]),
            ("horizon", "==", int(job["horizon"])),
            ("stage", "==", "rolling_validation"),
            ("native_track_eligible", "==", True),
        ],
    )

    if origin_surface_df.empty:
        raise ValueError(f"{job['job_id']}: no rolling-validation origins found.")

    origin_surface_df["origin_date"] = pd.to_datetime(
        origin_surface_df["origin_date"]
    ).dt.normalize()
    origin_surface_df["origin_observation_sequence_id"] = pd.to_numeric(
        origin_surface_df["origin_observation_sequence_id"],
        errors="raise",
    ).astype("int64")

    # Many Taxi Zones share one canonical forecast origin. Collapse to the time
    # point before choosing the first legal origin of each calendar month.
    origins_df = (
        origin_surface_df[
            ["origin_observation_sequence_id", "origin_date"]
        ]
        .drop_duplicates()
        .sort_values("origin_observation_sequence_id")
        .reset_index(drop=True)
    )
    origins_df["validation_month"] = origins_df["origin_date"].dt.to_period("M")

    schedule_df = (
        origins_df.groupby("validation_month", as_index=False)
        .first()
        .sort_values("origin_observation_sequence_id")
        .reset_index(drop=True)
    )

    if len(schedule_df) != TRANSFORMER_CADENCE_EXPECTED_REFITS_PER_JOB:
        raise ValueError(
            f"{job['job_id']}: expected "
            f"{TRANSFORMER_CADENCE_EXPECTED_REFITS_PER_JOB} validation months, "
            f"found {len(schedule_df)}."
        )

    schedule_df["validation_month"] = schedule_df["validation_month"].astype(str)
    schedule_df["refit_step"] = np.arange(len(schedule_df), dtype=int)
    schedule_df["next_refit_origin_id"] = (
        schedule_df["origin_observation_sequence_id"].shift(-1)
    )
    schedule_df["track"] = job["track"]
    schedule_df["metric"] = job["metric"]
    schedule_df["horizon"] = int(job["horizon"])
    schedule_df["candidate_id"] = job["candidate_id"]
    schedule_df["job_id"] = job["job_id"]
    schedule_df["batch"] = "cadence"
    schedule_df["contract_id"] = TRANSFORMER_CADENCE_CONTRACT_ID
    schedule_df["temporal_window"] = schedule_df["refit_step"].map(
        lambda step: f"cadence_refit_{int(step):02d}"
    )

    return schedule_df


cadence_schedule_parts = [
    build_transformer_cadence_schedule(job._asdict())
    for job in transformer_cadence_jobs_df.itertuples(index=False)
]
transformer_cadence_schedule_df = pd.concat(
    cadence_schedule_parts,
    ignore_index=True,
)

if len(transformer_cadence_schedule_df) != TRANSFORMER_CADENCE_EXPECTED_FITS:
    raise ValueError("Cadence schedule does not contain the expected 50 physical fits.")

if transformer_cadence_schedule_df.duplicated(["job_id", "refit_step"]).any():
    raise ValueError("Cadence schedule contains duplicate Job × Refit identities.")

atomic_write_parquet(
    transformer_cadence_schedule_df,
    TRANSFORMER_CADENCE_SCHEDULE_PATH,
)

# Keep the 50-row schedule as an audit artifact while presenting one concise
# summary row per target to the notebook reader.
transformer_cadence_schedule_display_df = (
    transformer_cadence_schedule_df
    .groupby(["job_id", "metric", "track", "horizon"], as_index=False)
    .agg(
        Refits=("refit_step", "nunique"),
        **{
            "First refit": ("origin_date", "min"),
            "Last refit": ("origin_date", "max"),
            "First origin ID": ("origin_observation_sequence_id", "min"),
            "Last origin ID": ("origin_observation_sequence_id", "max"),
        },
    )
)
transformer_cadence_schedule_display_df["metric"] = (
    transformer_cadence_schedule_display_df["metric"].map(TARGET_LABELS)
)
transformer_cadence_schedule_display_df["track"] = (
    transformer_cadence_schedule_display_df["track"].map(TRACK_LABELS)
)
transformer_cadence_schedule_display_df = (
    transformer_cadence_schedule_display_df.rename(
        columns={
            "metric": "Target",
            "track": "Track",
            "horizon": "Horizon",
        }
    )
)

display(transformer_cadence_schedule_display_df)

,job_id,Target,Track,Horizon,Refits,First refit,Last refit,First origin ID,Last origin ID
0,fhvhv_avg_trip_speed__h1,FHVHV average speed,Full-History Regime-Aware,1,10,2025-04-04,2026-01-01,4125,5481
1,fhvhv_trip_count__h2,FHVHV trips,Post-CP Primary,2,10,2025-04-04,2026-01-01,4124,5481
2,subway_ridership__h5,Subway ridership,Post-CP Primary,5,10,2025-04-04,2026-01-01,4121,5481
3,taxi_avg_trip_speed__h5,Taxi average speed,Full-History Regime-Aware,5,10,2025-04-04,2026-01-01,4121,5481
4,taxi_trip_count__h1,Taxi trips,Post-CP Primary,1,10,2025-04-04,2026-01-01,4125,5481


**Findings.** All five target-specific forecasting problems resolve to the expected **ten monthly refit points**, spanning the ordinary rolling-validation period from the first legal April 2025 forecast origin through the January 2026 boundary.

The exact first origin varies slightly with forecast horizon, as expected, while all five schedules end at the same final monthly boundary. No missing or duplicate Job × Refit identities appear, so the cadence comparison has a complete calendar-driven training schedule before any model accuracy is considered.

**4.5.2.4.2A — Load one cadence job on the exact benchmark-comparison surface.**

A cadence comparison is only useful if both policies are judged on the same physical future rows. This helper loads one selected Track × Target × Horizon problem, removes rows that cannot form the selected Transformer history, excludes the final holdout, and inner-joins the remaining rolling-validation rows to the already-frozen weekly benchmark.

The full forecasting matrices are never loaded across all targets at once. Cadence execution processes one target job at a time to keep memory use bounded.

This cell defines data preparation only; it does not fit a Transformer.

In [44]:
# ---------------------------------------------------------------------
# 4.5.2.4.2A — Load one cadence job on exact benchmark-aligned support
# ---------------------------------------------------------------------

def load_transformer_cadence_job(job):
    """Load one target sentinel and align validation rows to its benchmark.

    The returned rolling-validation surface is the exact physical row
    population on which fixed and monthly Transformer weights are compared.
    """
    track = job["track"]
    metric = job["metric"]
    horizon = int(job["horizon"])
    lookback_steps = int(job["lookback_steps"])
    matrix_path = FORECASTING_MATRIX_PATHS[track]

    matrix_columns = list(dict.fromkeys([
        "modeling_track", "stage", "forecast_sequence_id", "taxi_zone_id",
        "metric", "origin_observation_sequence_id", "origin_date",
        "target_observation_sequence_id", "target_date",
        "target_observed_value", "target_label_available",
        "native_track_eligible", "horizon",
        *TRANSFORMER_CONTEXT_SOURCE_FEATURES,
    ]))

    matrix_df = pd.read_parquet(
        matrix_path,
        columns=matrix_columns,
        filters=[
            ("modeling_track", "==", track),
            ("metric", "==", metric),
            ("horizon", "==", horizon),
            ("native_track_eligible", "==", True),
            ("stage", "in", ["initial_training", "rolling_validation"]),
        ],
    )

    if matrix_df.empty:
        raise ValueError(f"{job['job_id']}: forecasting matrix is empty.")

    matrix_df["origin_date"] = pd.to_datetime(matrix_df["origin_date"]).dt.normalize()
    matrix_df["target_date"] = pd.to_datetime(matrix_df["target_date"]).dt.normalize()

    for column in [
        "origin_observation_sequence_id",
        "target_observation_sequence_id",
    ]:
        matrix_df[column] = pd.to_numeric(
            matrix_df[column],
            errors="raise",
        ).astype("int64")

    matrix_df["_label_available"] = (
        matrix_df["target_label_available"].fillna(False).astype(bool)
        & matrix_df["target_observed_value"].notna()
    )

    # The chosen representation needs a complete historical sequence before an
    # origin can be learned from or scored.
    history_start_id = TRACK_HISTORY_START_SEQUENCE_ID[track] + lookback_steps - 1
    matrix_df["_history_eligible"] = (
        matrix_df["origin_observation_sequence_id"].ge(history_start_id)
    )

    validation_df = matrix_df.loc[
        matrix_df["stage"].eq("rolling_validation")
        & matrix_df["_label_available"]
        & matrix_df["_history_eligible"]
        & matrix_df["target_date"].lt(FINAL_HOLDOUT_START_DATE)
    ].copy()

    if validation_df.empty:
        raise ValueError(f"{job['job_id']}: no legal validation rows remain.")

    benchmark_info = advanced_model_benchmark_registry_df.loc[
        advanced_model_benchmark_registry_df["modeling_track"].eq(track)
        & advanced_model_benchmark_registry_df["metric"].eq(metric)
        & advanced_model_benchmark_registry_df["horizon"].astype(int).eq(horizon)
    ]
    if len(benchmark_info) != 1:
        raise ValueError(
            f"{job['job_id']}: expected exactly one frozen benchmark contract; "
            f"found {len(benchmark_info)}."
        )

    benchmark_id = str(benchmark_info.iloc[0]["baseline_id"])
    benchmark_label = str(benchmark_info.iloc[0]["baseline_label"])
    benchmark_column = f"baseline__{benchmark_id}"

    join_keys = [
        "forecast_sequence_id",
        "origin_observation_sequence_id",
        "target_observation_sequence_id",
        "horizon",
        "target_date",
    ]

    baseline_df = pd.read_parquet(
        BASELINE_PREDICTION_PATHS[track],
        columns=[
            "stage", "metric", *join_keys,
            "target_observed_value", "target_label_available",
            benchmark_column,
        ],
        filters=[
            ("metric", "==", metric),
            ("horizon", "==", horizon),
            ("stage", "==", "rolling_validation"),
        ],
    )
    baseline_df["target_date"] = pd.to_datetime(
        baseline_df["target_date"]
    ).dt.normalize()

    baseline_df = baseline_df.loc[
        baseline_df["target_label_available"].fillna(False).astype(bool)
        & baseline_df["target_observed_value"].notna()
        & baseline_df[benchmark_column].notna()
        & baseline_df["target_date"].lt(FINAL_HOLDOUT_START_DATE)
    ].copy()

    if validation_df.duplicated(join_keys).any():
        raise ValueError(f"{job['job_id']}: validation keys are not unique.")
    if baseline_df.duplicated(join_keys).any():
        raise ValueError(f"{job['job_id']}: benchmark keys are not unique.")

    benchmark_values_df = baseline_df[
        join_keys + ["target_observed_value", benchmark_column]
    ].rename(
        columns={"target_observed_value": "benchmark_target_observed_value"}
    )

    validation_df = validation_df.merge(
        benchmark_values_df,
        on=join_keys,
        how="inner",
        validate="one_to_one",
    )

    if validation_df.empty:
        raise ValueError(
            f"{job['job_id']}: benchmark alignment removed all validation rows."
        )

    if not np.allclose(
        validation_df["target_observed_value"].to_numpy(dtype=float),
        validation_df["benchmark_target_observed_value"].to_numpy(dtype=float),
        rtol=1e-10,
        atol=1e-10,
    ):
        raise ValueError(
            f"{job['job_id']}: forecasting matrix and benchmark targets disagree."
        )

    validation_df = validation_df.drop(
        columns="benchmark_target_observed_value"
    )

    return {
        "matrix_df": matrix_df,
        "validation_df": validation_df,
        "benchmark_id": benchmark_id,
        "benchmark_label": benchmark_label,
        "benchmark_column": benchmark_column,
    }

**4.5.2.4.2B — Build a leakage-safe payload for one monthly refit.**

At each scheduled boundary, a target may enter training only if its target observation would already have been available by that forecast origin. The latest 28 legal historical dates are reserved for the disposable epoch selector; earlier legal history trains that selector, and all legal pre-origin history is then eligible for the fresh final refit.

The same **192-origin bounded-screen sampler used in Chapter 3** is reused here. Complete forecast-origin panels are retained at approximately even positions across the legal training history rather than taking only the newest observations or sampling Taxi Zone rows independently.

Each monthly model scores only the validation rows it owns until the next month boundary. The ten monthly prediction files therefore form a non-overlapping reconstruction of the same complete validation surface.

This cell constructs raw model payloads only; preprocessing remains training-fitted inside the existing Transformer fit engine.

In [45]:
# ---------------------------------------------------------------------
# 4.5.2.4.2B — Build one leakage-safe monthly Transformer payload
# ---------------------------------------------------------------------

def build_transformer_cadence_payload(job_frames, job, refit_row):
    """Build selector, fresh-refit, and one-month evaluation arrays.

    Target observations enter training only after their canonical observation
    sequence has occurred. The existing Chapter 3 origin sampler then bounds
    training cost without breaking cross-zone panels or historical coverage.
    """
    matrix_df = job_frames["matrix_df"]
    validation_df = job_frames["validation_df"]
    metric = job["metric"]
    lookback_steps = int(job["lookback_steps"])
    refit_origin_id = int(refit_row["origin_observation_sequence_id"])

    legal_history_df = matrix_df.loc[
        matrix_df["_label_available"]
        & matrix_df["_history_eligible"]
        & matrix_df["target_observation_sequence_id"].le(refit_origin_id)
    ].copy()

    if legal_history_df.empty:
        raise ValueError(
            f"{job['job_id']} · refit {int(refit_row['refit_step'])}: "
            "no legal training history."
        )

    # Reserve the latest 28 legally observable historical dates for adaptive
    # epoch selection. These rows never fit the disposable selector itself.
    inner_end_date = legal_history_df["target_date"].max()
    inner_start_date = inner_end_date - pd.Timedelta(days=27)

    inner_df = legal_history_df.loc[
        legal_history_df["target_date"].between(
            inner_start_date,
            inner_end_date,
            inclusive="both",
        )
    ].copy()
    selector_train_df = legal_history_df.loc[
        legal_history_df["target_date"].lt(inner_start_date)
    ].copy()

    if selector_train_df.empty or inner_df.empty:
        raise ValueError(
            f"{job['job_id']} · refit {int(refit_row['refit_step'])}: "
            "adaptive selector produced an empty historical split."
        )

    if inner_df["target_date"].nunique() != 28:
        raise ValueError(
            f"{job['job_id']} · refit {int(refit_row['refit_step'])}: "
            "inner validation does not contain 28 complete historical dates."
        )

    if (
        selector_train_df["target_observation_sequence_id"].max()
        >= inner_df["target_observation_sequence_id"].min()
    ):
        raise ValueError(
            f"{job['job_id']} · refit {int(refit_row['refit_step'])}: "
            "selector training and inner validation are not strictly ordered."
        )

    # Reuse the exact bounded-screen sampling policy already used by Chapter 3.
    # It retains complete origins distributed across the legal history rather
    # than changing cadence methodology to a latest-only training sample.
    selector_train_df = thin_training_origins(
        selector_train_df,
        TRANSFORMER_MAX_TRAIN_ORIGINS,
        "origin_observation_sequence_id",
    )
    final_refit_df = thin_training_origins(
        legal_history_df,
        TRANSFORMER_MAX_TRAIN_ORIGINS,
        "origin_observation_sequence_id",
    )

    score_mask = validation_df["origin_observation_sequence_id"].ge(
        refit_origin_id
    )
    next_refit_origin_id = refit_row["next_refit_origin_id"]

    if pd.notna(next_refit_origin_id):
        score_mask &= validation_df["origin_observation_sequence_id"].lt(
            int(next_refit_origin_id)
        )

    evaluation_df = validation_df.loc[score_mask].copy()
    if evaluation_df.empty:
        raise ValueError(
            f"{job['job_id']} · refit {int(refit_row['refit_step'])}: "
            "monthly scoring slice is empty."
        )

    train_payload = _build_split_payload(
        selector_train_df,
        metric,
        lookback_steps,
    )
    inner_payload = _build_split_payload(
        inner_df,
        metric,
        lookback_steps,
    )
    final_payload = _build_split_payload(
        final_refit_df,
        metric,
        lookback_steps,
    )
    eval_payload = _build_split_payload(
        evaluation_df,
        metric,
        lookback_steps,
    )

    eval_keys = evaluation_df[
        [
            "forecast_sequence_id",
            "taxi_zone_id",
            "origin_observation_sequence_id",
            "origin_date",
            "target_observation_sequence_id",
            "target_date",
        ]
    ].copy()

    return {
        "train_sequence_values": train_payload["sequence_values"],
        "train_sequence_masks": train_payload["sequence_masks"],
        "train_context": train_payload["context"],
        "train_zone_index": train_payload["zone_index"],
        "train_target": train_payload["target"],

        "inner_sequence_values": inner_payload["sequence_values"],
        "inner_sequence_masks": inner_payload["sequence_masks"],
        "inner_context": inner_payload["context"],
        "inner_zone_index": inner_payload["zone_index"],
        "inner_target": inner_payload["target"],

        "final_sequence_values": final_payload["sequence_values"],
        "final_sequence_masks": final_payload["sequence_masks"],
        "final_context": final_payload["context"],
        "final_zone_index": final_payload["zone_index"],
        "final_target": final_payload["target"],

        "eval_sequence_values": eval_payload["sequence_values"],
        "eval_sequence_masks": eval_payload["sequence_masks"],
        "eval_context": eval_payload["context"],
        "eval_zone_index": eval_payload["zone_index"],
        "eval_target": eval_payload["target"],
        "eval_keys": eval_keys,
        "benchmark_prediction": pd.to_numeric(
            evaluation_df[job_frames["benchmark_column"]],
            errors="raise",
        ).to_numpy(dtype=np.float32),
    }

**4.5.2.4.2C — Preflight one cadence payload before launching training.**

Before spending time on 50 Transformer fits, build one representative monthly payload and pass it through the same input-contract checks used by Chapter 3.

This is a data and leakage preflight only. It verifies that selector training, historical inner validation, fresh final refit, and monthly scoring all produce aligned model inputs with the frozen target-specific lookback. It does not instantiate or train another Transformer.

In [46]:
# ---------------------------------------------------------------------
# 4.5.2.4.2C — Preflight one monthly cadence payload
# ---------------------------------------------------------------------

cadence_probe_job = transformer_cadence_jobs_df.iloc[0].to_dict()
cadence_probe_refit = (
    transformer_cadence_schedule_df.loc[
        transformer_cadence_schedule_df["job_id"].eq(
            cadence_probe_job["job_id"]
        )
    ]
    .sort_values("refit_step")
    .iloc[0]
    .to_dict()
)

cadence_probe_frames = load_transformer_cadence_job(cadence_probe_job)
cadence_probe_payload = build_transformer_cadence_payload(
    cadence_probe_frames,
    cadence_probe_job,
    cadence_probe_refit,
)

_require_fit_payload(cadence_probe_payload)

cadence_probe_rows_df = pd.DataFrame([
    {
        "Split": "Selector training",
        "Rows": len(cadence_probe_payload["train_target"]),
    },
    {
        "Split": "Inner validation",
        "Rows": len(cadence_probe_payload["inner_target"]),
    },
    {
        "Split": "Fresh final refit",
        "Rows": len(cadence_probe_payload["final_target"]),
    },
    {
        "Split": "Monthly scoring",
        "Rows": len(cadence_probe_payload["eval_target"]),
    },
])

if (
    pd.to_datetime(cadence_probe_payload["eval_keys"]["target_date"])
    >= FINAL_HOLDOUT_START_DATE
).any():
    raise ValueError("Cadence preflight leaked final-holdout targets.")

display(cadence_probe_rows_df)

print(
    "Cadence payload preflight passed: "
    f"{TARGET_LABELS[cadence_probe_job['metric']]} · "
    f"{TRACK_LABELS[cadence_probe_job['track']]} · "
    f"h={int(cadence_probe_job['horizon'])}."
)

cadence_probe_frames = None
cadence_probe_payload = None
gc.collect()

,Split,Rows
0,Selector training,49228
1,Inner validation,35915
2,Fresh final refit,49241
3,Monthly scoring,33583


Cadence payload preflight passed: Taxi trips · Post-CP Primary · h=1.


956

**Findings.** The representative Taxi-trip cadence payload passes the full preflight. Selector training contains **49,228 rows**, the historical inner-validation tail contains **35,915 rows**, the fresh final refit contains **49,241 rows**, and the first monthly scoring window contains **33,583 rows**.

All four model stages are therefore non-empty and aligned under the frozen Transformer input contract, and the monthly scoring surface remains outside the untouched final holdout. The relatively large row counts are consistent with the bounded **192-origin** rule because each retained forecast origin preserves its complete cross-zone panel rather than sampling individual Taxi Zone rows.

**4.5.2.4.3A — Execute or recover the 50 monthly Transformer fits.**

The cadence runner follows the same restartable convention as Chapter 3. Every logical fit is checked on disk before its large forecasting matrix is loaded. A complete checkpoint plus prediction, learning-curve, and model-state artifacts is reused; only missing or incomplete refits are trained.

Each physical fit still follows the frozen neural training procedure: a disposable selector chooses the epoch count from historical inner validation, that model is discarded, and a fresh Transformer with fresh preprocessing is trained for exactly the selected duration.

A crash therefore does not restart the 50-fit experiment. Completed monthly states survive and are recovered on the next run.

In [47]:
# ---------------------------------------------------------------------
# 4.5.2.4.3A — Execute or recover the 50 monthly Transformer fits
# ---------------------------------------------------------------------

TRANSFORMER_CADENCE_KEY_COLUMNS = [
    "job_id",
    "refit_step",
    "origin_observation_sequence_id",
]

cadence_representation_lookup = {
    item["representation_id"]: item
    for item in TRANSFORMER_REPRESENTATION_CANDIDATES
}
cadence_profile_lookup = {
    item["profile_id"]: item
    for item in TRANSFORMER_PROFILE_CANDIDATES
}

expected_cadence_roster_df = (
    transformer_cadence_schedule_df[
        TRANSFORMER_CADENCE_KEY_COLUMNS
    ]
    .drop_duplicates()
    .copy()
)

result_rows = []
recovered_fits = 0
new_fits = 0

for job_number, job_tuple in enumerate(
    transformer_cadence_jobs_df.itertuples(index=False),
    start=1,
):
    job = job_tuple._asdict()
    representation = cadence_representation_lookup[job["representation_id"]]
    profile = cadence_profile_lookup[job["profile_id"]]

    job_schedule_df = (
        transformer_cadence_schedule_df.loc[
            transformer_cadence_schedule_df["job_id"].eq(job["job_id"])
        ]
        .sort_values("refit_step")
        .reset_index(drop=True)
    )

    if len(job_schedule_df) != TRANSFORMER_CADENCE_EXPECTED_REFITS_PER_JOB:
        raise ValueError(f"{job['job_id']}: cadence schedule is incomplete.")

    print(
        f"[{job_number}/{TRANSFORMER_CADENCE_EXPECTED_JOBS}] "
        f"{TARGET_LABELS[job['metric']]} · "
        f"{TRACK_LABELS[job['track']]} · h={int(job['horizon'])}",
        flush=True,
    )

    job_frames = None

    for refit_row in job_schedule_df.to_dict("records"):
        logical_row = pd.Series({**refit_row, "status": "planned"})
        logical_key = {
            column: logical_row[column]
            for column in TRANSFORMER_LOGICAL_KEY_COLUMNS
        }
        token = logical_key_token(
            logical_row,
            TRANSFORMER_LOGICAL_KEY_COLUMNS,
        )

        checkpoint_path = TRANSFORMER_CADENCE_ROOT / f"fit_{token}.parquet"
        prediction_path = TRANSFORMER_CADENCE_ROOT / f"prediction_{token}.parquet"
        curve_path = TRANSFORMER_CADENCE_ROOT / f"curve_{token}.parquet"
        state_path = TRANSFORMER_CADENCE_ROOT / f"state_{token}.pt"

        # Recovery occurs before data preparation. A completed monthly state
        # should require artifact reads, not another large matrix build.
        recovered = recover_transformer_fit(
            checkpoint_path,
            TRANSFORMER_CADENCE_CONTRACT_ID,
            logical_key,
            prediction_path,
            curve_path,
            state_path,
        )

        if recovered is not None:
            result_rows.append(recovered)
            recovered_fits += 1
            continue

        if job_frames is None:
            job_frames = load_transformer_cadence_job(job)

        payload = build_transformer_cadence_payload(
            job_frames,
            job,
            refit_row,
        )

        result, _, _ = run_transformer_logical_fit(
            logical_row,
            representation,
            profile,
            payload,
            TRANSFORMER_CADENCE_ROOT,
        )
        result_rows.append(result)
        new_fits += 1

        if new_fits % 5 == 0:
            print(
                f"  New cadence fits completed this run: {new_fits}",
                flush=True,
            )

    job_frames = None
    gc.collect()
    clear_device_memory()

transformer_cadence_results_df = pd.DataFrame(result_rows)

validate_exact_success_roster(
    transformer_cadence_results_df,
    expected_cadence_roster_df,
    TRANSFORMER_CADENCE_KEY_COLUMNS,
)

if len(transformer_cadence_results_df) != TRANSFORMER_CADENCE_EXPECTED_FITS:
    raise ValueError(
        "Cadence execution did not recover or complete all 50 physical fits."
    )

transformer_cadence_results_df = (
    transformer_cadence_results_df
    .sort_values(["job_id", "refit_step"])
    .reset_index(drop=True)
)
atomic_write_parquet(
    transformer_cadence_results_df,
    TRANSFORMER_CADENCE_RESULTS_PATH,
)

cadence_execution_display_df = (
    transformer_cadence_results_df
    .groupby(["metric", "track", "horizon"], as_index=False)
    .agg(
        successful_refits=("status", "size"),
        min_selected_epoch=("selected_epoch", "min"),
        median_selected_epoch=("selected_epoch", "median"),
        max_selected_epoch=("selected_epoch", "max"),
        fit_minutes=("runtime_seconds", lambda values: values.sum() / 60.0),
    )
)

cadence_execution_display_df["metric"] = (
    cadence_execution_display_df["metric"].map(TARGET_LABELS)
)
cadence_execution_display_df["track"] = (
    cadence_execution_display_df["track"].map(TRACK_LABELS)
)
cadence_execution_display_df[
    ["median_selected_epoch", "fit_minutes"]
] = cadence_execution_display_df[
    ["median_selected_epoch", "fit_minutes"]
].round(3)

display(cadence_execution_display_df)

print(
    f"Cadence restart summary: {recovered_fits} recovered · "
    f"{new_fits} newly fitted · "
    f"{len(transformer_cadence_results_df)}/"
    f"{TRANSFORMER_CADENCE_EXPECTED_FITS} complete."
)

[1/5] Taxi trips · Post-CP Primary · h=1
[2/5] Taxi average speed · Full-History Regime-Aware · h=5
[3/5] FHVHV trips · Post-CP Primary · h=2
[4/5] FHVHV average speed · Full-History Regime-Aware · h=1
[5/5] Subway ridership · Post-CP Primary · h=5


,metric,track,horizon,successful_refits,min_selected_epoch,median_selected_epoch,max_selected_epoch,fit_minutes
0,FHVHV average speed,Full-History Regime-Aware,1,10,2,7.500,20,24.886
1,FHVHV trips,Post-CP Primary,2,10,5,12.500,24,20.086
2,Subway ridership,Post-CP Primary,5,10,4,14.500,21,11.802
3,Taxi average speed,Full-History Regime-Aware,5,10,6,14.500,24,9.902
4,Taxi trips,Post-CP Primary,1,10,3,14.000,23,32.669


Cadence restart summary: 50 recovered · 0 newly fitted · 50/50 complete.


**Findings.** The bounded cadence run completed successfully with **50 of 50 planned Transformer fits** and no failed or missing monthly states. This first execution trained all 50 fits from scratch; none needed to be recovered from an earlier interrupted run.

Adaptive training duration continues to vary meaningfully across targets and months. Median selected epochs range from **7.5 for FHVHV average speed** to **14.5 for Taxi average speed and Subway ridership**, while the complete set ranges from **2 to 24 epochs**. No fit selects the 30-epoch ceiling, providing no evidence that the allowed training envelope is truncating still-improving models.

The 50 fits accumulated approximately **99.345 fit-minutes**, almost exactly matching the pre-run projection of about 1.653 fit-hours. Taxi trips was the most expensive target at **32.669 minutes** across its ten states, followed by FHVHV average speed at **24.886 minutes** and FHVHV trips at **20.086 minutes**.

**4.5.2.4.3B — Define saved-state scoring for the fixed-weight policy.**

The validation-start-only policy needs no second training engine. These helpers restore the first monthly model's saved Transformer weights and training-fitted preprocessing, then score later validation rows with that state unchanged.

“Fixed” refers only to learned model state. Forecast-time history, weather, calendar, weekly STL seasonal state, congestion-pricing state, and other legal context still update for each later forecast row.

Inference is batched so a long validation month does not create one unnecessarily large device tensor. This cell defines scoring machinery only and performs no fitting.

In [48]:
# ---------------------------------------------------------------------
# 4.5.2.4.3B — Define fixed-state Transformer scoring helpers
# ---------------------------------------------------------------------

def restore_cadence_preprocessor(state):
    """Restore the exact training-fitted preprocessing state."""
    preprocessor = TransformerPreprocessor(pd.DataFrame(state["context_contract"]))
    saved = state["preprocessor_state"]

    array_fields = {
        "sequence_means_": "sequence_means",
        "sequence_stds_": "sequence_stds",
    }
    state_fields = {
        "context_medians_": "context_medians",
        "context_means_": "context_means",
        "context_stds_": "context_stds",
        "stl_training_support_": "stl_training_support",
        "stl_mean_": "stl_mean",
        "stl_std_": "stl_std",
        "target_mean_": "target_mean",
        "target_std_": "target_std",
    }

    for attr, key in array_fields.items():
        setattr(preprocessor, attr, np.asarray(saved[key], dtype=float))

    for attr, key in state_fields.items():
        setattr(preprocessor, attr, saved[key])

    preprocessor.is_fitted_ = True
    return preprocessor


def score_restored_transformer_rows(
    state,
    evaluation_df,
    metric,
    benchmark_column,
):
    """Score current legal rows with validation-start weights/preprocessing.

    The model-facing sequence and context are reconstructed from each later
    forecast row. Only learned parameters and training-derived statistics stay
    fixed at their validation-start values.
    """
    representation = state["representation"]
    profile = state["profile"]
    lookback_steps = int(representation["lookback_steps"])

    eval_payload = _build_split_payload(
        evaluation_df,
        metric,
        lookback_steps,
    )
    preprocessor = restore_cadence_preprocessor(state)

    prepared_sequence = preprocessor.transform_sequence(
        eval_payload["sequence_values"],
        eval_payload["sequence_masks"],
    )
    prepared_context = preprocessor.transform_context(eval_payload["context"])

    model = PatchTransformerRegressor(
        NEURAL_SEQUENCE_INPUT_DIM,
        len(state["context_feature_order"]),
        len(state["zone_ids"]),
        representation,
        profile,
    ).to(TORCH_DEVICE)
    model.load_state_dict(state["model_state_dict"])
    model.eval()

    predictions = []

    for start in range(
        0,
        len(evaluation_df),
        selected_transformer_batch_size,
    ):
        stop = min(
            start + selected_transformer_batch_size,
            len(evaluation_df),
        )

        with torch.no_grad():
            standardized = model(
                torch.as_tensor(
                    prepared_sequence[start:stop],
                    dtype=torch.float32,
                    device=TORCH_DEVICE,
                ),
                torch.as_tensor(
                    prepared_context[start:stop],
                    dtype=torch.float32,
                    device=TORCH_DEVICE,
                ),
                torch.as_tensor(
                    eval_payload["zone_index"][start:stop],
                    dtype=torch.long,
                    device=TORCH_DEVICE,
                ),
            ).cpu().numpy()

        predictions.append(standardized)

    prediction_raw = preprocessor.inverse_target(np.concatenate(predictions))
    prediction_final = apply_prediction_output_rule(prediction_raw)

    prediction_df = evaluation_df[
        [
            "forecast_sequence_id",
            "taxi_zone_id",
            "origin_observation_sequence_id",
            "origin_date",
            "target_observation_sequence_id",
            "target_date",
        ]
    ].copy()

    prediction_df["target_observed_value"] = eval_payload["target"]
    prediction_df["benchmark_prediction"] = pd.to_numeric(
        evaluation_df[benchmark_column],
        errors="raise",
    ).to_numpy(dtype=np.float32)
    prediction_df["prediction_raw"] = prediction_raw
    prediction_df["prediction_final"] = prediction_final

    del model, prepared_sequence, prepared_context
    clear_device_memory()

    return prediction_df


def cadence_fixed_prediction_path(job_id):
    """Return one durable fixed-policy prediction artifact per target."""
    token = hashlib.sha1(job_id.encode("utf-8")).hexdigest()[:12]
    return TRANSFORMER_CADENCE_FIXED_DIR / f"fixed_{token}.parquet"


def fixed_prediction_is_valid(path, validation_df):
    """Require one saved fixed forecast for every benchmark-aligned row."""
    if not path.exists():
        return False

    try:
        candidate = pd.read_parquet(path)
    except Exception:
        return False

    key_columns = [
        "forecast_sequence_id",
        "origin_observation_sequence_id",
        "target_observation_sequence_id",
        "target_date",
    ]
    required = set(
        key_columns
        + [
            "target_observed_value",
            "benchmark_prediction",
            "prediction_raw",
            "prediction_final",
            "cadence",
        ]
    )

    if not required.issubset(candidate.columns):
        return False
    if candidate.duplicated(key_columns).any():
        return False

    expected_keys = set(map(tuple, validation_df[key_columns].to_numpy()))
    actual_keys = set(map(tuple, candidate[key_columns].to_numpy()))

    return (
        len(candidate) == len(validation_df)
        and actual_keys == expected_keys
        and candidate["cadence"].eq("validation_start_only").all()
    )

**4.5.2.4.3C — Execute or recover the fixed-weight validation forecasts.**

For each target, the first validation month is copied directly from the first monthly fit so both cadence policies begin from the **same physical trained state and the same predictions**. Months two through ten restore that same state and score their current legal inputs without changing the weights or preprocessing.

One consolidated fixed-policy prediction artifact is saved per target. Valid artifacts are recovered on rerun, so this inference-only work is not repeated unnecessarily.

No additional Transformer training occurs in this cell.

In [49]:
# ---------------------------------------------------------------------
# 4.5.2.4.3C — Execute or recover fixed-weight validation predictions
# ---------------------------------------------------------------------

fixed_registry_rows = []

for job_number, job_tuple in enumerate(
    transformer_cadence_jobs_df.itertuples(index=False),
    start=1,
):
    job = job_tuple._asdict()
    job_frames = load_transformer_cadence_job(job)
    validation_df = job_frames["validation_df"]
    fixed_path = cadence_fixed_prediction_path(job["job_id"])

    if fixed_prediction_is_valid(fixed_path, validation_df):
        fixed_registry_rows.append({
            "job_id": job["job_id"],
            "metric": job["metric"],
            "track": job["track"],
            "horizon": int(job["horizon"]),
            "prediction_path": str(fixed_path),
            "status": "recovered",
        })
        print(
            f"[{job_number}/{TRANSFORMER_CADENCE_EXPECTED_JOBS}] "
            f"{TARGET_LABELS[job['metric']]} · fixed forecasts recovered.",
            flush=True,
        )
        continue

    first_refit = transformer_cadence_results_df.loc[
        transformer_cadence_results_df["job_id"].eq(job["job_id"])
        & transformer_cadence_results_df["refit_step"].astype(int).eq(0)
    ]

    if len(first_refit) != 1:
        raise ValueError(
            f"{job['job_id']}: expected exactly one validation-start state."
        )

    first_refit = first_refit.iloc[0]
    state_path = Path(first_refit["state_path"])
    first_prediction_path = Path(first_refit["prediction_path"])

    if not state_path.exists() or not first_prediction_path.exists():
        raise FileNotFoundError(
            f"{job['job_id']}: validation-start state artifacts are incomplete."
        )

    state = torch.load(
        state_path,
        map_location=TORCH_DEVICE,
        weights_only=False,
    )

    fixed_parts = []
    job_schedule_df = (
        transformer_cadence_schedule_df.loc[
            transformer_cadence_schedule_df["job_id"].eq(job["job_id"])
        ]
        .sort_values("refit_step")
        .reset_index(drop=True)
    )

    # Month one is not recomputed: it is literally the same prediction artifact
    # produced by the first monthly physical fit.
    first_month_df = pd.read_parquet(first_prediction_path).copy()
    first_month_df["cadence"] = "validation_start_only"
    first_month_df["source_refit_step"] = 0
    fixed_parts.append(first_month_df)

    for refit_row in job_schedule_df.iloc[1:].to_dict("records"):
        refit_origin_id = int(refit_row["origin_observation_sequence_id"])
        score_mask = validation_df["origin_observation_sequence_id"].ge(
            refit_origin_id
        )

        if pd.notna(refit_row["next_refit_origin_id"]):
            score_mask &= validation_df["origin_observation_sequence_id"].lt(
                int(refit_row["next_refit_origin_id"])
            )

        evaluation_df = validation_df.loc[score_mask].copy()
        if evaluation_df.empty:
            raise ValueError(
                f"{job['job_id']} · fixed window "
                f"{int(refit_row['refit_step'])}: empty scoring slice."
            )

        month_prediction_df = score_restored_transformer_rows(
            state,
            evaluation_df,
            job["metric"],
            job_frames["benchmark_column"],
        )
        month_prediction_df["track"] = job["track"]
        month_prediction_df["metric"] = job["metric"]
        month_prediction_df["horizon"] = int(job["horizon"])
        month_prediction_df["candidate_id"] = job["candidate_id"]
        month_prediction_df["temporal_window"] = refit_row["temporal_window"]
        month_prediction_df["cadence"] = "validation_start_only"
        month_prediction_df["source_refit_step"] = 0
        fixed_parts.append(month_prediction_df)

    fixed_prediction_df = pd.concat(fixed_parts, ignore_index=True)

    key_columns = [
        "forecast_sequence_id",
        "origin_observation_sequence_id",
        "target_observation_sequence_id",
        "target_date",
    ]

    if len(fixed_prediction_df) != len(validation_df):
        raise ValueError(
            f"{job['job_id']}: fixed policy does not cover the full validation surface."
        )
    if fixed_prediction_df.duplicated(key_columns).any():
        raise ValueError(
            f"{job['job_id']}: fixed policy produced duplicate forecast rows."
        )

    expected_keys = set(map(tuple, validation_df[key_columns].to_numpy()))
    actual_keys = set(map(tuple, fixed_prediction_df[key_columns].to_numpy()))
    if actual_keys != expected_keys:
        raise ValueError(
            f"{job['job_id']}: fixed forecast keys do not match validation."
        )

    atomic_write_parquet(fixed_prediction_df, fixed_path)

    fixed_registry_rows.append({
        "job_id": job["job_id"],
        "metric": job["metric"],
        "track": job["track"],
        "horizon": int(job["horizon"]),
        "prediction_path": str(fixed_path),
        "status": "computed",
    })

    print(
        f"[{job_number}/{TRANSFORMER_CADENCE_EXPECTED_JOBS}] "
        f"{TARGET_LABELS[job['metric']]} · fixed full-span forecasts saved.",
        flush=True,
    )

    state = None
    job_frames = None
    gc.collect()
    clear_device_memory()

transformer_cadence_fixed_registry_df = pd.DataFrame(fixed_registry_rows)

if len(transformer_cadence_fixed_registry_df) != TRANSFORMER_CADENCE_EXPECTED_JOBS:
    raise ValueError(
        "Expected one fixed-policy prediction artifact for each target."
    )

atomic_write_parquet(
    transformer_cadence_fixed_registry_df,
    TRANSFORMER_CADENCE_FIXED_REGISTRY_PATH,
)

fixed_registry_display_df = transformer_cadence_fixed_registry_df.copy()
fixed_registry_display_df["metric"] = (
    fixed_registry_display_df["metric"].map(TARGET_LABELS)
)
fixed_registry_display_df["track"] = (
    fixed_registry_display_df["track"].map(TRACK_LABELS)
)
fixed_registry_display_df = fixed_registry_display_df.rename(
    columns={
        "metric": "Target",
        "track": "Track",
        "horizon": "Horizon",
        "status": "Artifact status",
    }
)

display(
    fixed_registry_display_df[
        ["Target", "Track", "Horizon", "Artifact status"]
    ]
)

[1/5] Taxi trips · fixed forecasts recovered.
[2/5] Taxi average speed · fixed forecasts recovered.
[3/5] FHVHV trips · fixed forecasts recovered.
[4/5] FHVHV average speed · fixed forecasts recovered.
[5/5] Subway ridership · fixed forecasts recovered.


,Target,Track,Horizon,Artifact status
0,Taxi trips,Post-CP Primary,1,recovered
1,Taxi average speed,Full-History Regime-Aware,5,recovered
2,FHVHV trips,Post-CP Primary,2,recovered
3,FHVHV average speed,Full-History Regime-Aware,1,recovered
4,Subway ridership,Post-CP Primary,5,recovered


**Findings.** The validation-start-only comparison surface was reconstructed successfully for all **five targets**. Each target now has one complete fixed-weight forecast artifact covering the same rolling-validation population used by the monthly policy.

These forecasts required **no additional Transformer fitting**. The first monthly fitted state was reused as the validation-start model, while later legal history and context continued to update at prediction time with the learned Transformer weights and training-fitted preprocessing held fixed. The cadence comparison can therefore isolate the value of refreshing the learned model rather than confounding it with differences in forecast-time information.

**4.5.2.4.4A — Compare validation-start-only weights with monthly refitting.**

Now pair the two cadence policies on exactly the same physical forecast rows for each target.

**Raw MAE is the primary cadence-selection metric.** It measures the average absolute forecasting miss before the project's nonnegative output floor is applied. Lower MAE is better, and a positive “monthly improvement” means monthly refitting reduced that error relative to keeping the validation-start weights fixed.

Final MAE shows performance after the common nonnegative output rule. **Benchmark skill** reports improvement relative to the frozen “previous week, same Daypart” forecast: positive skill beats that benchmark and negative skill trails it. RMSE gives extra weight to unusually large misses, while bias shows whether errors tend to lean systematically high or low.

The comparison also counts negative raw forecasts and measures the actual fitting-cost multiplier. Native-unit errors remain target-specific; only percentage improvement and target wins are synthesized across the five mobility outcomes.

In [50]:
# ---------------------------------------------------------------------
# 4.5.2.4.4A — Compare the two cadence policies on exact common rows
# ---------------------------------------------------------------------

cadence_comparison_rows = []

for job_tuple in transformer_cadence_jobs_df.itertuples(index=False):
    job = job_tuple._asdict()

    monthly_result_df = (
        transformer_cadence_results_df.loc[
            transformer_cadence_results_df["job_id"].eq(job["job_id"])
        ]
        .sort_values("refit_step")
        .reset_index(drop=True)
    )

    monthly_prediction_df = pd.concat(
        [
            pd.read_parquet(Path(path))
            for path in monthly_result_df["prediction_path"].astype(str)
        ],
        ignore_index=True,
    )
    monthly_prediction_df["cadence"] = "monthly"

    fixed_registry_row = transformer_cadence_fixed_registry_df.loc[
        transformer_cadence_fixed_registry_df["job_id"].eq(job["job_id"])
    ]
    if len(fixed_registry_row) != 1:
        raise ValueError(
            f"{job['job_id']}: fixed prediction registry is not unique."
        )

    fixed_prediction_df = pd.read_parquet(
        fixed_registry_row.iloc[0]["prediction_path"]
    )

    pair_keys = [
        "forecast_sequence_id",
        "origin_observation_sequence_id",
        "target_observation_sequence_id",
        "target_date",
    ]

    monthly_df = monthly_prediction_df[
        pair_keys
        + [
            "target_observed_value",
            "benchmark_prediction",
            "prediction_raw",
            "prediction_final",
        ]
    ].rename(
        columns={
            "prediction_raw": "monthly_prediction_raw",
            "prediction_final": "monthly_prediction_final",
        }
    )

    fixed_df = fixed_prediction_df[
        pair_keys
        + [
            "target_observed_value",
            "benchmark_prediction",
            "prediction_raw",
            "prediction_final",
        ]
    ].rename(
        columns={
            "target_observed_value": "fixed_target_observed_value",
            "benchmark_prediction": "fixed_benchmark_prediction",
            "prediction_raw": "fixed_prediction_raw",
            "prediction_final": "fixed_prediction_final",
        }
    )

    if monthly_df.duplicated(pair_keys).any() or fixed_df.duplicated(pair_keys).any():
        raise ValueError(
            f"{job['job_id']}: cadence predictions are not unique by forecast row."
        )

    paired_df = monthly_df.merge(
        fixed_df,
        on=pair_keys,
        how="inner",
        validate="one_to_one",
    )

    if len(paired_df) != len(monthly_df) or len(paired_df) != len(fixed_df):
        raise ValueError(
            f"{job['job_id']}: cadence policies do not share exact row support."
        )

    if not np.allclose(
        paired_df["target_observed_value"].to_numpy(dtype=float),
        paired_df["fixed_target_observed_value"].to_numpy(dtype=float),
        rtol=1e-10,
        atol=1e-10,
    ):
        raise ValueError(
            f"{job['job_id']}: cadence policies disagree on observed targets."
        )

    if not np.allclose(
        paired_df["benchmark_prediction"].to_numpy(dtype=float),
        paired_df["fixed_benchmark_prediction"].to_numpy(dtype=float),
        rtol=1e-10,
        atol=1e-10,
    ):
        raise ValueError(
            f"{job['job_id']}: cadence policies disagree on benchmark forecasts."
        )

    actual = paired_df["target_observed_value"].to_numpy(dtype=float)
    benchmark = paired_df["benchmark_prediction"].to_numpy(dtype=float)
    fixed_raw = paired_df["fixed_prediction_raw"].to_numpy(dtype=float)
    fixed_final = paired_df["fixed_prediction_final"].to_numpy(dtype=float)
    monthly_raw = paired_df["monthly_prediction_raw"].to_numpy(dtype=float)
    monthly_final = paired_df["monthly_prediction_final"].to_numpy(dtype=float)

    benchmark_metrics = regression_metrics(actual, benchmark)
    fixed_raw_metrics = regression_metrics(actual, fixed_raw)
    fixed_final_metrics = regression_metrics(actual, fixed_final)
    monthly_raw_metrics = regression_metrics(actual, monthly_raw)
    monthly_final_metrics = regression_metrics(actual, monthly_final)

    fixed_raw_mae = fixed_raw_metrics["mae"]
    monthly_raw_mae = monthly_raw_metrics["mae"]
    benchmark_mae = benchmark_metrics["mae"]

    cadence_comparison_rows.append({
        "job_id": job["job_id"],
        "metric": job["metric"],
        "track": job["track"],
        "horizon": int(job["horizon"]),
        "representation_id": job["representation_id"],
        "profile_id": job["profile_id"],
        "common_rows": len(paired_df),

        "fixed_raw_mae": fixed_raw_mae,
        "monthly_raw_mae": monthly_raw_mae,
        "monthly_vs_fixed_raw_mae_improvement_pct": (
            100.0 * (fixed_raw_mae - monthly_raw_mae) / fixed_raw_mae
            if fixed_raw_mae > 0 else np.nan
        ),

        "fixed_final_mae": fixed_final_metrics["mae"],
        "monthly_final_mae": monthly_final_metrics["mae"],
        "benchmark_mae": benchmark_mae,

        "fixed_raw_skill_vs_benchmark_pct": (
            100.0 * (benchmark_mae - fixed_raw_mae) / benchmark_mae
            if benchmark_mae > 0 else np.nan
        ),
        "monthly_raw_skill_vs_benchmark_pct": (
            100.0 * (benchmark_mae - monthly_raw_mae) / benchmark_mae
            if benchmark_mae > 0 else np.nan
        ),

        "fixed_raw_rmse": fixed_raw_metrics["rmse"],
        "monthly_raw_rmse": monthly_raw_metrics["rmse"],
        "fixed_raw_bias": fixed_raw_metrics["bias"],
        "monthly_raw_bias": monthly_raw_metrics["bias"],

        "fixed_negative_raw_predictions": int((fixed_raw < 0).sum()),
        "monthly_negative_raw_predictions": int((monthly_raw < 0).sum()),

        "fixed_fit_seconds": float(
            monthly_result_df.loc[
                monthly_result_df["refit_step"].astype(int).eq(0),
                "runtime_seconds",
            ].iloc[0]
        ),
        "monthly_fit_seconds": float(
            monthly_result_df["runtime_seconds"].sum()
        ),
    })

transformer_cadence_comparison_df = pd.DataFrame(cadence_comparison_rows)

transformer_cadence_comparison_df["fit_time_multiplier"] = (
    transformer_cadence_comparison_df["monthly_fit_seconds"]
    / transformer_cadence_comparison_df["fixed_fit_seconds"]
)

if len(transformer_cadence_comparison_df) != TRANSFORMER_CADENCE_EXPECTED_JOBS:
    raise ValueError("Expected exactly five target-level cadence comparisons.")

atomic_write_parquet(
    transformer_cadence_comparison_df,
    TRANSFORMER_CADENCE_COMPARISON_PATH,
)

cadence_display_df = transformer_cadence_comparison_df[
    [
        "metric", "track", "horizon", "profile_id", "common_rows",
        "fixed_raw_mae", "monthly_raw_mae",
        "monthly_vs_fixed_raw_mae_improvement_pct",
        "fixed_raw_skill_vs_benchmark_pct",
        "monthly_raw_skill_vs_benchmark_pct",
        "fixed_final_mae", "monthly_final_mae",
        "fit_time_multiplier",
    ]
].copy()

cadence_display_df["metric"] = cadence_display_df["metric"].map(TARGET_LABELS)
cadence_display_df["track"] = cadence_display_df["track"].map(TRACK_LABELS)

cadence_display_df = cadence_display_df.rename(
    columns={
        "metric": "Target",
        "track": "Track",
        "horizon": "Horizon",
        "profile_id": "Transformer profile",
        "common_rows": "Common rows",
        "fixed_raw_mae": "Fixed raw MAE",
        "monthly_raw_mae": "Monthly raw MAE",
        "monthly_vs_fixed_raw_mae_improvement_pct": "Monthly raw-MAE improvement %",
        "fixed_raw_skill_vs_benchmark_pct": "Fixed benchmark skill %",
        "monthly_raw_skill_vs_benchmark_pct": "Monthly benchmark skill %",
        "fixed_final_mae": "Fixed final MAE",
        "monthly_final_mae": "Monthly final MAE",
        "fit_time_multiplier": "Fit-time multiplier",
    }
)

numeric_display_columns = [
    "Fixed raw MAE",
    "Monthly raw MAE",
    "Monthly raw-MAE improvement %",
    "Fixed benchmark skill %",
    "Monthly benchmark skill %",
    "Fixed final MAE",
    "Monthly final MAE",
    "Fit-time multiplier",
]
cadence_display_df[numeric_display_columns] = (
    cadence_display_df[numeric_display_columns].round(3)
)

display(cadence_display_df)

monthly_wins = int(
    transformer_cadence_comparison_df[
        "monthly_vs_fixed_raw_mae_improvement_pct"
    ].gt(0).sum()
)
median_monthly_improvement = float(
    transformer_cadence_comparison_df[
        "monthly_vs_fixed_raw_mae_improvement_pct"
    ].median()
)
median_fit_multiplier = float(
    transformer_cadence_comparison_df["fit_time_multiplier"].median()
)

print(
    "Cadence synthesis: "
    f"monthly wins {monthly_wins}/{TRANSFORMER_CADENCE_EXPECTED_JOBS} targets · "
    f"median raw-MAE improvement {median_monthly_improvement:.3f}% · "
    f"median fit-time multiplier {median_fit_multiplier:.3f}×."
)

,Target,Track,Horizon,Transformer profile,Common rows,Fixed raw MAE,Monthly raw MAE,Monthly raw-MAE improvement %,Fixed benchmark skill %,Monthly benchmark skill %,Fixed final MAE,Monthly final MAE,Fit-time multiplier
0,Taxi trips,Post-CP Primary,1,higher_capacity_regularized,352715,30.407,20.931,31.164,-50.186,-3.382,24.928,20.216,9.946
1,Taxi average speed,Full-History Regime-Aware,5,compact,168305,1.984,1.974,0.507,13.580,14.018,1.984,1.974,9.907
2,FHVHV trips,Post-CP Primary,2,compact,352715,102.508,73.923,27.886,-47.210,-6.160,101.347,73.443,21.196
3,FHVHV average speed,Full-History Regime-Aware,1,higher_capacity_regularized,352319,1.159,1.125,2.923,12.778,15.328,1.159,1.125,5.342
4,Subway ridership,Post-CP Primary,5,compact,204385,935.358,737.429,21.161,-72.720,-36.171,888.295,730.649,7.484


Cadence synthesis: monthly wins 5/5 targets · median raw-MAE improvement 21.161% · median fit-time multiplier 9.907×.


**Findings.** Monthly refitting improves raw MAE for **all five representative Transformer problems**, with a median improvement of **21.161%** relative to keeping the validation-start model fixed.

The improvement also survives the common nonnegative output rule: monthly final MAE is lower for every target. Benchmark-relative performance moves in the same direction as well. Both speed targets remain above the frozen weekly benchmark and improve further, while Taxi trips, FHVHV trips, and Subway ridership substantially reduce—but do not fully eliminate—their benchmark deficits.

The gain comes with a meaningful computational tradeoff. Monthly refitting requires a median **9.907×** the fitting time of validation-start-only training, with substantial variation across targets because adaptive epoch selection changes the amount of training performed at each monthly refresh.

The evidence therefore favors monthly refitting on more than one criterion: the improvement is **directionally consistent across all five targets**, remains present after output constraints are applied, and generally improves benchmark-relative performance despite the additional fitting cost.

**4.5.2.4.4B — Visualize the scale-free cadence effect by target.**

The comparison table preserves each target's native-unit errors, but the family-wide cadence question is easier to read on a common percentage scale.

Positive bars mean monthly refitting lowered raw MAE relative to keeping the validation-start weights fixed. Negative bars favor the fixed policy. The zero line therefore marks the point where the two approaches are equally accurate.

Hover retains the exact Track × Horizon problem, both raw MAEs, benchmark skill, and the measured fitting-cost multiplier.

In [51]:
# ---------------------------------------------------------------------
# 4.5.2.4.4B — Visualize the scale-free cadence effect by target
# ---------------------------------------------------------------------

chart_df = transformer_cadence_comparison_df.copy()
chart_df["Target"] = chart_df["metric"].map(TARGET_LABELS)
chart_df["Track"] = chart_df["track"].map(TRACK_LABELS)

cadence_fig = go.Figure(
    go.Bar(
        x=chart_df["monthly_vs_fixed_raw_mae_improvement_pct"],
        y=chart_df["Target"],
        orientation="h",
        marker_color=BRAND_COLORS["dark_teal"],
        customdata=np.column_stack([
            chart_df["Track"],
            chart_df["horizon"],
            chart_df["fixed_raw_mae"],
            chart_df["monthly_raw_mae"],
            chart_df["fixed_raw_skill_vs_benchmark_pct"],
            chart_df["monthly_raw_skill_vs_benchmark_pct"],
            chart_df["fit_time_multiplier"],
        ]),
        hovertemplate=(
            "<b>%{y}</b><br>"
            "Track: %{customdata[0]}<br>"
            "Horizon: %{customdata[1]:.0f}<br>"
            "Monthly raw-MAE improvement: %{x:.3f}%<br>"
            "Fixed raw MAE: %{customdata[2]:.3f}<br>"
            "Monthly raw MAE: %{customdata[3]:.3f}<br>"
            "Fixed benchmark skill: %{customdata[4]:.3f}%<br>"
            "Monthly benchmark skill: %{customdata[5]:.3f}%<br>"
            "Fit-time multiplier: %{customdata[6]:.3f}×"
            "<extra></extra>"
        ),
    )
)

cadence_fig.add_vline(
    x=0,
    line_dash="dash",
    line_color=BRAND_COLORS["terracotta"],
)

cadence_fig = apply_branding(cadence_fig)
cadence_fig.update_layout(
    title=(
        "Transformer Refit Cadence — "
        "Monthly Raw-MAE Improvement vs Fixed Weights"
    ),
    xaxis_title="Monthly improvement over validation-start-only raw MAE (%)",
    yaxis_title=None,
    height=430,
    margin={"l": 180, "r": 40, "t": 80, "b": 70},
)
cadence_fig.update_yaxes(autorange="reversed")
cadence_fig.show()

**Findings.** The visual pattern reveals an important difference between **demand forecasting and speed forecasting**.

Monthly refitting produces large gains for the three demand-oriented targets—Taxi trips, FHVHV trips, and Subway ridership—while the two average-speed targets improve much less. Taxi average speed is especially close to cadence-invariant, whereas the demand forecasts appear much more sensitive to keeping the learned Transformer state current.

This suggests that the value of monthly refitting is not simply a uniform percentage boost applied to every forecasting problem. Instead, the benefit is concentrated where the underlying mobility volume appears to change enough over the validation period that a model fitted only once becomes progressively stale.

The absence of any negative bars is also important: although the magnitude varies considerably by target, **monthly refresh never sacrifices raw-MAE accuracy in this screen**. The cadence decision is therefore supported by broad directional consistency even though the practical benefit is much larger for demand than for speed.

> ✅ Decision — Use monthly Transformer refitting
Monthly refitting is frozen as the common Transformer cadence across all five targets and forecast horizons\. Monthly refitting lowers raw MAE for all five representative forecasting problems, with a median improvement of 21\.161% over keeping the validation\-start weights fixed\. The advantage also remains after the common nonnegative output rule is applied\.

The computational cost is materially higher—a median 9\.907× fit\-time multiplier—but the accuracy gain is substantial for the three demand targets and positive for both speed targets\. No target becomes less accurate under monthly refitting\.

Each monthly refresh will continue to use the frozen 28\-day historical epoch selector → fresh preprocessing → fresh final Transformer refit procedure\. The cadence itself is now fixed and will not vary by target or horizon unless a downstream QA failure invalidates this evidence\.

### Section Summary

Section 4.5.2.4 compared keeping one validation-start Transformer fit with refreshing the model monthly as new legal history became available. Across all five representative forecasting problems, monthly refitting reduced raw MAE, with a median improvement of **21.161%** and especially large gains for Taxi trips, FHVHV trips, and Subway ridership; both speed targets also improved, though by smaller margins. The benefit remained after the nonnegative output rule and improved benchmark-relative performance for every target, at the cost of a median **9.907×** increase in fitting time. Based on this consistent accuracy gain, **monthly Transformer refitting was frozen as the common cadence across all targets and horizons**.

# 4.5.2.5 Choose the Transformer Training-History Track

The Transformer representation, target-specific capacity, and monthly refit cadence are now fixed. One within-family modeling choice remains unresolved: **how much historical data should each target learn from?**

The **Post-CP Primary** track trains only on mobility observed under congestion pricing. The **Full-History Regime-Aware** track can also learn from the longer 2023–2024 record while receiving the explicit pre/post-policy regime indicator as model context. More data is not automatically better: older history may provide useful recurring structure, but it may also describe a transportation system operating under a different policy regime.

The comparison is intentionally bounded. For each of the five targets, three monthly refits spread across ordinary validation—positions **0, 4, and 8**—represent early, middle, and later conditions. The completed cadence experiment already supplies those three fitted states on one modeling track. We therefore train only the **opposite-track mirror**, requiring **15 new Transformer fits rather than another complete two-track validation run**.

Within each target, the history representation, Transformer profile, forecast horizon, monthly position, 192-origin development cap, adaptive epoch-selection procedure, and stochastic starting seed remain fixed. Modeling track is the intended changed modeling choice.

Each track retains its own legally valid monthly forecast origin and native Taxi Zone population. Accuracy is compared only after the two sides are intersected to the **exact same future forecast rows**. The primary decision metric remains pooled raw MAE across the three sampled months; errors are recomputed from row-level predictions rather than averaging three already-aggregated monthly MAEs.

The Full-History model also receives an explicit integrity check. After bounded origin sampling, its selector-training and final-refit surfaces must still contain both pre-CP and post-CP observations. Post-CP Primary must contain only post-policy observations. This prevents the development cap from accidentally turning a nominal Full-History fit into an effectively Post-CP fit.

The final 2026 holdout remains untouched.

**4.5.2.5.1A — Freeze the matched training-history bakeoff.**

Build one matched comparison for each mobility target using the Transformer specification already frozen in Chapter 3 and the monthly cadence selected in Section 4.

The three sampled refit positions—0, 4, and 8—are taken directly from the completed cadence schedule. For each target, the existing cadence artifacts provide one side of the comparison. The new roster contains only the three opposite-track mirror fits.

The paired original and mirror fits also share the original fit's deterministic seed. This removes stochastic initialization as an avoidable source of difference while still allowing each modeling track to use its own legal training history and native forecasting population.

In [52]:
# ---------------------------------------------------------------------
# 4.5.2.5.1A — Freeze the matched training-history bakeoff
# ---------------------------------------------------------------------

TRANSFORMER_TRACK_BAKEOFF_REFIT_STEPS = [0, 4, 8]
TRANSFORMER_TRACK_BAKEOFF_CONTRACT_ID = (
    "transformer_track_bakeoff_v1_monthly_5target_3month_"
    "same_seed_192origin_regimeqa"
)
TRANSFORMER_TRACK_BAKEOFF_ROOT = INTERMEDIATE_DIR / "tr_track_v1"
TRANSFORMER_TRACK_BAKEOFF_RESULTS_PATH = (
    TRANSFORMER_TRACK_BAKEOFF_ROOT / "mirror_results.parquet"
)
TRANSFORMER_TRACK_BAKEOFF_COMPARISON_PATH = (
    TRANSFORMER_TRACK_BAKEOFF_ROOT / "comparison.parquet"
)

TRANSFORMER_TRACK_BAKEOFF_EXPECTED_TARGETS = len(FORECAST_TARGETS)
TRANSFORMER_TRACK_BAKEOFF_EXPECTED_MIRROR_FITS = (
    TRANSFORMER_TRACK_BAKEOFF_EXPECTED_TARGETS
    * len(TRANSFORMER_TRACK_BAKEOFF_REFIT_STEPS)
)

if "RESET_TRANSFORMER_TRACK_BAKEOFF" not in globals():
    RESET_TRANSFORMER_TRACK_BAKEOFF = False

if RESET_TRANSFORMER_TRACK_BAKEOFF:
    reset_experiment_artifacts(TRANSFORMER_TRACK_BAKEOFF_ROOT)

TRANSFORMER_TRACK_BAKEOFF_ROOT.mkdir(parents=True, exist_ok=True)

opposite_track = {
    "post_cp_primary": "full_history_regime_aware",
    "full_history_regime_aware": "post_cp_primary",
}

if set(transformer_cadence_jobs_df["metric"]) != set(FORECAST_TARGETS):
    raise ValueError(
        "Track bakeoff requires the five target-specific cadence jobs."
    )

if set(transformer_cadence_results_df["refit_step"].astype(int)) != set(range(10)):
    raise ValueError(
        "Track bakeoff requires the completed ten-refit monthly cadence evidence."
    )

target_rows = []
original_fit_rows = []
mirror_fit_rows = []

for job_tuple in transformer_cadence_jobs_df.itertuples(index=False):
    job = job_tuple._asdict()
    original_track = job["track"]

    if original_track not in opposite_track:
        raise ValueError(f"Unexpected ordinary modeling track: {original_track}")

    mirror_track = opposite_track[original_track]

    original_schedule_df = (
        transformer_cadence_schedule_df.loc[
            transformer_cadence_schedule_df["job_id"].eq(job["job_id"])
            & transformer_cadence_schedule_df["refit_step"].astype(int).isin(
                TRANSFORMER_TRACK_BAKEOFF_REFIT_STEPS
            )
        ]
        .sort_values("refit_step")
        .reset_index(drop=True)
    )

    original_results_df = (
        transformer_cadence_results_df.loc[
            transformer_cadence_results_df["job_id"].eq(job["job_id"])
            & transformer_cadence_results_df["refit_step"].astype(int).isin(
                TRANSFORMER_TRACK_BAKEOFF_REFIT_STEPS
            )
        ]
        .sort_values("refit_step")
        .reset_index(drop=True)
    )

    if len(original_schedule_df) != 3 or len(original_results_df) != 3:
        raise ValueError(
            f"{job['job_id']}: expected three completed original-track refits."
        )

    mirror_job = {
        **job,
        "track": mirror_track,
        "job_id": f"{job['job_id']}__{mirror_track}",
    }

    mirror_schedule_df = (
        build_transformer_cadence_schedule(mirror_job)
        .loc[
            lambda frame: frame["refit_step"].astype(int).isin(
                TRANSFORMER_TRACK_BAKEOFF_REFIT_STEPS
            )
        ]
        .sort_values("refit_step")
        .reset_index(drop=True)
    )

    if len(mirror_schedule_df) != 3:
        raise ValueError(
            f"{job['job_id']}: expected three opposite-track monthly positions."
        )

    if (
        original_schedule_df["validation_month"].tolist()
        != mirror_schedule_df["validation_month"].tolist()
    ):
        raise ValueError(
            f"{job['job_id']}: original and mirror schedules do not represent "
            "the same sampled calendar months."
        )

    target_rows.append({
        "job_id": job["job_id"],
        "metric": job["metric"],
        "horizon": int(job["horizon"]),
        "representation_id": job["representation_id"],
        "profile_id": job["profile_id"],
        "lookback_steps": int(job["lookback_steps"]),
        "patch_length": int(job["patch_length"]),
        "patch_stride": int(job["patch_stride"]),
        "original_track": original_track,
        "mirror_track": mirror_track,
        "sampled_months": ", ".join(original_schedule_df["validation_month"]),
    })

    for step in TRANSFORMER_TRACK_BAKEOFF_REFIT_STEPS:
        original_result = original_results_df.loc[
            original_results_df["refit_step"].astype(int).eq(step)
        ].iloc[0]

        mirror_schedule = mirror_schedule_df.loc[
            mirror_schedule_df["refit_step"].astype(int).eq(step)
        ].iloc[0]

        matched_seed = stable_transformer_fit_seed(
            original_result["contract_id"],
            original_result["track"],
            original_result["metric"],
            int(original_result["horizon"]),
            original_result["candidate_id"],
            original_result["temporal_window"],
        )

        original_fit_rows.append({
            **original_result.to_dict(),
            "source_job_id": job["job_id"],
            "matched_refit_step": int(step),
            "matched_fit_seed": int(matched_seed),
        })

        mirror_fit_rows.append({
            "batch": "track_bakeoff",
            "contract_id": TRANSFORMER_TRACK_BAKEOFF_CONTRACT_ID,
            "job_id": mirror_job["job_id"],
            "source_job_id": job["job_id"],
            "track": mirror_track,
            "metric": job["metric"],
            "horizon": int(job["horizon"]),
            "candidate_id": job["candidate_id"],
            "representation_id": job["representation_id"],
            "profile_id": job["profile_id"],
            "lookback_steps": int(job["lookback_steps"]),
            "patch_length": int(job["patch_length"]),
            "patch_stride": int(job["patch_stride"]),
            "refit_step": int(step),
            "validation_month": mirror_schedule["validation_month"],
            "origin_date": mirror_schedule["origin_date"],
            "origin_observation_sequence_id": int(
                mirror_schedule["origin_observation_sequence_id"]
            ),
            "next_refit_origin_id": mirror_schedule["next_refit_origin_id"],
            "temporal_window": f"track_bakeoff_refit_{int(step):02d}",
            "matched_original_track": original_track,
            "matched_original_temporal_window": original_result["temporal_window"],
            "fit_seed_override": int(matched_seed),
            "status": "planned",
        })

transformer_track_bakeoff_target_roster_df = pd.DataFrame(target_rows)
transformer_track_bakeoff_original_fits_df = pd.DataFrame(original_fit_rows)
transformer_track_bakeoff_mirror_roster_df = pd.DataFrame(mirror_fit_rows)

if len(transformer_track_bakeoff_target_roster_df) != TRANSFORMER_TRACK_BAKEOFF_EXPECTED_TARGETS:
    raise ValueError("Expected exactly five target-level track comparisons.")

if len(transformer_track_bakeoff_original_fits_df) != TRANSFORMER_TRACK_BAKEOFF_EXPECTED_MIRROR_FITS:
    raise ValueError("Expected exactly 15 reusable original-track fits.")

if len(transformer_track_bakeoff_mirror_roster_df) != TRANSFORMER_TRACK_BAKEOFF_EXPECTED_MIRROR_FITS:
    raise ValueError("Expected exactly 15 opposite-track mirror fits.")

if transformer_track_bakeoff_mirror_roster_df.duplicated(
    TRANSFORMER_LOGICAL_KEY_COLUMNS
).any():
    raise ValueError("Track-bakeoff mirror roster contains duplicate logical fits.")

track_plan_display_df = transformer_track_bakeoff_target_roster_df[
    [
        "metric", "horizon", "original_track", "mirror_track",
        "profile_id", "sampled_months",
    ]
].copy()

track_plan_display_df["metric"] = track_plan_display_df["metric"].map(TARGET_LABELS)
track_plan_display_df["original_track"] = (
    track_plan_display_df["original_track"].map(TRACK_LABELS)
)
track_plan_display_df["mirror_track"] = (
    track_plan_display_df["mirror_track"].map(TRACK_LABELS)
)

track_plan_display_df = track_plan_display_df.rename(
    columns={
        "metric": "Target",
        "horizon": "Horizon",
        "original_track": "Existing track",
        "mirror_track": "New mirror track",
        "profile_id": "Transformer profile",
        "sampled_months": "Sampled months",
    }
)

display(track_plan_display_df)

print(
    "Matched track bakeoff: "
    f"{TRANSFORMER_TRACK_BAKEOFF_EXPECTED_MIRROR_FITS} completed cadence fits reused + "
    f"{TRANSFORMER_TRACK_BAKEOFF_EXPECTED_MIRROR_FITS} new opposite-track mirror fits."
)

,Target,Horizon,Existing track,New mirror track,Transformer profile,Sampled months
0,Taxi trips,1,Post-CP Primary,Full-History Regime-Aware,higher_capacity_regularized,"2025-04, 2025-08, 2025-12"
1,Taxi average speed,5,Full-History Regime-Aware,Post-CP Primary,compact,"2025-04, 2025-08, 2025-12"
2,FHVHV trips,2,Post-CP Primary,Full-History Regime-Aware,compact,"2025-04, 2025-08, 2025-12"
3,FHVHV average speed,1,Full-History Regime-Aware,Post-CP Primary,higher_capacity_regularized,"2025-04, 2025-08, 2025-12"
4,Subway ridership,5,Post-CP Primary,Full-History Regime-Aware,compact,"2025-04, 2025-08, 2025-12"


Matched track bakeoff: 15 completed cadence fits reused + 15 new opposite-track mirror fits.


**Findings.** The matched track bakeoff covers all **five mobility targets** at three spread-out validation months: **April, August, and December 2025**. Fifteen completed cadence fits are reused, so only **15 opposite-track mirror fits** need to be trained.

Each target keeps its already-selected Transformer profile and forecast horizon. The experiment therefore changes the **training-history track**, not the underlying Transformer design.

**4.5.2.5.2A — Verify that the bounded screen still represents two genuinely different training histories.**

The 192-origin development cap makes this comparison affordable, but it must not erase the distinction we are trying to test.

The model-facing congestion-pricing indicator encodes **pre-CP as 0 and post-CP as 1**. After bounded sampling, Full-History selector training and fresh final refitting must still contain **both 0 and 1**. Post-CP Primary training must contain **only 1**.

The historical 28-day selector-validation tail and the sampled outer scoring month occur after congestion pricing began, so those surfaces should contain only post-CP observations under either track.

Before the mirror run begins, one representative payload from each track is checked explicitly. The same assertion is repeated for every newly trained mirror fit.

In [53]:
# ---------------------------------------------------------------------
# 4.5.2.5.2A — Verify policy-regime semantics before mirror fitting
# ---------------------------------------------------------------------

def transformer_regime_signature(context_frame):
    """Return the observed policy-regime states from model-facing context."""
    if "target_cp_regime" not in context_frame.columns:
        raise ValueError("Transformer context is missing target_cp_regime.")

    values = pd.to_numeric(
        context_frame["target_cp_regime"],
        errors="raise",
    ).to_numpy(dtype=float)

    if not np.isfinite(values).all():
        raise ValueError("Policy-regime context contains non-finite values.")

    signature = tuple(sorted(set(values.tolist())))

    if not set(signature).issubset({0.0, 1.0}):
        raise ValueError(f"Unexpected policy-regime values: {signature}")

    return signature


def validate_transformer_track_regime(payload, track):
    """Prove that bounded sampling preserves the intended training-history track.

    Full-History training must still contain both pre- and post-CP examples
    after the 192-origin sampler is applied. Post-CP training must contain only
    post-policy examples. Historical inner validation and outer scoring are
    post-CP for both ordinary tracks in this experiment.
    """
    signatures = {
        "train": transformer_regime_signature(payload["train_context"]),
        "inner": transformer_regime_signature(payload["inner_context"]),
        "final": transformer_regime_signature(payload["final_context"]),
        "eval": transformer_regime_signature(payload["eval_context"]),
    }

    if track == "full_history_regime_aware":
        if signatures["train"] != (0.0, 1.0) or signatures["final"] != (0.0, 1.0):
            raise ValueError(
                "Full-History bounded training no longer contains both CP regimes."
            )
    elif track == "post_cp_primary":
        if signatures["train"] != (1.0,) or signatures["final"] != (1.0,):
            raise ValueError(
                "Post-CP bounded training contains observations outside the post-CP regime."
            )
    else:
        raise ValueError(f"Unexpected ordinary modeling track: {track}")

    if signatures["inner"] != (1.0,) or signatures["eval"] != (1.0,):
        raise ValueError(
            "Track-bakeoff inner validation and scoring must remain post-CP."
        )

    return {
        "regime_train": "/".join(f"{value:.0f}" for value in signatures["train"]),
        "regime_inner": "/".join(f"{value:.0f}" for value in signatures["inner"]),
        "regime_final": "/".join(f"{value:.0f}" for value in signatures["final"]),
        "regime_eval": "/".join(f"{value:.0f}" for value in signatures["eval"]),
        "regime_qa_passed": True,
    }


regime_preflight_rows = []

for track in ["post_cp_primary", "full_history_regime_aware"]:
    probe_row = (
        transformer_track_bakeoff_mirror_roster_df.loc[
            transformer_track_bakeoff_mirror_roster_df["track"].eq(track)
        ]
        .sort_values(["refit_step", "metric"])
        .iloc[0]
        .to_dict()
    )

    probe_job = {
        **probe_row,
        "job_id": probe_row["job_id"],
    }

    probe_frames = load_transformer_cadence_job(probe_job)
    probe_payload = build_transformer_cadence_payload(
        probe_frames,
        probe_job,
        probe_row,
    )

    _require_fit_payload(probe_payload)

    qa = validate_transformer_track_regime(probe_payload, track)

    regime_preflight_rows.append({
        "Track": TRACK_LABELS[track],
        "Selector-train regimes": qa["regime_train"],
        "Inner-validation regimes": qa["regime_inner"],
        "Final-refit regimes": qa["regime_final"],
        "Scoring regimes": qa["regime_eval"],
        "Status": "PASS",
    })

    probe_frames = None
    probe_payload = None
    gc.collect()

transformer_track_bakeoff_regime_preflight_df = pd.DataFrame(regime_preflight_rows)
display(transformer_track_bakeoff_regime_preflight_df)

,Track,Selector-train regimes,Inner-validation regimes,Final-refit regimes,Scoring regimes,Status
0,Post-CP Primary,1,1,1,1,PASS
1,Full-History Regime-Aware,0/1,1,0/1,1,PASS


**Findings.** The regime checks pass for both modeling tracks. Post-CP training contains only **post-policy observations**, while Full-History training still contains **both pre- and post-CP observations** after the 192-origin development cap is applied.

The inner-validation and scoring periods are post-CP for both tracks, as intended. The bounded screen therefore preserves the actual distinction between the two training-history strategies rather than accidentally turning Full-History into another Post-CP sample.

**4.5.2.5.2B — Execute or recover the 15 opposite-track mirror fits.**

Each mirror fit uses the same restartable Transformer engine as the completed tournament and cadence screen. Recovery is attempted before loading the large forecasting matrix; a valid checkpoint therefore avoids both retraining and unnecessary payload construction.

For a new fit, policy-regime integrity is checked again after its exact monthly payload has been constructed. The mirror then receives the deterministic seed from its paired completed cadence fit before running the normal historical epoch selector and fresh final refit.

Progress is reported for every logical fit. Each completed line shows whether the result was newly trained or recovered, the selected epoch, fitting time, raw MAE, and skill relative to the frozen weekly benchmark.

In [54]:
# ---------------------------------------------------------------------
# 4.5.2.5.2B — Execute or recover the 15 opposite-track mirror fits
# ---------------------------------------------------------------------

import warnings

track_representation_lookup = {
    item["representation_id"]: item
    for item in TRANSFORMER_REPRESENTATION_CANDIDATES
}
track_profile_lookup = {
    item["profile_id"]: item
    for item in TRANSFORMER_PROFILE_CANDIDATES
}

expected_mirror_roster_df = (
    transformer_track_bakeoff_mirror_roster_df[
        TRANSFORMER_LOGICAL_KEY_COLUMNS
    ]
    .drop_duplicates()
    .copy()
)

mirror_result_rows = []
recovered_mirror_fits = 0
new_mirror_fits = 0
active_job_id = None
active_job_frames = None

ordered_mirror_roster_df = (
    transformer_track_bakeoff_mirror_roster_df
    .sort_values(["metric", "refit_step"])
    .reset_index(drop=True)
)

for fit_index, roster_row in ordered_mirror_roster_df.iterrows():
    logical_row = roster_row.copy()

    logical_key = {
        column: logical_row[column]
        for column in TRANSFORMER_LOGICAL_KEY_COLUMNS
    }
    token = logical_key_token(logical_row, TRANSFORMER_LOGICAL_KEY_COLUMNS)

    checkpoint_path = TRANSFORMER_TRACK_BAKEOFF_ROOT / f"fit_{token}.parquet"
    prediction_path = TRANSFORMER_TRACK_BAKEOFF_ROOT / f"prediction_{token}.parquet"
    curve_path = TRANSFORMER_TRACK_BAKEOFF_ROOT / f"curve_{token}.parquet"
    state_path = TRANSFORMER_TRACK_BAKEOFF_ROOT / f"state_{token}.pt"

    recovered = recover_transformer_fit(
        checkpoint_path,
        TRANSFORMER_TRACK_BAKEOFF_CONTRACT_ID,
        logical_key,
        prediction_path,
        curve_path,
        state_path,
    )

    target_label = TARGET_LABELS[logical_row["metric"]]
    track_label = TRACK_LABELS[logical_row["track"]]
    month_label = logical_row["validation_month"]
    progress_prefix = (
        f"[{fit_index + 1}/{TRANSFORMER_TRACK_BAKEOFF_EXPECTED_MIRROR_FITS}]"
    )

    if recovered is not None:
        if int(recovered.get("fit_seed_override", -1)) != int(
            logical_row["fit_seed_override"]
        ):
            raise ValueError(
                f"{target_label} · {month_label}: recovered mirror seed does not "
                "match its paired original-track fit."
            )

        if not bool(recovered.get("regime_qa_passed", False)):
            raise ValueError(
                f"{target_label} · {month_label}: recovered mirror lacks a passing "
                "policy-regime audit."
            )

        mirror_result_rows.append(recovered)
        recovered_mirror_fits += 1

        print(
            f"✓ {progress_prefix} RECOVERED · {target_label} · {track_label} · "
            f"{month_label} · epoch {int(recovered['selected_epoch'])} · "
            f"{float(recovered['runtime_seconds']):.1f}s · "
            f"raw MAE {float(recovered['raw_mae']):.3f} · "
            f"skill {float(recovered['benchmark_skill_pct']):+.3f}%",
            flush=True,
        )
        continue

    if active_job_id != logical_row["job_id"]:
        active_job_frames = None
        gc.collect()
        clear_device_memory()

        mirror_job = logical_row.to_dict()
        active_job_frames = load_transformer_cadence_job(mirror_job)
        active_job_id = logical_row["job_id"]

    payload = build_transformer_cadence_payload(
        active_job_frames,
        logical_row.to_dict(),
        logical_row.to_dict(),
    )
    _require_fit_payload(payload)

    regime_qa = validate_transformer_track_regime(
        payload,
        logical_row["track"],
    )

    for field, value in regime_qa.items():
        logical_row[field] = value

    representation = track_representation_lookup[
        logical_row["representation_id"]
    ]
    profile = track_profile_lookup[logical_row["profile_id"]]

    print(
        f"→ {progress_prefix} {target_label} · {track_label} · "
        f"h={int(logical_row['horizon'])} · {month_label} · fitting...",
        flush=True,
    )

    with warnings.catch_warnings():
        warnings.filterwarnings(
            "ignore",
            message=(
                r"enable_nested_tensor is True, but self\.use_nested_tensor is False "
                r"because encoder_layer\.norm_first was True"
            ),
            category=UserWarning,
        )

        result, _, _ = run_transformer_logical_fit(
            logical_row,
            representation,
            profile,
            payload,
            TRANSFORMER_TRACK_BAKEOFF_ROOT,
        )

    mirror_result_rows.append(result)
    new_mirror_fits += 1

    print(
        f"✓ {progress_prefix} NEW · epoch {int(result['selected_epoch'])} · "
        f"{float(result['runtime_seconds']):.1f}s · "
        f"raw MAE {float(result['raw_mae']):.3f} · "
        f"skill {float(result['benchmark_skill_pct']):+.3f}%",
        flush=True,
    )

active_job_frames = None
gc.collect()
clear_device_memory()

transformer_track_bakeoff_mirror_results_df = pd.DataFrame(mirror_result_rows)

validate_exact_success_roster(
    transformer_track_bakeoff_mirror_results_df,
    expected_mirror_roster_df,
    TRANSFORMER_LOGICAL_KEY_COLUMNS,
)

if len(transformer_track_bakeoff_mirror_results_df) != (
    TRANSFORMER_TRACK_BAKEOFF_EXPECTED_MIRROR_FITS
):
    raise ValueError(
        "Track bakeoff did not recover or complete all 15 mirror fits."
    )

if not transformer_track_bakeoff_mirror_results_df[
    "regime_qa_passed"
].fillna(False).astype(bool).all():
    raise ValueError(
        "At least one completed mirror fit failed policy-regime QA."
    )

atomic_write_parquet(
    transformer_track_bakeoff_mirror_results_df,
    TRANSFORMER_TRACK_BAKEOFF_RESULTS_PATH,
)

mirror_summary_df = (
    transformer_track_bakeoff_mirror_results_df
    .groupby(["metric", "track", "horizon"], as_index=False)
    .agg(
        refits=("refit_step", "nunique"),
        median_selected_epoch=("selected_epoch", "median"),
        fit_minutes=("runtime_seconds", lambda values: values.sum() / 60.0),
        train_regimes=(
            "regime_train",
            lambda values: ", ".join(sorted(set(values))),
        ),
        final_refit_regimes=(
            "regime_final",
            lambda values: ", ".join(sorted(set(values))),
        ),
    )
)

mirror_summary_df["metric"] = mirror_summary_df["metric"].map(TARGET_LABELS)
mirror_summary_df["track"] = mirror_summary_df["track"].map(TRACK_LABELS)
mirror_summary_df[["median_selected_epoch", "fit_minutes"]] = (
    mirror_summary_df[["median_selected_epoch", "fit_minutes"]].round(3)
)

mirror_summary_df = mirror_summary_df.rename(
    columns={
        "metric": "Target",
        "track": "Mirror track",
        "horizon": "Horizon",
        "refits": "Refits",
        "median_selected_epoch": "Median selected epoch",
        "fit_minutes": "Fit minutes",
        "train_regimes": "Selector-train regimes",
        "final_refit_regimes": "Final-refit regimes",
    }
)

display(mirror_summary_df)

print(
    f"Track-bakeoff restart summary: {recovered_mirror_fits} recovered · "
    f"{new_mirror_fits} newly fitted · "
    f"{len(transformer_track_bakeoff_mirror_results_df)}/"
    f"{TRANSFORMER_TRACK_BAKEOFF_EXPECTED_MIRROR_FITS} complete."
)

✓ [1/15] RECOVERED · FHVHV average speed · Post-CP Primary · 2025-04 · epoch 12 · 176.2s · raw MAE 1.093 · skill +21.693%
✓ [2/15] RECOVERED · FHVHV average speed · Post-CP Primary · 2025-08 · epoch 13 · 185.8s · raw MAE 0.863 · skill +14.967%
✓ [3/15] RECOVERED · FHVHV average speed · Post-CP Primary · 2025-12 · epoch 11 · 164.3s · raw MAE 1.345 · skill +20.146%
✓ [4/15] RECOVERED · FHVHV trips · Full-History Regime-Aware · 2025-04 · epoch 16 · 133.3s · raw MAE 82.284 · skill -44.091%
✓ [5/15] RECOVERED · FHVHV trips · Full-History Regime-Aware · 2025-08 · epoch 13 · 112.7s · raw MAE 66.156 · skill -36.732%
✓ [6/15] RECOVERED · FHVHV trips · Full-History Regime-Aware · 2025-12 · epoch 12 · 104.6s · raw MAE 105.097 · skill +14.523%
✓ [7/15] RECOVERED · Subway ridership · Full-History Regime-Aware · 2025-04 · epoch 17 · 83.4s · raw MAE 673.946 · skill -45.899%
✓ [8/15] RECOVERED · Subway ridership · Full-History Regime-Aware · 2025-08 · epoch 9 · 50.3s · raw MAE 568.633 · skill -54.234%

,Target,Mirror track,Horizon,Refits,Median selected epoch,Fit minutes,Selector-train regimes,Final-refit regimes
0,FHVHV average speed,Post-CP Primary,1,3,12.000,8.771,1,1
1,FHVHV trips,Full-History Regime-Aware,2,3,13.000,5.843,0/1,0/1
2,Subway ridership,Full-History Regime-Aware,5,3,17.000,3.609,0/1,0/1
3,Taxi average speed,Post-CP Primary,5,3,9.000,3.372,1,1
4,Taxi trips,Full-History Regime-Aware,1,3,17.000,12.907,0/1,0/1


Track-bakeoff restart summary: 15 recovered · 0 newly fitted · 15/15 complete.


**Findings.** All **15 of 15 mirror fits** completed successfully. Median selected training durations range from **9 to 17 epochs** across targets, with no sign that the 30-epoch ceiling is constraining the screen.

The full mirror run required about **34.5 fit-minutes**. Every Full-History fit retained both policy regimes in training, while every Post-CP fit remained post-policy only, confirming that the comparison stayed within its intended modeling contract.

**4.5.2.5.3A — Compare Post-CP and Full-History on the same future forecasts.**

The two modeling tracks do not have identical native Taxi Zone populations, so their raw prediction files cannot be compared safely by simply placing their aggregate MAEs side by side.

For each target, the three sampled original-track months and three opposite-track mirror months are first combined. The two sides are then inner-joined on the complete physical forecast identity. Raw MAE, final MAE, RMSE, bias, and benchmark skill are recomputed only after that exact row matching.

The three sampled months are **pooled at the row level before MAE is calculated**. A month with more valid forecasts therefore contributes more forecast errors rather than receiving the same weight as a much smaller month.

The signed percentage shown as **Full-History improvement** uses Post-CP raw MAE as its reference. Positive values mean Full-History lowers error; negative values mean Post-CP lowers error. The lower pooled raw MAE supplies the target-level proposal, while the common-support percentage shows how much of the smaller native prediction surface remains in the apples-to-apples comparison.

In [55]:
# ---------------------------------------------------------------------
# 4.5.2.5.3A — Compare Post-CP and Full-History on exact common rows
# ---------------------------------------------------------------------

TRACK_BAKEOFF_PREDICTION_KEY_COLUMNS = [
    "forecast_sequence_id",
    "taxi_zone_id",
    "origin_observation_sequence_id",
    "origin_date",
    "target_observation_sequence_id",
    "target_date",
]

track_comparison_rows = []

for target_row in transformer_track_bakeoff_target_roster_df.itertuples(index=False):
    original_fit_df = (
        transformer_track_bakeoff_original_fits_df.loc[
            transformer_track_bakeoff_original_fits_df["metric"].eq(
                target_row.metric
            )
        ]
        .sort_values("matched_refit_step")
        .reset_index(drop=True)
    )

    mirror_fit_df = (
        transformer_track_bakeoff_mirror_results_df.loc[
            transformer_track_bakeoff_mirror_results_df["metric"].eq(
                target_row.metric
            )
        ]
        .sort_values("refit_step")
        .reset_index(drop=True)
    )

    if len(original_fit_df) != 3 or len(mirror_fit_df) != 3:
        raise ValueError(
            f"{target_row.metric}: expected three original and three mirror "
            "prediction files."
        )

    original_prediction_df = pd.concat(
        [
            pd.read_parquet(Path(path))
            for path in original_fit_df["prediction_path"].astype(str)
        ],
        ignore_index=True,
    )

    mirror_prediction_df = pd.concat(
        [
            pd.read_parquet(Path(path))
            for path in mirror_fit_df["prediction_path"].astype(str)
        ],
        ignore_index=True,
    )

    if original_prediction_df.duplicated(
        TRACK_BAKEOFF_PREDICTION_KEY_COLUMNS
    ).any():
        raise ValueError(
            f"{target_row.metric}: original-track sampled forecasts are not unique."
        )

    if mirror_prediction_df.duplicated(
        TRACK_BAKEOFF_PREDICTION_KEY_COLUMNS
    ).any():
        raise ValueError(
            f"{target_row.metric}: mirror-track sampled forecasts are not unique."
        )

    original_compare_df = original_prediction_df[
        TRACK_BAKEOFF_PREDICTION_KEY_COLUMNS
        + [
            "target_observed_value",
            "benchmark_prediction",
            "prediction_raw",
            "prediction_final",
        ]
    ].rename(
        columns={
            "target_observed_value": "target_original",
            "benchmark_prediction": "benchmark_original",
            "prediction_raw": "prediction_raw_original",
            "prediction_final": "prediction_final_original",
        }
    )

    mirror_compare_df = mirror_prediction_df[
        TRACK_BAKEOFF_PREDICTION_KEY_COLUMNS
        + [
            "target_observed_value",
            "benchmark_prediction",
            "prediction_raw",
            "prediction_final",
        ]
    ].rename(
        columns={
            "target_observed_value": "target_mirror",
            "benchmark_prediction": "benchmark_mirror",
            "prediction_raw": "prediction_raw_mirror",
            "prediction_final": "prediction_final_mirror",
        }
    )

    common_df = original_compare_df.merge(
        mirror_compare_df,
        on=TRACK_BAKEOFF_PREDICTION_KEY_COLUMNS,
        how="inner",
        validate="one_to_one",
    )

    if common_df.empty:
        raise ValueError(
            f"{target_row.metric}: the two training-history tracks have no "
            "common forecasts."
        )

    if not np.allclose(
        common_df["target_original"].to_numpy(dtype=float),
        common_df["target_mirror"].to_numpy(dtype=float),
        rtol=1e-10,
        atol=1e-10,
    ):
        raise ValueError(
            f"{target_row.metric}: matched tracks disagree on observed targets."
        )

    if not np.allclose(
        common_df["benchmark_original"].to_numpy(dtype=float),
        common_df["benchmark_mirror"].to_numpy(dtype=float),
        rtol=1e-10,
        atol=1e-10,
    ):
        raise ValueError(
            f"{target_row.metric}: matched tracks disagree on frozen benchmark "
            "predictions."
        )

    actual = common_df["target_original"].to_numpy(dtype=float)
    benchmark = common_df["benchmark_original"].to_numpy(dtype=float)

    if target_row.original_track == "post_cp_primary":
        post_raw = common_df["prediction_raw_original"].to_numpy(dtype=float)
        post_final = common_df["prediction_final_original"].to_numpy(dtype=float)
        full_raw = common_df["prediction_raw_mirror"].to_numpy(dtype=float)
        full_final = common_df["prediction_final_mirror"].to_numpy(dtype=float)

    elif target_row.original_track == "full_history_regime_aware":
        full_raw = common_df["prediction_raw_original"].to_numpy(dtype=float)
        full_final = common_df["prediction_final_original"].to_numpy(dtype=float)
        post_raw = common_df["prediction_raw_mirror"].to_numpy(dtype=float)
        post_final = common_df["prediction_final_mirror"].to_numpy(dtype=float)

    else:
        raise ValueError(
            f"{target_row.metric}: unexpected original modeling track."
        )

    benchmark_metrics = regression_metrics(actual, benchmark)
    post_raw_metrics = regression_metrics(actual, post_raw)
    full_raw_metrics = regression_metrics(actual, full_raw)
    post_final_metrics = regression_metrics(actual, post_final)
    full_final_metrics = regression_metrics(actual, full_final)

    post_raw_mae = post_raw_metrics["mae"]
    full_raw_mae = full_raw_metrics["mae"]
    benchmark_mae = benchmark_metrics["mae"]

    if np.isclose(post_raw_mae, full_raw_mae, rtol=0.0, atol=1e-12):
        selected_track = "tie"
    elif full_raw_mae < post_raw_mae:
        selected_track = "full_history_regime_aware"
    else:
        selected_track = "post_cp_primary"

    smaller_native_surface = min(
        len(original_prediction_df),
        len(mirror_prediction_df),
    )

    common_support_pct = (
        100.0 * len(common_df) / smaller_native_surface
        if smaller_native_surface > 0
        else np.nan
    )

    track_comparison_rows.append({
        "metric": target_row.metric,
        "horizon": int(target_row.horizon),
        "representation_id": target_row.representation_id,
        "profile_id": target_row.profile_id,
        "sampled_months": target_row.sampled_months,

        "post_cp_native_rows": (
            len(original_prediction_df)
            if target_row.original_track == "post_cp_primary"
            else len(mirror_prediction_df)
        ),
        "full_history_native_rows": (
            len(original_prediction_df)
            if target_row.original_track == "full_history_regime_aware"
            else len(mirror_prediction_df)
        ),
        "common_rows": len(common_df),
        "common_support_pct_of_smaller_track": common_support_pct,

        "benchmark_mae": benchmark_mae,
        "post_cp_raw_mae": post_raw_mae,
        "full_history_raw_mae": full_raw_mae,

        "full_history_vs_post_cp_raw_mae_improvement_pct": (
            100.0 * (post_raw_mae - full_raw_mae) / post_raw_mae
            if post_raw_mae > 0
            else np.nan
        ),

        "post_cp_raw_skill_vs_benchmark_pct": (
            100.0 * (benchmark_mae - post_raw_mae) / benchmark_mae
            if benchmark_mae > 0
            else np.nan
        ),
        "full_history_raw_skill_vs_benchmark_pct": (
            100.0 * (benchmark_mae - full_raw_mae) / benchmark_mae
            if benchmark_mae > 0
            else np.nan
        ),

        "post_cp_final_mae": post_final_metrics["mae"],
        "full_history_final_mae": full_final_metrics["mae"],
        "post_cp_raw_rmse": post_raw_metrics["rmse"],
        "full_history_raw_rmse": full_raw_metrics["rmse"],
        "post_cp_raw_bias": post_raw_metrics["bias"],
        "full_history_raw_bias": full_raw_metrics["bias"],
        "post_cp_negative_raw_predictions": int((post_raw < 0).sum()),
        "full_history_negative_raw_predictions": int((full_raw < 0).sum()),
        "proposed_track": selected_track,
    })

transformer_track_bakeoff_comparison_df = pd.DataFrame(track_comparison_rows)

if len(transformer_track_bakeoff_comparison_df) != len(FORECAST_TARGETS):
    raise ValueError(
        "Expected exactly one matched track comparison per target."
    )

atomic_write_parquet(
    transformer_track_bakeoff_comparison_df,
    TRANSFORMER_TRACK_BAKEOFF_COMPARISON_PATH,
)

track_comparison_display_df = transformer_track_bakeoff_comparison_df[
    [
        "metric", "horizon", "post_cp_raw_mae", "full_history_raw_mae",
        "full_history_vs_post_cp_raw_mae_improvement_pct",
        "post_cp_raw_skill_vs_benchmark_pct",
        "full_history_raw_skill_vs_benchmark_pct",
        "common_rows", "common_support_pct_of_smaller_track",
        "proposed_track",
    ]
].copy()

track_comparison_display_df["metric"] = (
    track_comparison_display_df["metric"].map(TARGET_LABELS)
)

track_comparison_display_df["proposed_track"] = (
    track_comparison_display_df["proposed_track"]
    .map({
        "post_cp_primary": TRACK_LABELS["post_cp_primary"],
        "full_history_regime_aware": TRACK_LABELS["full_history_regime_aware"],
        "tie": "Tie",
    })
)

track_comparison_display_df = track_comparison_display_df.rename(
    columns={
        "metric": "Target",
        "horizon": "Horizon",
        "post_cp_raw_mae": "Post-CP raw MAE",
        "full_history_raw_mae": "Full-History raw MAE",
        "full_history_vs_post_cp_raw_mae_improvement_pct":
            "Full-History improvement %",
        "post_cp_raw_skill_vs_benchmark_pct":
            "Post-CP benchmark skill %",
        "full_history_raw_skill_vs_benchmark_pct":
            "Full-History benchmark skill %",
        "common_rows": "Common rows",
        "common_support_pct_of_smaller_track": "Common support %",
        "proposed_track": "Lower-MAE track",
    }
)

numeric_columns = [
    "Post-CP raw MAE",
    "Full-History raw MAE",
    "Full-History improvement %",
    "Post-CP benchmark skill %",
    "Full-History benchmark skill %",
    "Common support %",
]

track_comparison_display_df[numeric_columns] = (
    track_comparison_display_df[numeric_columns].round(3)
)

display(track_comparison_display_df)

winner_counts = (
    transformer_track_bakeoff_comparison_df["proposed_track"]
    .value_counts()
    .to_dict()
)

print(
    "Matched-track synthesis: "
    f"Post-CP wins {winner_counts.get('post_cp_primary', 0)} · "
    f"Full-History wins {winner_counts.get('full_history_regime_aware', 0)} · "
    f"ties {winner_counts.get('tie', 0)}."
)

,Target,Horizon,Post-CP raw MAE,Full-History raw MAE,Full-History improvement %,Post-CP benchmark skill %,Full-History benchmark skill %,Common rows,Common support %,Lower-MAE track
0,Taxi trips,1,20.221,20.463,-1.198,4.576,3.433,113104,100.000,Post-CP Primary
1,Taxi average speed,5,1.982,2.061,-4.021,17.923,14.622,54455,100.000,Post-CP Primary
2,FHVHV trips,2,78.429,84.700,-7.996,-1.380,-9.486,113358,100.000,Post-CP Primary
3,FHVHV average speed,1,1.101,1.091,0.917,19.260,20.001,112616,100.000,Full-History Regime-Aware
4,Subway ridership,5,859.160,793.440,7.649,-43.233,-32.277,66226,100.000,Full-History Regime-Aware


Matched-track synthesis: Post-CP wins 3 · Full-History wins 2 · ties 0.


**Findings.** The track comparison achieves **100% common forecast support for all five targets**, so every Post-CP versus Full-History result is based on the same future observations.

Post-CP produces the lower raw MAE for **Taxi trips, Taxi average speed, and FHVHV trips**. Full-History performs better for **FHVHV average speed and Subway ridership**.

The strongest differences are **7.996% in favor of Post-CP for FHVHV trips** and **7.649% in favor of Full-History for Subway ridership**. Taxi average speed also favors Post-CP by **4.021%**. Taxi trips and FHVHV average speed are much closer, with differences of only **1.198%** and **0.917%**, respectively.

Benchmark behavior tells the same broad story. Both speed targets beat the weekly benchmark under either track, while FHVHV trips and Subway ridership remain below it. Training-history choice improves those weaker forecasts, but does not fully solve them.

**4.5.2.5.3B — Show where additional history helps or hurts.**

The matched comparison is easiest to read on a common percentage scale.

Each bar shows how much raw MAE changes when the same target-specific Transformer is trained with Full-History rather than Post-CP history:

- **positive** values favor Full-History;
- **negative** values favor Post-CP;
- values near zero indicate little practical separation in this bounded comparison.

Color uses the same signed diverging visual grammar as the other forecasting track comparisons. Hover retains both native-unit MAEs, benchmark skill, exact common-row count, common-support rate, and the lower-MAE track.

In [56]:
# ---------------------------------------------------------------------
# 4.5.2.5.3B — Visualize the matched training-history effect by target
# ---------------------------------------------------------------------

track_chart_df = transformer_track_bakeoff_comparison_df.copy()
track_chart_df["Target"] = track_chart_df["metric"].map(TARGET_LABELS)

track_chart_df["Winner"] = track_chart_df["proposed_track"].map({
    "post_cp_primary": TRACK_LABELS["post_cp_primary"],
    "full_history_regime_aware": TRACK_LABELS["full_history_regime_aware"],
    "tie": "Tie",
})

effect = track_chart_df[
    "full_history_vs_post_cp_raw_mae_improvement_pct"
].to_numpy(dtype=float)

color_cap = max(1.0, float(np.nanmax(np.abs(effect))))

track_fig = go.Figure(
    go.Bar(
        x=effect,
        y=track_chart_df["Target"],
        orientation="h",
        marker={
            "color": effect,
            "colorscale": BRAND_DIVERGING_SEQUENCE,
            "cmin": -color_cap,
            "cmax": color_cap,
            "colorbar": {"title": "FH improvement %"},
        },
        text=[f"{value:+.1f}%" for value in effect],
        textposition="outside",
        customdata=np.column_stack([
            track_chart_df["post_cp_raw_mae"],
            track_chart_df["full_history_raw_mae"],
            track_chart_df["post_cp_raw_skill_vs_benchmark_pct"],
            track_chart_df["full_history_raw_skill_vs_benchmark_pct"],
            track_chart_df["common_rows"],
            track_chart_df["common_support_pct_of_smaller_track"],
            track_chart_df["Winner"],
        ]),
        hovertemplate=(
            "<b>%{y}</b><br>"
            "Full-History improvement over Post-CP: %{x:+.3f}%<br>"
            "Post-CP raw MAE: %{customdata[0]:.3f}<br>"
            "Full-History raw MAE: %{customdata[1]:.3f}<br>"
            "Post-CP benchmark skill: %{customdata[2]:+.3f}%<br>"
            "Full-History benchmark skill: %{customdata[3]:+.3f}%<br>"
            "Common rows: %{customdata[4]:,.0f}<br>"
            "Common support: %{customdata[5]:.3f}%<br>"
            "Lower-MAE track: %{customdata[6]}"
            "<extra></extra>"
        ),
    )
)

track_fig = apply_branding(track_fig)

track_fig.add_vline(
    x=0,
    line_dash="dash",
    line_color=BRAND_COLORS["terracotta"],
)

track_fig.update_layout(
    title="Transformer Training History — Full-History Improvement vs Post-CP",
    xaxis_title="Full-History improvement over Post-CP raw MAE (%)",
    yaxis_title=None,
    height=430,
    margin={"l": 180, "r": 90, "t": 80, "b": 70},
)

track_fig.update_yaxes(autorange="reversed")
track_fig.show()

**Findings.** The visual makes clear that **more historical data is not universally better** for the Transformer.

Full-History provides its clearest advantage for Subway ridership and only a very small advantage for FHVHV average speed. For the other three targets, restricting training to the post-congestion-pricing period performs better.

The result supports a **target-specific training-history choice** rather than forcing one history rule across the entire Transformer family. It also shows that some track differences are substantial, while Taxi trips and FHVHV average speed are close enough that their selections should be treated as modest advantages rather than dramatic wins.

> \> ✅ \*\*Decision — Use target\-specific Transformer training\-history tracks\*\*
\>
\> \*\*Post\-CP Primary:\*\* Taxi trips, Taxi average speed, and FHVHV trips\.
\>
\> \*\*Full\-History Regime\-Aware:\*\* FHVHV average speed and Subway ridership\.
\>
\> The choice is based on pooled raw MAE over exact common forecast rows across April, August, and December 2025\. The result is mixed rather than universal: some targets benefit from the longer historical record, while others forecast better using only post\-congestion\-pricing history\.
\>
\> These track choices are now frozen for all forecast horizons unless a downstream QA failure invalidates the evidence\.

### Section Summary

Section 4.5.2.5 compared Post-CP Primary with Full-History Regime-Aware training using three matched monthly windows per target and exact common forecast rows. The result was target-specific: **Post-CP performed better for Taxi trips, Taxi average speed, and FHVHV trips**, while **Full-History performed better for FHVHV average speed and Subway ridership**. Common support was 100% for all five comparisons, and the bounded Full-History fits retained both pre- and post-CP observations as intended. The evidence therefore supports using **different training-history tracks by target rather than forcing one history rule across the entire Transformer family**.

# 4.5.2.6 Run Full Transformer Rolling Validation

The Transformer design is now frozen at the target level: each target has one history representation, one capacity profile, one training-history track, and the shared monthly refit cadence. Full rolling validation expands those choices across horizons **1, 2, and 5**, producing **15 forecasting jobs × 10 monthly refits = 150 logical fits**.

Unlike the bounded development screens, production fitting uses **all legally available expanding history** at each monthly boundary. The 192-origin cap is removed, while the same 28-day historical epoch selector, fresh preprocessing, fresh final model, benchmark alignment, and untouched 2026 holdout remain in force.

Before launching the complete crawl, one exact late production fit is run as a timing and memory stress test. It is saved under the production contract, so it will count as one of the eventual 150 fits rather than being throwaway work.

**4.5.2.6.1A — Freeze the target-specific production contract.**

Combine the decisions already made in Chapters 3–5 into one production registry. Each target keeps its selected Transformer profile and training-history track across all three horizons; no architecture, cadence, or track choice is reopened here.

The reader-facing table stays at the target level. Internally, each row expands to three Target × Horizon jobs and 30 monthly fits.

In [57]:
# ---------------------------------------------------------------------
# 4.5.2.6.1A — Freeze the target-specific production contract
# ---------------------------------------------------------------------

TRANSFORMER_FULL_CONTRACT_ID = (
    "transformer_full_validation_v1_targettrack_monthly_uncapped_adaptive"
)
TRANSFORMER_FULL_ROOT = INTERMEDIATE_DIR / "tr_full_v1"
TRANSFORMER_FULL_SCHEDULE_PATH = TRANSFORMER_FULL_ROOT / "schedule.parquet"
TRANSFORMER_FULL_RESULTS_PATH = TRANSFORMER_FULL_ROOT / "results.parquet"

TRANSFORMER_FULL_EXPECTED_REFITS_PER_JOB = 10
TRANSFORMER_FULL_EXPECTED_JOBS = len(FORECAST_TARGETS) * len(FORECAST_HORIZONS)
TRANSFORMER_FULL_EXPECTED_FITS = (
    TRANSFORMER_FULL_EXPECTED_JOBS * TRANSFORMER_FULL_EXPECTED_REFITS_PER_JOB
)
TRANSFORMER_FULL_ROSTER_KEY_COLUMNS = [
    "job_id", "refit_step", "origin_observation_sequence_id",
]

if TRANSFORMER_FULL_EXPECTED_JOBS != 15 or TRANSFORMER_FULL_EXPECTED_FITS != 150:
    raise AssertionError(
        "Full Transformer validation must contain 15 jobs and 150 monthly fits."
    )

# RESET invalidates this production contract; it is not a run-permission flag.
# Missing current-contract artifacts are always eligible to be computed.
if "RESET_TRANSFORMER_FULL_VALIDATION" not in globals():
    RESET_TRANSFORMER_FULL_VALIDATION = False

if RESET_TRANSFORMER_FULL_VALIDATION:
    reset_experiment_artifacts(TRANSFORMER_FULL_ROOT)

TRANSFORMER_FULL_ROOT.mkdir(parents=True, exist_ok=True)

track_choice_df = (
    transformer_track_bakeoff_comparison_df[["metric", "proposed_track"]]
    .rename(columns={"proposed_track": "track"})
    .copy()
)

if track_choice_df["track"].eq("tie").any():
    raise ValueError(
        "A tied training-history choice cannot enter production validation."
    )

if not set(track_choice_df["track"]).issubset(
    {"post_cp_primary", "full_history_regime_aware"}
):
    raise ValueError(
        "Production track registry contains an unknown modeling track."
    )

transformer_full_target_registry_df = (
    transformer_selected_target_profile_df
    .merge(track_choice_df, on="metric", how="inner", validate="one_to_one")
    .copy()
)

if set(transformer_full_target_registry_df["metric"]) != set(FORECAST_TARGETS):
    raise ValueError(
        "Production registry must contain exactly the five forecasting targets."
    )

transformer_full_target_registry_df["candidate_id"] = (
    transformer_full_target_registry_df["representation_id"].astype(str)
    + "__"
    + transformer_full_target_registry_df["profile_id"].astype(str)
)

profile_labels = {
    "compact": "Compact",
    "higher_capacity_regularized": "Higher-capacity regularized",
}
representation_labels = {
    "two_week_overlapping_two_day": "Two-week overlapping two-day",
}

production_contract_display_df = (
    transformer_full_target_registry_df
    .assign(
        Target=lambda frame: frame["metric"].map(TARGET_LABELS),
        Track=lambda frame: frame["track"].map(TRACK_LABELS),
        **{
            "Transformer profile": lambda frame:
                frame["profile_id"].map(profile_labels),
            "History representation": lambda frame:
                frame["representation_id"].map(representation_labels),
            "Horizons": "1, 2, 5",
            "Planned monthly fits":
                len(FORECAST_HORIZONS) * TRANSFORMER_FULL_EXPECTED_REFITS_PER_JOB,
        },
    )[
        [
            "Target", "Track", "Transformer profile",
            "History representation", "Horizons", "Planned monthly fits",
        ]
    ]
)

if production_contract_display_df.isna().any().any():
    raise ValueError(
        "Reader-facing production contract contains an unmapped label."
    )

display(production_contract_display_df)

,Target,Track,Transformer profile,History representation,Horizons,Planned monthly fits
0,FHVHV average speed,Full-History Regime-Aware,Higher-capacity regularized,Two-week overlapping two-day,"1, 2, 5",30
1,FHVHV trips,Post-CP Primary,Compact,Two-week overlapping two-day,"1, 2, 5",30
2,Subway ridership,Full-History Regime-Aware,Compact,Two-week overlapping two-day,"1, 2, 5",30
3,Taxi average speed,Post-CP Primary,Compact,Two-week overlapping two-day,"1, 2, 5",30
4,Taxi trips,Post-CP Primary,Higher-capacity regularized,Two-week overlapping two-day,"1, 2, 5",30


**Findings.** The production contract preserves the target-specific choices established earlier. All five targets use the **two-week overlapping two-day** history representation across horizons **1, 2, and 5**, while capacity and training-history track vary by target.

Taxi trips and FHVHV average speed use the higher-capacity Transformer; the other three targets use the compact model. Three targets use **Post-CP Primary** training, while FHVHV average speed and Subway ridership use **Full-History Regime-Aware** training. Each target therefore contributes **30 monthly fits**, for 150 fits overall.

**4.5.2.6.1B — Build the 150-fit monthly production schedule.**

Expand the five frozen target definitions across horizons 1, 2, and 5 and the same ten calendar-driven monthly refit boundaries used in cadence testing. The schedule depends only on the legal forecasting calendar, never on future forecast errors.

Every Job × Month receives a stable production identity for checkpoint recovery. The 192-origin development cap is absent from this contract.

In [58]:
# ---------------------------------------------------------------------
# 4.5.2.6.1B — Build the 15-job, 150-fit monthly schedule
# ---------------------------------------------------------------------

job_rows = []

for target in transformer_full_target_registry_df.to_dict("records"):
    for horizon in FORECAST_HORIZONS:
        job_rows.append({
            **target,
            "horizon": int(horizon),
            "job_id": f"{target['metric']}__h{int(horizon)}",
        })

transformer_full_jobs_df = pd.DataFrame(job_rows)

if len(transformer_full_jobs_df) != TRANSFORMER_FULL_EXPECTED_JOBS:
    raise ValueError(
        "Production job registry does not contain exactly 15 jobs."
    )

if transformer_full_jobs_df.duplicated(["metric", "horizon"]).any():
    raise ValueError(
        "Production job registry contains duplicate Target × Horizon jobs."
    )

schedule_parts = []

for job in transformer_full_jobs_df.to_dict("records"):
    schedule = build_transformer_cadence_schedule(job).copy()

    # Reuse the already-audited monthly calendar, but bind every row to a new
    # production contract. The 192-origin development cap is not part of it.
    schedule["batch"] = "full_validation"
    schedule["contract_id"] = TRANSFORMER_FULL_CONTRACT_ID
    schedule["temporal_window"] = schedule["refit_step"].map(
        lambda step: f"full_refit_{int(step):02d}"
    )

    for column in [
        "representation_id", "lookback_steps", "patch_length",
        "patch_stride", "profile_id",
    ]:
        schedule[column] = job[column]

    schedule_parts.append(schedule)

transformer_full_schedule_df = (
    pd.concat(schedule_parts, ignore_index=True)
    .sort_values(["metric", "horizon", "refit_step"])
    .reset_index(drop=True)
)

if len(transformer_full_schedule_df) != TRANSFORMER_FULL_EXPECTED_FITS:
    raise ValueError(
        "Production schedule does not contain exactly 150 logical fits."
    )

if transformer_full_schedule_df.duplicated(
    TRANSFORMER_FULL_ROSTER_KEY_COLUMNS
).any():
    raise ValueError(
        "Production schedule contains duplicate Job × Month identities."
    )

refits_per_job = (
    transformer_full_schedule_df
    .groupby("job_id")["refit_step"]
    .nunique()
)

if not refits_per_job.eq(TRANSFORMER_FULL_EXPECTED_REFITS_PER_JOB).all():
    raise ValueError(
        "At least one production job does not contain all ten monthly refits."
    )

atomic_write_parquet(
    transformer_full_schedule_df,
    TRANSFORMER_FULL_SCHEDULE_PATH,
)

schedule_display_df = (
    transformer_full_schedule_df
    .groupby(["metric", "track"], as_index=False)
    .agg(
        **{
            "Horizons": (
                "horizon",
                lambda values: ", ".join(
                    map(str, sorted(set(map(int, values))))
                ),
            ),
            "First refit": ("origin_date", "min"),
            "Last refit": ("origin_date", "max"),
            "Planned logical fits": ("refit_step", "size"),
        }
    )
)

schedule_display_df["metric"] = schedule_display_df["metric"].map(TARGET_LABELS)
schedule_display_df["track"] = schedule_display_df["track"].map(TRACK_LABELS)

schedule_display_df = schedule_display_df.rename(
    columns={"metric": "Target", "track": "Track"}
)

display(schedule_display_df)

,Target,Track,Horizons,First refit,Last refit,Planned logical fits
0,FHVHV average speed,Full-History Regime-Aware,"1, 2, 5",2025-04-04,2026-01-01,30
1,FHVHV trips,Post-CP Primary,"1, 2, 5",2025-04-04,2026-01-01,30
2,Subway ridership,Full-History Regime-Aware,"1, 2, 5",2025-04-04,2026-01-01,30
3,Taxi average speed,Post-CP Primary,"1, 2, 5",2025-04-04,2026-01-01,30
4,Taxi trips,Post-CP Primary,"1, 2, 5",2025-04-04,2026-01-01,30


**Findings.** The production schedule contains the expected **15 Target × Horizon jobs** and **150 monthly fits**. Every target is evaluated at horizons 1, 2, and 5 across the same ten monthly refit positions, spanning **April 2025 through January 2026**.

Each target contributes 30 logical fits, with no duplicate schedule identities. The full-validation crawl is therefore aligned across targets and horizons before any expensive training begins.

**4.5.2.6.2A — Resolve the uncapped production splits without copying the full history.**

Production uses every legal training row, so the split logic now returns row positions into one loaded job matrix instead of copying very large training tables. The modeling rule is unchanged: the latest 28 historical days select training duration, and the fresh final refit then uses all labels legally known at that monthly boundary.

The helper also keeps the policy-regime check attached to the split: Full-History training must contain both pre- and post-CP observations, while Post-CP training must contain only post-policy observations.

In [59]:
# ---------------------------------------------------------------------
# 4.5.2.6.2A — Resolve uncapped production rows without duplicating them
# ---------------------------------------------------------------------

def resolve_full_transformer_split(job_frames, job, refit_row):
    """Resolve selector, final-refit, and monthly scoring rows for one fit.

    Large training partitions are returned as positional indices into the one
    job-level matrix. This keeps the full legal history without copying a
    second million-row DataFrame merely to describe the split.
    """
    matrix_df = job_frames["matrix_df"]
    validation_df = job_frames["validation_df"]
    refit_origin_id = int(refit_row["origin_observation_sequence_id"])

    legal_mask = (
        matrix_df["_label_available"]
        & matrix_df["_history_eligible"]
        & matrix_df["target_observation_sequence_id"].le(refit_origin_id)
    )

    legal_positions = np.flatnonzero(legal_mask.to_numpy())

    if len(legal_positions) == 0:
        raise ValueError(
            f"{job['job_id']}: no legal production training history."
        )

    legal_dates = matrix_df.iloc[legal_positions]["target_date"]
    inner_end_date = pd.Timestamp(legal_dates.max()).normalize()
    inner_start_date = inner_end_date - pd.Timedelta(days=27)

    target_dates = matrix_df["target_date"]
    inner_mask = legal_mask & target_dates.between(
        inner_start_date,
        inner_end_date,
        inclusive="both",
    )
    selector_mask = legal_mask & target_dates.lt(inner_start_date)

    selector_positions = np.flatnonzero(selector_mask.to_numpy())
    inner_positions = np.flatnonzero(inner_mask.to_numpy())
    final_positions = legal_positions

    if len(selector_positions) == 0 or len(inner_positions) == 0:
        raise ValueError(
            f"{job['job_id']}: production adaptive split is empty."
        )

    if matrix_df.iloc[inner_positions]["target_date"].nunique() != 28:
        raise ValueError(
            f"{job['job_id']}: inner validation must contain 28 dates."
        )

    if (
        matrix_df.iloc[selector_positions]["target_observation_sequence_id"].max()
        >= matrix_df.iloc[inner_positions]["target_observation_sequence_id"].min()
    ):
        raise ValueError(
            f"{job['job_id']}: selector training is not strictly historical."
        )

    score_mask = validation_df["origin_observation_sequence_id"].ge(
        refit_origin_id
    )
    next_refit_origin_id = refit_row["next_refit_origin_id"]

    if pd.notna(next_refit_origin_id):
        score_mask &= validation_df["origin_observation_sequence_id"].lt(
            int(next_refit_origin_id)
        )

    evaluation_df = validation_df.loc[score_mask].copy()

    if evaluation_df.empty:
        raise ValueError(
            f"{job['job_id']}: production monthly scoring slice is empty."
        )

    if pd.to_datetime(
        evaluation_df["target_date"]
    ).ge(FINAL_HOLDOUT_START_DATE).any():
        raise ValueError(
            f"{job['job_id']}: production split reaches the final holdout."
        )

    return {
        "selector_positions": selector_positions,
        "inner_positions": inner_positions,
        "final_positions": final_positions,
        "evaluation_df": evaluation_df,
        "inner_start_date": inner_start_date,
        "inner_end_date": inner_end_date,
    }


def production_regime_signature(values):
    """Return model-facing congestion-pricing states as a compact 0/1 signature."""
    regime = pd.Series(values, copy=False).astype("string")

    if not regime.isin(["pre_cp", "post_cp"]).all():
        raise ValueError(
            "Production split contains an unknown policy regime."
        )

    encoded = regime.map({"pre_cp": 0.0, "post_cp": 1.0})
    return tuple(sorted(set(encoded.astype(float).tolist())))


def validate_full_transformer_regime(job_frames, job, split):
    """Verify that uncapped production history still matches the selected track."""
    matrix_df = job_frames["matrix_df"]
    evaluation_df = split["evaluation_df"]

    signatures = {
        "selector": production_regime_signature(
            matrix_df.iloc[
                split["selector_positions"]
            ]["target_pre_post_cp"]
        ),
        "inner": production_regime_signature(
            matrix_df.iloc[
                split["inner_positions"]
            ]["target_pre_post_cp"]
        ),
        "final": production_regime_signature(
            matrix_df.iloc[
                split["final_positions"]
            ]["target_pre_post_cp"]
        ),
        "eval": production_regime_signature(
            evaluation_df["target_pre_post_cp"]
        ),
    }

    if job["track"] == "full_history_regime_aware":
        expected_training = (0.0, 1.0)
    elif job["track"] == "post_cp_primary":
        expected_training = (1.0,)
    else:
        raise ValueError(
            f"Unexpected production track: {job['track']}"
        )

    if (
        signatures["selector"] != expected_training
        or signatures["final"] != expected_training
    ):
        raise ValueError(
            f"{job['job_id']}: uncapped training history violates "
            "the selected track."
        )

    if signatures["inner"] != (1.0,) or signatures["eval"] != (1.0,):
        raise ValueError(
            f"{job['job_id']}: inner validation and scoring must remain post-CP."
        )

    return signatures

**4.5.2.6.2B — Stream sequence windows instead of materializing the full training tensor.**

Removing the 192-origin cap makes the production history much larger than the development screens. Building every 70-Daypart sequence window at once would spend several gigabytes on duplicated arrays before training even begins.

The production loader therefore keeps the same inputs and preprocessing rules but slices history from the existing dense mobility store one training batch at a time. Sequence scaling is still learned from every observed legal history cell; context, weekly STL state, target scaling, and Taxi Zone identity keep the frozen Transformer contract.

In [60]:
# ---------------------------------------------------------------------
# 4.5.2.6.2B — Stream full-history sequences through the frozen preprocessor
# ---------------------------------------------------------------------

TRANSFORMER_SEQUENCE_STATS_CHUNK_ROWS = 20_000


def prepare_full_transformer_job_cache(job):
    """Load one job once and prepare reusable row-level metadata.

    Context is reconstructed once per Target × Horizon job. Historical sequence
    windows remain in the compact dense mobility store and are sliced only for
    the batches currently being trained.
    """
    job_frames = load_transformer_cadence_job(job)
    matrix_df = job_frames["matrix_df"]

    zone_positions = matrix_df["taxi_zone_id"].map(
        transformer_zone_to_index
    )
    origin_positions = matrix_df["origin_observation_sequence_id"].map(
        transformer_sequence_to_index
    )

    required_history = matrix_df["_history_eligible"]

    if (
        zone_positions.isna().any()
        or origin_positions.loc[required_history].isna().any()
    ):
        raise ValueError(
            f"{job['job_id']}: production rows fall outside dense history state."
        )

    return {
        **job_frames,
        "context_frame": derive_transformer_context_frame(matrix_df),
        "target_values": pd.to_numeric(
            matrix_df["target_observed_value"],
            errors="coerce",
        ).to_numpy(dtype=np.float32),
        "zone_positions": zone_positions.fillna(-1).to_numpy(dtype=np.int64),
        "origin_positions": origin_positions.fillna(-1).to_numpy(dtype=np.int64),
    }


def streamed_sequence_statistics(
    job_cache,
    row_positions,
    metric,
    lookback_steps,
):
    """Compute observed-cell sequence mean/std without storing all windows."""
    channel_order = TRANSFORMER_SEQUENCE_VALUE_CHANNELS_BY_TARGET[metric]
    channel_positions = [
        FORECAST_TARGETS.index(channel)
        for channel in channel_order
    ]

    counts = np.zeros(5, dtype=np.int64)
    means = np.zeros(5, dtype=np.float64)
    m2 = np.zeros(5, dtype=np.float64)
    offsets = np.arange(int(lookback_steps), dtype=np.int64)

    for start in range(
        0,
        len(row_positions),
        TRANSFORMER_SEQUENCE_STATS_CHUNK_ROWS,
    ):
        chunk_rows = row_positions[
            start:start + TRANSFORMER_SEQUENCE_STATS_CHUNK_ROWS
        ]
        zones = job_cache["zone_positions"][chunk_rows]
        origins = job_cache["origin_positions"][chunk_rows]
        history_starts = origins - int(lookback_steps) + 1

        if (zones < 0).any() or (history_starts < 0).any():
            raise ValueError(
                "A production training row lacks a complete dense history window."
            )

        history_positions = history_starts[:, None] + offsets[None, :]

        values = transformer_history_values[
            zones[:, None],
            history_positions,
            :,
        ]
        masks = transformer_history_masks[
            zones[:, None],
            history_positions,
            :,
        ]

        values = values[:, :, channel_positions]
        masks = masks[:, :, channel_positions]

        # Combine chunk-level moments rather than concatenating every history
        # value. This preserves the full legal sample with bounded memory use.
        for channel_index in range(5):
            observed = values[..., channel_index][
                masks[..., channel_index]
                & np.isfinite(values[..., channel_index])
            ].astype(np.float64, copy=False)

            if observed.size == 0:
                continue

            chunk_count = int(observed.size)
            chunk_mean = float(observed.mean())
            chunk_m2 = float(np.square(observed - chunk_mean).sum())

            old_count = int(counts[channel_index])
            new_count = old_count + chunk_count
            delta = chunk_mean - means[channel_index]

            means[channel_index] += delta * chunk_count / new_count
            m2[channel_index] += (
                chunk_m2
                + delta * delta * old_count * chunk_count / new_count
            )
            counts[channel_index] = new_count

    if (counts == 0).any():
        raise ValueError(
            "At least one mobility-history channel has no observed training values."
        )

    raw_stds = np.sqrt(np.maximum(m2 / counts, 0.0))
    effective_stds = np.where(raw_stds > 1e-8, raw_stds, 1.0)

    return means, effective_stds


def fit_streaming_transformer_preprocessor(
    job_cache,
    row_positions,
    metric,
    lookback_steps,
):
    """Fit canonical preprocessing without materializing the large sequence tensor.

    Sequence statistics are learned in chunks from every legal observed history
    cell. The existing preprocessor still fits context, weekly-STL, and target
    state from the complete legal training rows.
    """
    sequence_means, sequence_stds = streamed_sequence_statistics(
        job_cache,
        row_positions,
        metric,
        lookback_steps,
    )

    # A tiny proxy allows the existing preprocessor to execute its unchanged
    # context/STL/target fitting logic without constructing all sequence windows.
    proxy_values = np.stack(
        [
            sequence_means - sequence_stds,
            sequence_means + sequence_stds,
        ],
        axis=0,
    )[:, None, :].astype(np.float32)
    proxy_masks = np.ones_like(proxy_values, dtype=bool)

    context_frame = job_cache["context_frame"].iloc[row_positions]
    target_values = job_cache["target_values"][row_positions]

    preprocessor = TransformerPreprocessor(
        transformer_context_contract_df
    )
    preprocessor.fit(
        proxy_values,
        proxy_masks,
        context_frame,
        target_values,
    )

    # The proxy exists only to reuse the canonical context/STL/target logic.
    # Replace its sequence state with the exact full-history streamed moments.
    preprocessor.sequence_means_ = sequence_means.copy()
    preprocessor.sequence_stds_ = sequence_stds.copy()

    return preprocessor


class StreamingTransformerDataset(Dataset):
    """Materialize sequence windows only for the current PyTorch batch."""

    def __init__(
        self,
        zone_positions,
        origin_positions,
        prepared_context,
        prepared_target,
        metric,
        lookback_steps,
        preprocessor,
    ):
        self.zone_positions = np.asarray(
            zone_positions,
            dtype=np.int64,
        )
        self.origin_positions = np.asarray(
            origin_positions,
            dtype=np.int64,
        )
        self.context = np.asarray(
            prepared_context,
            dtype=np.float32,
        )
        self.target = np.asarray(
            prepared_target,
            dtype=np.float32,
        )
        self.lookback_steps = int(lookback_steps)
        self.preprocessor = preprocessor
        self.offsets = np.arange(
            self.lookback_steps,
            dtype=np.int64,
        )

        target_first = TRANSFORMER_SEQUENCE_VALUE_CHANNELS_BY_TARGET[metric]
        self.channel_positions = np.asarray(
            [
                FORECAST_TARGETS.index(channel)
                for channel in target_first
            ],
            dtype=np.int64,
        )

        row_count = len(self.zone_positions)

        if not (
            len(self.origin_positions) == row_count
            and len(self.context) == row_count
            and len(self.target) == row_count
        ):
            raise ValueError(
                "Streaming Transformer inputs are not aligned by row."
            )

    def __len__(self):
        return len(self.zone_positions)

    def __getitem__(self, index):
        # Return only the row index. The custom collator can then construct one
        # whole batch of 70-step windows with vectorized array indexing.
        return int(index)

    def collate_batch(self, indices):
        indices = np.asarray(indices, dtype=np.int64)

        zones = self.zone_positions[indices]
        origins = self.origin_positions[indices]
        starts = origins - self.lookback_steps + 1

        if (zones < 0).any() or (starts < 0).any():
            raise ValueError(
                "A streamed batch requests an illegal history window."
            )

        positions = starts[:, None] + self.offsets[None, :]

        values = transformer_history_values[
            zones[:, None],
            positions,
            :,
        ]
        masks = transformer_history_masks[
            zones[:, None],
            positions,
            :,
        ]

        values = values[:, :, self.channel_positions]
        masks = masks[:, :, self.channel_positions]

        sequence = self.preprocessor.transform_sequence(
            values,
            masks,
        )

        return (
            torch.from_numpy(sequence),
            torch.from_numpy(self.context[indices]),
            torch.from_numpy(zones.astype(np.int64, copy=False)),
            torch.from_numpy(self.target[indices]),
        )


def make_streaming_transformer_loader(
    job_cache,
    row_positions,
    preprocessor,
    metric,
    lookback_steps,
    shuffle,
):
    """Build one loader without allocating a full in-memory sequence tensor."""
    context_frame = job_cache["context_frame"].iloc[row_positions]
    prepared_context = preprocessor.transform_context(context_frame)

    prepared_target = preprocessor.transform_target(
        job_cache["target_values"][row_positions]
    )

    dataset = StreamingTransformerDataset(
        job_cache["zone_positions"][row_positions],
        job_cache["origin_positions"][row_positions],
        prepared_context,
        prepared_target,
        metric,
        lookback_steps,
        preprocessor,
    )

    return DataLoader(
        dataset,
        batch_size=selected_transformer_batch_size,
        shuffle=bool(shuffle),
        num_workers=0,
        collate_fn=dataset.collate_batch,
    )


def make_streaming_transformer_eval_loader(
    evaluation_df,
    preprocessor,
    metric,
    lookback_steps,
):
    """Build a batch-streamed loader for one monthly scoring slice."""
    zone_positions = evaluation_df["taxi_zone_id"].map(
        transformer_zone_to_index
    )
    origin_positions = evaluation_df[
        "origin_observation_sequence_id"
    ].map(transformer_sequence_to_index)

    if zone_positions.isna().any() or origin_positions.isna().any():
        raise ValueError(
            "Evaluation rows fall outside the dense Transformer state."
        )

    context_frame = derive_transformer_context_frame(evaluation_df)
    prepared_context = preprocessor.transform_context(context_frame)

    target_raw = pd.to_numeric(
        evaluation_df["target_observed_value"],
        errors="raise",
    ).to_numpy(dtype=np.float32)
    prepared_target = preprocessor.transform_target(target_raw)

    dataset = StreamingTransformerDataset(
        zone_positions.to_numpy(dtype=np.int64),
        origin_positions.to_numpy(dtype=np.int64),
        prepared_context,
        prepared_target,
        metric,
        lookback_steps,
        preprocessor,
    )

    loader = DataLoader(
        dataset,
        batch_size=selected_transformer_batch_size,
        shuffle=False,
        num_workers=0,
        collate_fn=dataset.collate_batch,
    )

    return loader, target_raw

**4.5.2.6.2C — Define one restartable uncapped production fit.**

One production fit repeats the frozen lifecycle: historical epoch selection, disposal of the selector, fresh preprocessing on all legal history, a fresh Transformer trained for the selected duration, and forecasting of the month owned by that fitted state.

Predictions, selector curves, fitted state, and the small success checkpoint are written in that order. A fit is recoverable only after all companion artifacts exist, so an interruption cannot mark partial work as complete.

In [61]:
# ---------------------------------------------------------------------
# 4.5.2.6.2C — Execute and persist one uncapped production Transformer fit
# ---------------------------------------------------------------------

def full_transformer_artifact_paths(logical_row):
    """Return the durable companions for one exact production logical fit."""
    token = logical_key_token(
        logical_row,
        TRANSFORMER_LOGICAL_KEY_COLUMNS,
    )

    return {
        "checkpoint": TRANSFORMER_FULL_ROOT / f"fit_{token}.parquet",
        "prediction": TRANSFORMER_FULL_ROOT / f"prediction_{token}.parquet",
        "curve": TRANSFORMER_FULL_ROOT / f"curve_{token}.parquet",
        "state": TRANSFORMER_FULL_ROOT / f"state_{token}.pt",
    }


def score_streaming_transformer(model, loader, preprocessor):
    """Forecast one scoring slice in batches and return native-unit predictions."""
    standardized_batches = []
    model.eval()

    with torch.no_grad():
        for sequence, context, zones, _ in loader:
            standardized_batches.append(
                model(
                    sequence.to(TORCH_DEVICE),
                    context.to(TORCH_DEVICE),
                    zones.to(TORCH_DEVICE),
                ).cpu().numpy()
            )

    if not standardized_batches:
        raise ValueError(
            "Production scoring loader produced no batches."
        )

    standardized = np.concatenate(standardized_batches)
    return preprocessor.inverse_target(standardized)


def execute_one_full_transformer_fit(
    job,
    refit_row,
    job_cache,
    split,
):
    """Run selector → fresh uncapped refit → monthly forecast for one fit."""
    logical_row = {**refit_row, "status": "planned"}

    representation = next(
        item
        for item in TRANSFORMER_REPRESENTATION_CANDIDATES
        if item["representation_id"] == job["representation_id"]
    )
    profile = next(
        item
        for item in TRANSFORMER_PROFILE_CANDIDATES
        if item["profile_id"] == job["profile_id"]
    )
    paths = full_transformer_artifact_paths(logical_row)

    seed = stable_transformer_fit_seed(
        logical_row["contract_id"],
        logical_row["track"],
        logical_row["metric"],
        logical_row["horizon"],
        logical_row["candidate_id"],
        logical_row["temporal_window"],
    )

    seed_experiment(seed)
    reset_cuda_peak_memory()
    fit_started = time.perf_counter()

    # -------------------------------------------------------------
    # Phase 1 — choose training duration from legal historical data
    # -------------------------------------------------------------

    selector_preprocessor = fit_streaming_transformer_preprocessor(
        job_cache,
        split["selector_positions"],
        job["metric"],
        job["lookback_steps"],
    )

    selector_train_loader = make_streaming_transformer_loader(
        job_cache,
        split["selector_positions"],
        selector_preprocessor,
        job["metric"],
        job["lookback_steps"],
        shuffle=True,
    )
    selector_inner_loader = make_streaming_transformer_loader(
        job_cache,
        split["inner_positions"],
        selector_preprocessor,
        job["metric"],
        job["lookback_steps"],
        shuffle=False,
    )

    selector_model = PatchTransformerRegressor(
        NEURAL_SEQUENCE_INPUT_DIM,
        transformer_context_feature_count,
        len(transformer_zone_ids),
        representation,
        profile,
    ).to(TORCH_DEVICE)

    selected_epoch, curve_df = _select_transformer_epoch(
        selector_model,
        selector_train_loader,
        selector_inner_loader,
        profile,
    )

    del (
        selector_model,
        selector_train_loader,
        selector_inner_loader,
        selector_preprocessor,
    )
    clear_device_memory()

    # -------------------------------------------------------------
    # Phase 2 — fresh refit using every legally available row
    # -------------------------------------------------------------

    final_preprocessor = fit_streaming_transformer_preprocessor(
        job_cache,
        split["final_positions"],
        job["metric"],
        job["lookback_steps"],
    )

    final_loader = make_streaming_transformer_loader(
        job_cache,
        split["final_positions"],
        final_preprocessor,
        job["metric"],
        job["lookback_steps"],
        shuffle=True,
    )

    final_model = PatchTransformerRegressor(
        NEURAL_SEQUENCE_INPUT_DIM,
        transformer_context_feature_count,
        len(transformer_zone_ids),
        representation,
        profile,
    ).to(TORCH_DEVICE)

    _fit_transformer_fixed_epochs(
        final_model,
        final_loader,
        profile,
        selected_epoch,
    )

    # -------------------------------------------------------------
    # Phase 3 — score only the month owned by this fitted state
    # -------------------------------------------------------------

    evaluation_df = split["evaluation_df"]

    eval_loader, target_raw = make_streaming_transformer_eval_loader(
        evaluation_df,
        final_preprocessor,
        job["metric"],
        job["lookback_steps"],
    )

    prediction_raw = score_streaming_transformer(
        final_model,
        eval_loader,
        final_preprocessor,
    )
    prediction_final = apply_prediction_output_rule(
        prediction_raw
    )

    prediction_df = evaluation_df[
        [
            "forecast_sequence_id",
            "taxi_zone_id",
            "origin_observation_sequence_id",
            "origin_date",
            "target_observation_sequence_id",
            "target_date",
        ]
    ].copy()

    prediction_df["target_observed_value"] = target_raw
    prediction_df["benchmark_prediction"] = pd.to_numeric(
        evaluation_df[job_cache["benchmark_column"]],
        errors="raise",
    ).to_numpy(dtype=np.float32)
    prediction_df["prediction_raw"] = prediction_raw
    prediction_df["prediction_final"] = prediction_final

    prediction_df["track"] = logical_row["track"]
    prediction_df["metric"] = logical_row["metric"]
    prediction_df["horizon"] = int(logical_row["horizon"])
    prediction_df["candidate_id"] = logical_row["candidate_id"]
    prediction_df["temporal_window"] = logical_row["temporal_window"]
    prediction_df["job_id"] = logical_row["job_id"]
    prediction_df["refit_step"] = int(logical_row["refit_step"])

    # Selector curves remain independently identifiable after a kernel restart.
    curve_df["track"] = logical_row["track"]
    curve_df["metric"] = logical_row["metric"]
    curve_df["horizon"] = int(logical_row["horizon"])
    curve_df["candidate_id"] = logical_row["candidate_id"]
    curve_df["temporal_window"] = logical_row["temporal_window"]
    curve_df["job_id"] = logical_row["job_id"]
    curve_df["refit_step"] = int(logical_row["refit_step"])

    raw_metrics = regression_metrics(
        target_raw,
        prediction_raw,
    )
    benchmark_metrics = regression_metrics(
        target_raw,
        prediction_df["benchmark_prediction"],
    )

    benchmark_mae = benchmark_metrics["mae"]
    benchmark_skill_pct = (
        100.0 * (benchmark_mae - raw_metrics["mae"]) / benchmark_mae
        if benchmark_mae > 0
        else np.nan
    )

    regimes = validate_full_transformer_regime(
        job_cache,
        job,
        split,
    )

    result = {
        **logical_row,
        "status": "success",
        "fit_seed": int(seed),
        "selected_epoch": int(selected_epoch),
        "parameter_count": transformer_fit_parameter_count(final_model),
        "runtime_seconds": time.perf_counter() - fit_started,
        "peak_gpu_memory_mb": read_cuda_peak_memory_mb(),
        "selector_train_rows": int(len(split["selector_positions"])),
        "inner_validation_rows": int(len(split["inner_positions"])),
        "final_refit_rows": int(len(split["final_positions"])),
        "evaluation_rows": int(len(target_raw)),
        "training_origin_cap": np.nan,
        "production_uncapped": True,
        "selector_regimes": "/".join(
            f"{value:.0f}" for value in regimes["selector"]
        ),
        "final_refit_regimes": "/".join(
            f"{value:.0f}" for value in regimes["final"]
        ),
        "raw_mae": raw_metrics["mae"],
        "benchmark_mae": benchmark_mae,
        "benchmark_skill_pct": benchmark_skill_pct,
        "negative_raw_predictions": int(
            np.sum(np.asarray(prediction_raw) < 0)
        ),
        "prediction_path": str(paths["prediction"]),
        "curve_path": str(paths["curve"]),
        "state_path": str(paths["state"]),
    }

    # Companion artifacts land first; the success checkpoint is the final
    # commit. A crash cannot make a partial fit look recoverable.
    atomic_write_parquet(
        prediction_df,
        paths["prediction"],
    )
    atomic_write_parquet(
        curve_df,
        paths["curve"],
    )

    atomic_torch_save(
        {
            "contract_id": logical_row["contract_id"],
            "logical_key": {
                column: logical_row[column]
                for column in TRANSFORMER_LOGICAL_KEY_COLUMNS
            },
            "model_state_dict": final_model.state_dict(),
            "representation": dict(representation),
            "profile": dict(profile),
            "selected_epoch": int(selected_epoch),
            "context_feature_order": list(
                TRANSFORMER_CONTEXT_FEATURE_ORDER
            ),
            "sequence_value_channels": list(
                TRANSFORMER_SEQUENCE_VALUE_CHANNELS_BY_TARGET[
                    job["metric"]
                ]
            ),
            "sequence_mask_channels": list(
                TRANSFORMER_SEQUENCE_MASK_CHANNELS_BY_TARGET[
                    job["metric"]
                ]
            ),
            "zone_ids": np.asarray(
                transformer_zone_ids,
                dtype=np.int64,
            ),
            "sequence_ids": np.asarray(
                transformer_sequence_ids,
                dtype=np.int64,
            ),
            "context_contract":
                transformer_context_contract_df.to_dict("records"),
            "preprocessor_state":
                final_preprocessor.export_state(),
        },
        paths["state"],
    )

    atomic_write_parquet(
        pd.DataFrame([result]),
        paths["checkpoint"],
    )

    del final_model, final_loader, eval_loader, final_preprocessor
    clear_device_memory()

    return result

**4.5.2.6.3A — Audit the uncapped production split on both training-history tracks.**

Before training a large model, inspect one late monthly split from each selected track. This checks the actual uncapped row counts, the 28-day historical selector tail, the pre/post-policy composition of training, the monthly scoring population, and the final-holdout boundary.

This is a row-contract check only; it does not fit a Transformer.

In [62]:
# ---------------------------------------------------------------------
# 4.5.2.6.3A — Audit one uncapped split from each selected training-history track
# ---------------------------------------------------------------------

split_audit_rows = []

for track in ["post_cp_primary", "full_history_regime_aware"]:
    track_jobs = transformer_full_jobs_df.loc[
        transformer_full_jobs_df["track"].eq(track)
        & transformer_full_jobs_df["horizon"].astype(int).eq(1)
    ].copy()

    if track_jobs.empty:
        raise ValueError(
            f"No production h=1 job uses {track}."
        )

    job = track_jobs.iloc[0].to_dict()

    refit_row = (
        transformer_full_schedule_df.loc[
            transformer_full_schedule_df["job_id"].eq(job["job_id"])
        ]
        .sort_values("refit_step")
        .iloc[-1]
        .to_dict()
    )

    job_frames = load_transformer_cadence_job(job)
    split = resolve_full_transformer_split(
        job_frames,
        job,
        refit_row,
    )
    regimes = validate_full_transformer_regime(
        job_frames,
        job,
        split,
    )

    split_audit_rows.append({
        "Track": TRACK_LABELS[track],
        "Target": TARGET_LABELS[job["metric"]],
        "Selector rows": len(split["selector_positions"]),
        "Inner rows": len(split["inner_positions"]),
        "Final-refit rows": len(split["final_positions"]),
        "Monthly score rows": len(split["evaluation_df"]),
        "Training regimes": "/".join(
            f"{value:.0f}" for value in regimes["final"]
        ),
        "Holdout rows": int(
            pd.to_datetime(
                split["evaluation_df"]["target_date"]
            )
            .ge(FINAL_HOLDOUT_START_DATE)
            .sum()
        ),
    })

    job_frames = None
    split = None
    gc.collect()

transformer_full_split_audit_df = pd.DataFrame(
    split_audit_rows
)

display(transformer_full_split_audit_df)

,Track,Target,Selector rows,Inner rows,Final-refit rows,Monthly score rows,Training regimes,Holdout rows
0,Post-CP Primary,FHVHV trips,410707,34913,445620,4870,1,0
1,Full-History Regime-Aware,FHVHV average speed,1352614,34880,1387494,4858,0/1,0


**Findings.** The uncapped production splits pass the key training-history and leakage checks.

The Post-CP example uses **445,620 final-refit rows** and contains only post-policy observations. The Full-History example is much larger at **1,387,494 rows** and correctly contains both pre- and post-CP observations. Both retain roughly **35,000 rows** for the 28-day inner validation period.

Neither scoring surface contains any final-holdout rows. The production setup therefore preserves the intended distinction between the two training-history tracks while using substantially more training data than the earlier development screens.

**4.5.2.6.3B — Time one exact late production fit before launching all 150.**

Use a late **Full-History Regime-Aware + higher-capacity** job as the stress test because it combines the longer selected training history with the larger selected Transformer. The fit follows the exact production contract with no 192-origin cap.

The 24-hour target for 150 logical fits corresponds to an average budget of **9.6 minutes per fit**. This single late fit is intentionally conservative rather than a forecast of the exact 150-fit average. Because it is saved under its real production identity, the later crawl will recover it instead of training it again.

In [63]:
# ---------------------------------------------------------------------
# 4.5.2.6.3B — Run or recover one exact late production fit as a timing probe
# ---------------------------------------------------------------------

import warnings

probe_candidates_df = transformer_full_schedule_df.loc[
    transformer_full_schedule_df["track"].eq(
        "full_history_regime_aware"
    )
    & transformer_full_schedule_df["profile_id"].eq(
        "higher_capacity_regularized"
    )
].copy()

if probe_candidates_df.empty:
    raise ValueError(
        "The timing probe expects at least one Full-History "
        "higher-capacity production job."
    )

# Choose the latest h=5 state from the heaviest selected history/profile
# combination. This is a conservative stress test and a real production fit.
probe_row = (
    probe_candidates_df
    .sort_values(
        ["refit_step", "horizon"],
        ascending=[False, False],
    )
    .iloc[0]
    .to_dict()
)

probe_job = (
    transformer_full_jobs_df.loc[
        transformer_full_jobs_df["job_id"].eq(
            probe_row["job_id"]
        )
    ]
    .iloc[0]
    .to_dict()
)

logical_key = {
    column: probe_row[column]
    for column in TRANSFORMER_LOGICAL_KEY_COLUMNS
}
probe_paths = full_transformer_artifact_paths(
    probe_row
)

# Recovery comes before loading the large matrix. A completed production fit
# should cost only artifact validation after a notebook restart.
recovered_probe = recover_transformer_fit(
    probe_paths["checkpoint"],
    TRANSFORMER_FULL_CONTRACT_ID,
    logical_key,
    probe_paths["prediction"],
    probe_paths["curve"],
    probe_paths["state"],
)

if recovered_probe is None:
    probe_cache = prepare_full_transformer_job_cache(
        probe_job
    )
    probe_split = resolve_full_transformer_split(
        probe_cache,
        probe_job,
        probe_row,
    )
    validate_full_transformer_regime(
        probe_cache,
        probe_job,
        probe_split,
    )

    print(
        "→ Production timing probe · "
        f"{TARGET_LABELS[probe_job['metric']]} · "
        f"{TRACK_LABELS[probe_job['track']]} · "
        f"h={int(probe_job['horizon'])} · "
        f"{probe_row['validation_month']} · fitting...",
        flush=True,
    )

    with warnings.catch_warnings():
        warnings.filterwarnings(
            "ignore",
            message=(
                r"enable_nested_tensor is True, but self\.use_nested_tensor "
                r"is False because encoder_layer\.norm_first was True"
            ),
            category=UserWarning,
        )

        probe_result = execute_one_full_transformer_fit(
            probe_job,
            probe_row,
            probe_cache,
            probe_split,
        )

    probe_status = "NEW"

    probe_cache = None
    probe_split = None
    gc.collect()
    clear_device_memory()

else:
    probe_result = recovered_probe
    probe_status = "RECOVERED"

fit_budget_minutes = (
    24.0 * 60.0 / TRANSFORMER_FULL_EXPECTED_FITS
)
probe_minutes = (
    float(probe_result["runtime_seconds"]) / 60.0
)

transformer_full_timing_probe_df = pd.DataFrame([{
    "Status": probe_status,
    "Target": TARGET_LABELS[probe_result["metric"]],
    "Track": TRACK_LABELS[probe_result["track"]],
    "Horizon": int(probe_result["horizon"]),
    "Validation month": probe_result["validation_month"],
    "Final-refit rows": int(probe_result["final_refit_rows"]),
    "Selected epoch": int(probe_result["selected_epoch"]),
    "Runtime min": probe_minutes,
    "24h average-fit budget min": fit_budget_minutes,
    "Probe / average budget": probe_minutes / fit_budget_minutes,
    "Raw MAE": float(probe_result["raw_mae"]),
    "Benchmark skill %": float(
        probe_result["benchmark_skill_pct"]
    ),
}]).round(3)

display(transformer_full_timing_probe_df)

,Status,Target,Track,Horizon,Validation month,Final-refit rows,Selected epoch,Runtime min,24h average-fit budget min,Probe / average budget,Raw MAE,Benchmark skill %
0,RECOVERED,FHVHV average speed,Full-History Regime-Aware,5,2026-01,1386469,12,70.461,9.600,7.340,1.013,36.814


**Findings.** The uncapped production probe successfully trained the late Full-History FHVHV-speed model on **1,386,469 rows**, selecting **12 epochs**. Forecast accuracy was strong: raw MAE was **1.013**, beating the frozen weekly benchmark by **36.814%**.

Runtime, however, was **70.461 minutes**—**7.34×** the 9.6-minute average needed to keep the 150-fit production crawl within 24 hours. The production contract is therefore viable, but the Deepnote runtime is not practical for the full crawl; the remaining fits should be executed with the same restartable contract on the local GPU.

In [64]:
# # 4.5.2 portable full-validation exporter
# # Run INSIDE the live Deepnote notebook namespace after Section 6.2C has executed.
# # It creates one self-contained ZIP for the local GPU runner.

# from pathlib import Path
# import hashlib
# import json
# import shutil
# import zipfile

# import duckdb
# import numpy as np
# import pandas as pd

# PORTABLE_CONTRACT_ID = "transformer_full_portable_v1"
# PORTABLE_STAGING_ROOT = INTERMEDIATE_DIR / PORTABLE_CONTRACT_ID
# PORTABLE_ZIP_PATH = INTERMEDIATE_DIR / f"{PORTABLE_CONTRACT_ID}.zip"
# DELETE_STAGING_AFTER_ZIP = True

# REQUIRED_GLOBALS = [
#     "transformer_full_jobs_df", "transformer_full_schedule_df",
#     "transformer_context_contract_df",
#     "TRANSFORMER_SEQUENCE_VALUE_CHANNELS_BY_TARGET",
#     "TRANSFORMER_SEQUENCE_MASK_CHANNELS_BY_TARGET",
#     "TRANSFORMER_CONTEXT_FEATURE_ORDER", "TRACK_HISTORY_START_SEQUENCE_ID",
#     "selected_transformer_batch_size", "advanced_model_benchmark_registry_df",
#     "FORECASTING_MATRIX_PATHS", "BASELINE_PREDICTION_PATHS",
#     "NEURAL_HISTORY_STATE_PATH", "TRANSFORMER_FULL_ROOT",
#     "TRANSFORMER_FULL_CONTRACT_ID",
# ]

# missing_globals = [name for name in REQUIRED_GLOBALS if name not in globals()]
# if missing_globals:
#     raise RuntimeError(
#         "Run the authoritative 4.5.2 notebook through Section 6.2C first. "
#         f"Missing notebook state: {missing_globals}"
#     )

# if "In" not in globals():
#     raise RuntimeError(
#         "This exporter must run inside the live notebook kernel so it can capture "
#         "the exact executed modeling helper source."
#     )


# def _latest_executed_cell(marker):
#     """Return the executed notebook cell whose header contains one exact marker.

#     Only the first 20 lines are searched. This accommodates our standard
#     separator lines while preventing this exporter from matching itself merely
#     because SOURCE_MARKERS lists the same section names farther down the cell.
#     """
#     matches = []

#     for source in In:
#         if not isinstance(source, str):
#             continue

#         header = "\n".join(source.lstrip().splitlines()[:20])

#         if marker in header:
#             matches.append(source)

#     if not matches:
#         raise RuntimeError(
#             f"Required executed notebook cell was not found near a cell header: {marker}"
#         )

#     return matches[-1].rstrip() + "\n"


# def _jsonable(value):
#     """Convert the small frozen contract into ordinary JSON-safe Python values."""
#     if isinstance(value, dict):
#         return {str(key): _jsonable(item) for key, item in value.items()}
#     if isinstance(value, (list, tuple)):
#         return [_jsonable(item) for item in value]
#     if isinstance(value, np.integer):
#         return int(value)
#     if isinstance(value, np.floating):
#         return float(value)
#     if isinstance(value, pd.Timestamp):
#         return value.isoformat()
#     return value


# def _sql_quote(value):
#     """Escape one literal for the throwaway DuckDB export query."""
#     return str(value).replace("'", "''")


# def _sha256(path, chunk_size=8 * 1024 * 1024):
#     """Hash large bundle files without reading them fully into memory."""
#     digest = hashlib.sha256()
#     with Path(path).open("rb") as handle:
#         for chunk in iter(lambda: handle.read(chunk_size), b""):
#             digest.update(chunk)
#     return digest.hexdigest()


# # ---------------------------------------------------------------------
# # 1. Start a clean staging directory
# # ---------------------------------------------------------------------

# if PORTABLE_STAGING_ROOT.exists():
#     shutil.rmtree(PORTABLE_STAGING_ROOT)
# if PORTABLE_ZIP_PATH.exists():
#     PORTABLE_ZIP_PATH.unlink()

# data_dir = PORTABLE_STAGING_ROOT / "data"
# contract_dir = PORTABLE_STAGING_ROOT / "contract"
# runtime_dir = PORTABLE_STAGING_ROOT / "runtime"
# seed_dir = PORTABLE_STAGING_ROOT / "seed" / "tr_full_v1"

# for path in [data_dir, contract_dir, runtime_dir, seed_dir]:
#     path.mkdir(parents=True, exist_ok=True)


# # ---------------------------------------------------------------------
# # 2. Freeze the already-selected production roster and model contract
# # ---------------------------------------------------------------------

# transformer_full_jobs_df.to_parquet(
#     contract_dir / "transformer_full_jobs.parquet", index=False, compression="zstd"
# )
# transformer_full_schedule_df.to_parquet(
#     contract_dir / "transformer_full_schedule.parquet", index=False, compression="zstd"
# )
# transformer_context_contract_df.to_parquet(
#     contract_dir / "transformer_context_contract.parquet", index=False, compression="zstd"
# )

# selected_pairs = (
#     transformer_full_jobs_df[["track", "metric"]]
#     .drop_duplicates()
#     .sort_values(["track", "metric"])
# )

# benchmark_subset = advanced_model_benchmark_registry_df.merge(
#     selected_pairs,
#     left_on=["modeling_track", "metric"],
#     right_on=["track", "metric"],
#     how="inner",
# ).drop(columns="track")

# benchmark_subset.to_parquet(
#     contract_dir / "advanced_model_benchmark_registry.parquet",
#     index=False,
#     compression="zstd",
# )

# runtime_config = {
#     "portable_contract_id": PORTABLE_CONTRACT_ID,
#     "production_contract_id": TRANSFORMER_FULL_CONTRACT_ID,
#     "global_seed": int(GLOBAL_SEED),
#     "final_holdout_start_date": str(pd.Timestamp(FINAL_HOLDOUT_START_DATE).date()),
#     "forecast_targets": list(FORECAST_TARGETS),
#     "forecast_horizons": [int(value) for value in FORECAST_HORIZONS],
#     "track_labels": dict(TRACK_LABELS),
#     "target_labels": dict(TARGET_LABELS),
#     "daypart_order": list(DAYPART_ORDER),
#     "sequence_input_dim": int(NEURAL_SEQUENCE_INPUT_DIM),
#     "context_feature_count": int(transformer_context_feature_count),
#     "selected_batch_size": int(selected_transformer_batch_size),
#     "track_history_start_sequence_id": {
#         key: int(value) for key, value in TRACK_HISTORY_START_SEQUENCE_ID.items()
#     },
#     "sequence_value_channels_by_target": _jsonable(
#         TRANSFORMER_SEQUENCE_VALUE_CHANNELS_BY_TARGET
#     ),
#     "sequence_mask_channels_by_target": _jsonable(
#         TRANSFORMER_SEQUENCE_MASK_CHANNELS_BY_TARGET
#     ),
#     "context_feature_order": list(TRANSFORMER_CONTEXT_FEATURE_ORDER),
#     "representation_candidates": _jsonable(TRANSFORMER_REPRESENTATION_CANDIDATES),
#     "profile_candidates": _jsonable(TRANSFORMER_PROFILE_CANDIDATES),
#     "max_epochs": int(TRANSFORMER_MAX_EPOCHS),
#     "patience": int(TRANSFORMER_PATIENCE),
#     "min_delta": float(TRANSFORMER_MIN_DELTA),
#     "grad_clip_norm": float(TRANSFORMER_GRAD_CLIP_NORM),
#     "logical_key_columns": list(TRANSFORMER_LOGICAL_KEY_COLUMNS),
# }

# with (contract_dir / "runtime_config.json").open("w", encoding="utf-8") as handle:
#     json.dump(runtime_config, handle, indent=2)


# # ---------------------------------------------------------------------
# # 3. Copy dense history and export only selected target/track rows
# # ---------------------------------------------------------------------

# shutil.copy2(NEURAL_HISTORY_STATE_PATH, data_dir / "neural_mobility_history_state.npz")

# duck = duckdb.connect()
# try:
#     for track in ORDINARY_TRACKS:
#         track_metrics = (
#             selected_pairs.loc[selected_pairs["track"].eq(track), "metric"]
#             .astype(str)
#             .tolist()
#         )
#         if not track_metrics:
#             continue

#         metric_sql = ", ".join(f"'{_sql_quote(metric)}'" for metric in track_metrics)

#         matrix_source = _sql_quote(Path(FORECASTING_MATRIX_PATHS[track]).resolve())
#         matrix_target = _sql_quote(data_dir / f"forecasting_matrix__{track}.parquet")

#         # Keep every column for the selected production rows. This is slightly
#         # larger than a hand-pruned schema, but avoids maintaining a second copy
#         # of the feature contract solely for transport.
#         duck.execute(
#             f"""
#             COPY (
#                 SELECT *
#                 FROM read_parquet('{matrix_source}')
#                 WHERE metric IN ({metric_sql})
#                   AND horizon IN (1, 2, 5)
#                   AND native_track_eligible = TRUE
#                   AND stage IN ('initial_training', 'rolling_validation')
#             )
#             TO '{matrix_target}'
#             (FORMAT PARQUET, COMPRESSION ZSTD);
#             """
#         )

#         baseline_source = _sql_quote(Path(BASELINE_PREDICTION_PATHS[track]).resolve())
#         baseline_target = _sql_quote(data_dir / f"baseline_predictions__{track}.parquet")

#         duck.execute(
#             f"""
#             COPY (
#                 SELECT *
#                 FROM read_parquet('{baseline_source}')
#                 WHERE metric IN ({metric_sql})
#                   AND horizon IN (1, 2, 5)
#                   AND stage = 'rolling_validation'
#             )
#             TO '{baseline_target}'
#             (FORMAT PARQUET, COMPRESSION ZSTD);
#             """
#         )
# finally:
#     duck.close()


# # ---------------------------------------------------------------------
# # 4. Capture the EXACT executed notebook modeling helpers
# # ---------------------------------------------------------------------
# # The local runtime is assembled from the live executed notebook source rather
# # than maintained as a parallel modeling implementation.

# SOURCE_MARKERS = [
#     "# 4.5.2.0C — External-input, persistence, restart, and evaluation helpers",
#     "# 4.5.2.2.3A — Transformer preprocessor",
#     "# 4.5.2.2.3C — Patch-based Transformer regressor",
#     "# 4.5.2.3.0B — Select training duration and perform fresh final refit",
#     "# 4.5.2.3.0D — Reconstruct the finalized 52-value context",
#     "# 4.5.2.4.2A — Load one cadence job on exact benchmark-aligned support",
#     "# 4.5.2.6.2A — Resolve uncapped production rows without duplicating them",
#     "# 4.5.2.6.2B — Stream full-history sequences through the frozen preprocessor",
#     "# 4.5.2.6.2C — Execute and persist one uncapped production Transformer fit",
# ]

# runtime_header = r'''# Auto-generated from the executed 4.5.2 notebook.
# # Do not hand-edit: this module is a portable execution copy of notebook logic.

# from pathlib import Path
# import gc
# import hashlib
# import json
# import os
# import random
# import shutil
# import time
# import uuid
# import warnings

# import numpy as np
# import pandas as pd
# import torch
# from torch import nn
# from torch.utils.data import Dataset, DataLoader

# BUNDLE_ROOT = Path(
#     os.environ.get("TRANSFORMER_BUNDLE_ROOT", Path(__file__).resolve().parents[1])
# ).resolve()
# DATA_DIR = BUNDLE_ROOT / "data"
# CONTRACT_DIR = BUNDLE_ROOT / "contract"

# with (CONTRACT_DIR / "runtime_config.json").open("r", encoding="utf-8") as handle:
#     _cfg = json.load(handle)

# GLOBAL_SEED = int(_cfg["global_seed"])
# FINAL_HOLDOUT_START_DATE = pd.Timestamp(_cfg["final_holdout_start_date"])
# FORECAST_TARGETS = list(_cfg["forecast_targets"])
# FORECAST_HORIZONS = [int(value) for value in _cfg["forecast_horizons"]]
# ORDINARY_TRACKS = ["post_cp_primary", "full_history_regime_aware"]
# TRACK_LABELS = dict(_cfg["track_labels"])
# TARGET_LABELS = dict(_cfg["target_labels"])
# DAYPART_ORDER = list(_cfg["daypart_order"])

# NEURAL_SEQUENCE_INPUT_DIM = int(_cfg["sequence_input_dim"])
# transformer_context_feature_count = int(_cfg["context_feature_count"])
# selected_transformer_batch_size = int(_cfg["selected_batch_size"])

# TRANSFORMER_REPRESENTATION_CANDIDATES = list(_cfg["representation_candidates"])
# TRANSFORMER_PROFILE_CANDIDATES = list(_cfg["profile_candidates"])
# TRANSFORMER_SEQUENCE_VALUE_CHANNELS_BY_TARGET = dict(
#     _cfg["sequence_value_channels_by_target"]
# )
# TRANSFORMER_SEQUENCE_MASK_CHANNELS_BY_TARGET = dict(
#     _cfg["sequence_mask_channels_by_target"]
# )
# TRANSFORMER_CONTEXT_FEATURE_ORDER = list(_cfg["context_feature_order"])
# TRACK_HISTORY_START_SEQUENCE_ID = {
#     key: int(value) for key, value in _cfg["track_history_start_sequence_id"].items()
# }

# TRANSFORMER_MAX_EPOCHS = int(_cfg["max_epochs"])
# TRANSFORMER_PATIENCE = int(_cfg["patience"])
# TRANSFORMER_MIN_DELTA = float(_cfg["min_delta"])
# TRANSFORMER_GRAD_CLIP_NORM = float(_cfg["grad_clip_norm"])
# TRANSFORMER_LOGICAL_KEY_COLUMNS = list(_cfg["logical_key_columns"])

# TRANSFORMER_FULL_CONTRACT_ID = str(_cfg["production_contract_id"])
# TRANSFORMER_FULL_ROOT = Path("pipeline_data/4.5.2.intermediate_tables/tr_full_v1")
# TRANSFORMER_FULL_RESULTS_PATH = TRANSFORMER_FULL_ROOT / "results.parquet"

# FORECASTING_MATRIX_PATHS = {
#     track: DATA_DIR / f"forecasting_matrix__{track}.parquet"
#     for track in ORDINARY_TRACKS
# }
# BASELINE_PREDICTION_PATHS = {
#     track: DATA_DIR / f"baseline_predictions__{track}.parquet"
#     for track in ORDINARY_TRACKS
# }

# advanced_model_benchmark_registry_df = pd.read_parquet(
#     CONTRACT_DIR / "advanced_model_benchmark_registry.parquet"
# )
# transformer_context_contract_df = pd.read_parquet(
#     CONTRACT_DIR / "transformer_context_contract.parquet"
# )

# TORCH_DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# TORCH_DTYPE = torch.float32

# with np.load(DATA_DIR / "neural_mobility_history_state.npz", allow_pickle=False) as _state:
#     transformer_history_values = np.asarray(_state["values"], dtype=np.float32)
#     transformer_history_masks = np.asarray(_state["masks"]).astype(bool, copy=False)
#     transformer_zone_ids = np.asarray(_state["zone_ids"], dtype=np.int64)
#     transformer_sequence_ids = np.asarray(_state["sequence_ids"], dtype=np.int64)

# if transformer_history_values.shape != transformer_history_masks.shape:
#     raise ValueError("Portable history values/masks do not align.")
# if transformer_history_values.ndim != 3 or transformer_history_values.shape[-1] != 5:
#     raise ValueError("Portable history must have shape [zone, sequence, 5].")

# transformer_zone_to_index = {
#     int(zone_id): index for index, zone_id in enumerate(transformer_zone_ids)
# }
# transformer_sequence_to_index = {
#     int(sequence_id): index
#     for index, sequence_id in enumerate(transformer_sequence_ids)
# }
# '''

# runtime_source = [runtime_header]
# for marker in SOURCE_MARKERS:
#     runtime_source.append("\n\n" + _latest_executed_cell(marker))

# (runtime_dir / "transformer_runtime.py").write_text(
#     "".join(runtime_source), encoding="utf-8"
# )


# # ---------------------------------------------------------------------
# # 5. Carry forward any already-completed Deepnote production fit
# # ---------------------------------------------------------------------

# if Path(TRANSFORMER_FULL_ROOT).exists():
#     for source in Path(TRANSFORMER_FULL_ROOT).iterdir():
#         if source.is_file():
#             shutil.copy2(source, seed_dir / source.name)


# # ---------------------------------------------------------------------
# # 6. Build manifest, ZIP once, then delete staging to avoid duplication
# # ---------------------------------------------------------------------

# manifest_rows = []
# for path in sorted(PORTABLE_STAGING_ROOT.rglob("*")):
#     if path.is_file():
#         manifest_rows.append({
#             "path": path.relative_to(PORTABLE_STAGING_ROOT).as_posix(),
#             "bytes": int(path.stat().st_size),
#             "sha256": _sha256(path),
#         })

# manifest = {
#     "portable_contract_id": PORTABLE_CONTRACT_ID,
#     "production_contract_id": TRANSFORMER_FULL_CONTRACT_ID,
#     "expected_jobs": int(len(transformer_full_jobs_df)),
#     "expected_fits": int(len(transformer_full_schedule_df)),
#     "selected_batch_size": int(selected_transformer_batch_size),
#     "files": manifest_rows,
# }

# with (PORTABLE_STAGING_ROOT / "manifest.json").open("w", encoding="utf-8") as handle:
#     json.dump(manifest, handle, indent=2)

# # Parquet/NPZ are already compressed. ZIP_STORED avoids wasting CPU time trying
# # to recompress a multi-GB transport bundle.
# with zipfile.ZipFile(
#     PORTABLE_ZIP_PATH,
#     mode="w",
#     compression=zipfile.ZIP_STORED,
#     allowZip64=True,
# ) as archive:
#     for path in sorted(PORTABLE_STAGING_ROOT.rglob("*")):
#         if path.is_file():
#             archive.write(
#                 path,
#                 arcname=path.relative_to(PORTABLE_STAGING_ROOT).as_posix(),
#             )

# zip_gb = PORTABLE_ZIP_PATH.stat().st_size / (1024 ** 3)

# if DELETE_STAGING_AFTER_ZIP:
#     shutil.rmtree(PORTABLE_STAGING_ROOT)

# print(
#     "Portable Transformer bundle ready | "
#     f"jobs={len(transformer_full_jobs_df):,} | "
#     f"fits={len(transformer_full_schedule_df):,} | "
#     f"size={zip_gb:.2f} GB"
# )
# print(f"Download: {PORTABLE_ZIP_PATH}")


**4.5.2.6.3C — Run the complete monthly production validation**

The final Transformer validation now applies the frozen design to the complete ordinary-validation surface: **15 Target × Horizon jobs × 10 monthly refits = 150 logical fits**.

Each fit preserves the same lifecycle established above: choose training duration from a strictly historical 28-day inner-validation tail, discard that selector, refit preprocessing from all legally available history, train a fresh model for the selected number of epochs, and score only the month owned by that fitted state.

Recovery happens before expensive work begins. A complete current-contract experiment therefore performs **zero fitting** on rerun; an interrupted experiment resumes only its missing fits. Missing artifacts never trigger a large run silently—the explicit production toggle remains the safety gate. The final holdout remains excluded throughout.

In [65]:
# ---------------------------------------------------------------------
# 4.5.2.6.3C — Run or recover the complete monthly production validation
# ---------------------------------------------------------------------

def recover_full_transformer_roster():
    """Recover valid production fits and return the unfinished logical roster."""
    recovered_rows = []
    missing_rows = []

    for logical_row in transformer_full_schedule_df.to_dict("records"):
        paths = full_transformer_artifact_paths(logical_row)
        logical_key = {
            column: logical_row[column]
            for column in TRANSFORMER_LOGICAL_KEY_COLUMNS
        }

        recovered = recover_transformer_fit(
            paths["checkpoint"],
            TRANSFORMER_FULL_CONTRACT_ID,
            logical_key,
            prediction_path=paths["prediction"],
            curve_path=paths["curve"],
            state_path=paths["state"],
        )

        if recovered is None:
            missing_rows.append(logical_row)
        else:
            recovered_rows.append(recovered)

    return pd.DataFrame(recovered_rows), missing_rows


if RESET_TRANSFORMER_FULL_VALIDATION:
    reset_experiment_artifacts(
        TRANSFORMER_FULL_ROOT,
        consolidated_paths=[TRANSFORMER_FULL_RESULTS_PATH],
    )


# Recover first so Restart kernel → Run All becomes a zero-fit operation once
# every current-contract logical fit is safely stored.
recovered_df, missing_rows = recover_full_transformer_roster()

print(
    "Production restart state | "
    f"{len(recovered_df):,}/{len(transformer_full_schedule_df):,} complete | "
    f"{len(missing_rows):,} unfinished"
)


# Missing files alone never launch a large training run. The explicit toggle is
# the safety gate; when enabled, only unfinished current-contract fits execute.
if missing_rows and not RUN_TRANSFORMER_FULL_VALIDATION:
    raise RuntimeError(
        "Production validation is incomplete. Set "
        "RUN_TRANSFORMER_FULL_VALIDATION=True to execute only the missing "
        f"{len(missing_rows):,} logical fits."
    )


if missing_rows:
    missing_keys = {
        (row["job_id"], int(row["refit_step"]))
        for row in missing_rows
    }
    completed_before = len(recovered_df)
    completed_new = 0

    ordered_jobs = (
        transformer_full_jobs_df
        .sort_values(["metric", "horizon"])
        .to_dict("records")
    )

    for job in ordered_jobs:
        job_schedule = (
            transformer_full_schedule_df.loc[
                transformer_full_schedule_df["job_id"].eq(job["job_id"])
            ]
            .sort_values("refit_step")
            .reset_index(drop=True)
        )

        job_missing = [
            row
            for row in job_schedule.to_dict("records")
            if (row["job_id"], int(row["refit_step"])) in missing_keys
        ]

        # Do not build the million-row reusable job cache when all ten monthly
        # fits for this Target × Horizon job are already durable.
        if not job_missing:
            continue

        job_cache = prepare_full_transformer_job_cache(job)

        for refit_row in job_missing:
            split = resolve_full_transformer_split(
                job_cache,
                job,
                refit_row,
            )

            validate_full_transformer_regime(
                job_cache,
                job,
                split,
            )

            result = execute_one_full_transformer_fit(
                job,
                refit_row,
                job_cache,
                split,
            )

            completed_new += 1

            progress_message(
                completed_before + completed_new,
                len(transformer_full_schedule_df),
                (
                    f"{TARGET_LABELS[result['metric']]} · "
                    f"{TRACK_LABELS[result['track']]} · "
                    f"h={int(result['horizon'])} · "
                    f"{result['validation_month']} · "
                    f"epoch={int(result['selected_epoch'])} · "
                    f"MAE={float(result['raw_mae']):.3f}"
                ),
            )

        del job_cache
        clear_device_memory()


# Re-read durable checkpoints rather than trusting this kernel's in-memory
# results. The experiment counts as complete only when the exact frozen roster
# can be recovered from disk.
full_transformer_results_df, missing_rows = recover_full_transformer_roster()

if missing_rows:
    raise RuntimeError(
        f"Production validation still has {len(missing_rows):,} unfinished fits."
    )

full_transformer_results_df = (
    full_transformer_results_df
    .sort_values(["metric", "horizon", "refit_step"])
    .reset_index(drop=True)
)

validate_exact_success_roster(
    full_transformer_results_df,
    transformer_full_schedule_df,
    TRANSFORMER_LOGICAL_KEY_COLUMNS,
)

atomic_write_parquet(
    full_transformer_results_df,
    TRANSFORMER_FULL_RESULTS_PATH,
)

refits_per_job = full_transformer_results_df.groupby("job_id").size()

production_completion_df = pd.DataFrame([{
    "Completed fits": len(full_transformer_results_df),
    "Target × Horizon jobs": full_transformer_results_df["job_id"].nunique(),
    "Minimum refits per job": int(refits_per_job.min()),
    "Maximum refits per job": int(refits_per_job.max()),
    "Production rows capped": bool(
        full_transformer_results_df["production_uncapped"].eq(False).any()
    ),
}])

display(production_completion_df)

Production restart state | 150/150 complete | 0 unfinished


,Completed fits,Target × Horizon jobs,Minimum refits per job,Maximum refits per job,Production rows capped
0,150,15,10,10,False


**Findings.** The full production roster is complete: **150 successful monthly fits across 15 Target × Horizon jobs**, with exactly ten refits per job. Every fit uses uncapped expanding history under the frozen target-specific track, representation, capacity profile, and monthly cadence.

The checkpoint contract is also working as intended: once the complete roster is durable, rerunning this section recovers the existing production evidence and performs no model fitting.

### 4.5.2.6.4 Evaluate the complete production validation

The full rolling-validation crawl gives us two different kinds of evidence.

First, the row-level forecasts tell us whether the finalized Transformer improves on **Previous week · same Daypart** across targets and forecast horizons. Second, the saved selector curves and fit metadata show whether the adaptive training policy behaved sensibly across 150 independent monthly refits.

The evaluation below keeps those questions separate: **forecast accuracy first, training behavior second.**

**4.5.2.6.4A — Score the complete row-level validation surface**

Each monthly model owns one consecutive scoring segment, so production accuracy must be calculated after recombining the **row-level forecasts across all ten refits**. Simply averaging ten monthly MAEs would give every month equal weight even though their scored row counts differ.

We preserve three complementary views of accuracy:

- **Native-unit MAE** tells us the typical size of an error in the target's own units, such as trips, riders, or speed.
- **Relative MAE (%)** expresses total absolute forecast error as a percentage of total observed activity. This puts targets with very different numerical scales onto a comparable basis. Lower values are better.
- **Benchmark skill (%)** asks how much MAE the Transformer removes relative to **Previous week · same Daypart**. Positive values favor the Transformer.

Headline performance uses `prediction_final`, while `prediction_raw` remains available to show whether the nonnegative output rule materially changes the result.

In [66]:
# ---------------------------------------------------------------------
# 4.5.2.6.4A — Score the complete row-level validation surface
# ---------------------------------------------------------------------

prediction_paths = (
    full_transformer_results_df["prediction_path"]
    .drop_duplicates()
    .map(Path)
    .tolist()
)

missing_prediction_paths = [
    path
    for path in prediction_paths
    if not path.exists()
]

if missing_prediction_paths:
    raise FileNotFoundError(
        "Production prediction artifacts are missing: "
        f"{missing_prediction_paths[:5]}"
    )


duck = duckdb.connect()

duck.read_parquet(
    [str(path) for path in prediction_paths]
).create_view("transformer_production_predictions")


# ---------------------------------------------------------------------
# Exact-row QA
# ---------------------------------------------------------------------

prediction_qa_df = duck.execute("""
    SELECT
        COUNT(*) AS evaluation_rows,
        MIN(target_date) AS minimum_target_date,
        MAX(target_date) AS maximum_target_date,
        SUM(
            CASE
                WHEN target_observed_value IS NULL
                  OR prediction_raw IS NULL
                  OR prediction_final IS NULL
                  OR benchmark_prediction IS NULL
                THEN 1 ELSE 0
            END
        ) AS incomplete_rows
    FROM transformer_production_predictions
""").df()

duplicate_rows_df = duck.execute("""
    SELECT
        track,
        metric,
        horizon,
        forecast_sequence_id,
        taxi_zone_id,
        origin_observation_sequence_id,
        target_observation_sequence_id,
        COUNT(*) AS row_count
    FROM transformer_production_predictions
    GROUP BY
        track,
        metric,
        horizon,
        forecast_sequence_id,
        taxi_zone_id,
        origin_observation_sequence_id,
        target_observation_sequence_id
    HAVING COUNT(*) > 1
    LIMIT 1
""").df()

evaluation_rows = int(prediction_qa_df.loc[0, "evaluation_rows"])

if evaluation_rows == 0:
    raise ValueError("Production validation contains no scored forecast rows.")

if int(prediction_qa_df.loc[0, "incomplete_rows"]) != 0:
    raise ValueError("Production scoring contains incomplete comparison rows.")

if not duplicate_rows_df.empty:
    raise ValueError("Production scoring contains duplicate physical forecast rows.")

maximum_target_date = pd.Timestamp(
    prediction_qa_df.loc[0, "maximum_target_date"]
).normalize()

if maximum_target_date >= FINAL_HOLDOUT_START_DATE:
    raise ValueError("Production scoring reaches the untouched final holdout.")


# ---------------------------------------------------------------------
# Recombine monthly pieces before computing headline job metrics
# ---------------------------------------------------------------------

transformer_job_scorecard_df = duck.execute("""
    SELECT
        track,
        metric,
        horizon,
        COUNT(*) AS evaluation_rows,

        AVG(
            ABS(prediction_raw - target_observed_value)
        ) AS raw_mae,

        AVG(
            ABS(prediction_final - target_observed_value)
        ) AS final_mae,

        AVG(
            ABS(benchmark_prediction - target_observed_value)
        ) AS benchmark_mae,

        SQRT(
            AVG(POWER(prediction_final - target_observed_value, 2))
        ) AS final_rmse,

        AVG(
            prediction_final - target_observed_value
        ) AS final_bias,

        -- Relative MAE uses totals rather than averaging row-level
        -- percentages. This keeps large and small observations weighted
        -- according to the actual amount of mobility they represent.
        SUM(
            ABS(target_observed_value)
        ) AS observed_activity_total,

        SUM(
            ABS(prediction_final - target_observed_value)
        ) AS final_absolute_error_total,

        SUM(
            ABS(benchmark_prediction - target_observed_value)
        ) AS benchmark_absolute_error_total,

        SUM(
            CASE WHEN prediction_raw < 0 THEN 1 ELSE 0 END
        ) AS negative_raw_predictions

    FROM transformer_production_predictions
    GROUP BY track, metric, horizon
    ORDER BY metric, horizon
""").df()

if len(transformer_job_scorecard_df) != len(transformer_full_jobs_df):
    raise ValueError(
        "Production scorecard does not contain exactly one row per final job."
    )

if transformer_job_scorecard_df["benchmark_mae"].le(0).any():
    raise ValueError(
        "Benchmark-relative metrics require strictly positive benchmark MAE."
    )

if transformer_job_scorecard_df["observed_activity_total"].le(0).any():
    raise ValueError(
        "Relative MAE requires strictly positive observed activity."
    )

# Relative MAE answers a different question from benchmark skill:
# how large is total forecast error relative to the amount actually observed?
transformer_job_scorecard_df["relative_mae_pct"] = (
    100.0
    * transformer_job_scorecard_df["final_absolute_error_total"]
    / transformer_job_scorecard_df["observed_activity_total"]
)

transformer_job_scorecard_df["benchmark_relative_activity_mae_pct"] = (
    100.0
    * transformer_job_scorecard_df["benchmark_absolute_error_total"]
    / transformer_job_scorecard_df["observed_activity_total"]
)

# Positive values mean the Transformer removes this many Relative-MAE
# percentage points compared with the frozen weekly benchmark.
transformer_job_scorecard_df["relative_mae_advantage_pp"] = (
    transformer_job_scorecard_df["benchmark_relative_activity_mae_pct"]
    - transformer_job_scorecard_df["relative_mae_pct"]
)

transformer_job_scorecard_df["benchmark_relative_mae"] = (
    transformer_job_scorecard_df["final_mae"]
    / transformer_job_scorecard_df["benchmark_mae"]
)

transformer_job_scorecard_df["benchmark_skill_pct"] = (
    100.0
    * (
        transformer_job_scorecard_df["benchmark_mae"]
        - transformer_job_scorecard_df["final_mae"]
    )
    / transformer_job_scorecard_df["benchmark_mae"]
)

transformer_job_scorecard_df["floor_mae_change_pct"] = np.where(
    transformer_job_scorecard_df["raw_mae"].gt(0),
    100.0
    * (
        transformer_job_scorecard_df["final_mae"]
        - transformer_job_scorecard_df["raw_mae"]
    )
    / transformer_job_scorecard_df["raw_mae"],
    np.nan,
)


# Preserve month-level normalized evidence separately. This lets later
# diagnostics ask whether good pooled performance was stable through time.
transformer_monthly_scorecard_df = duck.execute("""
    SELECT
        track,
        metric,
        horizon,
        temporal_window,
        COUNT(*) AS evaluation_rows,

        AVG(
            ABS(prediction_final - target_observed_value)
        ) AS final_mae,

        AVG(
            ABS(benchmark_prediction - target_observed_value)
        ) AS benchmark_mae

    FROM transformer_production_predictions
    GROUP BY track, metric, horizon, temporal_window
    ORDER BY metric, horizon, temporal_window
""").df()

if transformer_monthly_scorecard_df["benchmark_mae"].le(0).any():
    raise ValueError(
        "Monthly benchmark-relative metrics require positive benchmark MAE."
    )

transformer_monthly_scorecard_df["benchmark_relative_mae"] = (
    transformer_monthly_scorecard_df["final_mae"]
    / transformer_monthly_scorecard_df["benchmark_mae"]
)

transformer_monthly_scorecard_df["benchmark_skill_pct"] = (
    100.0
    * (
        1.0
        - transformer_monthly_scorecard_df["benchmark_relative_mae"]
    )
)


TRANSFORMER_FULL_JOB_SCORECARD_PATH = (
    INTERMEDIATE_DIR / "transformer_full_job_scorecard.parquet"
)

TRANSFORMER_FULL_MONTHLY_SCORECARD_PATH = (
    INTERMEDIATE_DIR / "transformer_full_monthly_scorecard.parquet"
)

atomic_write_parquet(
    transformer_job_scorecard_df,
    TRANSFORMER_FULL_JOB_SCORECARD_PATH,
)

atomic_write_parquet(
    transformer_monthly_scorecard_df,
    TRANSFORMER_FULL_MONTHLY_SCORECARD_PATH,
)


scorecard_display_df = (
    transformer_job_scorecard_df
    .assign(
        Target=lambda frame: frame["metric"].map(TARGET_LABELS),
        Track=lambda frame: frame["track"].map(TRACK_LABELS),
    )
    .rename(columns={
        "horizon": "Horizon",
        "evaluation_rows": "Rows",
        "raw_mae": "Raw MAE",
        "final_mae": "Final MAE",
        "benchmark_mae": "Benchmark MAE",
        "benchmark_relative_mae": "Benchmark-relative MAE",
        "benchmark_skill_pct": "Benchmark skill %",
        "negative_raw_predictions": "Negative raw",
        "floor_mae_change_pct": "Floor MAE change %",
    })
)

display(
    scorecard_display_df[
        [
            "Target",
            "Horizon",
            "Track",
            "Rows",
            "Raw MAE",
            "Final MAE",
            "Benchmark MAE",
            "Benchmark-relative MAE",
            "Benchmark skill %",
            "Negative raw",
            "Floor MAE change %",
        ]
    ].round(3)
)

minimum_target_date = pd.Timestamp(
    prediction_qa_df.loc[0, "minimum_target_date"]
).date()

print(
    "Validation support | "
    f"{evaluation_rows:,} forecast rows | "
    f"{minimum_target_date} → {maximum_target_date.date()} | "
    "final holdout rows=0"
)

duck.close()

,Target,Horizon,Track,Rows,Raw MAE,Final MAE,Benchmark MAE,Benchmark-relative MAE,Benchmark skill %,Negative raw,Floor MAE change %
0,FHVHV average speed,1,Full-History Regime-Aware,352319,0.921,0.921,1.329,0.693,30.726,0.000,-0.000
1,FHVHV average speed,2,Full-History Regime-Aware,352319,0.973,0.973,1.329,0.732,26.773,0.000,-0.000
2,FHVHV average speed,5,Full-History Regime-Aware,352319,0.990,0.990,1.329,0.745,25.492,0.000,-0.000
3,FHVHV trips,1,Post-CP Primary,352715,58.599,58.437,69.633,0.839,16.080,"4,712.000",-0.276
4,FHVHV trips,2,Post-CP Primary,352715,62.165,62.050,69.633,0.891,10.891,"3,600.000",-0.186
5,FHVHV trips,5,Post-CP Primary,352715,63.114,62.633,69.633,0.899,10.054,"8,086.000",-0.763
6,Subway ridership,1,Full-History Regime-Aware,208859,448.634,447.209,531.790,0.841,15.905,"2,743.000",-0.318
7,Subway ridership,2,Full-History Regime-Aware,208859,479.791,475.990,531.790,0.895,10.493,"5,864.000",-0.792
8,Subway ridership,5,Full-History Regime-Aware,208859,479.341,477.380,531.790,0.898,10.231,"3,507.000",-0.409
9,Taxi average speed,1,Post-CP Primary,250689,2.168,2.168,2.951,0.735,26.529,0.000,0.000


Validation support | 4,551,891 forecast rows | 2025-04-05 → 2026-01-04 | final holdout rows=0


**Findings.** The complete production surface contains **4,551,891 row-level forecasts** from April 5, 2025 through January 4, 2026, with no rows entering the untouched final holdout. All **15 Target × Horizon jobs beat Previous week · same Daypart** after the ten monthly refits are recombined on their full scored populations.

The margin over the benchmark ranges from **5.365% for Taxi trips at h=2** to **30.726% for FHVHV average speed at h=1**. Both speed targets are especially strong, while the demand targets still improve on the benchmark but generally by smaller margins at longer horizons.

Negative raw forecasts occur only for the three demand targets. Applying the frozen zero floor improves headline MAE rather than worsening it, but the effect is modest relative to the overall benchmark gains: the largest change is **3.559% for Taxi trips at h=1**, while the FHVHV-trip and Subway changes remain below 1%. The Transformer's pooled advantage is therefore not simply an artifact of clipping invalid negative predictions.

**4.5.2.6.4B — How large are Transformer forecast errors relative to observed activity?**

Native-unit MAE is easy to interpret within one target, but it cannot be compared directly across trips, riders, and speed. **Relative MAE (%)** puts those forecasts on a common scale:

**Relative MAE = 100 × total absolute forecast error ÷ total absolute observed activity**

A Relative MAE of 10%, for example, means that the model's accumulated absolute error is equal to about 10% of the amount actually observed over the scored rows.

**Lower values are better.** The heatmap therefore shows where the finalized Transformer is most and least accurate relative to the scale of the mobility measure being forecast. This is different from benchmark skill: Relative MAE measures the size of the Transformer's error itself, while benchmark skill asks whether that error is better or worse than the weekly baseline.

In [67]:
# ---------------------------------------------------------------------
# 4.5.2.6.4B — Visualize Relative MAE across the final 15 jobs
# ---------------------------------------------------------------------

relative_mae_matrix_df = (
    transformer_job_scorecard_df
    .pivot(
        index="metric",
        columns="horizon",
        values="relative_mae_pct",
    )
    .reindex(
        index=FORECAST_TARGETS,
        columns=FORECAST_HORIZONS,
    )
)

if relative_mae_matrix_df.isna().any().any():
    raise ValueError(
        "Relative-MAE heatmap is missing at least one Target × Horizon job."
    )

relative_mae_values = relative_mae_matrix_df.to_numpy(dtype=float)
relative_mae_text = np.vectorize(
    lambda value: f"{value:.1f}%"
)(relative_mae_values)

fig = go.Figure(
    data=go.Heatmap(
        z=relative_mae_values,
        x=[f"h={horizon}" for horizon in FORECAST_HORIZONS],
        y=[TARGET_LABELS[metric] for metric in FORECAST_TARGETS],
        text=relative_mae_text,
        texttemplate="%{text}",
        colorscale=BRAND_SEQUENTIAL_HEATMAP_SCALE,
        zmin=0,
        colorbar={"title": "Relative MAE (%)"},
        hovertemplate=(
            "Target: %{y}<br>"
            "Horizon: %{x}<br>"
            "Relative MAE: %{z:.3f}%"
            "<extra></extra>"
        ),
    )
)

fig = apply_branding(fig)

fig.update_layout(
    title="Transformer Forecast Error Relative to Observed Activity",
    xaxis_title="Forecast horizon",
    yaxis_title="",
)

fig.show()

**Findings.** Once forecast error is expressed relative to the amount of mobility actually observed, **FHVHV average speed is the Transformer's most accurate target by a wide margin**, with Relative MAE staying at roughly **5.5%–5.9%** across all three horizons. Subway ridership is next, rising only modestly from **9.4% at h=1** to **10.1% at h=5**.

FHVHV trips also remain fairly stable as the forecast moves farther ahead, increasing from **11.3% to 12.1%**, while Taxi average speed changes only slightly from **15.7% to 16.1%**. These patterns suggest that most targets lose accuracy gradually rather than collapsing as horizon increases.

**Taxi trips are the clear exception and the hardest Transformer target on this scale.** Relative MAE is **15.7% at h=1**, jumps to **18.3% at h=2**, then improves somewhat to **17.1% at h=5**. That non-monotonic pattern is a useful reminder that a farther forecast is not automatically harder for every Target × Horizon job.

This view answers a different question from benchmark skill. A target can have a relatively large normalized forecasting error and still improve substantially on the weekly benchmark if that benchmark also struggles with the target. Relative MAE therefore tells us **how large the Transformer's error is**, while benchmark skill tells us **how much better that error is than the baseline**.

**4.5.2.6.4C — Compare benchmark-relative accuracy across targets and horizons**

A Taxi-trip MAE and a Subway-ridership MAE cannot be compared directly because they measure different quantities on very different scales. Benchmark skill solves that problem by asking the same question for every job:

**How much error did the Transformer remove—or add—relative to Previous week · same Daypart?**

Positive values favor the Transformer; negative values favor the benchmark. The heatmap therefore shows where the finalized model adds predictive value without collapsing incompatible target units into one artificial average.

In [68]:
# ---------------------------------------------------------------------
# 4.5.2.6.4C — Visualize benchmark-relative production accuracy
# ---------------------------------------------------------------------

skill_matrix_df = (
    transformer_job_scorecard_df
    .pivot(
        index="metric",
        columns="horizon",
        values="benchmark_skill_pct",
    )
    .reindex(
        index=FORECAST_TARGETS,
        columns=FORECAST_HORIZONS,
    )
)

if skill_matrix_df.isna().any().any():
    raise ValueError(
        "Benchmark-skill heatmap is missing at least one Target × Horizon job."
    )

skill_values = skill_matrix_df.to_numpy(dtype=float)
skill_bound = max(
    float(np.nanmax(np.abs(skill_values))),
    1e-9,
)

skill_text = np.vectorize(
    lambda value: f"{value:+.1f}%"
)(skill_values)

fig = go.Figure(
    data=go.Heatmap(
        z=skill_values,
        x=[f"h={horizon}" for horizon in FORECAST_HORIZONS],
        y=[TARGET_LABELS[metric] for metric in FORECAST_TARGETS],
        text=skill_text,
        texttemplate="%{text}",
        colorscale=list(reversed(BRAND_DIVERGING_SEQUENCE)),
        zmid=0,
        zmin=-skill_bound,
        zmax=skill_bound,
        colorbar={"title": "Skill %"},
        hovertemplate=(
            "Target: %{y}<br>"
            "Horizon: %{x}<br>"
            "Benchmark skill: %{z:.3f}%"
            "<extra></extra>"
        ),
    )
)

fig = apply_branding(fig)

fig.update_layout(
    title="Transformer Production Accuracy Relative to the Weekly Benchmark",
    xaxis_title="Forecast horizon",
    yaxis_title="",
)

fig.show()

**Findings.** Benchmark-relative accuracy is positive across the entire **5 Target × 3 Horizon** surface: none of the 15 final Transformer jobs loses to the weekly benchmark after pooling its complete rolling-validation rows.

The strongest results occur for the two speed targets. FHVHV average speed improves on the benchmark by **30.726%, 26.773%, and 25.492%** at h=1, h=2, and h=5, while Taxi average speed delivers a similarly consistent **26.529%, 25.585%, and 24.520%**.

The demand targets also remain positive, but their advantage generally narrows with forecast distance. FHVHV trips falls from **16.080% at h=1** to about 10% at h=2 and h=5; Subway follows a similar pattern. Taxi trips is the least uniform, with **19.091% at h=1**, **5.365% at h=2**, and **11.839% at h=5**. Across all five targets, h=1 is the strongest horizon, although the longer-horizon decline is much smaller for speed than for demand.

**4.5.2.6.4D — Audit adaptive training duration across production refits**

The monthly production models do not all receive the same training duration. Each fit independently uses its historical inner-validation tail to choose an epoch count, then trains a fresh final model for exactly that duration.

The completed production surface lets us check whether this policy behaved as intended. We want to see genuine variation when different targets and months need different amounts of optimization, while also checking whether selectors repeatedly run into the 30-epoch search ceiling. Repeated ceiling pressure would suggest that the search bound—not the validation evidence—was determining training duration.

This is a training diagnostic, not another model-selection experiment.

In [69]:
# ---------------------------------------------------------------------
# 4.5.2.6.4D — Audit adaptive production training duration
# ---------------------------------------------------------------------

curve_frames = []

for curve_path in full_transformer_results_df["curve_path"].map(Path):
    if not curve_path.exists():
        raise FileNotFoundError(f"Missing selector curve: {curve_path}")

    curve_frames.append(
        pd.read_parquet(
            curve_path,
            columns=[
                "metric",
                "horizon",
                "job_id",
                "refit_step",
                "epoch",
                "selected_epoch",
            ],
        )
    )

transformer_selector_curves_df = pd.concat(
    curve_frames,
    ignore_index=True,
)

transformer_selector_curves_df["selected_epoch"] = (
    transformer_selector_curves_df["selected_epoch"]
    .fillna(False)
    .astype(bool)
)

transformer_selector_curves_df["selected_epoch_value"] = np.where(
    transformer_selector_curves_df["selected_epoch"],
    transformer_selector_curves_df["epoch"],
    np.nan,
)

selector_run_summary_df = (
    transformer_selector_curves_df
    .groupby(
        ["metric", "horizon", "job_id", "refit_step"],
        as_index=False,
    )
    .agg(
        selector_epochs_run=("epoch", "max"),
        selected_marker_count=("selected_epoch", "sum"),
        curve_selected_epoch=("selected_epoch_value", "max"),
    )
)

if not selector_run_summary_df["selected_marker_count"].eq(1).all():
    raise ValueError(
        "Every production selector curve must contain exactly one selected-epoch marker."
    )

training_fit_audit_df = (
    full_transformer_results_df[
        [
            "metric",
            "horizon",
            "job_id",
            "refit_step",
            "selected_epoch",
            "runtime_seconds",
        ]
    ]
    .merge(
        selector_run_summary_df,
        on=["metric", "horizon", "job_id", "refit_step"],
        how="left",
        validate="one_to_one",
    )
)

if training_fit_audit_df["selector_epochs_run"].isna().any():
    raise ValueError("At least one production fit is missing its selector curve.")

if not np.array_equal(
    training_fit_audit_df["selected_epoch"].to_numpy(dtype=int),
    training_fit_audit_df["curve_selected_epoch"].to_numpy(dtype=int),
):
    raise ValueError(
        "A saved selector curve disagrees with its fit checkpoint's selected epoch."
    )

training_fit_audit_df["selector_hit_ceiling"] = (
    training_fit_audit_df["selector_epochs_run"]
    .eq(TRANSFORMER_MAX_EPOCHS)
)

transformer_training_duration_df = (
    training_fit_audit_df
    .groupby(["metric", "horizon"], as_index=False)
    .agg(
        refits=("selected_epoch", "size"),
        minimum_selected_epoch=("selected_epoch", "min"),
        median_selected_epoch=("selected_epoch", "median"),
        mean_selected_epoch=("selected_epoch", "mean"),
        maximum_selected_epoch=("selected_epoch", "max"),
        median_selector_epochs_run=("selector_epochs_run", "median"),
        ceiling_hits=("selector_hit_ceiling", "sum"),
        median_runtime_minutes=(
            "runtime_seconds",
            lambda values: np.median(values) / 60.0,
        ),
    )
)

duration_display_df = (
    transformer_training_duration_df
    .assign(
        Target=lambda frame: frame["metric"].map(TARGET_LABELS)
    )
    .rename(columns={
        "horizon": "Horizon",
        "refits": "Refits",
        "minimum_selected_epoch": "Min selected",
        "median_selected_epoch": "Median selected",
        "mean_selected_epoch": "Mean selected",
        "maximum_selected_epoch": "Max selected",
        "median_selector_epochs_run": "Median selector run",
        "ceiling_hits": "30-epoch ceiling hits",
        "median_runtime_minutes": "Median runtime min",
    })
)

display(
    duration_display_df[
        [
            "Target",
            "Horizon",
            "Refits",
            "Min selected",
            "Median selected",
            "Mean selected",
            "Max selected",
            "Median selector run",
            "30-epoch ceiling hits",
            "Median runtime min",
        ]
    ].round(3)
)


# Median selected epoch is the clearest cross-job summary of how long the
# adaptive rule typically decided each final model should train.
median_epoch_matrix_df = (
    transformer_training_duration_df
    .pivot(
        index="metric",
        columns="horizon",
        values="median_selected_epoch",
    )
    .reindex(
        index=FORECAST_TARGETS,
        columns=FORECAST_HORIZONS,
    )
)

if median_epoch_matrix_df.isna().any().any():
    raise ValueError(
        "Training-duration heatmap is missing at least one Target × Horizon job."
    )

median_epoch_values = median_epoch_matrix_df.to_numpy(dtype=float)
median_epoch_text = np.vectorize(
    lambda value: f"{value:.1f}"
)(median_epoch_values)

fig = go.Figure(
    data=go.Heatmap(
        z=median_epoch_values,
        x=[f"h={horizon}" for horizon in FORECAST_HORIZONS],
        y=[TARGET_LABELS[metric] for metric in FORECAST_TARGETS],
        colorscale=BRAND_SEQUENTIAL_HEATMAP_SCALE,
        text=median_epoch_text,
        texttemplate="%{text}",
        colorbar={"title": "Epoch"},
        hovertemplate=(
            "Target: %{y}<br>"
            "Horizon: %{x}<br>"
            "Median selected epoch: %{z:.3f}"
            "<extra></extra>"
        ),
    )
)

fig = apply_branding(fig)

fig.update_layout(
    title="Adaptive Training Duration Across Production Refits",
    xaxis_title="Forecast horizon",
    yaxis_title="",
)

fig.show()

print(
    "Adaptive-duration QA | "
    f"selected epoch range="
    f"{int(training_fit_audit_df['selected_epoch'].min())}–"
    f"{int(training_fit_audit_df['selected_epoch'].max())} | "
    f"selector ceiling hits="
    f"{int(training_fit_audit_df['selector_hit_ceiling'].sum())}/"
    f"{len(training_fit_audit_df)}"
)

,Target,Horizon,Refits,Min selected,Median selected,Mean selected,Max selected,Median selector run,30-epoch ceiling hits,Median runtime min
0,FHVHV average speed,1,10,5,8.500,9.000,15,13.500,0,20.111
1,FHVHV average speed,2,10,4,9.500,10.800,24,14.500,0,22.397
2,FHVHV average speed,5,10,2,9.500,8.900,15,14.500,0,23.224
3,FHVHV trips,1,10,7,16.500,16.000,23,21.500,0,3.926
4,FHVHV trips,2,10,6,7.500,10.100,19,12.500,0,3.790
5,FHVHV trips,5,10,6,12.500,11.900,21,17.500,0,4.550
6,Subway ridership,1,10,5,20.500,17.300,25,25.500,1,18.290
7,Subway ridership,2,10,4,11.000,11.000,19,16.000,0,10.838
8,Subway ridership,5,10,4,12.000,13.300,27,17.000,1,10.599
9,Taxi average speed,1,10,4,10.500,10.500,17,15.500,0,2.527


Adaptive-duration QA | selected epoch range=2–27 | selector ceiling hits=2/150


**Findings.** Adaptive training duration is meaningfully active rather than collapsing to one fixed epoch count. Across the 150 monthly refits, selected training duration ranges from **2 to 27 epochs**, while Target × Horizon median selections range from **6.5 epochs for Taxi average speed at h=5** to **20.5 for Subway ridership at h=1**.

Only **2 of 150 selectors** run all the way to the 30-epoch search ceiling, both within Subway jobs. This provides little evidence that the training envelope is generally cutting off models that still need substantially more training.

Runtime also reflects the amount of legal history available. The Full-History jobs are materially heavier, with median fit times ranging from roughly **10.6 to 23.2 minutes**, while the Post-CP jobs are generally around **1.4 to 5.1 minutes**. The variation in selected epochs and runtime reinforces why adaptive duration is preferable to forcing every Target × Horizon × Month fit through the same training budget.

### Section 6 Summary

Section 6 applies the frozen Transformer design to the complete ordinary rolling-validation surface: **15 Target × Horizon jobs × 10 monthly expanding-history refits = 150 successful fits** and **4,551,891 scored forecasts**. All 15 pooled jobs outperform Previous week · same Daypart, with benchmark skill ranging from **5.365% to 30.726%**. Relative MAE ranges from roughly **5.5% to 18.3%**, showing that FHVHV average speed is the most accurate target relative to its observed scale while Taxi trips remains the most difficult. Speed forecasting is also the most horizon-stable; demand gains are smaller and more variable at longer horizons. Adaptive training selects between **2 and 27 epochs**, with only **2 of 150 selectors** reaching the 30-epoch search limit. Negative raw forecasts occur only for demand targets and clipping changes MAE by at most **3.559%**, so the benchmark gains are not primarily driven by the output floor. The complete validation ends on **January 4, 2026**, leaving the final holdout untouched.

## 4.5.2.7 — Diagnose and Interpret the Final Transformer

Production validation tells us whether the finalized Transformer forecasts work. This chapter asks three follow-up questions that make that result easier to trust and interpret:

1. **How did training behave?** Did adaptive epoch selection stop at sensible points, or repeatedly press against the search ceiling?
2. **How stable was forecast performance?** Did pooled accuracy reflect broad month-to-month strength, or depend on a few unusually favorable periods?
3. **What information did the fitted models rely on?** Which broad input sources mattered most when the trained model made its forecasts?

These diagnostics reuse the completed production artifacts. The models are not redesigned, and the final holdout remains untouched.

### 4.5.2.7.1 — Training and Validation Behavior

The production models choose their own training duration from a historical inner-validation tail. Section 6 confirmed the selected epoch counts; here we look more closely at the **shape of the validation curves around those choices**.

A useful selector should usually improve before the chosen epoch and then flatten or deteriorate afterward. If validation error routinely keeps falling long after selection—or selectors repeatedly hit the 30-epoch ceiling—the stopping rule may be constraining the model rather than protecting it from over-training.

**4.5.2.7.1A — Compare validation behavior before and after the selected epoch**

Each saved selector curve contains training loss, historical validation loss, standardized validation MAE, and the epoch chosen for the fresh final refit.

To compare 150 fits with very different target scales, this analysis expresses each curve relative to **its own selected-epoch validation MAE**. Epoch `0` on the chart is therefore the chosen stopping point; negative positions occurred before selection and positive positions are the extra patience epochs observed afterward.

In [70]:
# ---------------------------------------------------------------------
# 4.5.2.7.1A — Compare validation behavior around the selected epoch
# ---------------------------------------------------------------------

training_curve_frames = []

for curve_path in full_transformer_results_df["curve_path"].map(Path):
    if not curve_path.exists():
        raise FileNotFoundError(f"Missing production selector curve: {curve_path}")

    training_curve_frames.append(
        pd.read_parquet(
            curve_path,
            columns=[
                "metric",
                "horizon",
                "job_id",
                "refit_step",
                "epoch",
                "train_huber",
                "inner_validation_huber",
                "inner_validation_standardized_mae",
                "selected_epoch",
            ],
        )
    )

transformer_training_curves_df = pd.concat(
    training_curve_frames,
    ignore_index=True,
)

transformer_training_curves_df["selected_epoch"] = (
    transformer_training_curves_df["selected_epoch"]
    .fillna(False)
    .astype(bool)
)

fit_keys = ["metric", "horizon", "job_id", "refit_step"]

selected_counts = (
    transformer_training_curves_df
    .groupby(fit_keys, observed=True)["selected_epoch"]
    .sum()
)

if not selected_counts.eq(1).all():
    raise ValueError(
        "Every production selector curve must contain exactly one selected epoch."
    )


# ---------------------------------------------------------------------
# Reduce each selector curve to three interpretable checkpoints
# ---------------------------------------------------------------------

ordered_curves_df = transformer_training_curves_df.sort_values(
    fit_keys + ["epoch"]
)

first_epoch_df = (
    ordered_curves_df
    .groupby(fit_keys, as_index=False, observed=True)
    .first()
    [fit_keys + ["epoch", "inner_validation_standardized_mae"]]
    .rename(columns={
        "epoch": "first_epoch",
        "inner_validation_standardized_mae": "first_validation_mae",
    })
)

selected_epoch_df = (
    ordered_curves_df.loc[ordered_curves_df["selected_epoch"]]
    [fit_keys + ["epoch", "inner_validation_standardized_mae"]]
    .rename(columns={
        "epoch": "selected_epoch_number",
        "inner_validation_standardized_mae": "selected_validation_mae",
    })
)

last_epoch_df = (
    ordered_curves_df
    .groupby(fit_keys, as_index=False, observed=True)
    .last()
    [fit_keys + ["epoch", "inner_validation_standardized_mae"]]
    .rename(columns={
        "epoch": "selector_epochs_run",
        "inner_validation_standardized_mae": "stopping_validation_mae",
    })
)

transformer_training_behavior_df = (
    first_epoch_df
    .merge(selected_epoch_df, on=fit_keys, validate="one_to_one")
    .merge(last_epoch_df, on=fit_keys, validate="one_to_one")
)

transformer_training_behavior_df["improvement_to_selected_pct"] = (
    100.0
    * (
        transformer_training_behavior_df["first_validation_mae"]
        - transformer_training_behavior_df["selected_validation_mae"]
    )
    / transformer_training_behavior_df["first_validation_mae"]
)

# Positive values mean validation was worse when patience finally stopped than
# it had been at the epoch chosen for the final model.
transformer_training_behavior_df["post_selection_change_pct"] = (
    100.0
    * (
        transformer_training_behavior_df["stopping_validation_mae"]
        - transformer_training_behavior_df["selected_validation_mae"]
    )
    / transformer_training_behavior_df["selected_validation_mae"]
)

transformer_training_behavior_df["epochs_after_selected"] = (
    transformer_training_behavior_df["selector_epochs_run"]
    - transformer_training_behavior_df["selected_epoch_number"]
)

transformer_training_behavior_df["selector_hit_ceiling"] = (
    transformer_training_behavior_df["selector_epochs_run"]
    .eq(TRANSFORMER_MAX_EPOCHS)
)


# ---------------------------------------------------------------------
# One compact Target × Horizon summary
# ---------------------------------------------------------------------

training_behavior_summary_df = (
    transformer_training_behavior_df
    .groupby(["metric", "horizon"], as_index=False, observed=True)
    .agg(
        refits=("refit_step", "size"),
        median_selected_epoch=("selected_epoch_number", "median"),
        median_improvement_pct=("improvement_to_selected_pct", "median"),
        median_post_selection_change_pct=("post_selection_change_pct", "median"),
        median_patience_epochs=("epochs_after_selected", "median"),
        ceiling_hits=("selector_hit_ceiling", "sum"),
    )
)

training_behavior_display_df = (
    training_behavior_summary_df
    .assign(Target=lambda frame: frame["metric"].map(TARGET_LABELS))
    .rename(columns={
        "horizon": "Horizon",
        "refits": "Refits",
        "median_selected_epoch": "Median selected epoch",
        "median_improvement_pct": "Median improvement to selection %",
        "median_post_selection_change_pct": "Median post-selection change %",
        "median_patience_epochs": "Median patience epochs",
        "ceiling_hits": "30-epoch ceiling hits",
    })
)

display(
    training_behavior_display_df[
        [
            "Target",
            "Horizon",
            "Refits",
            "Median selected epoch",
            "Median improvement to selection %",
            "Median post-selection change %",
            "Median patience epochs",
            "30-epoch ceiling hits",
        ]
    ].round(3)
)


# ---------------------------------------------------------------------
# Align curves around the chosen epoch for a scale-free visual diagnostic
# ---------------------------------------------------------------------

curve_relative_df = ordered_curves_df.merge(
    selected_epoch_df,
    on=fit_keys,
    how="left",
    validate="many_to_one",
)

curve_relative_df["relative_epoch"] = (
    curve_relative_df["epoch"]
    - curve_relative_df["selected_epoch_number"]
)

curve_relative_df["relative_validation_change_pct"] = (
    100.0
    * (
        curve_relative_df["inner_validation_standardized_mae"]
        - curve_relative_df["selected_validation_mae"]
    )
    / curve_relative_df["selected_validation_mae"]
)

# Five epochs on either side keep the chart focused on the stopping decision
# instead of letting long early training histories dominate the display.
curve_window_df = curve_relative_df.loc[
    curve_relative_df["relative_epoch"].between(-5, 5)
].copy()

aligned_curve_summary_df = (
    curve_window_df
    .groupby(["metric", "relative_epoch"], as_index=False, observed=True)
    .agg(
        median_relative_change_pct=("relative_validation_change_pct", "median"),
        supporting_fits=("job_id", "size"),
    )
)

fig = go.Figure()

# Five target trajectories need five visible categorical colors. The core
# sequence ends in "ice", which matches the branded plot background, so use
# the extended categorical palette for this multi-series diagnostic.
for metric_index, metric in enumerate(FORECAST_TARGETS):
    metric_df = aligned_curve_summary_df.loc[
        aligned_curve_summary_df["metric"].eq(metric)
    ]

    fig.add_trace(
        go.Scatter(
            x=metric_df["relative_epoch"],
            y=metric_df["median_relative_change_pct"],
            mode="lines+markers",
            name=TARGET_LABELS[metric],
            line={"color": BRAND_CLUSTER_PALETTE[metric_index]},
            marker={"color": BRAND_CLUSTER_PALETTE[metric_index]},
            hovertemplate=(
                "Target: %{fullData.name}<br>"
                "Epoch relative to selection: %{x}<br>"
                "Median validation change: %{y:.3f}%"
                "<extra></extra>"
            ),
        )
    )

fig = apply_branding(fig)
fig.add_vline(x=0, line_dash="dash")

fig.update_layout(
    title="Validation Behavior Around the Selected Training Epoch",
    xaxis_title="Epoch relative to selected epoch",
    yaxis_title="Validation MAE change vs selected epoch (%)",
    legend_title_text="Target",
)

fig.show()

,Target,Horizon,Refits,Median selected epoch,Median improvement to selection %,Median post-selection change %,Median patience epochs,30-epoch ceiling hits
0,FHVHV average speed,1,10,8.500,5.838,1.099,5.000,0
1,FHVHV average speed,2,10,9.500,5.898,1.195,5.000,0
2,FHVHV average speed,5,10,9.500,4.707,2.040,5.000,0
3,FHVHV trips,1,10,16.500,29.266,7.042,5.000,0
4,FHVHV trips,2,10,7.500,21.525,6.335,5.000,0
5,FHVHV trips,5,10,12.500,20.853,7.985,5.000,0
6,Subway ridership,1,10,20.500,36.253,6.305,5.000,1
7,Subway ridership,2,10,11.000,30.569,5.830,5.000,0
8,Subway ridership,5,10,12.000,27.467,4.160,5.000,1
9,Taxi average speed,1,10,10.500,4.871,0.950,5.000,0


**Findings.** The saved learning curves show that adaptive epoch selection is doing substantially more work for the demand targets than for speed. From the first epoch to the selected epoch, median validation MAE improves by roughly **21%–38%** for Taxi trips, FHVHV trips, and Subway ridership, compared with only about **3%–6%** for the two average-speed targets.

The five-epoch patience window also behaves as intended. Validation error typically flattens or worsens after the selected epoch rather than continuing to improve: median post-selection deterioration is only about **1%–2% for speed**, but rises to roughly **4%–8% for FHVHV trips and Subway** and exceeds **12% for Taxi trips at h=1 and h=5**. These are precisely the cases where continuing to train after the best historical validation point would be most costly.

Only **2 of 150 selectors** reach the 30-epoch ceiling, so there is little evidence that the search limit is generally constraining training. The wide range of selected durations instead suggests that the adaptive policy is responding to genuine differences across targets, horizons, and monthly refits; a larger epoch count should not be interpreted as a better model by itself.

### 4.5.2.7.2 — Forecast Performance and Stability

Section 6 established the pooled production scorecard. That tells us whether each Target × Horizon job works **overall**, but pooled accuracy can hide uneven performance through time.

This section therefore asks whether the Transformer's benchmark advantage is **stable across the ten monthly validation segments**. A model that wins consistently is stronger evidence than one whose pooled improvement depends on only one or two unusually favorable months.

**4.5.2.7.2A — Track benchmark-relative performance through time**

Benchmark skill remains the common scale: positive values mean the Transformer improves on **Previous week · same Daypart**, while negative values favor the benchmark.

The table summarizes stability for each final job. The heatmap then preserves all ten monthly results so we can see whether weak or strong performance is isolated, persistent, or concentrated around particular validation periods.

In [71]:
# ---------------------------------------------------------------------
# 4.5.2.7.2A — Track benchmark-relative performance through time
# ---------------------------------------------------------------------

monthly_stability_df = transformer_monthly_scorecard_df.copy()

expected_monthly_rows = (
    len(transformer_full_jobs_df)
    * full_transformer_results_df["refit_step"].nunique()
)

if len(monthly_stability_df) != expected_monthly_rows:
    raise ValueError(
        "Monthly stability evidence does not contain the complete "
        f"production surface: expected {expected_monthly_rows:,} rows, "
        f"found {len(monthly_stability_df):,}."
    )

monthly_stability_df["beat_benchmark"] = (
    monthly_stability_df["benchmark_skill_pct"].gt(0)
)

stability_summary_df = (
    monthly_stability_df
    .groupby(["metric", "horizon"], as_index=False, observed=True)
    .agg(
        validation_months=("temporal_window", "nunique"),
        months_beating_benchmark=("beat_benchmark", "sum"),
        median_skill_pct=("benchmark_skill_pct", "median"),
        worst_month_skill_pct=("benchmark_skill_pct", "min"),
        best_month_skill_pct=("benchmark_skill_pct", "max"),
    )
)

stability_display_df = (
    stability_summary_df
    .assign(Target=lambda frame: frame["metric"].map(TARGET_LABELS))
    .rename(columns={
        "horizon": "Horizon",
        "validation_months": "Months",
        "months_beating_benchmark": "Months beating benchmark",
        "median_skill_pct": "Median skill %",
        "worst_month_skill_pct": "Worst month %",
        "best_month_skill_pct": "Best month %",
    })
)

display(
    stability_display_df[
        [
            "Target",
            "Horizon",
            "Months",
            "Months beating benchmark",
            "Median skill %",
            "Worst month %",
            "Best month %",
        ]
    ].round(3)
)


# ---------------------------------------------------------------------
# Show every monthly comparison without emitting a 150-row table
# ---------------------------------------------------------------------

monthly_stability_df["job_label"] = (
    monthly_stability_df["metric"].map(TARGET_LABELS)
    + " · h="
    + monthly_stability_df["horizon"].astype(int).astype(str)
)

job_order = [
    f"{TARGET_LABELS[metric]} · h={horizon}"
    for metric in FORECAST_TARGETS
    for horizon in FORECAST_HORIZONS
]

month_order = sorted(
    monthly_stability_df["temporal_window"].astype(str).unique()
)

monthly_skill_matrix_df = (
    monthly_stability_df
    .pivot(
        index="job_label",
        columns="temporal_window",
        values="benchmark_skill_pct",
    )
    .reindex(index=job_order, columns=month_order)
)

if monthly_skill_matrix_df.isna().any().any():
    raise ValueError(
        "Monthly benchmark-skill heatmap is missing part of the 15 × 10 surface."
    )

monthly_skill_values = monthly_skill_matrix_df.to_numpy(dtype=float)

monthly_skill_bound = max(
    float(np.nanmax(np.abs(monthly_skill_values))),
    1e-9,
)

fig = go.Figure(
    data=go.Heatmap(
        z=monthly_skill_values,
        x=month_order,
        y=job_order,
        colorscale=list(reversed(BRAND_DIVERGING_SEQUENCE)),
        zmid=0,
        zmin=-monthly_skill_bound,
        zmax=monthly_skill_bound,
        colorbar={"title": "Skill %"},
        hovertemplate=(
            "Job: %{y}<br>"
            "Validation month: %{x}<br>"
            "Benchmark skill: %{z:.3f}%"
            "<extra></extra>"
        ),
    )
)

fig = apply_branding(fig)

fig.update_layout(
    title="Monthly Transformer Skill Relative to the Weekly Benchmark",
    xaxis_title="Validation month",
    yaxis_title="",
)

fig.show()

,Target,Horizon,Months,Months beating benchmark,Median skill %,Worst month %,Best month %
0,FHVHV average speed,1,10,10,29.785,22.086,42.968
1,FHVHV average speed,2,10,10,24.650,17.753,37.923
2,FHVHV average speed,5,10,10,25.550,15.977,36.814
3,FHVHV trips,1,10,7,14.946,-15.443,41.912
4,FHVHV trips,2,10,8,7.815,-32.090,31.666
5,FHVHV trips,5,10,6,7.457,-41.351,28.874
6,Subway ridership,1,10,7,11.981,-23.840,54.495
7,Subway ridership,2,10,6,15.031,-26.028,44.000
8,Subway ridership,5,10,7,9.768,-14.039,33.281
9,Taxi average speed,1,10,10,25.760,18.461,30.710


**Findings.** The monthly view shows an important distinction hidden by the uniformly positive pooled scorecard. Both speed targets are exceptionally stable: all **six speed Target × Horizon jobs beat the benchmark in all 10 validation months**, with median monthly skill around **25%–30%**.

Demand forecasting is more variable. Taxi trips at h=1 still wins in **9 of 10 months**, but the remaining Taxi-trip, FHVHV-trip, and Subway jobs beat the benchmark in only **6 to 8 of 10 months**. Their weakest periods tend to cluster near the beginning of validation and again in the final January window, while December is unusually strong across the demand targets.

Importantly, every Target × Horizon job still has **positive median monthly skill**, so the pooled Transformer advantage is not being created solely by one or two exceptional months. The evidence instead says that speed gains are broad and persistent, whereas demand gains are real but substantially more sensitive to when the forecast is made.

**4.5.2.7.2B — Check whether the nonnegative output rule is doing substantive work**

Trip counts and ridership cannot be negative, so the project applies a zero floor only after the raw Transformer forecast exists. That rule should ideally enforce physical validity without materially changing forecast accuracy.

The diagnostic below compares **how often raw forecasts are negative** with **how much the floor changes MAE**. Speed targets are included as a useful control because their raw forecasts should rarely, if ever, require clipping.

In [72]:
# ---------------------------------------------------------------------
# 4.5.2.7.2B — Diagnose the nonnegative output rule
# ---------------------------------------------------------------------

output_rule_df = transformer_job_scorecard_df.copy()

output_rule_df["negative_raw_rate_pct"] = (
    100.0
    * output_rule_df["negative_raw_predictions"]
    / output_rule_df["evaluation_rows"]
)

# Section 6 stored final-minus-raw MAE change. Reverse the sign here so a
# positive value reads naturally as "the floor improved MAE by this amount."
output_rule_df["floor_mae_improvement_pct"] = (
    -output_rule_df["floor_mae_change_pct"]
)

fig = go.Figure()

for metric_index, metric in enumerate(FORECAST_TARGETS):
    metric_df = output_rule_df.loc[
        output_rule_df["metric"].eq(metric)
    ].sort_values("horizon")

    fig.add_trace(
        go.Scatter(
            x=metric_df["negative_raw_rate_pct"],
            y=metric_df["floor_mae_improvement_pct"],
            mode="markers+text",
            text=[f"h={int(value)}" for value in metric_df["horizon"]],
            textposition="top center",
            name=TARGET_LABELS[metric],
            marker={
                "size": 11,
                "color": BRAND_COLOR_SEQUENCE[
                    metric_index % len(BRAND_COLOR_SEQUENCE)
                ],
            },
            hovertemplate=(
                "Target: %{fullData.name}<br>"
                "Negative raw forecasts: %{x:.3f}%<br>"
                "MAE improvement from floor: %{y:.3f}%"
                "<extra></extra>"
            ),
        )
    )

fig = apply_branding(fig)

fig.update_layout(
    title="How Much Does the Nonnegative Output Rule Affect Accuracy?",
    xaxis_title="Negative raw forecast rate (%)",
    yaxis_title="MAE improvement from zero floor (%)",
    legend_title_text="Target",
)

fig.show()

print(
    "Output-rule QA | "
    f"jobs with negative raw forecasts="
    f"{int(output_rule_df['negative_raw_predictions'].gt(0).sum())}/"
    f"{len(output_rule_df)} | "
    f"maximum negative rate="
    f"{output_rule_df['negative_raw_rate_pct'].max():.3f}% | "
    f"largest MAE change="
    f"{output_rule_df['floor_mae_improvement_pct'].abs().max():.3f}%"
)

Output-rule QA | jobs with negative raw forecasts=9/15 | maximum negative rate=19.581% | largest MAE change=3.559%


**Findings.** Negative raw forecasts occur in **9 of 15 jobs**, covering all three horizons for Taxi trips, FHVHV trips, and Subway ridership. Neither average-speed target produces negative forecasts.

Taxi trips is the main source of the issue: negative raw predictions account for **19.581% of h=1 rows**, **16.423% of h=2 rows**, and **11.487% of h=5 rows**. The corresponding rates for FHVHV trips and Subway remain below 3%.

The zero floor improves MAE in every affected job, but the largest improvement is only **3.559%**. For FHVHV trips and Subway, the change remains below 1%. The floor is therefore valuable primarily because negative demand forecasts are physically invalid; its modest accuracy benefit should not be confused with evidence that clipping is responsible for the Transformer's overall forecasting performance.

### 4.5.2.7.3 — Grouped Predictive Reliance

Forecast accuracy tells us whether the Transformer works. Grouped predictive reliance asks a different question:

**What broad sources of information does the finished Transformer actually depend on?**

The analysis uses **grouped row permutation**. One retained production model remains completely fixed while a coherent input group is reassigned across otherwise valid forecast rows. If error increases, the row-level correspondence carried by that group was helping the model.

Related values move together. The complete patched mobility history is permuted as one block; weekly-STL magnitude and state indicators move together; weather values stay attached to their missingness indicators; and Taxi Zone identity is permuted through its own embedding input.

This measures **predictive reliance, not causality**. A large weather reliance score, for example, does not mean experimentally changing weather would cause mobility to change by the measured amount.

The analysis uses only the **latest validated fitted state for each of the 15 final Target × Horizon jobs**. No model is retrained.

**4.5.2.7.3A — Freeze the grouped permutation contract**

Eight broad groups cover the Transformer's three model-facing inputs:

- **Recent mobility history** — the complete 70-step × 10-channel patched sequence.
- **Annual / calendar timing** — Daypart, weekday, weekend state, and annual Fourier position.
- **Weekly STL seasonal state** — seasonal offset, enablement, and missingness state.
- **Weather** — origin-time weather, forecast-time weather proxies, and their missingness indicators.
- **Congestion-pricing regime** — the model-facing policy-state indicator.
- **Multimodal / connected context** — auxiliary local mobility plus connected-zone mobility context and missingness state.
- **Spatial / network context** — CBD geography, distances, and network structure plus missingness state.
- **Taxi Zone identity** — the learned zone embedding input.

Each group is permuted three times on at most **5,000 rows per final job**. A group that does not vary on the sampled scoring surface is reported as **not permutation-identifiable / N/A**, rather than being assigned a misleading zero.

In [73]:
# ---------------------------------------------------------------------
# 4.5.2.7.3A — Freeze the grouped permutation contract
# ---------------------------------------------------------------------

TRANSFORMER_GROUP_RELIANCE_CONTRACT_ID = "transformer_group_reliance_v1"
TRANSFORMER_RELIANCE_MAX_ROWS = 5_000
TRANSFORMER_RELIANCE_REPEATS = 3

TRANSFORMER_GROUP_RELIANCE_ROOT = (
    INTERMEDIATE_DIR / "tr_group_reliance_v1"
)

TRANSFORMER_GROUP_RELIANCE_RESULTS_PATH = (
    TRANSFORMER_GROUP_RELIANCE_ROOT / "results.parquet"
)

RUN_TRANSFORMER_GROUPED_RELIANCE = True
RESET_TRANSFORMER_GROUPED_RELIANCE = False


def context_source_bundle(source_features):
    """Return context sources together with their paired missingness masks."""
    bundled = []

    for source in source_features:
        if source in TRANSFORMER_CONTEXT_FEATURE_ORDER:
            bundled.append(source)

        mask = f"missing_mask__{source}"

        if mask in TRANSFORMER_CONTEXT_FEATURE_ORDER:
            bundled.append(mask)

    return bundled


TRANSFORMER_CONTEXT_GROUP_FEATURES = {
    "Annual / calendar timing": [
        "target_daypart_sin",
        "target_daypart_cos",
        "target_weekday_sin",
        "target_weekday_cos",
        "target_is_weekend",
        "annual_fourier_sin",
        "annual_fourier_cos",
    ],
    "Weekly STL seasonal state": [
        "weekly_stl_seasonal_offset",
        "weekly_stl_enabled",
        "missing_mask__weekly_stl_seasonal_offset",
    ],
    "Weather": context_source_bundle(
        WEATHER_CONTEXT_SOURCES
    ),
    "Congestion-pricing regime": [
        "target_cp_regime",
    ],
    "Multimodal / connected context": context_source_bundle(
        MULTIMODAL_CONTEXT_SOURCES
        + SPATIAL_CONTEXT_SOURCES
    ),
    "Spatial / network context": context_source_bundle(
        GEOGRAPHIC_CONTEXT_SOURCES
    ),
}

context_group_fields = [
    feature
    for features in TRANSFORMER_CONTEXT_GROUP_FEATURES.values()
    for feature in features
]

duplicate_context_fields = pd.Series(
    context_group_fields
).duplicated().any()

missing_context_fields = sorted(
    set(TRANSFORMER_CONTEXT_FEATURE_ORDER)
    - set(context_group_fields)
)

unexpected_context_fields = sorted(
    set(context_group_fields)
    - set(TRANSFORMER_CONTEXT_FEATURE_ORDER)
)

if duplicate_context_fields:
    raise ValueError(
        "Grouped Transformer reliance assigns at least one context field twice."
    )

if missing_context_fields or unexpected_context_fields:
    raise ValueError(
        "Grouped Transformer reliance must partition all 52 context values "
        f"exactly once. Missing={missing_context_fields}; "
        f"unexpected={unexpected_context_fields}."
    )


reliance_contract_records = [{
    "Input group": "Recent mobility history",
    "Model-facing scope": "70 steps × 10 channels",
}]

for group, features in TRANSFORMER_CONTEXT_GROUP_FEATURES.items():
    reliance_contract_records.append({
        "Input group": group,
        "Model-facing scope": f"{len(features)} context fields",
    })

reliance_contract_records.append({
    "Input group": "Taxi Zone identity",
    "Model-facing scope": "1 embedding index",
})

transformer_reliance_contract_df = pd.DataFrame(
    reliance_contract_records
)

display(transformer_reliance_contract_df)

print(
    "Permutation contract | "
    f"groups={len(transformer_reliance_contract_df)} | "
    f"final models=15 | "
    f"rows/job≤{TRANSFORMER_RELIANCE_MAX_ROWS:,} | "
    f"repeats/group={TRANSFORMER_RELIANCE_REPEATS} | "
    "retraining=False"
)

,Input group,Model-facing scope
0,Recent mobility history,70 steps × 10 channels
1,Annual / calendar timing,7 context fields
2,Weekly STL seasonal state,3 context fields
3,Weather,8 context fields
4,Congestion-pricing regime,1 context fields
5,Multimodal / connected context,15 context fields
6,Spatial / network context,18 context fields
7,Taxi Zone identity,1 embedding index


Permutation contract | groups=8 | final models=15 | rows/job≤5,000 | repeats/group=3 | retraining=False


**Findings.** The grouped reliance contract covers **8 semantic input groups across all 15 final Target × Horizon models**. Each model is evaluated on at most **5,000 forecast rows**, with **3 row permutations per group**, while the fitted Transformer weights and preprocessing state remain unchanged.

The complete **52-value context** is partitioned exactly once across six context-based groups; recent mobility history and Taxi Zone identity remain separate model inputs. Groups without model-facing variation remain visible as **not permutation-identifiable / N/A** rather than being interpreted as zero reliance.

**4.5.2.7.3B — Restore the latest production models without retraining**

The production crawl saved a complete fitted state for every monthly refit. For interpretability we need only one current snapshot per final forecasting job, so this section retains the **latest validated monthly state** for each of the 15 Target × Horizon combinations.

Each state restores the Transformer weights, chosen representation and capacity profile, Taxi Zone embedding space, target-specific channel order, and training-only preprocessing statistics. No optimizer is created and no training routine is called.

In [74]:
# ---------------------------------------------------------------------
# 4.5.2.7.3B — Restore the latest production states without retraining
# ---------------------------------------------------------------------

latest_refit_by_job = (
    full_transformer_results_df
    .groupby("job_id", observed=True)["refit_step"]
    .transform("max")
)

transformer_latest_state_registry_df = (
    full_transformer_results_df.loc[
        full_transformer_results_df["refit_step"].eq(
            latest_refit_by_job
        ),
        [
            "job_id",
            "track",
            "metric",
            "horizon",
            "candidate_id",
            "temporal_window",
            "refit_step",
            "selected_epoch",
            "state_path",
        ],
    ]
    .sort_values(["metric", "horizon"])
    .reset_index(drop=True)
)

if len(transformer_latest_state_registry_df) != len(
    transformer_full_jobs_df
):
    raise RuntimeError(
        "Expected exactly one latest production state for each of 15 final jobs."
    )

if transformer_latest_state_registry_df["job_id"].duplicated().any():
    raise RuntimeError(
        "Latest Transformer state registry contains duplicate jobs."
    )

if not transformer_latest_state_registry_df["state_path"].map(
    lambda value: pd.notna(value) and Path(value).exists()
).all():
    raise RuntimeError(
        "At least one latest Transformer production state is missing."
    )


def restore_transformer_preprocessor(state):
    """Restore training-only preprocessing saved with one fitted Transformer."""
    preprocessor = TransformerPreprocessor(
        pd.DataFrame(state["context_contract"])
    )

    saved = state["preprocessor_state"]

    preprocessor.sequence_means_ = np.asarray(
        saved["sequence_means"],
        dtype=np.float64,
    )
    preprocessor.sequence_stds_ = np.asarray(
        saved["sequence_stds"],
        dtype=np.float64,
    )
    preprocessor.context_medians_ = np.asarray(
        saved["context_medians"],
        dtype=np.float64,
    )
    preprocessor.context_means_ = np.asarray(
        saved["context_means"],
        dtype=np.float64,
    )
    preprocessor.context_stds_ = np.asarray(
        saved["context_stds"],
        dtype=np.float64,
    )

    preprocessor.stl_training_support_ = int(
        saved["stl_training_support"]
    )
    preprocessor.stl_mean_ = float(saved["stl_mean"])
    preprocessor.stl_std_ = float(saved["stl_std"])
    preprocessor.weekly_stl_scaler_active_ = bool(
        saved["weekly_stl_scaler_active"]
    )

    preprocessor.target_mean_ = float(
        saved["target_mean"]
    )
    preprocessor.target_std_ = float(
        saved["target_std"]
    )
    preprocessor.is_fitted_ = True

    return preprocessor


def restore_transformer_model(state_path):
    """Restore one saved Transformer forecasting function for inference only."""
    state = torch.load(
        state_path,
        map_location=TORCH_DEVICE,
        weights_only=False,
    )

    if list(state["context_feature_order"]) != list(
        TRANSFORMER_CONTEXT_FEATURE_ORDER
    ):
        raise RuntimeError(
            "Saved Transformer context order differs from the current contract."
        )

    if not np.array_equal(
        np.asarray(state["zone_ids"], dtype=np.int64),
        np.asarray(transformer_zone_ids, dtype=np.int64),
    ):
        raise RuntimeError(
            "Saved Transformer Taxi Zone vocabulary differs from the current state."
        )

    model = PatchTransformerRegressor(
        NEURAL_SEQUENCE_INPUT_DIM,
        len(state["context_feature_order"]),
        len(state["zone_ids"]),
        state["representation"],
        state["profile"],
    ).to(TORCH_DEVICE)

    model.load_state_dict(
        state["model_state_dict"]
    )
    model.eval()

    preprocessor = restore_transformer_preprocessor(
        state
    )

    return state, model, preprocessor


print(
    "Latest-state registry | "
    f"{len(transformer_latest_state_registry_df)}/15 models available | "
    "retraining=False"
)

Latest-state registry | 15/15 models available | retraining=False


**Findings.** All **15 final Target × Horizon jobs** have one latest validated fitted state available for inference-only diagnostics. Restoring these states reproduces the saved forecasting function—model weights, preprocessing, representation, and Taxi Zone identity—without creating an optimizer or refitting any model.

**4.5.2.7.3C — Measure grouped predictive reliance**

For each final job, the latest scoring window is sampled deterministically to at most **5,000 rows**. The unmodified restored model first establishes baseline raw- and final-output MAE.

Each identifiable input group is then permuted three times. All values inside one semantic group use the **same row permutation**, preserving their internal structure while breaking their relationship to the forecast row.

The experiment is checkpointed by forecasting job. A restart therefore reuses completed attribution jobs instead of repeating their inference passes.

In [75]:
# ---------------------------------------------------------------------
# 4.5.2.7.3C — Measure grouped predictive reliance
# ---------------------------------------------------------------------

def model_facing_group_varies(values):
    """Return whether a sampled model-facing input group varies across rows."""
    array = np.asarray(values)

    if len(array) < 2:
        return False

    flattened = array.reshape(len(array), -1)

    if not np.isfinite(flattened).all():
        raise RuntimeError(
            "Permutation identifiability requires finite model-facing inputs."
        )

    return bool(np.any(flattened != flattened[0]))


def transformer_reliance_seed(job_id, input_group, repeat):
    """Create a deterministic permutation seed for one Job × Group × Repeat."""
    seed_text = (
        f"{TRANSFORMER_GROUP_RELIANCE_CONTRACT_ID}|"
        f"{job_id}|{input_group}|{repeat}|{GLOBAL_SEED}"
    )

    digest = hashlib.sha256(
        seed_text.encode("utf-8")
    ).digest()

    return int.from_bytes(
        digest[:4],
        "big",
    ) % (2**31 - 1)


def predict_prepared_transformer(
    model,
    preprocessor,
    sequence,
    context,
    zone_index,
):
    """Score prepared Transformer inputs in bounded inference batches."""
    standardized_batches = []

    with torch.no_grad():
        for start in range(
            0,
            len(sequence),
            selected_transformer_batch_size,
        ):
            stop = start + selected_transformer_batch_size

            standardized_batches.append(
                model(
                    torch.as_tensor(
                        sequence[start:stop],
                        dtype=torch.float32,
                        device=TORCH_DEVICE,
                    ),
                    torch.as_tensor(
                        context[start:stop],
                        dtype=torch.float32,
                        device=TORCH_DEVICE,
                    ),
                    torch.as_tensor(
                        zone_index[start:stop],
                        dtype=torch.long,
                        device=TORCH_DEVICE,
                    ),
                ).cpu().numpy()
            )

    standardized = np.concatenate(
        standardized_batches
    )

    prediction_raw = preprocessor.inverse_target(
        standardized
    )

    prediction_final = apply_prediction_output_rule(
        prediction_raw
    )

    return (
        prediction_raw.astype(float),
        prediction_final.astype(float),
    )


def prepare_transformer_reliance_sample(
    job,
    refit_row,
    state,
    preprocessor,
):
    """Recreate one latest scoring surface and materialize sampled inputs."""
    job_cache = prepare_full_transformer_job_cache(
        job
    )

    split = resolve_full_transformer_split(
        job_cache,
        job,
        refit_row,
    )

    evaluation_df = (
        split["evaluation_df"]
        .sort_values(
            [
                "target_date",
                "forecast_sequence_id",
                "taxi_zone_id",
            ],
            kind="stable",
        )
        .reset_index(drop=True)
    )

    # Even spacing preserves coverage across the scoring window without making
    # attribution runtime depend on the full monthly row count.
    if len(evaluation_df) > TRANSFORMER_RELIANCE_MAX_ROWS:
        sample_positions = np.linspace(
            0,
            len(evaluation_df) - 1,
            TRANSFORMER_RELIANCE_MAX_ROWS,
        ).round().astype(int)

        evaluation_df = (
            evaluation_df
            .iloc[np.unique(sample_positions)]
            .reset_index(drop=True)
        )

    zone_positions = evaluation_df["taxi_zone_id"].map(
        transformer_zone_to_index
    )

    origin_positions = evaluation_df[
        "origin_observation_sequence_id"
    ].map(
        transformer_sequence_to_index
    )

    if zone_positions.isna().any() or origin_positions.isna().any():
        raise RuntimeError(
            "Reliance sample contains a Taxi Zone or sequence outside dense history."
        )

    zone_index = zone_positions.to_numpy(
        dtype=np.int64
    )

    origin_index = origin_positions.to_numpy(
        dtype=np.int64
    )

    lookback_steps = int(
        job["lookback_steps"]
    )

    history_starts = (
        origin_index
        - lookback_steps
        + 1
    )

    if (history_starts < 0).any():
        raise RuntimeError(
            "Reliance sample requests an incomplete history window."
        )

    history_positions = (
        history_starts[:, None]
        + np.arange(
            lookback_steps,
            dtype=np.int64,
        )[None, :]
    )

    channel_positions = [
        FORECAST_TARGETS.index(channel)
        for channel in state["sequence_value_channels"]
    ]

    raw_sequence_values = transformer_history_values[
        zone_index[:, None],
        history_positions,
        :,
    ][:, :, channel_positions]

    raw_sequence_masks = transformer_history_masks[
        zone_index[:, None],
        history_positions,
        :,
    ][:, :, channel_positions]

    prepared_sequence = preprocessor.transform_sequence(
        raw_sequence_values,
        raw_sequence_masks,
    )

    prepared_context = preprocessor.transform_context(
        derive_transformer_context_frame(
            evaluation_df
        )
    )

    target = pd.to_numeric(
        evaluation_df["target_observed_value"],
        errors="raise",
    ).to_numpy(
        dtype=float
    )

    del job_cache

    return {
        "sequence": prepared_sequence,
        "context": prepared_context,
        "zone_index": zone_index,
        "target": target,
    }


def grouped_reliance_checkpoint_path(job_id):
    """Return one durable attribution checkpoint per final forecasting job."""
    token = hashlib.sha1(
        str(job_id).encode("utf-8")
    ).hexdigest()[:16]

    return (
        TRANSFORMER_GROUP_RELIANCE_ROOT
        / f"job_{token}.parquet"
    )


def recover_grouped_reliance_job(job_id):
    """Recover one complete current-contract reliance job when available."""
    path = grouped_reliance_checkpoint_path(
        job_id
    )

    if not path.exists():
        return None

    try:
        frame = pd.read_parquet(
            path
        )

        expected_rows = (
            len(transformer_reliance_contract_df)
            * TRANSFORMER_RELIANCE_REPEATS
        )

        if (
            len(frame) != expected_rows
            or not frame["contract_id"].eq(
                TRANSFORMER_GROUP_RELIANCE_CONTRACT_ID
            ).all()
            or not frame["job_id"].eq(job_id).all()
        ):
            return None

        return frame

    except (
        OSError,
        ValueError,
        KeyError,
        TypeError,
    ):
        return None


def run_transformer_grouped_reliance(state_row):
    """Run inference-only grouped permutation reliance for one final job."""
    job_match_df = transformer_full_jobs_df.loc[
        transformer_full_jobs_df["job_id"].eq(
            state_row.job_id
        )
    ]

    if len(job_match_df) != 1:
        raise RuntimeError(
            f"{state_row.job_id}: expected exactly one frozen production job."
        )

    job = job_match_df.iloc[0].to_dict()

    refit_match_df = transformer_full_schedule_df.loc[
        transformer_full_schedule_df["job_id"].eq(
            state_row.job_id
        )
        & transformer_full_schedule_df[
            "refit_step"
        ].astype(int).eq(
            int(state_row.refit_step)
        )
    ]

    if len(refit_match_df) != 1:
        raise RuntimeError(
            f"{state_row.job_id}: latest production refit is not unique."
        )

    refit_row = refit_match_df.iloc[0].to_dict()

    state, model, preprocessor = restore_transformer_model(
        state_row.state_path
    )

    logical_key = state["logical_key"]

    for column in TRANSFORMER_LOGICAL_KEY_COLUMNS:
        if str(logical_key[column]) != str(
            refit_row[column]
        ):
            raise RuntimeError(
                f"{state_row.job_id}: saved state disagrees with {column}."
            )

    sample = prepare_transformer_reliance_sample(
        job,
        refit_row,
        state,
        preprocessor,
    )

    baseline_raw, baseline_final = (
        predict_prepared_transformer(
            model,
            preprocessor,
            sample["sequence"],
            sample["context"],
            sample["zone_index"],
        )
    )

    baseline_raw_mae = regression_metrics(
        sample["target"],
        baseline_raw,
    )["mae"]

    baseline_final_mae = regression_metrics(
        sample["target"],
        baseline_final,
    )["mae"]

    context_index = {
        feature: index
        for index, feature in enumerate(
            state["context_feature_order"]
        )
    }

    records = []

    for input_group in transformer_reliance_contract_df[
        "Input group"
    ]:
        if input_group == "Recent mobility history":
            model_facing_values = sample[
                "sequence"
            ]

        elif input_group == "Taxi Zone identity":
            model_facing_values = sample[
                "zone_index"
            ][:, None]

        else:
            group_indices = [
                context_index[feature]
                for feature in TRANSFORMER_CONTEXT_GROUP_FEATURES[
                    input_group
                ]
            ]

            model_facing_values = sample[
                "context"
            ][:, group_indices]

        identifiable = model_facing_group_varies(
            model_facing_values
        )

        for repeat in range(
            1,
            TRANSFORMER_RELIANCE_REPEATS + 1,
        ):
            record = {
                "contract_id":
                    TRANSFORMER_GROUP_RELIANCE_CONTRACT_ID,
                "job_id":
                    state_row.job_id,
                "track":
                    state_row.track,
                "metric":
                    state_row.metric,
                "horizon":
                    int(state_row.horizon),
                "refit_step":
                    int(state_row.refit_step),
                "temporal_window":
                    state_row.temporal_window,
                "input_group":
                    input_group,
                "repeat":
                    repeat,
                "evaluation_rows":
                    len(sample["target"]),
                "permutation_identifiable":
                    identifiable,
                "baseline_raw_mae":
                    baseline_raw_mae,
                "baseline_final_mae":
                    baseline_final_mae,
                "perturbed_raw_mae":
                    np.nan,
                "perturbed_final_mae":
                    np.nan,
                "raw_mae_damage":
                    np.nan,
                "final_mae_damage":
                    np.nan,
                "retrained":
                    False,
            }

            if identifiable:
                rng = np.random.default_rng(
                    transformer_reliance_seed(
                        state_row.job_id,
                        input_group,
                        repeat,
                    )
                )

                permutation = rng.permutation(
                    len(sample["target"])
                )

                sequence = sample["sequence"]
                context = sample["context"]
                zone_index = sample["zone_index"]

                if input_group == "Recent mobility history":
                    sequence = sample[
                        "sequence"
                    ][permutation]

                elif input_group == "Taxi Zone identity":
                    zone_index = sample[
                        "zone_index"
                    ][permutation]

                else:
                    context = sample[
                        "context"
                    ].copy()

                    group_indices = [
                        context_index[feature]
                        for feature
                        in TRANSFORMER_CONTEXT_GROUP_FEATURES[
                            input_group
                        ]
                    ]

                    context[:, group_indices] = (
                        sample["context"][
                            permutation
                        ][:, group_indices]
                    )

                perturbed_raw, perturbed_final = (
                    predict_prepared_transformer(
                        model,
                        preprocessor,
                        sequence,
                        context,
                        zone_index,
                    )
                )

                perturbed_raw_mae = regression_metrics(
                    sample["target"],
                    perturbed_raw,
                )["mae"]

                perturbed_final_mae = regression_metrics(
                    sample["target"],
                    perturbed_final,
                )["mae"]

                record.update({
                    "perturbed_raw_mae":
                        perturbed_raw_mae,
                    "perturbed_final_mae":
                        perturbed_final_mae,
                    "raw_mae_damage":
                        perturbed_raw_mae
                        - baseline_raw_mae,
                    "final_mae_damage":
                        perturbed_final_mae
                        - baseline_final_mae,
                })

            records.append(
                record
            )

    del model, preprocessor, sample
    clear_device_memory()

    return pd.DataFrame(
        records
    )


TRANSFORMER_GROUP_RELIANCE_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

if RESET_TRANSFORMER_GROUPED_RELIANCE:
    reset_experiment_artifacts(
        TRANSFORMER_GROUP_RELIANCE_ROOT,
        consolidated_paths=[
            TRANSFORMER_GROUP_RELIANCE_RESULTS_PATH
        ],
    )


reliance_job_frames = []

for job_number, state_row in enumerate(
    transformer_latest_state_registry_df.itertuples(
        index=False
    ),
    start=1,
):
    recovered = recover_grouped_reliance_job(
        state_row.job_id
    )

    if recovered is not None:
        reliance_job_frames.append(
            recovered
        )

        print(
            f"Grouped reliance: {job_number}/15 recovered · "
            f"{TARGET_LABELS[state_row.metric]} · "
            f"h={int(state_row.horizon)}"
        )

        continue

    if not RUN_TRANSFORMER_GROUPED_RELIANCE:
        raise RuntimeError(
            "Grouped Transformer reliance is incomplete. Set "
            "RUN_TRANSFORMER_GROUPED_RELIANCE=True to execute missing jobs."
        )

    completed = run_transformer_grouped_reliance(
        state_row
    )

    atomic_write_parquet(
        completed,
        grouped_reliance_checkpoint_path(
            state_row.job_id
        ),
    )

    reliance_job_frames.append(
        completed
    )

    print(
        f"Grouped reliance: {job_number}/15 complete · "
        f"{TARGET_LABELS[state_row.metric]} · "
        f"h={int(state_row.horizon)}"
    )


transformer_grouped_reliance_repeat_df = pd.concat(
    reliance_job_frames,
    ignore_index=True,
)

expected_reliance_rows = (
    len(transformer_latest_state_registry_df)
    * len(transformer_reliance_contract_df)
    * TRANSFORMER_RELIANCE_REPEATS
)

if (
    len(transformer_grouped_reliance_repeat_df)
    != expected_reliance_rows
):
    raise RuntimeError(
        "Grouped reliance result does not contain the complete planned surface."
    )

if transformer_grouped_reliance_repeat_df[
    "retrained"
].any():
    raise RuntimeError(
        "Grouped reliance must remain inference-only."
    )

atomic_write_parquet(
    transformer_grouped_reliance_repeat_df,
    TRANSFORMER_GROUP_RELIANCE_RESULTS_PATH,
)

print(
    "Grouped reliance complete | "
    f"jobs="
    f"{transformer_grouped_reliance_repeat_df['job_id'].nunique()} | "
    f"groups="
    f"{transformer_grouped_reliance_repeat_df['input_group'].nunique()} | "
    f"repeat-level comparisons="
    f"{len(transformer_grouped_reliance_repeat_df):,}"
)

Grouped reliance: 1/15 recovered · FHVHV average speed · h=1
Grouped reliance: 2/15 recovered · FHVHV average speed · h=2
Grouped reliance: 3/15 recovered · FHVHV average speed · h=5
Grouped reliance: 4/15 recovered · FHVHV trips · h=1
Grouped reliance: 5/15 recovered · FHVHV trips · h=2
Grouped reliance: 6/15 recovered · FHVHV trips · h=5
Grouped reliance: 7/15 recovered · Subway ridership · h=1
Grouped reliance: 8/15 recovered · Subway ridership · h=2
Grouped reliance: 9/15 recovered · Subway ridership · h=5
Grouped reliance: 10/15 recovered · Taxi average speed · h=1
Grouped reliance: 11/15 recovered · Taxi average speed · h=2
Grouped reliance: 12/15 recovered · Taxi average speed · h=5
Grouped reliance: 13/15 recovered · Taxi trips · h=1
Grouped reliance: 14/15 recovered · Taxi trips · h=2
Grouped reliance: 15/15 recovered · Taxi trips · h=5
Grouped reliance complete | jobs=15 | groups=8 | repeat-level comparisons=360


**Findings.** The grouped permutation run covers the complete **15-job × 8-group × 3-repeat** attribution surface and uses only restored production states. No forecasting model is retrained; completed Job-level attribution blocks are reused on restart.

*Interpretive findings follow after the repeat-level results are summarized below.*

**4.5.2.7.3D — Which information sources matter most overall?**

Raw permutation damage is expressed in each target's native error units, so it cannot be averaged directly across Taxi trips, vehicle speeds, FHVHV demand, and Subway ridership.

For cross-target comparison, positive final-output MAE damage is therefore converted into a **within-job reliance share**. For each job with usable positive damage, the identifiable groups sum to 100%. Negative permutation damage contributes zero share rather than being interpreted as evidence that an input is negatively important.

This gives us a common question across all 15 final models: **what fraction of measured predictive reliance belongs to each broad information source?**

In [76]:
# ---------------------------------------------------------------------
# 4.5.2.7.3D — Summarize broad predictive reliance
# ---------------------------------------------------------------------

transformer_grouped_reliance_job_df = (
    transformer_grouped_reliance_repeat_df
    .groupby(
        [
            "job_id",
            "track",
            "metric",
            "horizon",
            "input_group",
        ],
        as_index=False,
        observed=True,
    )
    .agg(
        permutation_identifiable=(
            "permutation_identifiable",
            "max",
        ),
        mean_raw_mae_damage=(
            "raw_mae_damage",
            "mean",
        ),
        mean_final_mae_damage=(
            "final_mae_damage",
            "mean",
        ),
    )
)

transformer_grouped_reliance_job_df[
    "positive_final_damage"
] = np.where(
    transformer_grouped_reliance_job_df[
        "permutation_identifiable"
    ],
    transformer_grouped_reliance_job_df[
        "mean_final_mae_damage"
    ].clip(lower=0),
    np.nan,
)

job_positive_totals = (
    transformer_grouped_reliance_job_df
    .groupby(
        "job_id",
        observed=True,
    )["positive_final_damage"]
    .transform("sum")
)

transformer_grouped_reliance_job_df[
    "reliance_share_pct"
] = np.where(
    job_positive_totals.gt(0),
    100.0
    * transformer_grouped_reliance_job_df[
        "positive_final_damage"
    ]
    / job_positive_totals,
    np.nan,
)


# Identify the highest-share group within each job only when normalization exists.
identifiable_share_df = (
    transformer_grouped_reliance_job_df
    .loc[
        transformer_grouped_reliance_job_df[
            "reliance_share_pct"
        ].notna()
    ]
    .copy()
)

top_group_by_job_df = (
    identifiable_share_df.loc[
        identifiable_share_df
        .groupby(
            "job_id",
            observed=True,
        )["reliance_share_pct"]
        .idxmax()
    ]
    [["job_id", "input_group"]]
)

top_group_counts = (
    top_group_by_job_df[
        "input_group"
    ].value_counts()
)

transformer_reliance_overall_df = (
    transformer_grouped_reliance_job_df
    .groupby(
        "input_group",
        as_index=False,
        observed=True,
    )
    .agg(
        identifiable_jobs=(
            "permutation_identifiable",
            "sum",
        ),
        mean_reliance_share_pct=(
            "reliance_share_pct",
            "mean",
        ),
        median_reliance_share_pct=(
            "reliance_share_pct",
            "median",
        ),
    )
)

transformer_reliance_overall_df[
    "top_group_jobs"
] = (
    transformer_reliance_overall_df[
        "input_group"
    ]
    .map(top_group_counts)
    .fillna(0)
    .astype(int)
)

transformer_reliance_overall_df = (
    transformer_reliance_overall_df
    .sort_values(
        "mean_reliance_share_pct",
        ascending=False,
        na_position="last",
    )
    .reset_index(drop=True)
)

reliance_display_df = (
    transformer_reliance_overall_df
    .rename(columns={
        "input_group": "Input group",
        "identifiable_jobs": "Identifiable jobs",
        "mean_reliance_share_pct": "Mean reliance share %",
        "median_reliance_share_pct": "Median reliance share %",
        "top_group_jobs": "Jobs where #1",
    })
)

display(
    reliance_display_df.round(3)
)


bar_df = (
    transformer_reliance_overall_df
    .dropna(
        subset=["mean_reliance_share_pct"]
    )
    .sort_values(
        "mean_reliance_share_pct"
    )
)

bar_colors = [
    BRAND_COLOR_SEQUENCE[
        index % len(BRAND_COLOR_SEQUENCE)
    ]
    for index in range(len(bar_df))
]

fig = go.Figure(
    go.Bar(
        x=bar_df["mean_reliance_share_pct"],
        y=bar_df["input_group"],
        orientation="h",
        marker_color=bar_colors,
        text=bar_df[
            "mean_reliance_share_pct"
        ].map(
            lambda value: f"{value:.1f}%"
        ),
        textposition="outside",
        hovertemplate=(
            "Input group: %{y}<br>"
            "Mean reliance share: %{x:.3f}%"
            "<extra></extra>"
        ),
    )
)

fig = apply_branding(fig)

fig.update_layout(
    title="Average Predictive Reliance Across Final Transformer Jobs",
    xaxis_title="Mean within-job reliance share (%)",
    yaxis_title="",
    showlegend=False,
)

fig.show()

,Input group,Identifiable jobs,Mean reliance share %,Median reliance share %,Jobs where #1
0,Recent mobility history,15,86.454,89.549,15
1,Annual / calendar timing,15,6.445,4.301,0
2,Multimodal / connected context,15,3.157,1.699,0
3,Spatial / network context,15,1.778,1.578,0
4,Taxi Zone identity,15,1.488,1.148,0
5,Weekly STL seasonal state,15,0.577,0.368,0
6,Weather,15,0.100,0.016,0
7,Congestion-pricing regime,0,NaN,NaN,0


**Findings.** Recent mobility history overwhelmingly dominates the Transformer's measured predictive reliance. It accounts for **86.454% of reliance on average**, has a **89.549% median share**, and ranks as the single most important group in **all 15 final Target × Horizon models**.

The remaining measured reliance is much more dispersed: annual/calendar timing contributes **6.445% on average**, multimodal/connected context **3.157%**, spatial/network context **1.778%**, Taxi Zone identity **1.488%**, weekly-STL seasonal state **0.577%**, and weather only **0.100%**.

The congestion-pricing regime is **not permutation-identifiable in any of the 15 latest scoring surfaces**, because those rows do not provide model-facing variation in that indicator. This should be read as **N/A, not zero importance**.

These percentages describe how much the fitted models' forecast accuracy deteriorates when broad input groups are disrupted. They measure **predictive reliance**, not causal effects: a large share does not mean that changing that input in the real world would cause an equivalent change in mobility.

**4.5.2.7.3E — How does reliance differ by forecast target?**

An overall average can hide important differences between mobility outcomes. The same Transformer architecture may lean on recent history for one target while using calendar, weather, or connected-mobility context more heavily for another.

The heatmap therefore averages the within-job reliance shares across the three horizons for each target. Blank cells indicate that the group was not permutation-identifiable on that target's latest scoring surface—not that the model had zero dependence on it.

In [85]:
# ---------------------------------------------------------------------
# 4.5.2.7.3E — Compare reliance across forecast targets
# ---------------------------------------------------------------------

target_reliance_df = (
    transformer_grouped_reliance_job_df
    .groupby(
        ["metric", "input_group"],
        as_index=False,
        observed=True,
    )
    .agg(
        mean_reliance_share_pct=(
            "reliance_share_pct",
            "mean",
        ),
        identifiable_jobs=(
            "permutation_identifiable",
            "sum",
        ),
    )
)

reliance_group_order = (
    transformer_reliance_overall_df[
        "input_group"
    ].tolist()
)

target_reliance_matrix_df = (
    target_reliance_df
    .pivot(
        index="metric",
        columns="input_group",
        values="mean_reliance_share_pct",
    )
    .reindex(
        index=FORECAST_TARGETS,
        columns=reliance_group_order,
    )
)

target_reliance_values = (
    target_reliance_matrix_df
    .to_numpy(dtype=float)
)

target_reliance_text = np.empty(
    target_reliance_values.shape,
    dtype=object,
)

for row_index in range(
    target_reliance_values.shape[0]
):
    for column_index in range(
        target_reliance_values.shape[1]
    ):
        value = target_reliance_values[
            row_index,
            column_index,
        ]

        target_reliance_text[
            row_index,
            column_index,
        ] = (
            "—"
            if not np.isfinite(value)
            else f"{value:.1f}%"
        )

fig = go.Figure(
    data=go.Heatmap(
        z=target_reliance_values,
        x=reliance_group_order,
        y=[
            TARGET_LABELS[metric]
            for metric in FORECAST_TARGETS
        ],
        text=target_reliance_text,
        texttemplate="%{text}",
        colorscale=BRAND_SEQUENTIAL_HEATMAP_SCALE,
        colorbar={"title": "Share %"},
        hovertemplate=(
            "Target: %{y}<br>"
            "Input group: %{x}<br>"
            "Mean reliance share: %{z:.3f}%"
            "<extra></extra>"
        ),
    )
)

fig = apply_branding(fig)

fig.update_layout(
    title="Transformer Predictive Reliance by Forecast Target",
    xaxis_title="",
    yaxis_title="",
    xaxis_tickangle=-25,
)

fig.show()

**Findings.** Recent mobility history is the dominant information source for **every target**, but the degree of dependence differs substantially. FHVHV trips is the most history-driven at **97.2%**, followed by Subway ridership at **93.4%** and Taxi trips at **91.4%**.

The two speed targets use supporting context more heavily. Taxi average speed assigns about **80.1%** of measured reliance to recent history, with **11.8%** coming from annual/calendar timing and **4.8%** from multimodal/connected context. FHVHV average speed is the least history-dominated target at **70.1%** and shows the broadest contextual mix: **15.5% calendar**, **7.8% multimodal/connected**, **3.3% spatial/network**, and **2.7% Taxi Zone identity**.

Taxi trips shows a smaller but visible spatial component at **3.4%**, while weekly-STL reliance peaks at only about **1.2%** for Taxi trips and FHVHV trips. Weather remains negligible across all five targets, never exceeding **0.4%** of measured reliance.

The congestion-pricing regime remains non-identifiable on these latest scoring surfaces and should therefore remain blank/N/A rather than being interpreted as zero reliance.

**4.5.2.7.3F — Which recent-history signals drive the Transformer?**

The grouped analysis establishes a striking result: **Recent mobility history dominates the finalized Transformer**, accounting for 86.454% of measured predictive reliance on average and ranking first in all 15 Target × Horizon jobs.

That result is important, but it is still too broad to tell us **what inside the history window is doing the work**.

The complete sequence input contains **70 historical Dayparts × 10 channels = 700 model-facing cells**: five mobility-value histories and their five matching observation masks. A large permutation effect for that entire block does **not** mean that every historical feature is individually important, and dividing 86.454% by 700 would not produce a meaningful feature importance. Correlated lags can substitute for one another, several signals can work together, and perturbing the entire sequence deliberately removes much more information than perturbing one small part of it.

We therefore repeat the same inference-only test at two finer and more comparable levels:

1. **Source-metric history** — scramble one mobility measure across its complete 70-position history while keeping its observation mask attached. Each candidate moves the same **70 values + 70 masks = 140 model-facing cells**.
2. **Metric × historical position** — scramble one mobility value at one historical position together with its mask. Every candidate moves exactly **2 model-facing cells**, producing **5 mobility measures × 70 positions = 350 comparable history pairs**.

**Zero Dayparts back is the forecast origin; 69 Dayparts back is the oldest position in the two-week sequence.**

The experiment restores the same latest validated fitted model for each of the 15 final jobs, recreates the same bounded scoring surface used by grouped reliance, and applies the same row permutation to every candidate within a Job × Repeat. This makes differences within a repeat attributable to **which part of history was disrupted**, rather than to a different random shuffle.

Because the Transformer creates overlapping patches internally, perturbation occurs **before patch construction**. A historical position therefore flows through every patch that normally contains it; we are measuring reliance on the original time-series signal rather than inventing a separate artificial patch-level feature.

These finer scores are normalized **within each analysis level**. They reveal where recent-history dependence is concentrated, but they are **not expected to add back to the broad 86.454% Recent mobility history result**. Neural interactions and correlated lags make permutation effects non-additive.

As throughout Section 7.3, this is **predictive reliance, not causality**. No Transformer is retrained.

In [89]:
# ---------------------------------------------------------------------
# 4.5.2.7.3F — Run hierarchical recent-history perturbation
# ---------------------------------------------------------------------

TRANSFORMER_HISTORY_DETAIL_CONTRACT_ID = "transformer_recent_history_detail_v1"

TRANSFORMER_HISTORY_DETAIL_ROOT = (
    INTERMEDIATE_DIR / "tr_recent_history_detail_v1"
)

TRANSFORMER_HISTORY_DETAIL_RESULTS_PATH = (
    TRANSFORMER_HISTORY_DETAIL_ROOT / "results.parquet"
)

RUN_TRANSFORMER_HISTORY_DETAIL = False
RESET_TRANSFORMER_HISTORY_DETAIL = False

TRANSFORMER_HISTORY_DETAIL_REPEATS = TRANSFORMER_RELIANCE_REPEATS
TRANSFORMER_HISTORY_DETAIL_MAX_ROWS = TRANSFORMER_RELIANCE_MAX_ROWS

history_lookbacks = sorted(
    transformer_full_jobs_df["lookback_steps"]
    .astype(int)
    .unique()
    .tolist()
)

if history_lookbacks != [70]:
    raise RuntimeError(
        "Fine-grained Transformer history reliance expects the frozen "
        f"70-step representation; found {history_lookbacks}."
    )

TRANSFORMER_HISTORY_STEPS = history_lookbacks[0]
TRANSFORMER_HISTORY_VALUE_CHANNELS = len(FORECAST_TARGETS)

SOURCE_HISTORY_UNITS = TRANSFORMER_HISTORY_VALUE_CHANNELS
POSITION_HISTORY_UNITS = (
    TRANSFORMER_HISTORY_VALUE_CHANNELS
    * TRANSFORMER_HISTORY_STEPS
)
HISTORY_UNITS_PER_REPEAT = (
    SOURCE_HISTORY_UNITS
    + POSITION_HISTORY_UNITS
)

EXPECTED_HISTORY_DETAIL_ROWS = (
    len(transformer_latest_state_registry_df)
    * TRANSFORMER_HISTORY_DETAIL_REPEATS
    * HISTORY_UNITS_PER_REPEAT
)

print(
    "Recent-history detail contract | "
    f"jobs={len(transformer_latest_state_registry_df)} | "
    f"history steps={TRANSFORMER_HISTORY_STEPS} | "
    f"source histories={SOURCE_HISTORY_UNITS} | "
    f"Metric × Position pairs={POSITION_HISTORY_UNITS} | "
    f"repeats={TRANSFORMER_HISTORY_DETAIL_REPEATS} | "
    f"planned perturbations={EXPECTED_HISTORY_DETAIL_ROWS:,}"
)


# ---------------------------------------------------------------------
# Deterministic Job × Repeat permutations
# ---------------------------------------------------------------------

def transformer_history_detail_seed(job_id, repeat):
    """Create one repeat-level permutation shared by every history candidate."""
    seed_text = (
        f"{TRANSFORMER_HISTORY_DETAIL_CONTRACT_ID}|"
        f"{job_id}|{repeat}|{GLOBAL_SEED}"
    )

    digest = hashlib.sha256(
        seed_text.encode("utf-8")
    ).digest()

    return int.from_bytes(
        digest[:4],
        "big",
    ) % (2**31 - 1)


def transformer_history_detail_checkpoint_path(job_id, repeat):
    """Return one durable checkpoint for a completed Job × Repeat block."""
    token = hashlib.sha1(
        f"{job_id}|{repeat}".encode("utf-8")
    ).hexdigest()[:16]

    return (
        TRANSFORMER_HISTORY_DETAIL_ROOT
        / f"job_repeat_{token}.parquet"
    )


# ---------------------------------------------------------------------
# Build equal-grain history candidates from the saved channel order
# ---------------------------------------------------------------------

def build_transformer_history_units(state):
    """
    Define source-history and Metric × Position perturbations.

    The saved sequence order is target-specific, so channel indices must come
    from the fitted state rather than assuming one universal five-channel order.
    """
    sequence_value_channels = list(
        state["sequence_value_channels"]
    )

    if (
        len(sequence_value_channels) != TRANSFORMER_HISTORY_VALUE_CHANNELS
        or set(sequence_value_channels) != set(FORECAST_TARGETS)
    ):
        raise RuntimeError(
            "Saved Transformer state does not contain the expected five "
            "mobility-history value channels."
        )

    units = []

    for source_metric in FORECAST_TARGETS:
        value_channel = sequence_value_channels.index(
            source_metric
        )

        # transform_sequence() stores all standardized values first and then
        # their five masks in the same order.
        mask_channel = (
            value_channel
            + TRANSFORMER_HISTORY_VALUE_CHANNELS
        )

        units.append({
            "analysis_level": "source_metric_history",
            "unit_id": f"history__{source_metric}",
            "source_metric": source_metric,
            "history_position_index": np.nan,
            "dayparts_back": np.nan,
            "value_channel": value_channel,
            "mask_channel": mask_channel,
            "model_facing_cells": (
                2 * TRANSFORMER_HISTORY_STEPS
            ),
        })

        for position in range(
            TRANSFORMER_HISTORY_STEPS
        ):
            dayparts_back = (
                TRANSFORMER_HISTORY_STEPS
                - 1
                - position
            )

            units.append({
                "analysis_level": "metric_position",
                "unit_id": (
                    f"{source_metric}__"
                    f"{dayparts_back:02d}_dayparts_back"
                ),
                "source_metric": source_metric,
                "history_position_index": position,
                "dayparts_back": dayparts_back,
                "value_channel": value_channel,
                "mask_channel": mask_channel,
                "model_facing_cells": 2,
            })

    if len(units) != HISTORY_UNITS_PER_REPEAT:
        raise RuntimeError(
            "History-detail unit construction does not match the frozen "
            f"{HISTORY_UNITS_PER_REPEAT}-unit contract."
        )

    return units


def history_unit_values(sequence, unit):
    """Extract exactly the model-facing values moved by one perturbation."""
    channels = [
        int(unit["value_channel"]),
        int(unit["mask_channel"]),
    ]

    if unit["analysis_level"] == "source_metric_history":
        return sequence[:, :, channels]

    position = int(
        unit["history_position_index"]
    )

    return sequence[:, position, channels]


def apply_history_permutation(
    working_sequence,
    baseline_sequence,
    unit,
    permutation,
):
    """
    Replace one history unit with values from the repeat's donor rows.

    Value and observation-mask information move together so the perturbation
    breaks row correspondence without inventing impossible value/mask pairs.
    """
    channels = [
        int(unit["value_channel"]),
        int(unit["mask_channel"]),
    ]

    if unit["analysis_level"] == "source_metric_history":
        working_sequence[:, :, channels] = (
            baseline_sequence[permutation][:, :, channels]
        )

    else:
        position = int(
            unit["history_position_index"]
        )

        working_sequence[:, position, channels] = (
            baseline_sequence[
                permutation,
                position,
            ][:, channels]
        )


def restore_history_unit(
    working_sequence,
    baseline_sequence,
    unit,
):
    """Restore the perturbed slice before testing the next candidate."""
    channels = [
        int(unit["value_channel"]),
        int(unit["mask_channel"]),
    ]

    if unit["analysis_level"] == "source_metric_history":
        working_sequence[:, :, channels] = (
            baseline_sequence[:, :, channels]
        )

    else:
        position = int(
            unit["history_position_index"]
        )

        working_sequence[:, position, channels] = (
            baseline_sequence[:, position, channels]
        )


# ---------------------------------------------------------------------
# Recovery validates a complete Job × Repeat before trusting it
# ---------------------------------------------------------------------

def recover_transformer_history_detail_repeat(job_id, repeat):
    """Recover one complete current-contract Job × Repeat checkpoint."""
    path = transformer_history_detail_checkpoint_path(
        job_id,
        repeat,
    )

    if not path.exists():
        return None

    try:
        frame = pd.read_parquet(
            path
        )

        if (
            len(frame) != HISTORY_UNITS_PER_REPEAT
            or not frame["contract_id"].eq(
                TRANSFORMER_HISTORY_DETAIL_CONTRACT_ID
            ).all()
            or not frame["job_id"].eq(job_id).all()
            or not frame["repeat"].eq(repeat).all()
            or frame["retrained"].any()
        ):
            return None

        source_rows = frame[
            "analysis_level"
        ].eq(
            "source_metric_history"
        ).sum()

        position_rows = frame[
            "analysis_level"
        ].eq(
            "metric_position"
        ).sum()

        if (
            source_rows != SOURCE_HISTORY_UNITS
            or position_rows != POSITION_HISTORY_UNITS
            or frame["unit_id"].duplicated().any()
        ):
            return None

        return frame

    except (
        OSError,
        ValueError,
        KeyError,
        TypeError,
    ):
        return None


def validate_transformer_history_detail(frame):
    """Fail loudly unless the complete fine-grained attribution surface exists."""
    required_columns = {
        "contract_id",
        "job_id",
        "track",
        "metric",
        "horizon",
        "refit_step",
        "temporal_window",
        "analysis_level",
        "unit_id",
        "source_metric",
        "history_position_index",
        "dayparts_back",
        "model_facing_cells",
        "repeat",
        "evaluation_rows",
        "permutation_identifiable",
        "baseline_raw_mae",
        "baseline_final_mae",
        "perturbed_raw_mae",
        "perturbed_final_mae",
        "raw_mae_damage",
        "final_mae_damage",
        "raw_mae_damage_pct",
        "final_mae_damage_pct",
        "retrained",
    }

    missing_columns = sorted(
        required_columns - set(frame.columns)
    )

    if missing_columns:
        raise RuntimeError(
            "Transformer history-detail output is missing columns: "
            f"{missing_columns}"
        )

    if len(frame) != EXPECTED_HISTORY_DETAIL_ROWS:
        raise RuntimeError(
            "Transformer history-detail output is incomplete: "
            f"expected {EXPECTED_HISTORY_DETAIL_ROWS:,} rows, "
            f"found {len(frame):,}."
        )

    if not frame["contract_id"].eq(
        TRANSFORMER_HISTORY_DETAIL_CONTRACT_ID
    ).all():
        raise RuntimeError(
            "History-detail output belongs to another contract."
        )

    if frame["retrained"].any():
        raise RuntimeError(
            "Recent-history attribution must remain inference-only."
        )

    duplicate_key = [
        "job_id",
        "repeat",
        "unit_id",
    ]

    if frame.duplicated(
        duplicate_key
    ).any():
        raise RuntimeError(
            "History-detail output contains duplicate Job × Repeat × Unit rows."
        )

    expected_jobs = set(
        transformer_latest_state_registry_df[
            "job_id"
        ]
    )

    if set(frame["job_id"]) != expected_jobs:
        raise RuntimeError(
            "History-detail output does not cover the exact 15-job roster."
        )

    counts = frame.groupby(
        ["job_id", "repeat"],
        observed=True,
    ).size()

    if not counts.eq(
        HISTORY_UNITS_PER_REPEAT
    ).all():
        raise RuntimeError(
            "At least one Job × Repeat is missing a history perturbation."
        )

    identifiable = frame[
        "permutation_identifiable"
    ]

    finite_columns = [
        "baseline_raw_mae",
        "baseline_final_mae",
    ]

    if not np.isfinite(
        frame[finite_columns].to_numpy(
            dtype=float
        )
    ).all():
        raise RuntimeError(
            "History-detail baseline MAE contains a non-finite value."
        )

    identifiable_values = frame.loc[
        identifiable,
        [
            "perturbed_raw_mae",
            "perturbed_final_mae",
            "raw_mae_damage",
            "final_mae_damage",
            "raw_mae_damage_pct",
            "final_mae_damage_pct",
        ],
    ].to_numpy(
        dtype=float
    )

    if (
        len(identifiable_values)
        and not np.isfinite(
            identifiable_values
        ).all()
    ):
        raise RuntimeError(
            "An identifiable history perturbation has non-finite scoring output."
        )


# ---------------------------------------------------------------------
# One inference-only Job × Repeat
# ---------------------------------------------------------------------

def run_transformer_history_detail_repeat(
    state_row,
    job,
    refit_row,
    state,
    model,
    preprocessor,
    sample,
    baseline_raw_mae,
    baseline_final_mae,
    repeat,
):
    """Measure all 355 history candidates under one shared row permutation."""
    if baseline_raw_mae <= 0 or baseline_final_mae <= 0:
        raise RuntimeError(
            f"{state_row.job_id}: history attribution requires positive baseline MAE."
        )

    units = build_transformer_history_units(
        state
    )

    baseline_sequence = sample[
        "sequence"
    ]

    working_sequence = baseline_sequence.copy()

    rng = np.random.default_rng(
        transformer_history_detail_seed(
            state_row.job_id,
            repeat,
        )
    )

    permutation = rng.permutation(
        len(sample["target"])
    )

    records = []

    for unit in units:
        model_facing_values = history_unit_values(
            baseline_sequence,
            unit,
        )

        identifiable = model_facing_group_varies(
            model_facing_values
        )

        record = {
            "contract_id":
                TRANSFORMER_HISTORY_DETAIL_CONTRACT_ID,
            "job_id":
                state_row.job_id,
            "track":
                state_row.track,
            "metric":
                state_row.metric,
            "horizon":
                int(state_row.horizon),
            "refit_step":
                int(state_row.refit_step),
            "temporal_window":
                state_row.temporal_window,
            "analysis_level":
                unit["analysis_level"],
            "unit_id":
                unit["unit_id"],
            "source_metric":
                unit["source_metric"],
            "history_position_index":
                unit["history_position_index"],
            "dayparts_back":
                unit["dayparts_back"],
            "model_facing_cells":
                int(unit["model_facing_cells"]),
            "repeat":
                int(repeat),
            "evaluation_rows":
                len(sample["target"]),
            "permutation_identifiable":
                identifiable,
            "baseline_raw_mae":
                baseline_raw_mae,
            "baseline_final_mae":
                baseline_final_mae,
            "perturbed_raw_mae":
                np.nan,
            "perturbed_final_mae":
                np.nan,
            "raw_mae_damage":
                np.nan,
            "final_mae_damage":
                np.nan,
            "raw_mae_damage_pct":
                np.nan,
            "final_mae_damage_pct":
                np.nan,
            "retrained":
                False,
        }

        if identifiable:
            apply_history_permutation(
                working_sequence,
                baseline_sequence,
                unit,
                permutation,
            )

            try:
                perturbed_raw, perturbed_final = (
                    predict_prepared_transformer(
                        model,
                        preprocessor,
                        working_sequence,
                        sample["context"],
                        sample["zone_index"],
                    )
                )

            finally:
                # Always restore the baseline input, even if scoring raises,
                # so one failed candidate cannot contaminate the next one.
                restore_history_unit(
                    working_sequence,
                    baseline_sequence,
                    unit,
                )

            perturbed_raw_mae = regression_metrics(
                sample["target"],
                perturbed_raw,
            )["mae"]

            perturbed_final_mae = regression_metrics(
                sample["target"],
                perturbed_final,
            )["mae"]

            raw_damage = (
                perturbed_raw_mae
                - baseline_raw_mae
            )

            final_damage = (
                perturbed_final_mae
                - baseline_final_mae
            )

            record.update({
                "perturbed_raw_mae":
                    perturbed_raw_mae,
                "perturbed_final_mae":
                    perturbed_final_mae,
                "raw_mae_damage":
                    raw_damage,
                "final_mae_damage":
                    final_damage,
                "raw_mae_damage_pct":
                    100.0
                    * raw_damage
                    / baseline_raw_mae,
                "final_mae_damage_pct":
                    100.0
                    * final_damage
                    / baseline_final_mae,
            })

        records.append(
            record
        )

    return pd.DataFrame(
        records
    )


# ---------------------------------------------------------------------
# Recovery-first execution
# ---------------------------------------------------------------------

TRANSFORMER_HISTORY_DETAIL_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

if RESET_TRANSFORMER_HISTORY_DETAIL:
    reset_experiment_artifacts(
        TRANSFORMER_HISTORY_DETAIL_ROOT,
        consolidated_paths=[
            TRANSFORMER_HISTORY_DETAIL_RESULTS_PATH
        ],
    )


# A validated consolidated artifact is the cheapest restart path.
transformer_history_detail_repeat_df = None

if (
    not RESET_TRANSFORMER_HISTORY_DETAIL
    and TRANSFORMER_HISTORY_DETAIL_RESULTS_PATH.exists()
):
    try:
        recovered_full_df = pd.read_parquet(
            TRANSFORMER_HISTORY_DETAIL_RESULTS_PATH
        )

        validate_transformer_history_detail(
            recovered_full_df
        )

        transformer_history_detail_repeat_df = (
            recovered_full_df
            .sort_values(
                [
                    "metric",
                    "horizon",
                    "repeat",
                    "analysis_level",
                    "source_metric",
                    "dayparts_back",
                ],
                na_position="first",
            )
            .reset_index(drop=True)
        )

        print(
            "Recent-history detail recovered from consolidated result | "
            f"{len(transformer_history_detail_repeat_df):,}/"
            f"{EXPECTED_HISTORY_DETAIL_ROWS:,} comparisons"
        )

    except (
        OSError,
        ValueError,
        KeyError,
        TypeError,
        RuntimeError,
    ):
        transformer_history_detail_repeat_df = None


if transformer_history_detail_repeat_df is None:
    recovered_frames = []
    missing_keys = []

    for state_row in transformer_latest_state_registry_df.itertuples(
        index=False
    ):
        for repeat in range(
            1,
            TRANSFORMER_HISTORY_DETAIL_REPEATS + 1,
        ):
            recovered = recover_transformer_history_detail_repeat(
                state_row.job_id,
                repeat,
            )

            if recovered is None:
                missing_keys.append(
                    (state_row.job_id, repeat)
                )
            else:
                recovered_frames.append(
                    recovered
                )

    completed_blocks = (
        len(transformer_latest_state_registry_df)
        * TRANSFORMER_HISTORY_DETAIL_REPEATS
        - len(missing_keys)
    )

    print(
        "Recent-history detail restart state | "
        f"{completed_blocks}/"
        f"{len(transformer_latest_state_registry_df) * TRANSFORMER_HISTORY_DETAIL_REPEATS} "
        f"Job × Repeat blocks complete | "
        f"{len(missing_keys)} unfinished"
    )

    if missing_keys and not RUN_TRANSFORMER_HISTORY_DETAIL:
        raise RuntimeError(
            "Fine-grained Transformer history attribution is incomplete. "
            "Set RUN_TRANSFORMER_HISTORY_DETAIL=True to execute only the "
            f"{len(missing_keys)} missing Job × Repeat blocks."
        )


    # Restore each fitted model only when it still owns unfinished repeats.
    for job_number, state_row in enumerate(
        transformer_latest_state_registry_df.itertuples(
            index=False
        ),
        start=1,
    ):
        missing_repeats = [
            repeat
            for job_id, repeat in missing_keys
            if job_id == state_row.job_id
        ]

        if not missing_repeats:
            continue

        job_match_df = transformer_full_jobs_df.loc[
            transformer_full_jobs_df["job_id"].eq(
                state_row.job_id
            )
        ]

        refit_match_df = transformer_full_schedule_df.loc[
            transformer_full_schedule_df["job_id"].eq(
                state_row.job_id
            )
            & transformer_full_schedule_df["refit_step"].astype(int).eq(
                int(state_row.refit_step)
            )
        ]

        if len(job_match_df) != 1 or len(refit_match_df) != 1:
            raise RuntimeError(
                f"{state_row.job_id}: production Job or latest Refit is not unique."
            )

        job = job_match_df.iloc[0].to_dict()
        refit_row = refit_match_df.iloc[0].to_dict()

        state, model, preprocessor = restore_transformer_model(
            state_row.state_path
        )

        logical_key = state["logical_key"]

        for column in TRANSFORMER_LOGICAL_KEY_COLUMNS:
            if str(logical_key[column]) != str(
                refit_row[column]
            ):
                raise RuntimeError(
                    f"{state_row.job_id}: saved state disagrees with {column}."
                )

        sample = prepare_transformer_reliance_sample(
            job,
            refit_row,
            state,
            preprocessor,
        )

        baseline_raw, baseline_final = predict_prepared_transformer(
            model,
            preprocessor,
            sample["sequence"],
            sample["context"],
            sample["zone_index"],
        )

        baseline_raw_mae = regression_metrics(
            sample["target"],
            baseline_raw,
        )["mae"]

        baseline_final_mae = regression_metrics(
            sample["target"],
            baseline_final,
        )["mae"]

        for repeat in missing_repeats:
            completed = run_transformer_history_detail_repeat(
                state_row,
                job,
                refit_row,
                state,
                model,
                preprocessor,
                sample,
                baseline_raw_mae,
                baseline_final_mae,
                repeat,
            )

            # Atomic promotion makes the presence of this file the durable
            # success checkpoint for exactly one Job × Repeat block.
            atomic_write_parquet(
                completed,
                transformer_history_detail_checkpoint_path(
                    state_row.job_id,
                    repeat,
                ),
            )

            print(
                f"Recent-history detail · job {job_number}/15 · "
                f"{TARGET_LABELS[state_row.metric]} · "
                f"h={int(state_row.horizon)} · "
                f"repeat {repeat}/{TRANSFORMER_HISTORY_DETAIL_REPEATS} complete"
            )

        del (
            state,
            model,
            preprocessor,
            sample,
            baseline_raw,
            baseline_final,
        )

        clear_device_memory()


    # Re-read durable files rather than trusting this kernel's in-memory work.
    final_frames = []

    for state_row in transformer_latest_state_registry_df.itertuples(
        index=False
    ):
        for repeat in range(
            1,
            TRANSFORMER_HISTORY_DETAIL_REPEATS + 1,
        ):
            recovered = recover_transformer_history_detail_repeat(
                state_row.job_id,
                repeat,
            )

            if recovered is None:
                raise RuntimeError(
                    "Recent-history detail remains incomplete after execution: "
                    f"{state_row.job_id} · repeat={repeat}."
                )

            final_frames.append(
                recovered
            )

    transformer_history_detail_repeat_df = pd.concat(
        final_frames,
        ignore_index=True,
    )

    validate_transformer_history_detail(
        transformer_history_detail_repeat_df
    )

    atomic_write_parquet(
        transformer_history_detail_repeat_df,
        TRANSFORMER_HISTORY_DETAIL_RESULTS_PATH,
    )


print(
    "Recent-history detail complete | "
    f"jobs={transformer_history_detail_repeat_df['job_id'].nunique()} | "
    f"repeat-level perturbations={len(transformer_history_detail_repeat_df):,} | "
    "retraining=False"
)

Recent-history detail contract | jobs=15 | history steps=70 | source histories=5 | Metric × Position pairs=350 | repeats=3 | planned perturbations=15,975
Recent-history detail restart state | 0/45 Job × Repeat blocks complete | 45 unfinished
/root/venv/lib/python3.11/site-packages/torch/nn/modules/transformer.py:282: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(f"enable_nested_tensor is True, but self.use_nested_tensor is False because {why_not_sparsity_fast_path}")
Recent-history detail · job 1/15 · FHVHV average speed · h=1 · repeat 1/3 complete
Recent-history detail · job 1/15 · FHVHV average speed · h=1 · repeat 2/3 complete
Recent-history detail · job 1/15 · FHVHV average speed · h=1 · repeat 3/3 complete
/root/venv/lib/python3.11/site-packages/torch/nn/modules/transformer.py:282: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first

**Findings.** The drill-down completed across the full **15-job Transformer roster**, with **3 repeated perturbations for each of 355 recent-history candidates per job**, producing **15,975 inference-only comparisons**. No model was retrained, so the following results describe how the already-fitted finalists respond when specific pieces of their recent-history input are disrupted.

**4.5.2.7.3G — Where inside recent history is predictive reliance concentrated?**

The repeat-level perturbations are now converted into **within-history reliance shares** using the same logic as the broader grouped analysis: only positive final-output MAE damage contributes to the normalized share.

Normalization is performed separately for the two analysis levels:

- the five complete **source-metric histories** sum to 100% within each job when positive damage exists;
- the 350 equal-grain **Metric × Position pairs** sum to 100% within each job when positive damage exists.

This separation matters. The source-level view asks **which mobility streams matter?** The pair-level view asks **which specific historical signals matter?** Neither should be added back to the broad Recent mobility history percentage from Section 7.3D.

In [103]:
# ---------------------------------------------------------------------
# 4.5.2.7.3G — Normalize fine-grained history reliance within each level
# ---------------------------------------------------------------------

history_detail_keys = [
    "job_id", "track", "metric", "horizon", "analysis_level", "unit_id",
    "source_metric", "history_position_index", "dayparts_back", "model_facing_cells",
]

# Average the three repeat-level perturbations before comparing units. This
# leaves one row per Job × Analysis level × History unit.
transformer_history_detail_job_df = (
    transformer_history_detail_repeat_df
    .groupby(history_detail_keys, as_index=False, observed=True, dropna=False)
    .agg(
        permutation_identifiable=("permutation_identifiable", "max"),
        mean_raw_mae_damage=("raw_mae_damage", "mean"),
        mean_final_mae_damage=("final_mae_damage", "mean"),
        mean_final_mae_damage_pct=("final_mae_damage_pct", "mean"),
    )
)

# Negative permutation damage is not evidence of "negative importance."
# Only positive measured damage contributes to the normalized reliance share.
identifiable = transformer_history_detail_job_df["permutation_identifiable"]
positive_damage = transformer_history_detail_job_df["mean_final_mae_damage"].clip(lower=0)

transformer_history_detail_job_df["positive_final_damage"] = np.where(
    identifiable, positive_damage, np.nan
)

# Normalize source-history and Metric × Position analyses separately. Their
# permutation effects are non-additive, so neither level should be forced to
# reproduce the broad Recent mobility history percentage from Section 7.3D.
level_positive_totals = (
    transformer_history_detail_job_df
    .groupby(["job_id", "analysis_level"], observed=True)["positive_final_damage"]
    .transform("sum")
)

transformer_history_detail_job_df["within_history_reliance_share_pct"] = np.where(
    level_positive_totals.gt(0),
    100.0
    * transformer_history_detail_job_df["positive_final_damage"]
    / level_positive_totals,
    np.nan,
)

share_audit_df = (
    transformer_history_detail_job_df
    .groupby(["job_id", "analysis_level"], as_index=False, observed=True)
    .agg(
        positive_damage_total=("positive_final_damage", "sum"),
        reliance_share_total=("within_history_reliance_share_pct", "sum"),
    )
)

normalizable = share_audit_df["positive_damage_total"].gt(0)

if not np.allclose(
    share_audit_df.loc[normalizable, "reliance_share_total"],
    100.0,
    atol=1e-8,
):
    raise RuntimeError(
        "Fine-grained history reliance shares do not normalize to 100%."
    )

print(
    "History-detail normalization verified | "
    f"job-level units={len(transformer_history_detail_job_df):,} | "
    "source-history and Metric × Position levels normalized separately"
)

History-detail normalization verified | job-level units=5,325 | source-history and Metric × Position levels normalized separately


**Findings.** The fine-grained history analysis reduces the completed repeat-level experiment to **5,325 Job × History-unit results** and successfully normalizes both analysis levels. Each job's five complete mobility histories sum to 100% within the source-level view, while its 350 Metric × Position pairs separately sum to 100% within the position-level view.

These percentages therefore answer **where the Transformer's recent-history reliance is concentrated**, not how much of the model's total predictive power each feature contributes. The two views are intentionally kept separate because correlated lags and neural interactions make permutation effects non-additive.

**4.5.2.7.3H — Which mobility histories matter most?**

The first drill-down keeps each mobility stream's full two-week history intact. This asks whether the Transformer mainly depends on the target's **own** recent history or whether histories from other transportation modes contribute substantial predictive information.

Because every source candidate contains the same 70 values and 70 matching masks, the comparison is much fairer than comparing the original 700-cell history block with much smaller context groups.

In [105]:
# ---------------------------------------------------------------------
# 4.5.2.7.3H — Compare complete mobility-history sources
# ---------------------------------------------------------------------

metric_history_df = transformer_history_detail_job_df.loc[
    transformer_history_detail_job_df["analysis_level"].eq("source_metric_history")
].copy()

# Mean share gives the overall ranking; median share is retained because a
# source can matter enormously for a few jobs while contributing little to most.
source_history_summary_df = (
    metric_history_df
    .groupby("source_metric", as_index=False, observed=True)
    .agg(
        identifiable_jobs=("permutation_identifiable", "sum"),
        mean_reliance_share_pct=("within_history_reliance_share_pct", "mean"),
        median_reliance_share_pct=("within_history_reliance_share_pct", "median"),
        mean_mae_damage_pct=("mean_final_mae_damage_pct", "mean"),
    )
    .sort_values("mean_reliance_share_pct", ascending=False)
    .reset_index(drop=True)
)

source_history_display_df = (
    source_history_summary_df
    .assign(**{
        "History source": lambda frame: frame["source_metric"].map(TARGET_LABELS)
    })
    .rename(columns={
        "identifiable_jobs": "Identifiable jobs",
        "mean_reliance_share_pct": "Mean within-history share %",
        "median_reliance_share_pct": "Median within-history share %",
        "mean_mae_damage_pct": "Mean MAE damage %",
    })
)

display(
    source_history_display_df[
        [
            "History source", "Identifiable jobs", "Mean within-history share %",
            "Median within-history share %", "Mean MAE damage %",
        ]
    ].round(3)
)

bar_df = source_history_summary_df.sort_values("mean_reliance_share_pct")

fig = go.Figure(
    go.Bar(
        x=bar_df["mean_reliance_share_pct"],
        y=bar_df["source_metric"].map(TARGET_LABELS),
        orientation="h",
        marker_color=[
            BRAND_CLUSTER_PALETTE[FORECAST_TARGETS.index(metric)]
            for metric in bar_df["source_metric"]
        ],
        text=bar_df["mean_reliance_share_pct"].map(lambda value: f"{value:.1f}%"),
        textposition="outside",
        hovertemplate=(
            "History source: %{y}<br>"
            "Mean within-history reliance: %{x:.3f}%"
            "<extra></extra>"
        ),
    )
)

fig = apply_branding(fig)
fig.update_layout(
    title="Which Mobility Histories Does the Transformer Rely On?",
    xaxis_title="Average within-history reliance share (%)",
    yaxis_title="",
    showlegend=False,
)
fig.show()

,History source,Identifiable jobs,Mean within-history share %,Median within-history share %,Mean MAE damage %
0,FHVHV average speed,15,24.831,3.337,53.006
1,FHVHV trips,15,21.726,3.811,79.630
2,Subway ridership,15,19.196,0.104,173.285
3,Taxi trips,15,18.373,0.703,103.007
4,Taxi average speed,15,15.873,2.063,14.240


**Findings.** Averaged across all 15 final jobs, **FHVHV average-speed history carries the largest within-history reliance share at 24.831%**, followed by FHVHV trips at **21.726%**, Subway ridership at **19.196%**, Taxi trips at **18.373%**, and Taxi average speed at **15.873%**.

The much smaller median shares—only **0.104% to 3.811%** depending on source—show why the overall ranking needs care. A history stream can be extremely important for the jobs that use it while contributing very little elsewhere. In other words, this is not evidence that every Transformer broadly prefers FHVHV history; the target-specific heatmap below shows that the models are actually much more specialized.

**4.5.2.7.3I — Does each forecast mostly use its own history?**

An overall source ranking can hide very different behavior by forecasting target. This matrix therefore asks a more direct question:

**For each thing being forecast, which mobility history does its Transformer actually depend on?**

The diagonal represents **own-history reliance**. Off-diagonal cells reveal **cross-modal history** that the fitted Transformer finds useful.

In [107]:
# ---------------------------------------------------------------------
# 4.5.2.7.3I — Compare history sources within each forecast target
# ---------------------------------------------------------------------

target_source_history_df = (
    metric_history_df
    .groupby(["metric", "source_metric"], as_index=False, observed=True)[
        "within_history_reliance_share_pct"
    ]
    .mean()
    .pivot(
        index="source_metric",
        columns="metric",
        values="within_history_reliance_share_pct",
    )
    .reindex(index=FORECAST_TARGETS, columns=FORECAST_TARGETS)
)

if target_source_history_df.isna().any().any():
    raise RuntimeError(
        "Target × History Source matrix is missing an expected cell."
    )

source_target_values = target_source_history_df.to_numpy(dtype=float)
source_target_text = np.vectorize(
    lambda value: f"{value:.1f}%"
)(source_target_values)

target_labels = [TARGET_LABELS[metric] for metric in FORECAST_TARGETS]

# The diagonal is own-history reliance; off-diagonal cells reveal cross-modal
# history that improves a different target's forecast.
fig = go.Figure(
    go.Heatmap(
        z=source_target_values,
        x=target_labels,
        y=target_labels,
        text=source_target_text,
        texttemplate="%{text}",
        colorscale=BRAND_SEQUENTIAL_HEATMAP_SCALE,
        zmin=0,
        colorbar={"title": "Within-history<br>reliance (%)"},
        hovertemplate=(
            "Forecast target: %{x}<br>"
            "History source: %{y}<br>"
            "Reliance share: %{z:.3f}%"
            "<extra></extra>"
        ),
    )
)

fig = apply_branding(fig)
fig.update_layout(
    title="Own-History and Cross-Modal Reliance in the Final Transformer",
    xaxis_title="Forecast target",
    yaxis_title="History source",
)
fig.show()

**Findings.** The target-level view makes the pattern much clearer: **four of the five Transformers rely overwhelmingly on their own recent history**. Own-history reliance reaches **89.9% for Taxi trips, 96.4% for FHVHV trips, 95.2% for FHVHV average speed, and 95.0% for Subway ridership**.

**Taxi average speed is the important exception.** Its own history still leads at **72.2%**, but another **23.0% comes from FHVHV average-speed history**. That is the only large cross-modal relationship in the matrix; every other off-diagonal contribution is 4.0% or less.

The practical interpretation is therefore not that the Transformer indiscriminately mixes every transportation mode. It mostly follows the recent history of the thing being forecast, while the two surface-speed measures share substantial additional information with one another.

**4.5.2.7.3J — Which points in recent history matter most?**

The source-level analysis still moves an entire 70-position history at once. The next view uses the finest common perturbation grain: **one mobility value at one historical position together with its observation mask**.

Every candidate therefore moves exactly two model-facing cells. This lets us ask whether the Transformer's history dependence is concentrated near the forecast origin, around recurring weekly positions, or spread across the full two-week window.

The x-axis runs chronologically from the oldest history at the left to the forecast origin at the right.

In [109]:
# ---------------------------------------------------------------------
# 4.5.2.7.3J — Map reliance across source metric × historical position
# ---------------------------------------------------------------------

metric_position_df = transformer_history_detail_job_df.loc[
    transformer_history_detail_job_df["analysis_level"].eq("metric_position")
].copy()

position_summary_df = (
    metric_position_df
    .groupby(["source_metric", "dayparts_back"], as_index=False, observed=True)[
        "within_history_reliance_share_pct"
    ]
    .mean()
)

# Display history chronologically: oldest input at left, forecast origin at right.
position_order = list(range(TRANSFORMER_HISTORY_STEPS - 1, -1, -1))
position_labels = [str(position) for position in position_order]

history_position_matrix_df = (
    position_summary_df
    .pivot(
        index="source_metric",
        columns="dayparts_back",
        values="within_history_reliance_share_pct",
    )
    .reindex(index=FORECAST_TARGETS, columns=position_order)
)

if history_position_matrix_df.isna().any().any():
    raise RuntimeError(
        "Source × Historical Position matrix is missing an expected cell."
    )

position_values = history_position_matrix_df.to_numpy(dtype=float)

fig = go.Figure(
    go.Heatmap(
        z=position_values,
        x=position_labels,
        y=[TARGET_LABELS[metric] for metric in FORECAST_TARGETS],
        colorscale=BRAND_SEQUENTIAL_HEATMAP_SCALE,
        zmin=0,
        colorbar={"title": "Within-history<br>reliance (%)"},
        hovertemplate=(
            "History source: %{y}<br>"
            "Dayparts back: %{x}<br>"
            "Mean reliance share: %{z:.3f}%"
            "<extra></extra>"
        ),
    )
)

fig = apply_branding(fig)
fig.update_layout(
    title="Where in the Two-Week History Does Transformer Reliance Concentrate?",
    xaxis_title="Dayparts back from forecast origin",
    yaxis_title="History source",
)

major_positions = [69, 60, 50, 40, 30, 20, 10, 0]
major_labels = [str(value) for value in major_positions]
fig.update_xaxes(
    tickmode="array",
    tickvals=major_labels,
    ticktext=major_labels,
)

fig.show()

**Findings.** Reliance is **not spread smoothly across the two-week sequence**. The forecast origin itself—**0 Dayparts back**—is the strongest individual position for Taxi trips, FHVHV trips, FHVHV average speed, and Subway ridership. Its mean within-history shares reach **1.982%, 4.636%, 2.444%, and 3.220%**, respectively.

Older history still matters, but in distinct peaks rather than through a simple gradual decay. Positions around **30 and 65 Dayparts back** repeatedly stand out across several mobility streams, while nearby positions around 3–5 Dayparts back also remain prominent. Taxi average speed is more diffuse than the other sources: its strongest individual position is only about **0.513%**.

The Transformer therefore appears to use a mixture of the most recent signal and selected older temporal anchors, rather than treating all 70 historical positions equally.

**4.5.2.7.3K — Which individual history signals are strongest?**

The heatmap shows the shape of reliance across all 350 equal-grain candidates. A ranked view makes the strongest individual **Metric × Position** pairs explicit.

These values are averaged across the 15 final forecasting jobs after each job is normalized within its own 350-pair history surface, so large-unit targets such as Subway ridership cannot dominate the ranking simply because their native MAE is larger.

In [113]:
# ---------------------------------------------------------------------
# 4.5.2.7.3K — Rank the strongest Metric × Position history pairs
# ---------------------------------------------------------------------

pair_summary_df = (
    metric_position_df
    .groupby(["source_metric", "dayparts_back"], as_index=False, observed=True)
    .agg(
        mean_reliance_share_pct=("within_history_reliance_share_pct", "mean"),
        median_reliance_share_pct=("within_history_reliance_share_pct", "median"),
        identifiable_jobs=("permutation_identifiable", "sum"),
    )
    .sort_values("mean_reliance_share_pct", ascending=False)
    .reset_index(drop=True)
)

pair_summary_df["pair_label"] = (
    pair_summary_df["source_metric"].map(TARGET_LABELS)
    + " · "
    + pair_summary_df["dayparts_back"].astype(int).astype(str)
    + " back"
)

# Reverse the Top-10 subset only for plotting so the strongest signal is
# visually placed at the top of the horizontal ranking.
top_history_pairs_df = (
    pair_summary_df
    .head(10)
    .sort_values("mean_reliance_share_pct")
)

fig = go.Figure(
    go.Bar(
        x=top_history_pairs_df["mean_reliance_share_pct"],
        y=top_history_pairs_df["pair_label"],
        orientation="h",
        marker_color=BRAND_COLORS["dark_teal"],
        text=top_history_pairs_df["mean_reliance_share_pct"].map(
            lambda value: f"{value:.2f}%"
        ),
        textposition="outside",
        customdata=top_history_pairs_df[
            ["median_reliance_share_pct", "identifiable_jobs"]
        ],
        hovertemplate=(
            "%{y}<br>"
            "Mean reliance share: %{x:.3f}%<br>"
            "Median share: %{customdata[0]:.3f}%<br>"
            "Identifiable jobs: %{customdata[1]:.0f}"
            "<extra></extra>"
        ),
    )
)

fig = apply_branding(fig)
fig.update_layout(
    title="Strongest Individual Signals in the Transformer's Recent History",
    xaxis_title="Average within-history reliance share (%)",
    yaxis_title="",
    showlegend=False,
)
fig.show()

**Findings.** The Transformer's strongest individual history signals are concentrated at the **forecast origin itself**. FHVHV trips at 0 Dayparts back leads at **4.64% of average within-history reliance**, followed by Subway ridership at the origin (**3.22%**), FHVHV average speed at the origin (**2.44%**), and Taxi trips at the origin (**1.98%**). This says the model places substantial weight on the most recently observed mobility state rather than treating all points in its two-week history equally.

The dependence is not purely immediate, however. Several older positions also appear among the strongest signals, including **Subway ridership 65 Dayparts back (1.78%)**, FHVHV trips and FHVHV average speed **30 Dayparts back (1.66% and 1.63%)**, and additional signals 3 and 33 Dayparts back. The useful history is therefore spread across both the newest observations and a smaller set of older positions rather than declining smoothly with age.

These percentages are **within-history predictive-reliance shares averaged across the final jobs**. They identify which equal-grain Metric × Position inputs matter most within the history block; they should not be interpreted as causal effects or added back to the broader Recent mobility history reliance result.

**4.5.2.7.3L — Is recent-history reliance broad or concentrated in a few signals?**

The top-10 ranking identifies the strongest individual history signals, but it does not tell us how quickly the remaining reliance falls away.

Each final Transformer job exposes **350 equal-grain Metric × Position pairs: 5 mobility measures × 70 historical positions**. We therefore rank all 350 pairs separately within each job and measure how much of that job's positive fine-grained history reliance is captured by its strongest 5, 10, 25, and 50 pairs.

If a small number of pairs captures a large share of the total, the Transformer's history dependence is concentrated. If the cumulative share rises slowly, useful information is distributed more broadly across the two-week sequence.

In [117]:
# ---------------------------------------------------------------------
# 4.5.2.7.3L — Measure concentration of equal-grain history reliance
# ---------------------------------------------------------------------

CONCENTRATION_CUTOFFS = [5, 10, 25, 50]
EXPECTED_METRIC_POSITION_PAIRS = 5 * 70

# The concentration question must be answered within each final job first.
# Otherwise a few globally strong pairs could hide very different reliance
# patterns across targets and horizons.
pairs_per_job = metric_position_df.groupby("job_id", observed=True).size()

if not pairs_per_job.eq(EXPECTED_METRIC_POSITION_PAIRS).all():
    raise RuntimeError(
        "Every final Transformer job must contain exactly "
        f"{EXPECTED_METRIC_POSITION_PAIRS} equal-grain Metric × Position pairs. "
        f"Observed range: {pairs_per_job.min()}–{pairs_per_job.max()}."
    )

if metric_position_df["within_history_reliance_share_pct"].isna().any():
    raise RuntimeError(
        "Equal-grain history concentration requires a reliance share "
        "for every Metric × Position pair."
    )


# ---------------------------------------------------------------------
# Rank all 350 pairs independently inside each final job
# ---------------------------------------------------------------------

concentration_records = []

for job_id, job_df in metric_position_df.groupby("job_id", observed=True):
    ranked_shares = np.sort(
        job_df["within_history_reliance_share_pct"].to_numpy(dtype=float)
    )[::-1]

    # These are normalized within-history shares, so the complete set should
    # reconstruct approximately 100% before cumulative concentration is read.
    if not np.isclose(ranked_shares.sum(), 100.0, atol=1e-6):
        raise RuntimeError(
            f"{job_id}: within-history reliance shares sum to "
            f"{ranked_shares.sum():.6f}% rather than 100%."
        )

    cumulative_shares = np.cumsum(ranked_shares)

    for cutoff in CONCENTRATION_CUTOFFS:
        concentration_records.append({
            "job_id": job_id,
            "top_n": cutoff,
            "cumulative_reliance_pct": float(cumulative_shares[cutoff - 1]),
        })

history_concentration_df = pd.DataFrame(concentration_records)


# ---------------------------------------------------------------------
# Summarize concentration across the 15 final Target × Horizon jobs
# ---------------------------------------------------------------------

history_concentration_summary_df = (
    history_concentration_df
    .groupby("top_n", as_index=False, observed=True)
    .agg(
        mean_cumulative_reliance_pct=("cumulative_reliance_pct", "mean"),
        median_cumulative_reliance_pct=("cumulative_reliance_pct", "median"),
        minimum_cumulative_reliance_pct=("cumulative_reliance_pct", "min"),
        maximum_cumulative_reliance_pct=("cumulative_reliance_pct", "max"),
    )
)

history_concentration_display_df = (
    history_concentration_summary_df
    .rename(columns={
        "top_n": "Strongest pairs",
        "mean_cumulative_reliance_pct": "Mean cumulative reliance %",
        "median_cumulative_reliance_pct": "Median cumulative reliance %",
        "minimum_cumulative_reliance_pct": "Minimum cumulative reliance %",
        "maximum_cumulative_reliance_pct": "Maximum cumulative reliance %",
    })
)

display(history_concentration_display_df.round(3))


# ---------------------------------------------------------------------
# Show how quickly reliance accumulates among the strongest history pairs
# ---------------------------------------------------------------------

fig = go.Figure(
    go.Bar(
        x=history_concentration_summary_df["top_n"],
        y=history_concentration_summary_df["mean_cumulative_reliance_pct"],
        text=history_concentration_summary_df[
            "mean_cumulative_reliance_pct"
        ].map(lambda value: f"{value:.1f}%"),
        textposition="outside",
        marker_color=BRAND_COLOR_SEQUENCE[0],
        customdata=history_concentration_summary_df[
            [
                "median_cumulative_reliance_pct",
                "minimum_cumulative_reliance_pct",
                "maximum_cumulative_reliance_pct",
            ]
        ],
        hovertemplate=(
            "Top %{x} Metric × Position pairs<br>"
            "Mean cumulative reliance: %{y:.3f}%<br>"
            "Median: %{customdata[0]:.3f}%<br>"
            "Minimum job: %{customdata[1]:.3f}%<br>"
            "Maximum job: %{customdata[2]:.3f}%"
            "<extra></extra>"
        ),
    )
)

fig = apply_branding(fig)

fig.update_layout(
    title="How Much Reliance Is Captured by the Strongest History Signals?",
    xaxis_title="Number of strongest Metric × Position pairs",
    yaxis_title="Average cumulative within-history reliance (%)",
    showlegend=False,
)

fig.update_yaxes(range=[0, 105])

fig.show()

,Strongest pairs,Mean cumulative reliance %,Median cumulative reliance %,Minimum cumulative reliance %,Maximum cumulative reliance %
0,5,41.275,42.873,12.785,65.469
1,10,54.704,58.229,22.073,80.558
2,25,71.529,73.605,42.742,90.089
3,50,83.602,84.609,62.491,94.694


**Findings.** Reliance within the recent-history block is concentrated, but not confined to only a handful of inputs. Across the finalized Transformer jobs, the **5 strongest Metric × Position pairs account for 41.275% of within-history reliance on average**, rising to **54.704% for the top 10**, **71.529% for the top 25**, and **83.602% for the top 50**.

This means the Transformer usually gets more than half of its history-based predictive signal from a relatively small set of lag-position combinations, while still drawing meaningful information from a longer tail of historical inputs. The median values closely track the means, suggesting that this concentration pattern is typical rather than being driven by only a few unusual jobs.

There is nevertheless substantial variation across Target × Horizon models. The top 5 pairs capture anywhere from **12.785% to 65.469%** of within-history reliance, and even the top 50 range from **62.491% to 94.694%**. Some forecasts therefore depend on a very compact set of historical signals, while others distribute reliance much more broadly across the available sequence. The result supports the use of a richer history window without implying that every historical position contributes equally.

### Section 7 Summary

Chapter 7 shows that the finalized Transformer is not behaving like one fixed training recipe or one uniformly stable forecasting system. Adaptive selection chooses materially different training durations across monthly refits, and validation generally worsens during the five-epoch patience window after the selected point, supporting the early-stopping design. Monthly performance is especially stable for both speed targets, which beat the weekly benchmark in **every validation month**, while demand forecasts remain positive overall but experience intermittent weak periods, particularly near the edges of the validation window. Grouped permutation provides a clear model-level explanation of where the forecasts draw information: **recent mobility history contributes 86.454% of measured reliance on average and ranks first in all 15 jobs**, while speed forecasts use calendar and other supporting context more heavily than the demand targets. Within that history block, reliance is concentrated but not extremely sparse: the **10 strongest Metric × Position pairs capture 54.704% of within-history reliance on average, while the top 50 capture 83.602%**, with substantial variation across Target × Horizon jobs. The Transformer therefore tends to emphasize a relatively small set of especially informative historical positions while still drawing useful signal from a broader sequence. These reliance results describe predictive dependence rather than causal importance, and the congestion-pricing regime remains non-identifiable on the latest post-policy scoring surfaces.

## 4.5.2.8 — Freeze, Package, and Close the Transformer Forecasting Handoff

The Transformer branch is fully specified before this chapter begins. The temporal representation, target-specific capacity profiles, monthly refitting cadence, training-history tracks, adaptive training policy, and ordinary-validation evidence have already been established.

No model fitting occurs here. This final chapter turns those completed results into a compact, reusable handoff for downstream forecasting work.

The closeout has four responsibilities:

1. **Freeze the final 15-job candidate roster** — bind every Target × Horizon job to its selected representation, capacity profile, modeling track, validation evidence, and latest retained fitted state.
2. **Preserve the model contract** — record the architecture, model-facing inputs, preprocessing semantics, output rule, and production-training policy without requiring later notebooks to reconstruct them from notebook state.
3. **Package the evidence** — retain the complete production schedule, adaptive-training records, learning curves, grouped predictive-reliance results, and row-level ordinary-validation forecasts.
4. **Verify the handoff** — prove that the packaged forecasts reproduce the notebook's final scorecard, obey the output contract, remain outside the final holdout, and contain the complete frozen artifact roster.

This closes **within-Transformer development only**. Overall forecasting-family selection and final-holdout evaluation remain outside this notebook.


**4.5.2.8A — Build the final 15-job Transformer candidate registry**

The Transformer branch advances a coordinated **15-job Target × Horizon roster**, not one universal configuration.

All targets use the selected two-week overlapping representation, but model capacity and training-history strategy remain target-specific. The registry below binds those frozen choices to the full rolling-validation scorecard and the latest fitted state retained for each job.

The latest state is a reproducibility reference rather than a new evaluation surface: the complete ordinary-validation evidence still comes from all ten monthly refits per job.


In [78]:
# ---------------------------------------------------------------------
# 4.5.2.8A — Build the final 15-job Transformer candidate registry
# ---------------------------------------------------------------------

TRANSFORMER_FINAL_DIR = Path("pipeline_data") / "4.5.2.final_tables"
TRANSFORMER_FINAL_DIR.mkdir(parents=True, exist_ok=True)

TRANSFORMER_FINAL_MODEL_REGISTRY_PATH = (
    TRANSFORMER_FINAL_DIR / "transformer_final_model_registry.parquet"
)

EXPECTED_TRANSFORMER_REPRESENTATION_ID = "two_week_overlapping_two_day"

EXPECTED_TRANSFORMER_PROFILE_BY_TARGET = {
    "taxi_trip_count": "higher_capacity_regularized",
    "taxi_avg_trip_speed": "compact",
    "fhvhv_trip_count": "compact",
    "fhvhv_avg_trip_speed": "higher_capacity_regularized",
    "subway_ridership": "compact",
}

EXPECTED_TRANSFORMER_TRACK_BY_TARGET = {
    "taxi_trip_count": "post_cp_primary",
    "taxi_avg_trip_speed": "post_cp_primary",
    "fhvhv_trip_count": "post_cp_primary",
    "fhvhv_avg_trip_speed": "full_history_regime_aware",
    "subway_ridership": "full_history_regime_aware",
}

TRANSFORMER_FINAL_PROFILE_LABELS = {
    "compact": "Compact",
    "balanced": "Balanced",
    "higher_capacity_regularized": "Higher capacity + regularization",
}

required_job_columns = {
    "job_id", "track", "metric", "horizon",
    "candidate_id", "representation_id", "profile_id",
}
require_external_columns(
    transformer_full_jobs_df,
    required_job_columns,
    "transformer_full_jobs_df",
)

require_external_columns(
    transformer_job_scorecard_df,
    {"track", "metric", "horizon", "final_mae", "benchmark_mae", "benchmark_skill_pct"},
    "transformer_job_scorecard_df",
)

require_external_columns(
    full_transformer_results_df,
    {"job_id", "refit_step", "state_path", "selected_epoch", "parameter_count", "status"},
    "full_transformer_results_df",
)


# ---------------------------------------------------------------------
# Recover the latest fitted state for each production job
# ---------------------------------------------------------------------
# Refit step increases chronologically, so the highest completed step is the
# fitted state containing the largest legal expanding-history training surface.

latest_transformer_states_df = (
    full_transformer_results_df.loc[
        full_transformer_results_df["status"].eq("success")
    ]
    .sort_values(["job_id", "refit_step"])
    .groupby("job_id", observed=True)
    .tail(1)
    [[
        "job_id", "refit_step", "temporal_window",
        "selected_epoch", "parameter_count", "state_path",
    ]]
    .rename(columns={
        "refit_step": "latest_refit_step",
        "temporal_window": "latest_temporal_window",
        "selected_epoch": "latest_selected_epoch",
        "state_path": "latest_state_path",
    })
    .reset_index(drop=True)
)

if len(latest_transformer_states_df) != 15:
    raise RuntimeError(
        "Expected one latest successful fitted state for each of 15 Transformer jobs."
    )


# ---------------------------------------------------------------------
# Join frozen job identity, final validation evidence, and latest state
# ---------------------------------------------------------------------

scorecard_fields = [
    "track", "metric", "horizon",
    "final_mae", "raw_mae", "benchmark_mae", "benchmark_skill_pct",
]
scorecard_fields = [
    column for column in scorecard_fields
    if column in transformer_job_scorecard_df.columns
]

final_transformer_candidate_registry_df = (
    transformer_full_jobs_df
    .merge(
        transformer_job_scorecard_df[scorecard_fields],
        on=["track", "metric", "horizon"],
        how="left",
        validate="one_to_one",
    )
    .merge(
        latest_transformer_states_df,
        on="job_id",
        how="left",
        validate="one_to_one",
    )
)

final_transformer_candidate_registry_df["model_family"] = "transformer"
final_transformer_candidate_registry_df["model_class"] = "PatchTransformerRegressor"
final_transformer_candidate_registry_df["refit_cadence"] = "monthly"
final_transformer_candidate_registry_df["production_contract_id"] = (
    TRANSFORMER_FULL_CONTRACT_ID
)
final_transformer_candidate_registry_df["model_definition_id"] = (
    "transformer__"
    + final_transformer_candidate_registry_df["representation_id"].astype(str)
    + "__"
    + final_transformer_candidate_registry_df["profile_id"].astype(str)
)

# Fail loudly if the closeout roster drifted from the choices already frozen
# earlier in the notebook.
if len(final_transformer_candidate_registry_df) != 15:
    raise RuntimeError("The final Transformer registry must contain exactly 15 jobs.")

if final_transformer_candidate_registry_df["job_id"].duplicated().any():
    raise RuntimeError("The final Transformer registry contains duplicate job IDs.")

expected_target_horizons = {
    (metric, horizon)
    for metric in FORECAST_TARGETS
    for horizon in FORECAST_HORIZONS
}
observed_target_horizons = set(zip(
    final_transformer_candidate_registry_df["metric"],
    final_transformer_candidate_registry_df["horizon"].astype(int),
))

if observed_target_horizons != expected_target_horizons:
    raise RuntimeError("The final Transformer registry does not contain the exact 15-job roster.")

if not final_transformer_candidate_registry_df["representation_id"].eq(
    EXPECTED_TRANSFORMER_REPRESENTATION_ID
).all():
    raise RuntimeError("The final registry drifted from the frozen Transformer representation.")

for metric in FORECAST_TARGETS:
    target_rows = final_transformer_candidate_registry_df.loc[
        final_transformer_candidate_registry_df["metric"].eq(metric)
    ]

    if not target_rows["profile_id"].eq(
        EXPECTED_TRANSFORMER_PROFILE_BY_TARGET[metric]
    ).all():
        raise RuntimeError(
            f"{metric}: final Transformer profile does not match the frozen choice."
        )

    if not target_rows["track"].eq(
        EXPECTED_TRANSFORMER_TRACK_BY_TARGET[metric]
    ).all():
        raise RuntimeError(
            f"{metric}: final Transformer track does not match the frozen choice."
        )

if final_transformer_candidate_registry_df[
    ["final_mae", "benchmark_mae", "benchmark_skill_pct", "latest_state_path"]
].isna().any().any():
    raise RuntimeError(
        "The final Transformer registry is missing performance or fitted-state evidence."
    )

atomic_write_parquet(
    final_transformer_candidate_registry_df,
    TRANSFORMER_FINAL_MODEL_REGISTRY_PATH,
)

transformer_candidate_display_df = (
    final_transformer_candidate_registry_df
    .assign(
        Target=lambda df: df["metric"].map(TARGET_LABELS),
        Track=lambda df: df["track"].map(TRACK_LABELS),
        Profile=lambda df: df["profile_id"].map(TRANSFORMER_FINAL_PROFILE_LABELS),
    )
    [[
        "Target", "horizon", "Track", "Profile",
        "final_mae", "benchmark_skill_pct", "latest_selected_epoch",
    ]]
    .rename(columns={
        "horizon": "Horizon",
        "final_mae": "Final MAE",
        "benchmark_skill_pct": "Benchmark skill (%)",
        "latest_selected_epoch": "Latest selected epoch",
    })
    .sort_values(["Target", "Horizon"])
    .reset_index(drop=True)
)

display(transformer_candidate_display_df.round(3))


,Target,Horizon,Track,Profile,Final MAE,Benchmark skill (%),Latest selected epoch
0,FHVHV average speed,1,Full-History Regime-Aware,Higher capacity + regularization,0.921,30.726,15
1,FHVHV average speed,2,Full-History Regime-Aware,Higher capacity + regularization,0.973,26.773,17
2,FHVHV average speed,5,Full-History Regime-Aware,Higher capacity + regularization,0.990,25.492,12
3,FHVHV trips,1,Post-CP Primary,Compact,58.437,16.080,15
4,FHVHV trips,2,Post-CP Primary,Compact,62.050,10.891,15
5,FHVHV trips,5,Post-CP Primary,Compact,62.633,10.054,15
6,Subway ridership,1,Full-History Regime-Aware,Compact,447.209,15.905,25
7,Subway ridership,2,Full-History Regime-Aware,Compact,475.990,10.493,19
8,Subway ridership,5,Full-History Regime-Aware,Compact,477.380,10.231,21
9,Taxi average speed,1,Post-CP Primary,Compact,2.168,26.529,7


**Findings.** The final Transformer registry contains the complete **15 Target × Horizon jobs** with no missing or duplicated combinations. All 15 use the selected **two-week overlapping two-day representation**.

Capacity remains target-specific: **9 jobs use Compact** and **6 use Higher capacity + regularization**. Training history is also target-specific: **9 jobs use Post-CP Primary** and **6 use Full-History Regime-Aware**. Each candidate carries its full-validation performance record and one latest retained fitted state.

These are the Transformer candidates advancing from this notebook; no cross-family winner is selected here.


**4.5.2.8B — Freeze the reusable Transformer model and input contracts**

The candidate registry records **what advances**. Separate contracts preserve **how those candidates are defined**.

The handoff records the selected patch geometry and Transformer capacity settings, the target-specific 10-channel mobility-history ordering, the shared 52-value context, Taxi Zone identity, the forecast-output rule, and the exact context contract inherited from the modeling pipeline.

The fitted-state files themselves remain in the production checkpoint directory and are referenced by the final registry rather than copied again. This keeps the handoff compact while preserving reproducibility.


In [79]:
# ---------------------------------------------------------------------
# 4.5.2.8B — Freeze model, input, context, and output contracts
# ---------------------------------------------------------------------

TRANSFORMER_MODEL_DEFINITION_PATH = (
    TRANSFORMER_FINAL_DIR / "transformer_model_definition_contract.parquet"
)
TRANSFORMER_CONTEXT_CONTRACT_PATH = (
    TRANSFORMER_FINAL_DIR / "transformer_context_contract.parquet"
)
TRANSFORMER_INPUT_REGISTRY_PATH = (
    TRANSFORMER_FINAL_DIR / "transformer_input_registry.parquet"
)
TRANSFORMER_OUTPUT_CONTRACT_PATH = (
    TRANSFORMER_FINAL_DIR / "transformer_forecast_output_contract.parquet"
)

representation_lookup = {
    item["representation_id"]: item
    for item in TRANSFORMER_REPRESENTATION_CANDIDATES
}
profile_lookup = {
    item["profile_id"]: item
    for item in TRANSFORMER_PROFILE_CANDIDATES
}

selected_definition_pairs_df = (
    final_transformer_candidate_registry_df[
        ["representation_id", "profile_id", "model_definition_id"]
    ]
    .drop_duplicates()
    .sort_values(["representation_id", "profile_id"])
    .reset_index(drop=True)
)

model_definition_rows = []

for row in selected_definition_pairs_df.itertuples(index=False):
    representation = representation_lookup[row.representation_id]
    profile = profile_lookup[row.profile_id]

    lookback_steps = int(representation["lookback_steps"])
    patch_length = int(representation["patch_length"])
    patch_stride = int(representation["patch_stride"])
    n_patches = 1 + (lookback_steps - patch_length) // patch_stride

    model_definition_rows.append({
        "model_definition_id": row.model_definition_id,
        "model_family": "transformer",
        "model_class": "PatchTransformerRegressor",
        "architecture": "PatchTST-inspired encoder-only Transformer",
        "representation_id": row.representation_id,
        "profile_id": row.profile_id,
        "lookback_steps": lookback_steps,
        "patch_length": patch_length,
        "patch_stride": patch_stride,
        "patch_count": n_patches,
        "sequence_input_dim": int(NEURAL_SEQUENCE_INPUT_DIM),
        "context_input_dim": int(transformer_context_feature_count),
        "d_model": int(profile["d_model"]),
        "n_heads": int(profile["n_heads"]),
        "n_layers": int(profile["n_layers"]),
        "d_ff": int(profile["d_ff"]),
        "dropout": float(profile["dropout"]),
        "zone_embedding_dim": int(profile["zone_embedding_dim"]),
        "learning_rate": float(profile["learning_rate"]),
        "weight_decay": float(profile["weight_decay"]),
        "pooling": "mean",
        "positional_representation": "learned",
        "causal_attention_mask": False,
        "loss": "Huber",
        "optimizer": "AdamW",
    })

transformer_model_definition_contract_df = pd.DataFrame(model_definition_rows)

if len(transformer_model_definition_contract_df) != 2:
    raise RuntimeError(
        "Expected exactly two final Transformer model definitions: "
        "one representation paired with the two selected profiles."
    )


# ---------------------------------------------------------------------
# Build one explicit model-facing input registry
# ---------------------------------------------------------------------
# Sequence ordering changes by forecast target, so preserving only a generic
# ten-channel statement would be insufficient for later reproduction.

input_rows = []

for metric in FORECAST_TARGETS:
    value_channels = list(
        TRANSFORMER_SEQUENCE_VALUE_CHANNELS_BY_TARGET[metric]
    )
    mask_channels = list(
        TRANSFORMER_SEQUENCE_MASK_CHANNELS_BY_TARGET[metric]
    )

    if len(value_channels) != 5 or len(mask_channels) != 5:
        raise RuntimeError(
            f"{metric}: expected five sequence values and five masks."
        )

    for index, feature in enumerate(value_channels):
        input_rows.append({
            "target_metric": metric,
            "component": "sequence",
            "input_index": index,
            "input_name": feature,
            "semantic_role": "historical mobility value",
            "preprocessing": "training-only observed-cell standardization",
        })

    for index, feature in enumerate(mask_channels, start=5):
        input_rows.append({
            "target_metric": metric,
            "component": "sequence",
            "input_index": index,
            "input_name": feature,
            "semantic_role": "historical observation mask",
            "preprocessing": "binary mask paired with historical mobility value",
        })

for index, feature in enumerate(TRANSFORMER_CONTEXT_FEATURE_ORDER):
    input_rows.append({
        "target_metric": "shared",
        "component": "context",
        "input_index": index,
        "input_name": feature,
        "semantic_role": (
            "state-aware weekly STL"
            if feature in {
                "weekly_stl_seasonal_offset",
                "weekly_stl_enabled",
                "missing_mask__weekly_stl_seasonal_offset",
            }
            else "supporting context"
        ),
        "preprocessing": "See transformer_context_contract.parquet",
    })

input_rows.append({
    "target_metric": "shared",
    "component": "zone_identity",
    "input_index": 0,
    "input_name": "taxi_zone_id",
    "semantic_role": "Taxi Zone identity",
    "preprocessing": "learned embedding",
})

transformer_input_registry_df = pd.DataFrame(input_rows)

expected_input_rows = 5 * 10 + 52 + 1

if len(transformer_input_registry_df) != expected_input_rows:
    raise RuntimeError(
        f"Expected {expected_input_rows} model-facing input-registry rows."
    )


# ---------------------------------------------------------------------
# Freeze output semantics separately from architecture
# ---------------------------------------------------------------------

transformer_forecast_output_contract_df = pd.DataFrame([
    {
        "field": "actual",
        "role": "Observed target value",
        "rule": "Native-unit observed value on legal ordinary-validation support",
    },
    {
        "field": "benchmark_prediction",
        "role": "Frozen comparison forecast",
        "rule": "Previous week · same Daypart",
    },
    {
        "field": "prediction_raw",
        "role": "Unconstrained Transformer forecast",
        "rule": "Inverse-transformed native-unit model output",
    },
    {
        "field": "prediction_final",
        "role": "Production Transformer forecast",
        "rule": "max(0, prediction_raw)",
    },
])

atomic_write_parquet(
    transformer_model_definition_contract_df,
    TRANSFORMER_MODEL_DEFINITION_PATH,
)
atomic_write_parquet(
    transformer_context_contract_df.copy(),
    TRANSFORMER_CONTEXT_CONTRACT_PATH,
)
atomic_write_parquet(
    transformer_input_registry_df,
    TRANSFORMER_INPUT_REGISTRY_PATH,
)
atomic_write_parquet(
    transformer_forecast_output_contract_df,
    TRANSFORMER_OUTPUT_CONTRACT_PATH,
)

contract_inventory_df = pd.DataFrame([
    {
        "Contract": "Final model definitions",
        "Rows": len(transformer_model_definition_contract_df),
        "Meaning": "Exact selected representation × capacity definitions",
    },
    {
        "Contract": "Context",
        "Rows": len(transformer_context_contract_df),
        "Meaning": "Shared 52-value context semantics and preprocessing",
    },
    {
        "Contract": "Model-facing inputs",
        "Rows": len(transformer_input_registry_df),
        "Meaning": "Target-specific sequence + shared context + Taxi Zone identity",
    },
    {
        "Contract": "Forecast outputs",
        "Rows": len(transformer_forecast_output_contract_df),
        "Meaning": "Actual, benchmark, raw prediction, and final prediction semantics",
    },
])

display(contract_inventory_df)


,Contract,Rows,Meaning
0,Final model definitions,2,Exact selected representation × capacity defin...
1,Context,52,Shared 52-value context semantics and preproce...
2,Model-facing inputs,103,Target-specific sequence + shared context + Ta...
3,Forecast outputs,4,"Actual, benchmark, raw prediction, and final p..."


**Findings.** The reusable Transformer contract is now explicit rather than implicit in notebook state. The final branch uses **two model definitions**: the same 70-step overlapping-patch representation paired with either the Compact or Higher capacity + regularization profile.

The model-facing input registry preserves each target's **10 ordered sequence channels**, the shared **52-value context**, and Taxi Zone identity. The separate context contract retains the state-aware weekly-STL rules, while the output contract preserves both the unconstrained `prediction_raw` and production `prediction_final = max(0, prediction_raw)`.


**4.5.2.8C — Package production, training, development, and predictive-reliance evidence**

The model registry describes the final candidates. These artifacts preserve the evidence behind them.

The package retains the complete 150-fit monthly production schedule and results, every saved adaptive-training curve, the five target-level frozen modeling choices, and both grains of grouped predictive-reliance evidence.

This is evidence packaging rather than another model-selection step: the objects written below are the same completed results analyzed in Chapters 6 and 7.


In [80]:
# ---------------------------------------------------------------------
# 4.5.2.8C — Package development, training, and reliance evidence
# ---------------------------------------------------------------------

TRANSFORMER_PRODUCTION_SCHEDULE_PATH = (
    TRANSFORMER_FINAL_DIR / "transformer_production_schedule.parquet"
)
TRANSFORMER_TRAINING_POLICY_PATH = (
    TRANSFORMER_FINAL_DIR / "transformer_training_policy_evidence.parquet"
)
TRANSFORMER_TRAINING_CURVE_PATH = (
    TRANSFORMER_FINAL_DIR / "transformer_training_curve_diagnostic.parquet"
)
TRANSFORMER_RELIANCE_REPEAT_PATH = (
    TRANSFORMER_FINAL_DIR / "transformer_grouped_reliance_repeat.parquet"
)
TRANSFORMER_RELIANCE_SUMMARY_PATH = (
    TRANSFORMER_FINAL_DIR / "transformer_grouped_reliance_summary.parquet"
)
TRANSFORMER_DEVELOPMENT_SUMMARY_PATH = (
    TRANSFORMER_FINAL_DIR / "transformer_development_summary.parquet"
)

transformer_training_policy_evidence_df = (
    full_transformer_results_df
    .sort_values(["metric", "horizon", "refit_step"])
    .reset_index(drop=True)
)

transformer_training_curve_diagnostic_df = (
    transformer_training_curves_df
    .sort_values(["job_id", "refit_step", "epoch"])
    .reset_index(drop=True)
)

transformer_grouped_reliance_repeat_handoff_df = (
    transformer_grouped_reliance_repeat_df
    .copy()
    .reset_index(drop=True)
)

transformer_grouped_reliance_summary_handoff_df = (
    transformer_grouped_reliance_job_df
    .copy()
    .reset_index(drop=True)
)


# ---------------------------------------------------------------------
# Compress the frozen development choices to one readable row per target
# ---------------------------------------------------------------------
# Horizons share target-level representation, profile, track, and cadence, so
# five rows document the decisions without averaging incompatible native units.

development_rows = []

for metric in FORECAST_TARGETS:
    target_registry = final_transformer_candidate_registry_df.loc[
        final_transformer_candidate_registry_df["metric"].eq(metric)
    ]

    if len(target_registry) != 3:
        raise RuntimeError(
            f"{metric}: expected exactly three final horizons."
        )

    invariants = [
        "track",
        "representation_id",
        "profile_id",
        "refit_cadence",
    ]

    if any(
        target_registry[column].nunique() != 1
        for column in invariants
    ):
        raise RuntimeError(
            f"{metric}: target-level Transformer choices are not invariant."
        )

    development_rows.append({
        "metric": metric,
        "target_label": TARGET_LABELS[metric],
        "track": target_registry["track"].iloc[0],
        "track_label": TRACK_LABELS[target_registry["track"].iloc[0]],
        "representation_id": target_registry["representation_id"].iloc[0],
        "profile_id": target_registry["profile_id"].iloc[0],
        "profile_label": TRANSFORMER_FINAL_PROFILE_LABELS[
            target_registry["profile_id"].iloc[0]
        ],
        "refit_cadence": target_registry["refit_cadence"].iloc[0],
        "horizons": "1, 2, 5",
        "production_jobs": 3,
    })

transformer_development_summary_df = pd.DataFrame(development_rows)

atomic_write_parquet(
    transformer_full_schedule_df
    .sort_values(["metric", "horizon", "refit_step"])
    .reset_index(drop=True),
    TRANSFORMER_PRODUCTION_SCHEDULE_PATH,
)
atomic_write_parquet(
    transformer_training_policy_evidence_df,
    TRANSFORMER_TRAINING_POLICY_PATH,
)
atomic_write_parquet(
    transformer_training_curve_diagnostic_df,
    TRANSFORMER_TRAINING_CURVE_PATH,
)
atomic_write_parquet(
    transformer_grouped_reliance_repeat_handoff_df,
    TRANSFORMER_RELIANCE_REPEAT_PATH,
)
atomic_write_parquet(
    transformer_grouped_reliance_summary_handoff_df,
    TRANSFORMER_RELIANCE_SUMMARY_PATH,
)
atomic_write_parquet(
    transformer_development_summary_df,
    TRANSFORMER_DEVELOPMENT_SUMMARY_PATH,
)

evidence_inventory_df = pd.DataFrame([
    {
        "Evidence": "Production schedule",
        "Rows": len(transformer_full_schedule_df),
    },
    {
        "Evidence": "Production training policy",
        "Rows": len(transformer_training_policy_evidence_df),
    },
    {
        "Evidence": "Adaptive selector curves",
        "Rows": len(transformer_training_curve_diagnostic_df),
    },
    {
        "Evidence": "Grouped reliance · repeat",
        "Rows": len(transformer_grouped_reliance_repeat_handoff_df),
    },
    {
        "Evidence": "Grouped reliance · job summary",
        "Rows": len(transformer_grouped_reliance_summary_handoff_df),
    },
    {
        "Evidence": "Target development summary",
        "Rows": len(transformer_development_summary_df),
    },
])

display(evidence_inventory_df)


,Evidence,Rows
0,Production schedule,150
1,Production training policy,150
2,Adaptive selector curves,2471
3,Grouped reliance · repeat,360
4,Grouped reliance · job summary,120
5,Target development summary,5


**Findings.** The evidence package preserves the complete **150-fit production schedule and training-policy record**, all adaptive selector learning curves, and the five target-level frozen Transformer configurations.

Grouped predictive reliance is retained at both required grains: **360 Job × Group × Repeat rows** and **120 Job × Group summary rows**. These artifacts preserve the evidence needed to audit training behavior and model-facing information use without rerunning the production crawl.


**4.5.2.8D — Build the evaluation-ready row-level forecast handoff**

The production crawl stored forecasts as 150 durable monthly prediction artifacts. Downstream comparison should not have to understand that checkpoint layout.

This section consolidates those existing forecasts into one row-level ordinary-validation table. Each row preserves forecast and Taxi Zone identity, timing, Target × Horizon identity, the selected Transformer definition, monthly-refit ownership, the observed value, frozen benchmark, raw forecast, and nonnegative final forecast.

The consolidation uses DuckDB so the large forecast surface can be written without materializing millions of rows in notebook memory.

Before the artifact is accepted, it is rescored from the packaged rows themselves. Its row counts and final MAE must reproduce the Chapter 6 scorecard for every one of the 15 jobs, and no final-holdout observation is permitted.


In [81]:
# ---------------------------------------------------------------------
# 4.5.2.8D — Build the evaluation-ready row-level forecast handoff
# ---------------------------------------------------------------------

TRANSFORMER_SELECTED_PREDICTIONS_PATH = (
    TRANSFORMER_FINAL_DIR / "transformer_selected_validation_predictions.parquet"
)

successful_fit_df = full_transformer_results_df.loc[
    full_transformer_results_df["status"].eq("success")
].copy()

prediction_paths = successful_fit_df["prediction_path"].map(Path).tolist()

if len(prediction_paths) != 150 or len(set(map(str, prediction_paths))) != 150:
    raise RuntimeError(
        "Expected exactly 150 unique production prediction artifacts."
    )

missing_prediction_paths = [
    str(path)
    for path in prediction_paths
    if not path.exists()
]

if missing_prediction_paths:
    raise FileNotFoundError(
        "Production prediction artifacts are missing: "
        f"{missing_prediction_paths[:5]}"
    )

candidate_metadata_df = final_transformer_candidate_registry_df[[
    "job_id",
    "track",
    "metric",
    "horizon",
    "representation_id",
    "profile_id",
    "model_definition_id",
    "production_contract_id",
    "refit_cadence",
]].copy()


def _duckdb_sql_path(path):
    """Escape one filesystem path for a local DuckDB COPY statement."""
    return str(Path(path)).replace("'", "''")


# ---------------------------------------------------------------------
# Stream the 150 monthly artifacts into one downstream Parquet table
# ---------------------------------------------------------------------
# A temporary file is published only after the complete COPY succeeds, so an
# interrupted consolidation cannot masquerade as a valid downstream handoff.

temp_prediction_path = TRANSFORMER_SELECTED_PREDICTIONS_PATH.with_name(
    f".{TRANSFORMER_SELECTED_PREDICTIONS_PATH.stem}."
    f"{uuid.uuid4().hex}.tmp.parquet"
)

duck = duckdb.connect()

try:
    duck.read_parquet(
        [str(path) for path in prediction_paths]
    ).create_view("transformer_prediction_parts")

    duck.register(
        "transformer_candidate_metadata",
        candidate_metadata_df,
    )

    output_path_sql = _duckdb_sql_path(temp_prediction_path)

    duck.execute(f"""
        COPY (
            SELECT
                p.forecast_sequence_id,
                p.taxi_zone_id,
                p.origin_observation_sequence_id,
                p.origin_date,
                p.target_observation_sequence_id,
                p.target_date,
                p.track AS modeling_track,
                p.metric,
                CAST(p.horizon AS INTEGER) AS horizon,
                p.job_id,
                p.candidate_id,
                p.temporal_window,
                CAST(p.refit_step AS INTEGER) AS refit_step,

                p.target_observed_value AS actual,
                p.benchmark_prediction,
                p.prediction_raw,
                p.prediction_final,

                m.representation_id,
                m.profile_id,
                m.model_definition_id,
                m.production_contract_id,
                m.refit_cadence,

                'transformer' AS model_family,
                'PatchTransformerRegressor' AS model_class,
                'Previous week · same Daypart' AS benchmark_label,
                TRUE AS evaluation_eligible,
                TRUE AS benchmark_common_support
            FROM transformer_prediction_parts AS p
            INNER JOIN transformer_candidate_metadata AS m
                ON p.job_id = m.job_id
               AND p.track = m.track
               AND p.metric = m.metric
               AND CAST(p.horizon AS INTEGER) = CAST(m.horizon AS INTEGER)
        )
        TO '{output_path_sql}'
        (FORMAT PARQUET, COMPRESSION ZSTD);
    """)

    os.replace(
        temp_prediction_path,
        TRANSFORMER_SELECTED_PREDICTIONS_PATH,
    )

    # Audit the physical handoff without loading the large row-level artifact
    # into notebook memory.
    handoff_path_sql = _duckdb_sql_path(
        TRANSFORMER_SELECTED_PREDICTIONS_PATH
    )
    holdout_start = pd.Timestamp(
        FINAL_HOLDOUT_START_DATE
    ).strftime("%Y-%m-%d")

    transformer_handoff_audit_df = duck.execute(f"""
        SELECT
            COUNT(*) AS forecast_rows,
            COUNT(DISTINCT job_id) AS jobs,
            MIN(target_date) AS min_target_date,
            MAX(target_date) AS max_target_date,

            SUM(CASE WHEN
                actual IS NULL
                OR benchmark_prediction IS NULL
                OR prediction_raw IS NULL
                OR prediction_final IS NULL
                THEN 1 ELSE 0 END
            ) AS missing_score_rows,

            SUM(CASE WHEN
                NOT isfinite(actual)
                OR NOT isfinite(benchmark_prediction)
                OR NOT isfinite(prediction_raw)
                OR NOT isfinite(prediction_final)
                THEN 1 ELSE 0 END
            ) AS nonfinite_score_rows,

            SUM(CASE WHEN
                target_date >= DATE '{holdout_start}'
                THEN 1 ELSE 0 END
            ) AS final_holdout_rows,

            SUM(CASE WHEN
                ABS(prediction_final - GREATEST(0.0, prediction_raw)) > 1e-6
                THEN 1 ELSE 0 END
            ) AS output_rule_violations

        FROM read_parquet('{handoff_path_sql}')
    """).df()

    transformer_handoff_duplicate_rows = int(
        duck.execute(f"""
            SELECT COALESCE(SUM(row_count - 1), 0)
            FROM (
                SELECT
                    job_id,
                    forecast_sequence_id,
                    taxi_zone_id,
                    origin_observation_sequence_id,
                    target_observation_sequence_id,
                    COUNT(*) AS row_count
                FROM read_parquet('{handoff_path_sql}')
                GROUP BY ALL
                HAVING COUNT(*) > 1
            )
        """).fetchone()[0]
    )

    transformer_handoff_scorecard_df = duck.execute(f"""
        SELECT
            modeling_track AS track,
            metric,
            CAST(horizon AS INTEGER) AS horizon,
            COUNT(*) AS evaluation_rows,
            AVG(ABS(prediction_raw - actual)) AS raw_mae,
            AVG(ABS(prediction_final - actual)) AS final_mae,
            AVG(ABS(benchmark_prediction - actual)) AS benchmark_mae
        FROM read_parquet('{handoff_path_sql}')
        GROUP BY modeling_track, metric, horizon
        ORDER BY metric, horizon
    """).df()

finally:
    duck.close()

if temp_prediction_path.exists():
    temp_prediction_path.unlink()


# ---------------------------------------------------------------------
# Reproduce the Chapter 6 scoring evidence from the exported rows
# ---------------------------------------------------------------------

transformer_handoff_scorecard_df["benchmark_skill_pct"] = (
    100.0
    * (
        transformer_handoff_scorecard_df["benchmark_mae"]
        - transformer_handoff_scorecard_df["final_mae"]
    )
    / transformer_handoff_scorecard_df["benchmark_mae"]
)

expected_evaluation_rows_df = (
    full_transformer_results_df
    .groupby(
        ["track", "metric", "horizon"],
        observed=True,
    )["evaluation_rows"]
    .sum()
    .rename("expected_evaluation_rows")
    .reset_index()
)

scorecard_reference_df = (
    transformer_job_scorecard_df[
        ["track", "metric", "horizon", "final_mae", "benchmark_mae"]
    ]
    .rename(columns={
        "final_mae": "chapter6_final_mae",
        "benchmark_mae": "chapter6_benchmark_mae",
    })
)

transformer_handoff_reproduction_df = (
    transformer_handoff_scorecard_df
    .merge(
        expected_evaluation_rows_df,
        on=["track", "metric", "horizon"],
        how="left",
        validate="one_to_one",
    )
    .merge(
        scorecard_reference_df,
        on=["track", "metric", "horizon"],
        how="left",
        validate="one_to_one",
    )
)

transformer_handoff_reproduction_df["row_count_match"] = (
    transformer_handoff_reproduction_df["evaluation_rows"]
    .eq(transformer_handoff_reproduction_df["expected_evaluation_rows"])
)

transformer_handoff_reproduction_df["final_mae_match"] = np.isclose(
    transformer_handoff_reproduction_df["final_mae"],
    transformer_handoff_reproduction_df["chapter6_final_mae"],
    rtol=1e-9,
    atol=1e-9,
)

transformer_handoff_reproduction_df["benchmark_mae_match"] = np.isclose(
    transformer_handoff_reproduction_df["benchmark_mae"],
    transformer_handoff_reproduction_df["chapter6_benchmark_mae"],
    rtol=1e-9,
    atol=1e-9,
)

if not transformer_handoff_reproduction_df[
    ["row_count_match", "final_mae_match", "benchmark_mae_match"]
].all().all():
    raise RuntimeError(
        "The exported row-level Transformer handoff does not reproduce "
        "the Chapter 6 scorecard."
    )

handoff_display_df = (
    transformer_handoff_reproduction_df
    .assign(
        Target=lambda df: df["metric"].map(TARGET_LABELS),
        Track=lambda df: df["track"].map(TRACK_LABELS),
        MAE_delta=lambda df: df["final_mae"] - df["chapter6_final_mae"],
    )
    [[
        "Target",
        "horizon",
        "Track",
        "evaluation_rows",
        "final_mae",
        "chapter6_final_mae",
        "MAE_delta",
    ]]
    .rename(columns={
        "horizon": "Horizon",
        "evaluation_rows": "Rows",
        "final_mae": "Handoff MAE",
        "chapter6_final_mae": "Chapter 6 MAE",
        "MAE_delta": "MAE difference",
    })
    .sort_values(["Target", "Horizon"])
    .reset_index(drop=True)
)

display(handoff_display_df.round(3))


,Target,Horizon,Track,Rows,Handoff MAE,Chapter 6 MAE,MAE difference
0,FHVHV average speed,1,Full-History Regime-Aware,352319,0.921,0.921,0.000
1,FHVHV average speed,2,Full-History Regime-Aware,352319,0.973,0.973,0.000
2,FHVHV average speed,5,Full-History Regime-Aware,352319,0.990,0.990,0.000
3,FHVHV trips,1,Post-CP Primary,352715,58.437,58.437,0.000
4,FHVHV trips,2,Post-CP Primary,352715,62.050,62.050,0.000
5,FHVHV trips,5,Post-CP Primary,352715,62.633,62.633,0.000
6,Subway ridership,1,Full-History Regime-Aware,208859,447.209,447.209,0.000
7,Subway ridership,2,Full-History Regime-Aware,208859,475.990,475.990,0.000
8,Subway ridership,5,Full-History Regime-Aware,208859,477.380,477.380,0.000
9,Taxi average speed,1,Post-CP Primary,250689,2.168,2.168,0.000


**Findings.** The downstream row-level handoff contains **4,551,891 forecasts** covering target dates from **April 5, 2025 through January 4, 2026**.

All **15 Target × Horizon jobs** reproduce their Chapter 6 evaluated-row counts and final MAE within the notebook's strict numerical tolerance; the displayed MAE difference is **0.000 for every job**. The exported table therefore preserves the scoring evidence rather than merely pointing downstream to a separate summary.

The physical forecast keys are unique, all actual, benchmark, raw-prediction, and final-prediction values are present and finite, and `prediction_final = max(0, prediction_raw)` holds with **zero violations**. Most importantly, **zero rows fall on or after January 5, 2026**, so the evaluation-ready handoff preserves the untouched final-holdout boundary.

**4.5.2.8E — Verify the downstream handoff and write the final manifest**

The final QA evaluates the package as a downstream consumer would see it.

A successful closeout requires the exact 15-job roster, all 150 successful monthly production fits, ten refits per job, the frozen target-specific representation/profile/track choices, uncapped production history, complete latest fitted states, the 10-channel + 52-context model-facing contract, complete grouped predictive-reliance evidence, and a row-level forecast surface that reproduces Chapter 6 without touching the final holdout.

The manifest is written only after those checks pass. It indexes the substantive handoff artifacts rather than copying the production checkpoint directory into a second large archive.


In [82]:
# ---------------------------------------------------------------------
# 4.5.2.8E — Final closeout QA and downstream manifest
# ---------------------------------------------------------------------

TRANSFORMER_CLOSEOUT_QA_PATH = (
    TRANSFORMER_FINAL_DIR / "transformer_closeout_qa.parquet"
)
TRANSFORMER_HANDOFF_MANIFEST_PATH = (
    TRANSFORMER_FINAL_DIR / "transformer_handoff_manifest.parquet"
)


def add_closeout_check(rows, check, passed, evidence):
    """Append one machine-readable closeout assertion with readable evidence."""
    rows.append({
        "check": check,
        "passed": bool(passed),
        "evidence": str(evidence),
    })


qa_rows = []

refits_per_job = (
    full_transformer_results_df
    .groupby("job_id", observed=True)["refit_step"]
    .nunique()
)

state_paths = (
    final_transformer_candidate_registry_df["latest_state_path"]
    .map(Path)
)

stl_bundle = {
    "weekly_stl_seasonal_offset",
    "weekly_stl_enabled",
    "missing_mask__weekly_stl_seasonal_offset",
}

handoff_audit = transformer_handoff_audit_df.iloc[0]

add_closeout_check(
    qa_rows,
    "Final candidate jobs",
    len(final_transformer_candidate_registry_df) == 15,
    f"{len(final_transformer_candidate_registry_df)}/15 jobs",
)

add_closeout_check(
    qa_rows,
    "Unique Target × Horizon roster",
    set(zip(
        final_transformer_candidate_registry_df["metric"],
        final_transformer_candidate_registry_df["horizon"].astype(int),
    )) == expected_target_horizons,
    "5 targets × horizons 1, 2, and 5",
)

add_closeout_check(
    qa_rows,
    "Frozen representation",
    final_transformer_candidate_registry_df["representation_id"]
    .eq(EXPECTED_TRANSFORMER_REPRESENTATION_ID)
    .all(),
    EXPECTED_TRANSFORMER_REPRESENTATION_ID,
)

add_closeout_check(
    qa_rows,
    "Only ordinary modeling tracks retained",
    set(final_transformer_candidate_registry_df["track"]).issubset({
        "post_cp_primary",
        "full_history_regime_aware",
    }),
    sorted(
        final_transformer_candidate_registry_df["track"]
        .unique()
        .tolist()
    ),
)

add_closeout_check(
    qa_rows,
    "No Pre-CP Reference candidate",
    not final_transformer_candidate_registry_df["track"]
    .eq("pre_cp_reference")
    .any(),
    "0 Pre-CP Reference jobs",
)

add_closeout_check(
    qa_rows,
    "Production fits",
    len(full_transformer_results_df) == 150,
    f"{len(full_transformer_results_df)}/150 fits",
)

add_closeout_check(
    qa_rows,
    "All production fits successful",
    full_transformer_results_df["status"].eq("success").all(),
    (
        f"{int(full_transformer_results_df['status'].eq('success').sum())}"
        "/150 successful"
    ),
)

add_closeout_check(
    qa_rows,
    "Ten monthly refits per job",
    len(refits_per_job) == 15 and refits_per_job.eq(10).all(),
    f"range={int(refits_per_job.min())}–{int(refits_per_job.max())} refits/job",
)

add_closeout_check(
    qa_rows,
    "Production history uncapped",
    full_transformer_results_df["production_uncapped"]
    .fillna(False)
    .astype(bool)
    .all(),
    "All 150 production fits use complete legal expanding history",
)

add_closeout_check(
    qa_rows,
    "Latest fitted state per job",
    len(state_paths) == 15 and state_paths.map(Path.exists).all(),
    f"{int(state_paths.map(Path.exists).sum())}/15 state files present",
)

add_closeout_check(
    qa_rows,
    "Sequence input dimension",
    int(NEURAL_SEQUENCE_INPUT_DIM) == 10,
    f"{int(NEURAL_SEQUENCE_INPUT_DIM)} channels",
)

add_closeout_check(
    qa_rows,
    "Context input dimension",
    int(transformer_context_feature_count) == 52
    and len(TRANSFORMER_CONTEXT_FEATURE_ORDER) == 52,
    f"{len(TRANSFORMER_CONTEXT_FEATURE_ORDER)} context values",
)

add_closeout_check(
    qa_rows,
    "Weekly STL semantic bundle retained",
    stl_bundle.issubset(set(TRANSFORMER_CONTEXT_FEATURE_ORDER)),
    ", ".join(sorted(stl_bundle)),
)

add_closeout_check(
    qa_rows,
    "Grouped reliance repeat evidence",
    len(transformer_grouped_reliance_repeat_handoff_df) == 360,
    f"{len(transformer_grouped_reliance_repeat_handoff_df)}/360 rows",
)

add_closeout_check(
    qa_rows,
    "Grouped reliance job summary",
    len(transformer_grouped_reliance_summary_handoff_df) == 120,
    f"{len(transformer_grouped_reliance_summary_handoff_df)}/120 rows",
)

add_closeout_check(
    qa_rows,
    "Row-level handoff covers all jobs",
    int(handoff_audit["jobs"]) == 15,
    f"{int(handoff_audit['jobs'])}/15 jobs",
)

add_closeout_check(
    qa_rows,
    "Row-level forecast keys unique",
    transformer_handoff_duplicate_rows == 0,
    f"{transformer_handoff_duplicate_rows:,} duplicate rows",
)

add_closeout_check(
    qa_rows,
    "Complete scoring fields",
    int(handoff_audit["missing_score_rows"]) == 0
    and int(handoff_audit["nonfinite_score_rows"]) == 0,
    (
        f"missing={int(handoff_audit['missing_score_rows']):,}; "
        f"non-finite={int(handoff_audit['nonfinite_score_rows']):,}"
    ),
)

add_closeout_check(
    qa_rows,
    "Prediction output rule",
    int(handoff_audit["output_rule_violations"]) == 0,
    f"{int(handoff_audit['output_rule_violations']):,} violations",
)

add_closeout_check(
    qa_rows,
    "Chapter 6 scorecard reproduced",
    transformer_handoff_reproduction_df[
        ["row_count_match", "final_mae_match", "benchmark_mae_match"]
    ].all().all(),
    "15/15 jobs match row counts, final MAE, and benchmark MAE",
)

add_closeout_check(
    qa_rows,
    "Final holdout untouched",
    int(handoff_audit["final_holdout_rows"]) == 0,
    (
        f"0 rows on/after "
        f"{pd.Timestamp(FINAL_HOLDOUT_START_DATE).date()}"
    ),
)

transformer_closeout_qa_df = pd.DataFrame(qa_rows)

# Write QA before checking package materialization so the QA artifact itself can
# participate in the final file-level integrity check.
atomic_write_parquet(
    transformer_closeout_qa_df,
    TRANSFORMER_CLOSEOUT_QA_PATH,
)

substantive_artifacts = [
    (
        "transformer_final_model_registry.parquet",
        TRANSFORMER_FINAL_MODEL_REGISTRY_PATH,
        "Final 15-job Transformer candidate registry",
        len(final_transformer_candidate_registry_df),
    ),
    (
        "transformer_model_definition_contract.parquet",
        TRANSFORMER_MODEL_DEFINITION_PATH,
        "Selected patch geometry and Transformer capacity definitions",
        len(transformer_model_definition_contract_df),
    ),
    (
        "transformer_context_contract.parquet",
        TRANSFORMER_CONTEXT_CONTRACT_PATH,
        "Exact 52-value context and preprocessing contract",
        len(transformer_context_contract_df),
    ),
    (
        "transformer_input_registry.parquet",
        TRANSFORMER_INPUT_REGISTRY_PATH,
        "Target-specific sequence, context, and Taxi Zone inputs",
        len(transformer_input_registry_df),
    ),
    (
        "transformer_forecast_output_contract.parquet",
        TRANSFORMER_OUTPUT_CONTRACT_PATH,
        "Actual, benchmark, raw, and final forecast semantics",
        len(transformer_forecast_output_contract_df),
    ),
    (
        "transformer_production_schedule.parquet",
        TRANSFORMER_PRODUCTION_SCHEDULE_PATH,
        "Complete 150-fit monthly production schedule",
        len(transformer_full_schedule_df),
    ),
    (
        "transformer_training_policy_evidence.parquet",
        TRANSFORMER_TRAINING_POLICY_PATH,
        "Production adaptive-training and fit evidence",
        len(transformer_training_policy_evidence_df),
    ),
    (
        "transformer_training_curve_diagnostic.parquet",
        TRANSFORMER_TRAINING_CURVE_PATH,
        "Saved selector training and inner-validation curves",
        len(transformer_training_curve_diagnostic_df),
    ),
    (
        "transformer_grouped_reliance_repeat.parquet",
        TRANSFORMER_RELIANCE_REPEAT_PATH,
        "Job × Input Group × Repeat predictive-reliance evidence",
        len(transformer_grouped_reliance_repeat_handoff_df),
    ),
    (
        "transformer_grouped_reliance_summary.parquet",
        TRANSFORMER_RELIANCE_SUMMARY_PATH,
        "Job × Input Group predictive-reliance summary",
        len(transformer_grouped_reliance_summary_handoff_df),
    ),
    (
        "transformer_development_summary.parquet",
        TRANSFORMER_DEVELOPMENT_SUMMARY_PATH,
        "Five target-level frozen Transformer choices",
        len(transformer_development_summary_df),
    ),
    (
        "transformer_selected_validation_predictions.parquet",
        TRANSFORMER_SELECTED_PREDICTIONS_PATH,
        "Evaluation-ready row-level ordinary-validation forecasts",
        int(handoff_audit["forecast_rows"]),
    ),
    (
        "transformer_closeout_qa.parquet",
        TRANSFORMER_CLOSEOUT_QA_PATH,
        "Machine-readable final handoff QA",
        len(transformer_closeout_qa_df),
    ),
]

missing_final_artifacts = [
    name
    for name, path, _, _ in substantive_artifacts
    if not Path(path).exists()
]

if missing_final_artifacts:
    raise FileNotFoundError(
        "Final Transformer handoff artifacts are missing: "
        f"{missing_final_artifacts}"
    )

if not transformer_closeout_qa_df["passed"].all():
    failed_checks = transformer_closeout_qa_df.loc[
        ~transformer_closeout_qa_df["passed"],
        ["check", "evidence"],
    ]

    raise RuntimeError(
        "Transformer closeout QA failed:\n"
        + failed_checks.to_string(index=False)
    )


# ---------------------------------------------------------------------
# Write the final manifest only after substantive QA succeeds
# ---------------------------------------------------------------------

transformer_handoff_manifest_df = pd.DataFrame([
    {
        "artifact": name,
        "path": str(path),
        "purpose": purpose,
        "rows": int(rows),
    }
    for name, path, purpose, rows in substantive_artifacts
])

if transformer_handoff_manifest_df["artifact"].duplicated().any():
    raise RuntimeError(
        "Transformer handoff manifest contains duplicate artifact names."
    )

if transformer_handoff_manifest_df["path"].duplicated().any():
    raise RuntimeError(
        "Transformer handoff manifest contains duplicate artifact paths."
    )

atomic_write_parquet(
    transformer_handoff_manifest_df,
    TRANSFORMER_HANDOFF_MANIFEST_PATH,
)

if not TRANSFORMER_HANDOFF_MANIFEST_PATH.exists():
    raise FileNotFoundError(
        "Transformer handoff manifest was not materialized."
    )

display(transformer_closeout_qa_df)

print(
    "Transformer handoff written and verified | "
    f"jobs={len(final_transformer_candidate_registry_df):,} | "
    f"fits={len(full_transformer_results_df):,} | "
    f"forecast rows={int(handoff_audit['forecast_rows']):,} | "
    f"artifacts={len(transformer_handoff_manifest_df):,}"
)


,check,passed,evidence
0,Final candidate jobs,True,15/15 jobs
1,Unique Target × Horizon roster,True,"5 targets × horizons 1, 2, and 5"
2,Frozen representation,True,two_week_overlapping_two_day
3,Only ordinary modeling tracks retained,True,"['full_history_regime_aware', 'post_cp_primary']"
4,No Pre-CP Reference candidate,True,0 Pre-CP Reference jobs
5,Production fits,True,150/150 fits
6,All production fits successful,True,150/150 successful
7,Ten monthly refits per job,True,range=10–10 refits/job
8,Production history uncapped,True,All 150 production fits use complete legal exp...
9,Latest fitted state per job,True,15/15 state files present


Transformer handoff written and verified | jobs=15 | fits=150 | forecast rows=4,551,891 | artifacts=13


**Findings.** All final Transformer closeout checks pass. The package contains the complete **15-job candidate roster** and **150/150 successful production fits**, with exactly **10 monthly refits per job** and one latest fitted state retained for every candidate.

The model-facing contract remains fixed at **10 ordered sequence channels, 52 supporting-context values, and Taxi Zone identity**. Production training is uncapped, the state-aware weekly-STL bundle remains intact, and grouped predictive reliance is complete at both required grains.

The row-level handoff is unique, finite, obeys `prediction_final = max(0, prediction_raw)`, reproduces the Chapter 6 scoring evidence for all 15 jobs, and contains **zero final-holdout rows**.

The final manifest indexes **13 substantive Transformer handoff artifacts**. The manifest itself is stored alongside them as the fourteenth file in `4.5.2.final_tables`.


### Section 8 Summary

Chapter 8 closes the Transformer branch without fitting another model. It freezes the complete **15 Target × Horizon candidate roster**, preserves the architecture and model-facing input contracts, packages all **150 production fits** and grouped predictive-reliance evidence, and consolidates **4,551,891 ordinary-validation forecasts** into one evaluation-ready handoff. All **21 final closeout QA checks pass**: the exported forecasts reproduce the Chapter 6 scorecard, forecast keys and scoring fields are valid, the nonnegative output rule is preserved, and no row enters the January 5, 2026 final holdout. The handoff contains **13 substantive artifacts plus the manifest**, giving Chapter 4.6 a self-contained Transformer candidate without requiring it to understand the production checkpoint structure.

## Close

Notebook 4.5.2 completes the project's Transformer-based ordinary-forecasting branch.

The final system uses a **PatchTST-inspired encoder-only Transformer** with a **70-step two-week overlapping history**, **10 sequence channels**, **52 supporting-context values**, learned Taxi Zone identity, monthly expanding-history refitting, and leakage-safe adaptive training-duration selection. Capacity and training-history track remain target-specific across the five mobility outcomes.

Across **4,551,891 ordinary-validation forecasts**, all **15 Target × Horizon jobs outperform Previous week · same Daypart on pooled validation**, with benchmark skill ranging from **5.365% to 30.726%**. Relative MAE ranges from roughly **5.5% to 18.3%**: the two speed targets are especially strong and stable, while demand forecasting remains more variable across months and horizons. Adaptive training selects between **2 and 27 epochs**, and only **2 of 150 fits** reach the 30-epoch search ceiling.

Grouped predictive reliance shows that the Transformer draws primarily on **recent mobility history**, which accounts for **86.454% of measured reliance on average** and ranks first in all 15 final jobs. Supporting calendar, multimodal, spatial, Taxi Zone, weekly-STL, and weather information contributes smaller shares; congestion-pricing regime is not permutation-identifiable on the latest scoring surfaces.

All **21 closeout QA checks pass**, and the final handoff preserves the complete candidate registry, model and input contracts, production evidence, predictive-reliance diagnostics, and row-level validation forecasts while leaving the **January 5, 2026 final holdout untouched**.

Overall forecasting-family selection now passes to Chapter 4.6. Final-holdout evaluation and Chapter 5 counterfactual forecasting remain outside this notebook.

### Final File Inventory

The finalized Transformer handoff contains **13 substantive reusable artifacts plus one manifest**. The row-level ordinary-validation forecast table is the only large downstream asset; the model registries, contracts, training evidence, predictive-reliance summaries, and QA records remain compact.

| Artifact | Purpose | Rows | File size |
|---|---|---:|---:|
| `transformer_final_model_registry.parquet` | Final 15-job Transformer candidate registry | 15 | 26.700 KB |
| `transformer_model_definition_contract.parquet` | Selected patch geometry and Transformer capacity definitions | 2 | 18.064 KB |
| `transformer_context_contract.parquet` | Exact 52-value context and preprocessing contract | 52 | 5.083 KB |
| `transformer_input_registry.parquet` | Target-specific sequence, context, and Taxi Zone inputs | 103 | 6.012 KB |
| `transformer_forecast_output_contract.parquet` | Actual, benchmark, raw, and final forecast semantics | 4 | 2.839 KB |
| `transformer_production_schedule.parquet` | Complete 150-fit monthly production schedule | 150 | 14.362 KB |
| `transformer_training_policy_evidence.parquet` | Production adaptive-training and fit evidence | 150 | 42.835 KB |
| `transformer_training_curve_diagnostic.parquet` | Saved selector training and inner-validation curves | 2,471 | 72.984 KB |
| `transformer_grouped_reliance_repeat.parquet` | Job × Input Group × Repeat predictive-reliance evidence | 360 | 24.755 KB |
| `transformer_grouped_reliance_summary.parquet` | Job × Input Group predictive-reliance summary | 120 | 11.473 KB |
| `transformer_development_summary.parquet` | Five target-level frozen Transformer choices | 5 | 7.632 KB |
| `transformer_selected_validation_predictions.parquet` | Evaluation-ready row-level ordinary-validation forecasts | 4,551,891 | 55.023 MB |
| `transformer_closeout_qa.parquet` | Machine-readable final handoff QA | 21 | 3.337 KB |
| `transformer_handoff_manifest.parquet` | Index of the final Transformer handoff artifacts | 13 | 4.722 KB |

The **13-row manifest indexes the other 13 final artifacts rather than indexing itself**. Retained fitted Transformer states and restartable production checkpoints remain in the intermediate production-artifact tree for audit and reproducibility; they are referenced by the final registry rather than duplicated into the compact downstream handoff.

The final handoff is therefore ready for **4.6.1 cross-family model comparison** while preserving the complete Transformer modeling contract and the **4,551,891-row** validation surface needed for exact common-row comparison.

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=5c43663c-345d-4c21-9332-dd4f928af3ba' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>